
# Biotech Disclosure Sentiment Analysis Foundation

This notebook is the primary implementation surface for a notebook-first biotech disclosure sentiment analysis system.

## Architecture Summary

- One worker node is reserved for each disclosure type in scope.
- Retrieval is responsible for finding the most recent relevant disclosure candidate for each type.
- Shared worker-analysis utilities prepare evidence-backed analysis packets from one selected document at a time.
- Disclosure-specific worker prompts now specialize the five worker nodes on top of the shared analysis layer.
- Arbiter nodes will later reconcile cross-document outputs from the workers.
- One master node will later package arbiter outputs into the only UI-facing payload.

## Scope of This Step

Implemented now:
- shared configuration and typed contracts
- retrieval request models
- retrieval adapter interfaces
- candidate normalization and validation models
- provenance and audit helpers
- explicit relevance, freshness, and selection logic
- structured retrieval error handling
- mock adapters for all five disclosure types
- deterministic preprocessing, chunking, embedding hooks, and lightweight graph-aware retrieval
- shared worker-analysis contracts, rubric definitions, evidence helpers, and a swappable analysis-client interface
- disclosure-specific worker prompts, worker subclasses, registry wiring, and runnable mock demos for all five disclosure types

Intentionally not implemented yet:
- live SEC, ClinicalTrials.gov, FDA, or investor-relations integrations
- real network calls for analysis models
- arbiter logic
- master-node aggregation and final UI assembly
- tests


## 2. Imports

In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from datetime import date, datetime, time, timezone
from enum import Enum
import json
import subprocess
import logging
from pathlib import Path
from typing import Any, ClassVar, Sequence, Type

from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

import os
import time

import anthropic
import dotenv
import requests
from bs4 import BeautifulSoup
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

## 3. Global Configuration

In [ ]:
LOGGER_NAME = "biotech_disclosure_pipeline"
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(LOGGER_NAME)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GLOBAL_CONFIG: dict[str, Any] = {
    "project_name": "Biotech Disclosure Sentiment Analysis",
    "notebook_version": "0.5.0-arbiter-layer",
    "paths": {
        "project_root": PROJECT_ROOT,
        "data_root": PROJECT_ROOT / "data",
        "raw_documents": PROJECT_ROOT / "data" / "raw_documents",
        "processed_documents": PROJECT_ROOT / "data" / "processed_documents",
        "cache": PROJECT_ROOT / "data" / "cache",
    },
    "disclosure_types": [
        "material_event",
        "clinical_trial_update",
        "fda_review",
        "financing_dilution",
        "investor_communication",
    ],
    "scoring": {
        "sentiment_min": -1.0,
        "sentiment_max": 1.0,
        "tone_min": 0.0,
        "tone_max": 1.0,
    },
    "feature_flags": {
        "enable_retrieval": True,
        "enable_mock_retrieval": False,
        "enable_embeddings": True,
        "enable_graph_context": False,
        "enable_analysis": True,
        "enable_mock_analysis": False,
        "analysis_client_mode": "moonshot",
        "enable_langchain": False,
        "enable_langgraph": False,
    },
    "retrieval_defaults": {
        "max_candidates": 4,
        "min_usable_text_chars": 40,
        "minimum_relevance_score": 55.0,
        "freshness_bonus_max": 20.0,
        "freshness_rank_decay": 4.0,
        "default_source_preferences": [
            "official_regulatory",
            "issuer_published",
            "permitted_secondary",
            "unknown",
        ],
    },
    "future_model_config": {
        "worker_model_name": "moonshot-v1-128k",
        "arbiter_model_name": "claude-opus-4-20250514",
        "embedding_model_name": "voyage-3-large",
    },
    "future_source_adapters": {
        "material_event": "implementation_specific_adapter",
        "clinical_trial_update": "implementation_specific_adapter",
        "fda_review": "implementation_specific_adapter",
        "financing_dilution": "implementation_specific_adapter",
        "investor_communication": "implementation_specific_adapter",
    },
}

logger.info("Loaded arbiter-layer notebook config for %s", GLOBAL_CONFIG["project_name"])

In [ ]:
# --- Credential and Environment Loading ---
dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True))

SEC_EDGAR_USER_AGENT = os.environ.get("SEC_EDGAR_USER_AGENT", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
OPENFDA_API_KEY = os.environ.get("OPENFDA_API_KEY", "")
MOONSHOT_API_KEY = os.environ.get("MOONSHOT_API_KEY", "")

if SEC_EDGAR_USER_AGENT:
    logger.info("SEC EDGAR User-Agent loaded from environment.")
else:
    logger.warning("SEC_EDGAR_USER_AGENT not set — live EDGAR adapters will fail.")

if ANTHROPIC_API_KEY:
    logger.info("ANTHROPIC_API_KEY loaded from environment.")
else:
    logger.warning("ANTHROPIC_API_KEY not set — Anthropic analysis client will fail.")

if MOONSHOT_API_KEY:
    logger.info("MOONSHOT_API_KEY loaded from environment.")
else:
    logger.warning("MOONSHOT_API_KEY not set — Moonshot analysis client will fail.")

if OPENFDA_API_KEY:
    logger.info("OPENFDA_API_KEY loaded (optional rate limit boost).")
CLAUDE_CLI_PATH = "/home/kyanaman/.local/bin/claude"
if Path(CLAUDE_CLI_PATH).is_file():
    logger.info("Claude CLI found at %s", CLAUDE_CLI_PATH)
else:
    logger.warning("Claude CLI not found at %s — CLI analysis client will fail.", CLAUDE_CLI_PATH)

## 4. Disclosure Type Registry

In [ ]:
DISCLOSURE_TYPE_REGISTRY: list[dict[str, str]] = [
    {
        "key": "material_event",
        "label": "Material event 8-K / Reg FD press release",
        "worker_class_name": "MaterialEventWorker",
        "adapter_class_name": "MockMaterialEventRetrievalAdapter",
        "description": "Latest material event filing or issuer press release candidate.",
    },
    {
        "key": "clinical_trial_update",
        "label": "Clinical trial registry update / ClinicalTrials.gov update",
        "worker_class_name": "ClinicalTrialUpdateWorker",
        "adapter_class_name": "MockClinicalTrialUpdateRetrievalAdapter",
        "description": "Latest relevant registry update for the issuer or mapped program.",
    },
    {
        "key": "fda_review",
        "label": "FDA review materials / regulatory review documents",
        "worker_class_name": "FDAReviewWorker",
        "adapter_class_name": "MockFDAReviewRetrievalAdapter",
        "description": "Latest relevant FDA review package or official review material.",
    },
    {
        "key": "financing_dilution",
        "label": "Financing / dilution disclosure",
        "worker_class_name": "FinancingDilutionWorker",
        "adapter_class_name": "MockFinancingDilutionRetrievalAdapter",
        "description": "Latest financing or dilution-bearing disclosure candidate.",
    },
    {
        "key": "investor_communication",
        "label": "Investor presentation / corporate update deck / fireside chat / webcast transcript",
        "worker_class_name": "InvestorCommunicationWorker",
        "adapter_class_name": "MockInvestorCommunicationRetrievalAdapter",
        "description": "Latest investor-facing communication with usable text content.",
    },
]

DISCLOSURE_TYPE_KEYS = [entry["key"] for entry in DISCLOSURE_TYPE_REGISTRY]
DISCLOSURE_TYPE_LABELS = {entry["key"]: entry["label"] for entry in DISCLOSURE_TYPE_REGISTRY}


## 5. Shared Enums and Constants

In [ ]:
class DocumentType(str, Enum):
    """Supported disclosure types for the first system version."""

    MATERIAL_EVENT = "material_event"
    CLINICAL_TRIAL_UPDATE = "clinical_trial_update"
    FDA_REVIEW = "fda_review"
    FINANCING_DILUTION = "financing_dilution"
    INVESTOR_COMMUNICATION = "investor_communication"


class SentimentLabel(str, Enum):
    """Normalized sentiment labels used across workers and arbiters."""

    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"
    MIXED = "mixed"
    INSUFFICIENT_EVIDENCE = "insufficient_evidence"


class ProcessingStatus(str, Enum):
    """Lifecycle status values shared across retrieval and analysis outputs."""

    PENDING = "pending"
    SUCCESS = "success"
    PARTIAL = "partial"
    NO_DOCUMENT = "no_document"
    RETRIEVAL_FAILED = "retrieval_failed"
    SELECTION_FAILED = "selection_failed"
    EXTRACTION_FAILED = "extraction_failed"
    ANALYSIS_FAILED = "analysis_failed"


class SourceFamily(str, Enum):
    """High-level source classification without provider-specific assumptions."""

    OFFICIAL_REGULATORY = "official_regulatory"
    ISSUER_PUBLISHED = "issuer_published"
    PERMITTED_SECONDARY = "permitted_secondary"
    UNKNOWN = "unknown"


class ArbiterKind(str, Enum):
    """Supported arbiter responsibilities for the initial node layout."""

    SUMMARY = "summary"
    SENTIMENT = "sentiment"


DEFAULT_LANGUAGE = "en"
DEFAULT_SOURCE_FAMILY_ORDER = [
    SourceFamily.OFFICIAL_REGULATORY,
    SourceFamily.ISSUER_PUBLISHED,
    SourceFamily.PERMITTED_SECONDARY,
    SourceFamily.UNKNOWN,
]
DEFAULT_MAX_CANDIDATES = int(GLOBAL_CONFIG["retrieval_defaults"]["max_candidates"])
DEFAULT_MIN_USABLE_TEXT_CHARS = int(GLOBAL_CONFIG["retrieval_defaults"]["min_usable_text_chars"])
DEFAULT_MINIMUM_RELEVANCE_SCORE = float(GLOBAL_CONFIG["retrieval_defaults"]["minimum_relevance_score"])
DEFAULT_FRESHNESS_BONUS_MAX = float(GLOBAL_CONFIG["retrieval_defaults"]["freshness_bonus_max"])
DEFAULT_FRESHNESS_RANK_DECAY = float(GLOBAL_CONFIG["retrieval_defaults"]["freshness_rank_decay"])
SENTIMENT_SCORE_MIN = float(GLOBAL_CONFIG["scoring"]["sentiment_min"])
SENTIMENT_SCORE_MAX = float(GLOBAL_CONFIG["scoring"]["sentiment_max"])
TONE_SCORE_MIN = float(GLOBAL_CONFIG["scoring"]["tone_min"])
TONE_SCORE_MAX = float(GLOBAL_CONFIG["scoring"]["tone_max"])
MIN_DATETIME_UTC = datetime(1970, 1, 1, tzinfo=timezone.utc)


def now_utc() -> datetime:
    """Return the current UTC timestamp."""
    return datetime.now(timezone.utc)


assert DISCLOSURE_TYPE_KEYS == [item.value for item in DocumentType]


## 6. Typed Data Models

In [ ]:
def _validate_optional_range(value: float | None, minimum: float, maximum: float, field_name: str) -> float | None:
    if value is None:
        return value
    if not minimum <= value <= maximum:
        raise ValueError(f"{field_name} must be between {minimum} and {maximum}.")
    return value


class ContractModel(BaseModel):
    """Base contract model with strict validation for scaffold development."""

    model_config = ConfigDict(extra="forbid", validate_assignment=True)


class PipelineConfig(ContractModel):
    """Typed notebook configuration shared across retrieval and later analysis nodes."""

    project_name: str
    notebook_version: str
    project_root: Path
    data_root: Path
    raw_document_dir: Path
    processed_document_dir: Path
    cache_dir: Path
    disclosure_types: list[DocumentType]
    sentiment_score_min: float
    sentiment_score_max: float
    tone_score_min: float
    tone_score_max: float
    default_max_candidates: int
    min_usable_text_chars: int
    minimum_relevance_score: float
    freshness_bonus_max: float
    freshness_rank_decay: float
    default_source_preferences: list[SourceFamily]
    enable_retrieval: bool = True
    enable_mock_retrieval: bool = True
    enable_embeddings: bool = False
    enable_graph_context: bool = False
    enable_analysis: bool = True
    enable_mock_analysis: bool = True
    enable_langchain: bool = False
    enable_langgraph: bool = False
    analysis_client_mode: str = "cli"
    worker_model_name: str = "configurable_placeholder"
    arbiter_model_name: str = "configurable_placeholder"
    embedding_model_name: str = "configurable_placeholder"
    source_adapter_placeholders: dict[str, str] = Field(default_factory=dict)


class DocumentMetadata(ContractModel):
    """Normalized metadata for a single selected document candidate."""

    document_id: str
    ticker: str
    document_type: DocumentType
    company_name: str | None = None
    title: str | None = None
    source_name: str | None = None
    source_identifier: str | None = None
    source_family: SourceFamily = SourceFamily.UNKNOWN
    source_url: str | None = None
    published_at: datetime | None = None
    updated_at: datetime | None = None
    event_date: date | None = None
    version_label: str | None = None
    file_type: str | None = None
    language: str = DEFAULT_LANGUAGE
    content_hash: str | None = None
    retrieved_at: datetime | None = None
    is_structured_source: bool = False
    is_mock_data: bool = False
    external_ids: dict[str, str] = Field(default_factory=dict)
    raw_metadata: dict[str, Any] = Field(default_factory=dict)
    notes: list[str] = Field(default_factory=list)


class PipelineError(ContractModel):
    """Structured pipeline issue record for warnings and failures."""

    error_code: str
    message: str
    stage: str
    document_type: DocumentType | None = None
    candidate_id: str | None = None
    adapter_name: str | None = None
    recoverable: bool = True
    details: dict[str, Any] = Field(default_factory=dict)


class ProvenanceRecord(ContractModel):
    """Trace record describing how a retrieval object moved through the pipeline."""

    stage: str
    adapter_name: str
    candidate_id: str | None = None
    document_type: DocumentType | None = None
    source_name: str | None = None
    source_identifier: str | None = None
    source_url: str | None = None
    retrieved_at: datetime
    note: str | None = None
    ranking_notes: list[str] = Field(default_factory=list)
    validation_notes: list[str] = Field(default_factory=list)
    selection_reason: str | None = None
    is_mock_data: bool = False
    metadata: dict[str, Any] = Field(default_factory=dict)


class SentimentAssessment(ContractModel):
    """Standard sentiment assessment shape shared across outputs."""

    label: SentimentLabel = SentimentLabel.INSUFFICIENT_EVIDENCE
    score: float | None = None
    confidence: float | None = None
    rationale: str | None = None

    @field_validator("score")
    @classmethod
    def validate_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX, "score")

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence")


class ToneAssessment(ContractModel):
    """Fogging, hedging, and promotional tone scores for one analysis unit."""

    fogging_score: float | None = None
    hedging_score: float | None = None
    promotional_score: float | None = None
    confidence: float | None = None
    rationale: str | None = None

    @field_validator("fogging_score", "hedging_score", "promotional_score")
    @classmethod
    def validate_tone_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, TONE_SCORE_MIN, TONE_SCORE_MAX, "tone score")

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence")


class DisclosureCardPayload(ContractModel):
    """Per-disclosure UI card shape produced only by the master node."""

    document_type: DocumentType
    status: ProcessingStatus
    title: str | None = None
    source_name: str | None = None
    source_url: str | None = None
    summary: str | None = None
    sentiment_label: SentimentLabel | None = None
    sentiment_score: float | None = None
    fogging_score: float | None = None
    hedging_score: float | None = None
    promotional_score: float | None = None
    caveats: list[str] = Field(default_factory=list)

    @field_validator("sentiment_score")
    @classmethod
    def validate_sentiment_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX, "sentiment_score")

    @field_validator("fogging_score", "hedging_score", "promotional_score")
    @classmethod
    def validate_tone_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, TONE_SCORE_MIN, TONE_SCORE_MAX, "ui tone score")


class FinalUIPayload(ContractModel):
    """Final UI-facing payload emitted exclusively by the master node."""

    ticker: str
    generated_at: datetime = Field(default_factory=now_utc)
    overall_summary: str | None = None
    overall_sentiment_label: SentimentLabel | None = None
    overall_sentiment_score: float | None = None
    overall_fogging_score: float | None = None
    overall_hedging_score: float | None = None
    overall_promotional_score: float | None = None
    disclosures: list[DisclosureCardPayload] = Field(default_factory=list)
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    system_warnings: list[str] = Field(default_factory=list)
    provenance_links: list[str] = Field(default_factory=list)

    @field_validator("overall_sentiment_score")
    @classmethod
    def validate_sentiment_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX, "overall_sentiment_score")

    @field_validator("overall_fogging_score", "overall_hedging_score", "overall_promotional_score")
    @classmethod
    def validate_tone_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, TONE_SCORE_MIN, TONE_SCORE_MAX, "overall tone score")


PIPELINE_CONFIG = PipelineConfig(
    project_name=GLOBAL_CONFIG["project_name"],
    notebook_version=GLOBAL_CONFIG["notebook_version"],
    project_root=GLOBAL_CONFIG["paths"]["project_root"],
    data_root=GLOBAL_CONFIG["paths"]["data_root"],
    raw_document_dir=GLOBAL_CONFIG["paths"]["raw_documents"],
    processed_document_dir=GLOBAL_CONFIG["paths"]["processed_documents"],
    cache_dir=GLOBAL_CONFIG["paths"]["cache"],
    disclosure_types=[DocumentType(item) for item in GLOBAL_CONFIG["disclosure_types"]],
    sentiment_score_min=GLOBAL_CONFIG["scoring"]["sentiment_min"],
    sentiment_score_max=GLOBAL_CONFIG["scoring"]["sentiment_max"],
    tone_score_min=GLOBAL_CONFIG["scoring"]["tone_min"],
    tone_score_max=GLOBAL_CONFIG["scoring"]["tone_max"],
    default_max_candidates=GLOBAL_CONFIG["retrieval_defaults"]["max_candidates"],
    min_usable_text_chars=GLOBAL_CONFIG["retrieval_defaults"]["min_usable_text_chars"],
    minimum_relevance_score=GLOBAL_CONFIG["retrieval_defaults"]["minimum_relevance_score"],
    freshness_bonus_max=GLOBAL_CONFIG["retrieval_defaults"]["freshness_bonus_max"],
    freshness_rank_decay=GLOBAL_CONFIG["retrieval_defaults"]["freshness_rank_decay"],
    default_source_preferences=[SourceFamily(item) for item in GLOBAL_CONFIG["retrieval_defaults"]["default_source_preferences"]],
    enable_retrieval=GLOBAL_CONFIG["feature_flags"]["enable_retrieval"],
    enable_mock_retrieval=GLOBAL_CONFIG["feature_flags"]["enable_mock_retrieval"],
    enable_embeddings=GLOBAL_CONFIG["feature_flags"]["enable_embeddings"],
    enable_graph_context=GLOBAL_CONFIG["feature_flags"]["enable_graph_context"],
    enable_analysis=GLOBAL_CONFIG["feature_flags"]["enable_analysis"],
    enable_mock_analysis=GLOBAL_CONFIG["feature_flags"]["enable_mock_analysis"],
    enable_langchain=GLOBAL_CONFIG["feature_flags"]["enable_langchain"],
    enable_langgraph=GLOBAL_CONFIG["feature_flags"]["enable_langgraph"],
    analysis_client_mode=GLOBAL_CONFIG["feature_flags"].get("analysis_client_mode", "cli"),
    worker_model_name=GLOBAL_CONFIG["future_model_config"]["worker_model_name"],
    arbiter_model_name=GLOBAL_CONFIG["future_model_config"]["arbiter_model_name"],
    embedding_model_name=GLOBAL_CONFIG["future_model_config"]["embedding_model_name"],
    source_adapter_placeholders=GLOBAL_CONFIG["future_source_adapters"],
)


## 7. Retrieval Design Notes

Retrieval assumptions for this notebook step:

- Retrieval is provider-agnostic; live source integrations are deferred behind adapter interfaces.
- One retrieval request targets exactly one disclosure type.
- A candidate is considered selectable only if it is valid, relevant, and not superseded by a fresher duplicate.
- Duplicate handling is heuristic and explicit: source identifier first, then source URL, then normalized title/date fallback.
- Freshness is determined by `updated_at`, then `published_at`, then `event_date`.
- Ties are broken deterministically and recorded in ranking notes so selection remains auditable.
- Recoverable failures produce structured retrieval errors instead of crashing the notebook.
- Mock adapters are used only to prove normalization, validation, ranking, and selection logic.


## 8. Retrieval Query and Request Models

In [ ]:
class SourcePreferencePolicy(ContractModel):
    """Provider-agnostic retrieval source preferences."""

    preferred_source_families: list[SourceFamily] = Field(default_factory=lambda: DEFAULT_SOURCE_FAMILY_ORDER.copy())
    prefer_structured_source: bool = True
    allow_secondary_sources: bool = True
    allow_unknown_sources: bool = True
    allow_mock_data: bool = True


class RetrievalRequest(ContractModel):
    """Generic retrieval request used by all disclosure adapters."""

    request_id: str
    ticker: str
    company_name: str | None = None
    company_aliases: list[str] = Field(default_factory=list)
    document_type: DocumentType
    start_date: date | None = None
    end_date: date | None = None
    max_candidates: int = DEFAULT_MAX_CANDIDATES
    minimum_text_chars: int = DEFAULT_MIN_USABLE_TEXT_CHARS
    minimum_relevance_score: float = DEFAULT_MINIMUM_RELEVANCE_SCORE
    freshness_bonus_max: float = DEFAULT_FRESHNESS_BONUS_MAX
    freshness_rank_decay: float = DEFAULT_FRESHNESS_RANK_DECAY
    source_preferences: SourcePreferencePolicy = Field(default_factory=SourcePreferencePolicy)
    retrieval_notes: list[str] = Field(default_factory=list)
    filters: dict[str, Any] = Field(default_factory=dict)
    requested_at: datetime = Field(default_factory=now_utc)
    is_mock_request: bool = False

    @field_validator("ticker")
    @classmethod
    def uppercase_ticker(cls, value: str) -> str:
        return value.strip().upper()

    @field_validator("max_candidates")
    @classmethod
    def validate_max_candidates(cls, value: int) -> int:
        if value < 1:
            raise ValueError("max_candidates must be at least 1.")
        return value

    @model_validator(mode="after")
    def validate_date_bounds(self) -> "RetrievalRequest":
        if self.start_date and self.end_date and self.end_date < self.start_date:
            raise ValueError("end_date must be on or after start_date.")
        return self


## 9. Candidate Normalization Models

In [ ]:
def normalize_document_text(raw_text: str | None) -> str | None:
    """Return whitespace-normalized text or None if the input is empty."""
    if raw_text is None:
        return None
    normalized = " ".join(raw_text.split())
    return normalized or None


class RawRetrievalCandidateMetadata(ContractModel):
    """Generic raw metadata returned by an adapter before normalization."""

    raw_candidate_id: str
    adapter_name: str
    document_type: DocumentType
    ticker: str | None = None
    company_name: str | None = None
    source_name: str | None = None
    source_identifier: str | None = None
    source_url: str | None = None
    title: str | None = None
    published_at: datetime | None = None
    updated_at: datetime | None = None
    event_date: date | None = None
    source_family: SourceFamily = SourceFamily.UNKNOWN
    is_structured_source: bool = False
    is_mock_data: bool = False
    raw_metadata: dict[str, Any] = Field(default_factory=dict)


class FetchedDocumentContent(ContractModel):
    """Generic fetched content paired with one raw retrieval candidate."""

    raw_candidate_id: str
    document_text: str | None = None
    content_type: str | None = None
    fetched_at: datetime = Field(default_factory=now_utc)
    fetch_status: ProcessingStatus = ProcessingStatus.SUCCESS
    fetch_notes: list[str] = Field(default_factory=list)
    is_mock_data: bool = False
    raw_payload: dict[str, Any] = Field(default_factory=dict)


class DocumentValidationResult(ContractModel):
    """Validation outcome for a normalized retrieval candidate."""

    candidate_id: str
    status: ProcessingStatus = ProcessingStatus.PENDING
    is_valid: bool = False
    is_partial: bool = False
    has_usable_text: bool = False
    has_minimum_metadata: bool = False
    type_matches_request: bool = False
    identity_matches_request: bool = False
    validation_notes: list[str] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)


class CandidateEvaluation(ContractModel):
    """Transparent heuristic scores used for retrieval selection."""

    candidate_id: str
    is_relevant: bool = False
    relevance_score: float = 0.0
    freshness_score: float = 0.0
    total_score: float = 0.0
    type_compatibility_score: float = 0.0
    identity_alignment_score: float = 0.0
    title_alignment_score: float = 0.0
    structured_source_bonus: float = 0.0
    source_preference_bonus: float = 0.0
    duplicate_key: str | None = None
    effective_timestamp: datetime | None = None
    evaluation_notes: list[str] = Field(default_factory=list)


class RetrievalCandidate(ContractModel):
    """Normalized internal retrieval candidate used by selection logic."""

    candidate_id: str
    adapter_name: str
    document_type: DocumentType
    ticker: str | None = None
    company_name: str | None = None
    source_name: str | None = None
    source_identifier: str | None = None
    source_url: str | None = None
    source_family: SourceFamily = SourceFamily.UNKNOWN
    title: str | None = None
    published_at: datetime | None = None
    updated_at: datetime | None = None
    event_date: date | None = None
    retrieved_at: datetime
    document_text: str | None = None
    is_structured_source: bool = False
    is_mock_data: bool = False
    relevance_notes: list[str] = Field(default_factory=list)
    freshness_notes: list[str] = Field(default_factory=list)
    validation_notes: list[str] = Field(default_factory=list)
    selection_notes: list[str] = Field(default_factory=list)
    raw_metadata: dict[str, Any] = Field(default_factory=dict)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)
    validation: DocumentValidationResult | None = None
    evaluation: CandidateEvaluation | None = None
    duplicate_key: str | None = None
    superseded_by_candidate_id: str | None = None

    def effective_timestamp(self) -> datetime | None:
        if self.updated_at is not None:
            return self.updated_at
        if self.published_at is not None:
            return self.published_at
        if self.event_date is not None:
            return datetime.combine(self.event_date, time.min, tzinfo=timezone.utc)
        return None


class RetrievalSelectionDecision(ContractModel):
    """Selection audit trail for most-recent relevant candidate choice."""

    selected_candidate_id: str | None = None
    selected_reason: str | None = None
    rejected_candidate_ids: list[str] = Field(default_factory=list)
    ranking_notes: list[str] = Field(default_factory=list)
    tie_break_notes: list[str] = Field(default_factory=list)
    ambiguity_notes: list[str] = Field(default_factory=list)


class RetrievalResult(ContractModel):
    """Retrieval output for one request and one disclosure type."""

    request: RetrievalRequest
    adapter_name: str
    status: ProcessingStatus = ProcessingStatus.PENDING
    selected_candidate: RetrievalCandidate | None = None
    baseline_candidate: RetrievalCandidate | None = None
    candidates: list[RetrievalCandidate] = Field(default_factory=list)
    selection_decision: RetrievalSelectionDecision = Field(default_factory=RetrievalSelectionDecision)
    issues: list[PipelineError] = Field(default_factory=list)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)
    is_mock_result: bool = False
    retrieved_at: datetime = Field(default_factory=now_utc)


## 10. Provenance and Audit Models

In [ ]:
def build_provenance_record(
    *,
    stage: str,
    adapter_name: str,
    candidate_id: str | None = None,
    document_type: DocumentType | None = None,
    source_name: str | None = None,
    source_identifier: str | None = None,
    source_url: str | None = None,
    note: str | None = None,
    ranking_notes: list[str] | None = None,
    validation_notes: list[str] | None = None,
    selection_reason: str | None = None,
    is_mock_data: bool = False,
    metadata: dict[str, Any] | None = None,
) -> ProvenanceRecord:
    """Build a provider-agnostic provenance record for retrieval auditing."""
    return ProvenanceRecord(
        stage=stage,
        adapter_name=adapter_name,
        candidate_id=candidate_id,
        document_type=document_type,
        source_name=source_name,
        source_identifier=source_identifier,
        source_url=source_url,
        retrieved_at=now_utc(),
        note=note,
        ranking_notes=ranking_notes or [],
        validation_notes=validation_notes or [],
        selection_reason=selection_reason,
        is_mock_data=is_mock_data,
        metadata=metadata or {},
    )


def append_candidate_provenance(
    candidate: RetrievalCandidate,
    *,
    stage: str,
    note: str | None = None,
    ranking_notes: list[str] | None = None,
    validation_notes: list[str] | None = None,
    selection_reason: str | None = None,
    metadata: dict[str, Any] | None = None,
) -> RetrievalCandidate:
    """Append one audit trail record to a normalized candidate."""
    candidate.provenance.append(
        build_provenance_record(
            stage=stage,
            adapter_name=candidate.adapter_name,
            candidate_id=candidate.candidate_id,
            document_type=candidate.document_type,
            source_name=candidate.source_name,
            source_identifier=candidate.source_identifier,
            source_url=candidate.source_url,
            note=note,
            ranking_notes=ranking_notes,
            validation_notes=validation_notes,
            selection_reason=selection_reason,
            is_mock_data=candidate.is_mock_data,
            metadata=metadata,
        )
    )
    return candidate


def build_document_metadata_from_candidate(candidate: RetrievalCandidate) -> DocumentMetadata:
    """Convert a selected retrieval candidate into downstream document metadata."""
    return DocumentMetadata(
        document_id=candidate.candidate_id,
        ticker=candidate.ticker or "UNKNOWN",
        document_type=candidate.document_type,
        company_name=candidate.company_name,
        title=candidate.title,
        source_name=candidate.source_name,
        source_identifier=candidate.source_identifier,
        source_family=candidate.source_family,
        source_url=candidate.source_url,
        published_at=candidate.published_at,
        updated_at=candidate.updated_at,
        event_date=candidate.event_date,
        retrieved_at=candidate.retrieved_at,
        is_structured_source=candidate.is_structured_source,
        is_mock_data=candidate.is_mock_data,
        raw_metadata=candidate.raw_metadata,
        notes=(candidate.validation_notes + candidate.selection_notes),
    )


## 11. Structured Retrieval Error Handling

In [ ]:
class RetrievalErrorCode(str, Enum):
    """Retrieval-specific error codes surfaced by adapters and selection logic."""

    NO_CANDIDATES_FOUND = "no_candidates_found"
    CANDIDATES_FOUND_BUT_NONE_VALID = "candidates_found_but_none_valid"
    AMBIGUOUS_RECENT_CANDIDATES = "ambiguous_recent_candidates"
    MISSING_DOCUMENT_TEXT = "missing_document_text"
    INSUFFICIENT_METADATA = "insufficient_metadata"
    DOCUMENT_TYPE_MISMATCH = "document_type_mismatch"
    REQUEST_IDENTITY_MISMATCH = "request_identity_mismatch"
    UNSUPPORTED_DOCUMENT_TYPE = "unsupported_document_type"
    ADAPTER_FAILURE = "adapter_failure"


class RetrievalError(PipelineError):
    """Structured retrieval-layer error object."""

    error_code: RetrievalErrorCode
    is_mock_data: bool = False


def make_retrieval_error(
    code: RetrievalErrorCode,
    *,
    message: str,
    stage: str,
    document_type: DocumentType | None = None,
    candidate_id: str | None = None,
    adapter_name: str | None = None,
    recoverable: bool = True,
    is_mock_data: bool = False,
    details: dict[str, Any] | None = None,
) -> RetrievalError:
    """Create a retrieval error without assuming provider-specific fields."""
    return RetrievalError(
        error_code=code,
        message=message,
        stage=stage,
        document_type=document_type,
        candidate_id=candidate_id,
        adapter_name=adapter_name,
        recoverable=recoverable,
        is_mock_data=is_mock_data,
        details=details or {},
    )


## 12. Retrieval Adapter Interfaces

In [ ]:
class BaseRetrievalAdapter(ABC):
    """Base retrieval adapter interface for one disclosure type."""

    adapter_name: ClassVar[str] = "base_retrieval_adapter"
    document_type: ClassVar[DocumentType]

    @abstractmethod
    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        """Return raw candidate metadata records for the request."""

    @abstractmethod
    def fetch_document(
        self,
        raw_candidate: RawRetrievalCandidateMetadata,
        request: RetrievalRequest,
    ) -> FetchedDocumentContent:
        """Return document content for one raw candidate."""

    def normalize_candidate(
        self,
        raw_candidate: RawRetrievalCandidateMetadata,
        fetched_document: FetchedDocumentContent,
        request: RetrievalRequest,
    ) -> RetrievalCandidate:
        """Normalize one raw candidate and fetched document into the internal schema."""
        normalized_text = normalize_document_text(fetched_document.document_text)
        candidate = RetrievalCandidate(
            candidate_id=f"{self.adapter_name}:{raw_candidate.raw_candidate_id}",
            adapter_name=self.adapter_name,
            document_type=raw_candidate.document_type,
            ticker=(raw_candidate.ticker or request.ticker or "").upper() or None,
            company_name=raw_candidate.company_name or request.company_name,
            source_name=raw_candidate.source_name,
            source_identifier=raw_candidate.source_identifier,
            source_url=raw_candidate.source_url,
            source_family=raw_candidate.source_family,
            title=raw_candidate.title,
            published_at=raw_candidate.published_at,
            updated_at=raw_candidate.updated_at,
            event_date=raw_candidate.event_date,
            retrieved_at=fetched_document.fetched_at,
            document_text=normalized_text,
            is_structured_source=raw_candidate.is_structured_source,
            is_mock_data=bool(raw_candidate.is_mock_data or fetched_document.is_mock_data or request.is_mock_request),
            raw_metadata={**raw_candidate.raw_metadata, **fetched_document.raw_payload},
        )
        candidate.duplicate_key = build_candidate_duplicate_key(candidate)
        append_candidate_provenance(
            candidate,
            stage="normalize",
            note="Candidate normalized into internal retrieval schema.",
            metadata={"fetch_status": fetched_document.fetch_status.value},
        )
        return candidate

    def validate_document(
        self,
        candidate: RetrievalCandidate,
        request: RetrievalRequest,
    ) -> DocumentValidationResult:
        """Validate one normalized candidate using generic notebook logic."""
        return validate_candidate_document(candidate, request)

    def retrieve(self, request: RetrievalRequest) -> RetrievalResult:
        """Run a full retrieval pass using provider-agnostic notebook logic."""
        result = RetrievalResult(
            request=request,
            adapter_name=self.adapter_name,
            is_mock_result=request.is_mock_request,
        )

        if request.document_type != self.document_type:
            result.status = ProcessingStatus.RETRIEVAL_FAILED
            result.issues.append(
                make_retrieval_error(
                    RetrievalErrorCode.UNSUPPORTED_DOCUMENT_TYPE,
                    message="Request document type does not match adapter type.",
                    stage="retrieve",
                    document_type=request.document_type,
                    adapter_name=self.adapter_name,
                    recoverable=False,
                    is_mock_data=request.is_mock_request,
                    details={
                        "request_document_type": request.document_type.value,
                        "adapter_document_type": self.document_type.value,
                    },
                )
            )
            return result

        try:
            raw_candidates = self.search_candidates(request)[: request.max_candidates]
        except Exception as exc:
            result.status = ProcessingStatus.RETRIEVAL_FAILED
            result.issues.append(
                make_retrieval_error(
                    RetrievalErrorCode.ADAPTER_FAILURE,
                    message=f"Adapter search failed: {exc}",
                    stage="search_candidates",
                    document_type=request.document_type,
                    adapter_name=self.adapter_name,
                    recoverable=True,
                    is_mock_data=request.is_mock_request,
                )
            )
            return result

        if not raw_candidates:
            result.status = ProcessingStatus.NO_DOCUMENT
            result.issues.append(
                make_retrieval_error(
                    RetrievalErrorCode.NO_CANDIDATES_FOUND,
                    message="Adapter returned no candidates for the request.",
                    stage="search_candidates",
                    document_type=request.document_type,
                    adapter_name=self.adapter_name,
                    recoverable=True,
                    is_mock_data=request.is_mock_request,
                )
            )
            result.provenance.append(
                build_provenance_record(
                    stage="search_candidates",
                    adapter_name=self.adapter_name,
                    document_type=request.document_type,
                    note="No candidates returned.",
                    is_mock_data=request.is_mock_request,
                )
            )
            return result

        normalized_candidates: list[RetrievalCandidate] = []
        for raw_candidate in raw_candidates:
            try:
                fetched_document = self.fetch_document(raw_candidate, request)
                candidate = self.normalize_candidate(raw_candidate, fetched_document, request)
                validation = self.validate_document(candidate, request)
                candidate.validation = validation
                candidate.validation_notes.extend(validation.validation_notes)
                append_candidate_provenance(
                    candidate,
                    stage="validate",
                    note="Candidate validated against generic notebook rules.",
                    validation_notes=validation.validation_notes,
                    metadata={
                        "is_valid": validation.is_valid,
                        "is_partial": validation.is_partial,
                        "status": validation.status.value,
                    },
                )
                normalized_candidates.append(candidate)
            except Exception as exc:
                result.issues.append(
                    make_retrieval_error(
                        RetrievalErrorCode.ADAPTER_FAILURE,
                        message=f"Candidate normalization failed: {exc}",
                        stage="normalize_candidate",
                        document_type=request.document_type,
                        candidate_id=getattr(raw_candidate, "raw_candidate_id", None),
                        adapter_name=self.adapter_name,
                        recoverable=True,
                        is_mock_data=request.is_mock_request,
                    )
                )

        selection_result = select_most_recent_relevant_candidate(
            candidates=normalized_candidates,
            request=request,
            adapter_name=self.adapter_name,
        )
        selection_result.issues.extend(result.issues)
        selection_result.is_mock_result = bool(
            request.is_mock_request or any(candidate.is_mock_data for candidate in normalized_candidates)
        )
        selection_result.provenance.insert(
            0,
            build_provenance_record(
                stage="retrieve",
                adapter_name=self.adapter_name,
                document_type=request.document_type,
                note=f"Adapter processed {len(raw_candidates)} raw candidate(s).",
                is_mock_data=selection_result.is_mock_result,
            ),
        )
        return selection_result


## 13. Candidate Relevance and Freshness Logic

In [ ]:
def normalize_token(value: str | None) -> str:
    """Lowercase and collapse whitespace for simple identity matching."""
    if not value:
        return ""
    return " ".join(value.lower().split())


def candidate_effective_timestamp(candidate: RetrievalCandidate) -> datetime:
    """Return the effective timestamp used for freshness ranking."""
    return candidate.effective_timestamp() or MIN_DATETIME_UTC


def source_preference_index(source_family: SourceFamily, request: RetrievalRequest) -> int:
    """Return the configured source-family order index for tie-breaking."""
    preferences = request.source_preferences.preferred_source_families
    if source_family in preferences:
        return preferences.index(source_family)
    return len(preferences)


def build_candidate_duplicate_key(candidate: RetrievalCandidate) -> str:
    """Build a simple duplicate-group key for superseded-version handling."""
    if candidate.source_identifier:
        return f"source_identifier::{normalize_token(candidate.source_identifier)}"
    if candidate.source_url:
        return f"source_url::{normalize_token(candidate.source_url)}"
    normalized_title = normalize_token(candidate.title) or "untitled"
    date_key = ""
    if candidate.event_date is not None:
        date_key = candidate.event_date.isoformat()
    elif candidate.published_at is not None:
        date_key = candidate.published_at.date().isoformat()
    elif candidate.updated_at is not None:
        date_key = candidate.updated_at.date().isoformat()
    return f"title_date::{candidate.document_type.value}::{normalized_title}::{date_key}"


def _identity_alignment_score(candidate: RetrievalCandidate, request: RetrievalRequest) -> tuple[float, list[str], bool]:
    notes: list[str] = []
    request_ticker = normalize_token(request.ticker)
    candidate_ticker = normalize_token(candidate.ticker)
    if candidate_ticker:
        if candidate_ticker == request_ticker:
            notes.append("Ticker matches request.")
            return 25.0, notes, True
        notes.append("Ticker does not match request.")
        return -25.0, notes, False

    alias_tokens = [normalize_token(request.company_name)] if request.company_name else []
    alias_tokens.extend(normalize_token(alias) for alias in request.company_aliases if alias)
    metadata_blob = normalize_token(" ".join(filter(None, [candidate.company_name, candidate.title, candidate.source_name])))
    if any(alias and alias in metadata_blob for alias in alias_tokens):
        notes.append("Company name or alias aligns with candidate metadata.")
        return 18.0, notes, True

    notes.append("Identity could not be confirmed from candidate metadata.")
    return 0.0, notes, False


def _title_alignment_score(candidate: RetrievalCandidate, request: RetrievalRequest) -> tuple[float, list[str]]:
    notes: list[str] = []
    score = 0.0
    normalized_title = normalize_token(candidate.title)
    if not normalized_title:
        notes.append("Title is missing; no title-alignment bonus applied.")
        return score, notes

    if normalize_token(request.ticker) and normalize_token(request.ticker) in normalized_title:
        score += 5.0
        notes.append("Title contains the requested ticker.")

    alias_tokens = [normalize_token(request.company_name)] if request.company_name else []
    alias_tokens.extend(normalize_token(alias) for alias in request.company_aliases if alias)
    if any(alias and alias in normalized_title for alias in alias_tokens):
        score += 5.0
        notes.append("Title contains the company name or alias.")

    if score == 0.0:
        notes.append("Title provides no direct identity bonus.")
    return score, notes


def evaluate_candidate_relevance(candidate: RetrievalCandidate, request: RetrievalRequest) -> CandidateEvaluation:
    """Assign transparent heuristic relevance scores without LLM calls."""
    notes: list[str] = []

    if candidate.document_type == request.document_type:
        type_score = 50.0
        notes.append("Document type matches request.")
    else:
        type_score = 0.0
        notes.append("Document type does not match request.")

    identity_score, identity_notes, identity_ok = _identity_alignment_score(candidate, request)
    notes.extend(identity_notes)

    title_score, title_notes = _title_alignment_score(candidate, request)
    notes.extend(title_notes)

    if request.source_preferences.prefer_structured_source and candidate.is_structured_source:
        structured_bonus = 5.0
        notes.append("Structured source bonus applied.")
    else:
        structured_bonus = 0.0

    preference_index = source_preference_index(candidate.source_family, request)
    source_bonus_lookup = {0: 5.0, 1: 3.0, 2: 1.0}
    source_preference_bonus = source_bonus_lookup.get(preference_index, 0.0)
    if source_preference_bonus:
        notes.append("Source-family preference bonus applied.")

    if candidate.source_family == SourceFamily.PERMITTED_SECONDARY and not request.source_preferences.allow_secondary_sources:
        notes.append("Secondary sources are disabled for this request.")
    if candidate.source_family == SourceFamily.UNKNOWN and not request.source_preferences.allow_unknown_sources:
        notes.append("Unknown source families are disabled for this request.")

    relevance_score = type_score + identity_score + title_score + structured_bonus + source_preference_bonus
    validation_ok = bool(candidate.validation and candidate.validation.is_valid)
    source_allowed = not (
        (candidate.source_family == SourceFamily.PERMITTED_SECONDARY and not request.source_preferences.allow_secondary_sources)
        or (candidate.source_family == SourceFamily.UNKNOWN and not request.source_preferences.allow_unknown_sources)
        or (candidate.is_mock_data and not request.source_preferences.allow_mock_data)
    )

    is_relevant = bool(
        validation_ok
        and source_allowed
        and type_score > 0.0
        and identity_score >= 0.0
        and relevance_score >= request.minimum_relevance_score
    )
    if not validation_ok:
        notes.append("Candidate failed validation and cannot be selected.")
    if not source_allowed:
        notes.append("Candidate source is not allowed by request preferences.")
    if is_relevant:
        notes.append("Candidate cleared the minimum relevance threshold.")
    else:
        notes.append("Candidate did not clear the minimum relevance threshold.")

    evaluation = CandidateEvaluation(
        candidate_id=candidate.candidate_id,
        is_relevant=is_relevant,
        relevance_score=relevance_score,
        freshness_score=0.0,
        total_score=relevance_score,
        type_compatibility_score=type_score,
        identity_alignment_score=identity_score,
        title_alignment_score=title_score,
        structured_source_bonus=structured_bonus,
        source_preference_bonus=source_preference_bonus,
        duplicate_key=build_candidate_duplicate_key(candidate),
        effective_timestamp=candidate.effective_timestamp(),
        evaluation_notes=notes,
    )
    candidate.evaluation = evaluation
    candidate.relevance_notes.extend(notes)
    candidate.duplicate_key = evaluation.duplicate_key
    return evaluation


def assign_candidate_freshness(candidates: Sequence[RetrievalCandidate], request: RetrievalRequest) -> list[RetrievalCandidate]:
    """Apply freshness scores after validation and duplicate handling."""
    ranked_for_freshness = sorted(
        [candidate for candidate in candidates if candidate.superseded_by_candidate_id is None],
        key=lambda candidate: (
            -candidate_effective_timestamp(candidate).timestamp(),
            source_preference_index(candidate.source_family, request),
            -int(candidate.is_structured_source),
            candidate.candidate_id,
        ),
    )

    for rank, candidate in enumerate(ranked_for_freshness):
        freshness_score = 0.0
        effective_timestamp = candidate.evaluation.effective_timestamp if candidate.evaluation else candidate.effective_timestamp()
        if effective_timestamp is not None:
            freshness_score = max(0.0, request.freshness_bonus_max - (rank * request.freshness_rank_decay))
            candidate.freshness_notes.append(
                f"Freshness rank {rank + 1} among {len(ranked_for_freshness)} non-superseded candidates."
            )
        else:
            candidate.freshness_notes.append("No timestamp available; freshness bonus set to 0.")

        if candidate.evaluation is None:
            candidate.evaluation = evaluate_candidate_relevance(candidate, request)
        candidate.evaluation.freshness_score = freshness_score
        candidate.evaluation.total_score = candidate.evaluation.relevance_score + freshness_score
    return list(candidates)


## 14. Most Recent Relevant Selection Logic

In [ ]:
def validate_candidate_document(candidate: RetrievalCandidate, request: RetrievalRequest) -> DocumentValidationResult:
    """Validate one normalized candidate using generic retrieval rules."""
    notes: list[str] = []
    issues: list[PipelineError] = []

    normalized_text = normalize_document_text(candidate.document_text)
    has_usable_text = bool(normalized_text and len(normalized_text) >= request.minimum_text_chars)
    has_minimum_metadata = bool(candidate.title or candidate.source_identifier or candidate.source_url)
    type_matches_request = candidate.document_type == request.document_type

    identity_score, identity_notes, identity_matches_request = _identity_alignment_score(candidate, request)
    notes.extend(identity_notes)

    if has_usable_text:
        notes.append("Candidate contains usable text.")
    else:
        notes.append("Candidate text is missing or too short.")
        issues.append(
            make_retrieval_error(
                RetrievalErrorCode.MISSING_DOCUMENT_TEXT,
                message="Candidate is missing usable text.",
                stage="validate_document",
                document_type=request.document_type,
                candidate_id=candidate.candidate_id,
                adapter_name=candidate.adapter_name,
                is_mock_data=candidate.is_mock_data,
            )
        )

    if has_minimum_metadata:
        notes.append("Candidate has minimally sufficient metadata.")
    else:
        notes.append("Candidate is missing key metadata fields.")
        issues.append(
            make_retrieval_error(
                RetrievalErrorCode.INSUFFICIENT_METADATA,
                message="Candidate is missing title, source identifier, and source URL.",
                stage="validate_document",
                document_type=request.document_type,
                candidate_id=candidate.candidate_id,
                adapter_name=candidate.adapter_name,
                is_mock_data=candidate.is_mock_data,
            )
        )

    if type_matches_request:
        notes.append("Candidate document type matches the request.")
    else:
        notes.append("Candidate document type does not match the request.")
        issues.append(
            make_retrieval_error(
                RetrievalErrorCode.DOCUMENT_TYPE_MISMATCH,
                message="Candidate document type does not match the request.",
                stage="validate_document",
                document_type=request.document_type,
                candidate_id=candidate.candidate_id,
                adapter_name=candidate.adapter_name,
                is_mock_data=candidate.is_mock_data,
            )
        )

    if identity_matches_request:
        notes.append("Candidate identity aligns with the request.")
    else:
        notes.append("Candidate identity could not be confirmed for the request.")
        if identity_score < 0.0:
            issues.append(
                make_retrieval_error(
                    RetrievalErrorCode.REQUEST_IDENTITY_MISMATCH,
                    message="Candidate ticker conflicts with the request ticker.",
                    stage="validate_document",
                    document_type=request.document_type,
                    candidate_id=candidate.candidate_id,
                    adapter_name=candidate.adapter_name,
                    is_mock_data=candidate.is_mock_data,
                )
            )

    is_valid = has_usable_text and type_matches_request and (identity_matches_request or identity_score == 0.0)
    is_partial = is_valid and not has_minimum_metadata

    if is_valid and is_partial:
        status = ProcessingStatus.PARTIAL
    elif is_valid:
        status = ProcessingStatus.SUCCESS
    elif not has_usable_text:
        status = ProcessingStatus.EXTRACTION_FAILED
    else:
        status = ProcessingStatus.SELECTION_FAILED

    return DocumentValidationResult(
        candidate_id=candidate.candidate_id,
        status=status,
        is_valid=is_valid,
        is_partial=is_partial,
        has_usable_text=has_usable_text,
        has_minimum_metadata=has_minimum_metadata,
        type_matches_request=type_matches_request,
        identity_matches_request=identity_matches_request or identity_score == 0.0,
        validation_notes=notes,
        issues=issues,
    )


def deduplicate_candidates(
    candidates: Sequence[RetrievalCandidate],
    request: RetrievalRequest,
) -> list[RetrievalCandidate]:
    """Mark older duplicates as superseded and keep the freshest candidate per duplicate group."""
    grouped: dict[str, list[RetrievalCandidate]] = {}
    for candidate in candidates:
        duplicate_key = candidate.duplicate_key or build_candidate_duplicate_key(candidate)
        candidate.duplicate_key = duplicate_key
        grouped.setdefault(duplicate_key, []).append(candidate)

    deduplicated: list[RetrievalCandidate] = []
    for duplicate_key, group in grouped.items():
        ordered_group = sorted(
            group,
            key=lambda candidate: (
                -candidate_effective_timestamp(candidate).timestamp(),
                source_preference_index(candidate.source_family, request),
                -int(candidate.is_structured_source),
                candidate.candidate_id,
            ),
        )
        winner = ordered_group[0]
        deduplicated.append(winner)
        for superseded_candidate in ordered_group[1:]:
            superseded_candidate.superseded_by_candidate_id = winner.candidate_id
            superseded_candidate.selection_notes.append(
                f"Superseded by {winner.candidate_id} within duplicate group {duplicate_key}."
            )
    return deduplicated


def candidate_priority_signature(candidate: RetrievalCandidate, request: RetrievalRequest) -> tuple[Any, ...]:
    """Return the explicit tie-break signature used by selection."""
    evaluation = candidate.evaluation or CandidateEvaluation(candidate_id=candidate.candidate_id)
    return (
        round(evaluation.total_score, 6),
        candidate_effective_timestamp(candidate).timestamp(),
        -source_preference_index(candidate.source_family, request),
        int(candidate.is_structured_source),
        (candidate.updated_at or MIN_DATETIME_UTC).timestamp(),
        (candidate.published_at or MIN_DATETIME_UTC).timestamp(),
    )


def selection_sort_key(candidate: RetrievalCandidate, request: RetrievalRequest) -> tuple[Any, ...]:
    """Return the deterministic sorting key for winner selection."""
    evaluation = candidate.evaluation or CandidateEvaluation(candidate_id=candidate.candidate_id)
    return (
        -evaluation.total_score,
        -candidate_effective_timestamp(candidate).timestamp(),
        source_preference_index(candidate.source_family, request),
        -int(candidate.is_structured_source),
        -(candidate.updated_at or MIN_DATETIME_UTC).timestamp(),
        -(candidate.published_at or MIN_DATETIME_UTC).timestamp(),
        candidate.candidate_id,
    )


def select_most_recent_relevant_candidate(
    *,
    candidates: Sequence[RetrievalCandidate],
    request: RetrievalRequest,
    adapter_name: str,
) -> RetrievalResult:
    """Select the most recent relevant valid candidate with explicit ranking notes."""
    result = RetrievalResult(
        request=request,
        adapter_name=adapter_name,
        candidates=list(candidates),
        is_mock_result=bool(request.is_mock_request or any(candidate.is_mock_data for candidate in candidates)),
    )

    if not candidates:
        result.status = ProcessingStatus.NO_DOCUMENT
        result.issues.append(
            make_retrieval_error(
                RetrievalErrorCode.NO_CANDIDATES_FOUND,
                message="No normalized candidates were available for selection.",
                stage="select_candidate",
                document_type=request.document_type,
                adapter_name=adapter_name,
                is_mock_data=result.is_mock_result,
            )
        )
        return result

    for candidate in result.candidates:
        if candidate.validation is None:
            candidate.validation = validate_candidate_document(candidate, request)
            candidate.validation_notes.extend(candidate.validation.validation_notes)
        if candidate.evaluation is None:
            evaluate_candidate_relevance(candidate, request)

    deduplicated_candidates = deduplicate_candidates(result.candidates, request)
    assign_candidate_freshness(deduplicated_candidates, request)

    valid_candidates = [candidate for candidate in deduplicated_candidates if candidate.validation and candidate.validation.is_valid]
    relevant_candidates = [candidate for candidate in valid_candidates if candidate.evaluation and candidate.evaluation.is_relevant]

    if not valid_candidates:
        result.status = ProcessingStatus.SELECTION_FAILED
        result.issues.append(
            make_retrieval_error(
                RetrievalErrorCode.CANDIDATES_FOUND_BUT_NONE_VALID,
                message="Candidates were found, but none passed validation.",
                stage="select_candidate",
                document_type=request.document_type,
                adapter_name=adapter_name,
                is_mock_data=result.is_mock_result,
                details={"candidate_count": len(result.candidates)},
            )
        )
        result.selection_decision.rejected_candidate_ids = [candidate.candidate_id for candidate in result.candidates]
        return result

    if not relevant_candidates:
        result.status = ProcessingStatus.SELECTION_FAILED
        result.issues.append(
            make_retrieval_error(
                RetrievalErrorCode.CANDIDATES_FOUND_BUT_NONE_VALID,
                message="Validated candidates existed, but none were relevant enough to select.",
                stage="select_candidate",
                document_type=request.document_type,
                adapter_name=adapter_name,
                is_mock_data=result.is_mock_result,
                details={"valid_candidate_count": len(valid_candidates)},
            )
        )
        result.selection_decision.rejected_candidate_ids = [candidate.candidate_id for candidate in result.candidates]
        return result

    ranked_candidates = sorted(relevant_candidates, key=lambda candidate: selection_sort_key(candidate, request))
    ranking_notes: list[str] = []
    for rank, candidate in enumerate(ranked_candidates, start=1):
        evaluation = candidate.evaluation or CandidateEvaluation(candidate_id=candidate.candidate_id)
        ranking_notes.append(
            " | ".join(
                [
                    f"rank={rank}",
                    f"candidate_id={candidate.candidate_id}",
                    f"total_score={evaluation.total_score:.1f}",
                    f"relevance_score={evaluation.relevance_score:.1f}",
                    f"freshness_score={evaluation.freshness_score:.1f}",
                    f"effective_timestamp={candidate_effective_timestamp(candidate).isoformat()}",
                ]
            )
        )

    winner = ranked_candidates[0]
    ambiguity_notes: list[str] = []
    tie_break_notes: list[str] = [
        "Tie-break order: total_score desc, effective_timestamp desc, source preference asc, structured source desc, updated_at desc, published_at desc, candidate_id asc."
    ]
    if len(ranked_candidates) > 1:
        top_signature = candidate_priority_signature(ranked_candidates[0], request)
        second_signature = candidate_priority_signature(ranked_candidates[1], request)
        if top_signature == second_signature:
            ambiguity_notes.append(
                "Top candidates matched on all intended ranking dimensions before candidate_id tie-break. Deterministic selection used candidate_id."
            )
            result.issues.append(
                make_retrieval_error(
                    RetrievalErrorCode.AMBIGUOUS_RECENT_CANDIDATES,
                    message="Multiple top candidates remained tied after ranking and required candidate_id tie-break.",
                    stage="select_candidate",
                    document_type=request.document_type,
                    adapter_name=adapter_name,
                    is_mock_data=result.is_mock_result,
                    details={
                        "top_candidate_ids": [ranked_candidates[0].candidate_id, ranked_candidates[1].candidate_id],
                    },
                )
            )

    selection_reason = (
        f"Selected {winner.candidate_id} as the highest-ranked relevant candidate after validation, duplicate handling, and freshness ranking."
    )
    winner.selection_notes.append(selection_reason)
    append_candidate_provenance(
        winner,
        stage="select_candidate",
        note="Candidate selected as retrieval winner.",
        ranking_notes=ranking_notes,
        selection_reason=selection_reason,
    )

    result.selected_candidate = winner
    result.selection_decision = RetrievalSelectionDecision(
        selected_candidate_id=winner.candidate_id,
        selected_reason=selection_reason,
        rejected_candidate_ids=[candidate.candidate_id for candidate in result.candidates if candidate.candidate_id != winner.candidate_id],
        ranking_notes=ranking_notes,
        tie_break_notes=tie_break_notes,
        ambiguity_notes=ambiguity_notes,
    )
    result.provenance.append(
        build_provenance_record(
            stage="select_candidate",
            adapter_name=adapter_name,
            candidate_id=winner.candidate_id,
            document_type=request.document_type,
            source_name=winner.source_name,
            source_identifier=winner.source_identifier,
            source_url=winner.source_url,
            note="Selection completed.",
            ranking_notes=ranking_notes,
            selection_reason=selection_reason,
            is_mock_data=result.is_mock_result,
        )
    )

    if winner.validation and winner.validation.is_partial:
        result.status = ProcessingStatus.PARTIAL
    elif ambiguity_notes:
        result.status = ProcessingStatus.PARTIAL
    else:
        result.status = ProcessingStatus.SUCCESS
    return result


## 15. Mock Retrieval Adapters

In [ ]:
def make_mock_raw_candidate(
    *,
    raw_candidate_id: str,
    adapter_name: str,
    document_type: DocumentType,
    ticker: str | None,
    company_name: str | None,
    source_name: str,
    source_identifier: str | None,
    source_url: str | None,
    title: str | None,
    source_family: SourceFamily,
    published_at: datetime | None,
    updated_at: datetime | None,
    event_date: date | None,
    is_structured_source: bool,
    raw_metadata: dict[str, Any] | None = None,
) -> RawRetrievalCandidateMetadata:
    """Create a fake raw candidate for notebook demos."""
    return RawRetrievalCandidateMetadata(
        raw_candidate_id=raw_candidate_id,
        adapter_name=adapter_name,
        document_type=document_type,
        ticker=ticker,
        company_name=company_name,
        source_name=source_name,
        source_identifier=source_identifier,
        source_url=source_url,
        title=title,
        source_family=source_family,
        published_at=published_at,
        updated_at=updated_at,
        event_date=event_date,
        is_structured_source=is_structured_source,
        is_mock_data=True,
        raw_metadata=raw_metadata or {"demo_data": True},
    )


def make_mock_fetched_document(
    *,
    raw_candidate_id: str,
    document_text: str | None,
    content_type: str = "text/plain",
    fetch_notes: list[str] | None = None,
    raw_payload: dict[str, Any] | None = None,
) -> FetchedDocumentContent:
    """Create fake fetched content for notebook demos."""
    return FetchedDocumentContent(
        raw_candidate_id=raw_candidate_id,
        document_text=document_text,
        content_type=content_type,
        fetched_at=now_utc(),
        fetch_status=ProcessingStatus.SUCCESS,
        fetch_notes=fetch_notes or ["Synthetic fetched document for notebook demo."],
        is_mock_data=True,
        raw_payload=raw_payload or {"demo_payload": True},
    )


class MockBaseRetrievalAdapter(BaseRetrievalAdapter):
    """Base class for notebook-only mock retrieval adapters."""

    def build_demo_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        raise NotImplementedError

    def build_demo_document_store(self, request: RetrievalRequest) -> dict[str, FetchedDocumentContent]:
        raise NotImplementedError

    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        return self.build_demo_candidates(request)[: request.max_candidates]

    def fetch_document(
        self,
        raw_candidate: RawRetrievalCandidateMetadata,
        request: RetrievalRequest,
    ) -> FetchedDocumentContent:
        document_store = self.build_demo_document_store(request)
        return document_store[raw_candidate.raw_candidate_id]


class MockMaterialEventRetrievalAdapter(MockBaseRetrievalAdapter):
    adapter_name = "mock_material_event_adapter"
    document_type = DocumentType.MATERIAL_EVENT

    def build_demo_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        return [
            make_mock_raw_candidate(
                raw_candidate_id="material_001_latest",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock SEC Filing Feed",
                source_identifier="event-material-001",
                source_url="https://example.com/mock/material/latest",
                title=f"{request.ticker} files material event update",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 21, 16, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 21, 18, 0, tzinfo=timezone.utc),
                event_date=date(2026, 2, 21),
                is_structured_source=True,
                raw_metadata={"demo_variant": "latest_duplicate_winner"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="material_001_older",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock SEC Filing Feed",
                source_identifier="event-material-001",
                source_url="https://example.com/mock/material/older",
                title=f"{request.ticker} files material event update",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 21, 15, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 21, 15, 30, tzinfo=timezone.utc),
                event_date=date(2026, 2, 21),
                is_structured_source=True,
                raw_metadata={"demo_variant": "older_duplicate_loser"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="material_002_wrong_ticker",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker="OTHR",
                company_name="Other Bio Corp",
                source_name="Mock Issuer IR Site",
                source_identifier="event-material-002",
                source_url="https://example.com/mock/material/other",
                title="OTHR announces unrelated material update",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=datetime(2026, 2, 22, 10, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 22, 11, 0, tzinfo=timezone.utc),
                event_date=date(2026, 2, 22),
                is_structured_source=False,
                raw_metadata={"demo_variant": "identity_mismatch"},
            ),
        ]

    def build_demo_document_store(self, request: RetrievalRequest) -> dict[str, FetchedDocumentContent]:
        return {
            "material_001_latest": make_mock_fetched_document(
                raw_candidate_id="material_001_latest",
                document_text=(
                    f"{request.company_name or request.ticker} disclosed a material event update with board-approved next steps and operational timing details."
                ),
                raw_payload={"demo_variant": "latest_duplicate_winner"},
            ),
            "material_001_older": make_mock_fetched_document(
                raw_candidate_id="material_001_older",
                document_text=(
                    f"{request.company_name or request.ticker} disclosed an earlier material event version before the latest amendment."
                ),
                raw_payload={"demo_variant": "older_duplicate_loser"},
            ),
            "material_002_wrong_ticker": make_mock_fetched_document(
                raw_candidate_id="material_002_wrong_ticker",
                document_text="Other Bio Corp published a separate material update unrelated to the requested issuer.",
                raw_payload={"demo_variant": "identity_mismatch"},
            ),
        }


class MockClinicalTrialUpdateRetrievalAdapter(MockBaseRetrievalAdapter):
    adapter_name = "mock_clinical_trial_update_adapter"
    document_type = DocumentType.CLINICAL_TRIAL_UPDATE

    def build_demo_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        return [
            make_mock_raw_candidate(
                raw_candidate_id="trial_001_latest",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Clinical Trial Registry",
                source_identifier="trial-record-001",
                source_url="https://example.com/mock/trial/latest",
                title=f"{request.company_name or request.ticker} phase 2 study registry update",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 25, 9, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 25, 14, 0, tzinfo=timezone.utc),
                event_date=date(2026, 2, 25),
                is_structured_source=True,
                raw_metadata={"demo_variant": "latest_registry_record"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="trial_001_older",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Clinical Trial Registry",
                source_identifier="trial-record-001",
                source_url="https://example.com/mock/trial/older",
                title=f"{request.company_name or request.ticker} phase 2 study registry update",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 1, 20, 9, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 1, 20, 12, 0, tzinfo=timezone.utc),
                event_date=date(2026, 1, 20),
                is_structured_source=True,
                raw_metadata={"demo_variant": "older_registry_record"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="trial_002_missing_text",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Clinical Trial Registry",
                source_identifier="trial-record-002",
                source_url="https://example.com/mock/trial/missing-text",
                title=f"{request.company_name or request.ticker} safety follow-up entry",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 27, 10, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 27, 11, 0, tzinfo=timezone.utc),
                event_date=date(2026, 2, 27),
                is_structured_source=True,
                raw_metadata={"demo_variant": "missing_text_should_fail"},
            ),
        ]

    def build_demo_document_store(self, request: RetrievalRequest) -> dict[str, FetchedDocumentContent]:
        return {
            "trial_001_latest": make_mock_fetched_document(
                raw_candidate_id="trial_001_latest",
                document_text=(
                    f"Registry entry for {request.company_name or request.ticker} was updated with revised enrollment timing, contact details, and outcome-measure wording."
                ),
            ),
            "trial_001_older": make_mock_fetched_document(
                raw_candidate_id="trial_001_older",
                document_text=(
                    f"Earlier registry entry for {request.company_name or request.ticker} before the most recent study record update."
                ),
            ),
            "trial_002_missing_text": make_mock_fetched_document(
                raw_candidate_id="trial_002_missing_text",
                document_text="Too short",
                raw_payload={"demo_variant": "missing_text_should_fail"},
            ),
        }


class MockFDAReviewRetrievalAdapter(MockBaseRetrievalAdapter):
    adapter_name = "mock_fda_review_adapter"
    document_type = DocumentType.FDA_REVIEW

    def build_demo_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        return [
            make_mock_raw_candidate(
                raw_candidate_id="fda_001_official",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock FDA Review Archive",
                source_identifier="fda-review-001",
                source_url="https://example.com/mock/fda/official",
                title=f"{request.company_name or request.ticker} review package summary",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 28, 8, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 28, 8, 30, tzinfo=timezone.utc),
                event_date=date(2026, 2, 28),
                is_structured_source=True,
                raw_metadata={"demo_variant": "official_should_win"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="fda_001_secondary",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Secondary Review Blog",
                source_identifier="fda-review-001-secondary",
                source_url="https://example.com/mock/fda/secondary",
                title=f"{request.company_name or request.ticker} review package summary",
                source_family=SourceFamily.PERMITTED_SECONDARY,
                published_at=datetime(2026, 2, 28, 8, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 28, 8, 30, tzinfo=timezone.utc),
                event_date=date(2026, 2, 28),
                is_structured_source=False,
                raw_metadata={"demo_variant": "secondary_same_day_loser"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="fda_000_older",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock FDA Review Archive",
                source_identifier="fda-review-000",
                source_url="https://example.com/mock/fda/older",
                title=f"{request.company_name or request.ticker} briefing materials",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 18, 8, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 18, 8, 15, tzinfo=timezone.utc),
                event_date=date(2026, 2, 18),
                is_structured_source=True,
                raw_metadata={"demo_variant": "older_official"},
            ),
        ]

    def build_demo_document_store(self, request: RetrievalRequest) -> dict[str, FetchedDocumentContent]:
        return {
            "fda_001_official": make_mock_fetched_document(
                raw_candidate_id="fda_001_official",
                document_text=(
                    f"Official review material for {request.company_name or request.ticker} summarized benefit-risk discussion, review discipline findings, and open regulatory questions."
                ),
            ),
            "fda_001_secondary": make_mock_fetched_document(
                raw_candidate_id="fda_001_secondary",
                document_text=(
                    f"Secondary commentary described the same review package for {request.company_name or request.ticker}, but it is not the official source."
                ),
            ),
            "fda_000_older": make_mock_fetched_document(
                raw_candidate_id="fda_000_older",
                document_text=(
                    f"Earlier official review material for {request.company_name or request.ticker} before the latest package was posted."
                ),
            ),
        }


class MockFinancingDilutionRetrievalAdapter(MockBaseRetrievalAdapter):
    adapter_name = "mock_financing_dilution_adapter"
    document_type = DocumentType.FINANCING_DILUTION

    def build_demo_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        return [
            make_mock_raw_candidate(
                raw_candidate_id="fin_002_latest_filing",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock SEC Filing Feed",
                source_identifier="financing-event-002",
                source_url="https://example.com/mock/financing/latest",
                title=f"{request.ticker} financing update filing",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 3, 1, 16, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 3, 1, 16, 30, tzinfo=timezone.utc),
                event_date=date(2026, 3, 1),
                is_structured_source=True,
                raw_metadata={"demo_variant": "latest_filing_should_win"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="fin_002_same_event_pr",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Issuer IR Site",
                source_identifier="financing-event-002-pr",
                source_url="https://example.com/mock/financing/pr",
                title=f"{request.company_name or request.ticker} announces financing",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=datetime(2026, 3, 1, 15, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 3, 1, 15, 5, tzinfo=timezone.utc),
                event_date=date(2026, 3, 1),
                is_structured_source=False,
                raw_metadata={"demo_variant": "same_day_pr_loser"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="fin_001_older",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock SEC Filing Feed",
                source_identifier="financing-event-001",
                source_url="https://example.com/mock/financing/older",
                title=f"{request.ticker} prior financing filing",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=datetime(2026, 2, 10, 12, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 10, 12, 30, tzinfo=timezone.utc),
                event_date=date(2026, 2, 10),
                is_structured_source=True,
                raw_metadata={"demo_variant": "older_event"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="fin_003_missing_text",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Issuer IR Site",
                source_identifier="financing-event-003",
                source_url="https://example.com/mock/financing/missing-text",
                title=f"{request.company_name or request.ticker} ATM update",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=datetime(2026, 3, 2, 9, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 3, 2, 9, 10, tzinfo=timezone.utc),
                event_date=date(2026, 3, 2),
                is_structured_source=False,
                raw_metadata={"demo_variant": "missing_text_should_fail"},
            ),
        ]

    def build_demo_document_store(self, request: RetrievalRequest) -> dict[str, FetchedDocumentContent]:
        return {
            "fin_002_latest_filing": make_mock_fetched_document(
                raw_candidate_id="fin_002_latest_filing",
                document_text=(
                    f"{request.company_name or request.ticker} filed a financing update describing aggregate proceeds, security type, and closing conditions."
                ),
            ),
            "fin_002_same_event_pr": make_mock_fetched_document(
                raw_candidate_id="fin_002_same_event_pr",
                document_text=(
                    f"{request.company_name or request.ticker} issued a same-day press release about the financing event summarized in the filing."
                ),
            ),
            "fin_001_older": make_mock_fetched_document(
                raw_candidate_id="fin_001_older",
                document_text=(
                    f"Older financing filing for {request.company_name or request.ticker} before the latest capital raise disclosure."
                ),
            ),
            "fin_003_missing_text": make_mock_fetched_document(
                raw_candidate_id="fin_003_missing_text",
                document_text=None,
                raw_payload={"demo_variant": "missing_text_should_fail"},
            ),
        }


class MockInvestorCommunicationRetrievalAdapter(MockBaseRetrievalAdapter):
    adapter_name = "mock_investor_communication_adapter"
    document_type = DocumentType.INVESTOR_COMMUNICATION

    def build_demo_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        return [
            make_mock_raw_candidate(
                raw_candidate_id="investor_002_audio_only",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Issuer Events Page",
                source_identifier="investor-event-002",
                source_url="https://example.com/mock/investor/audio-only",
                title=f"{request.company_name or request.ticker} webcast replay",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=datetime(2026, 3, 5, 13, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 3, 5, 13, 5, tzinfo=timezone.utc),
                event_date=date(2026, 3, 5),
                is_structured_source=False,
                raw_metadata={"demo_variant": "audio_only_should_fail"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="investor_001_deck_transcript",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Issuer Events Page",
                source_identifier="investor-event-001",
                source_url="https://example.com/mock/investor/deck-transcript",
                title=f"{request.company_name or request.ticker} corporate update transcript",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=datetime(2026, 2, 25, 17, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 25, 17, 15, tzinfo=timezone.utc),
                event_date=date(2026, 2, 25),
                is_structured_source=False,
                raw_metadata={"demo_variant": "latest_valid_should_win"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="investor_001_secondary",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Secondary Transcript Host",
                source_identifier="investor-event-001-secondary",
                source_url="https://example.com/mock/investor/secondary",
                title=f"{request.company_name or request.ticker} corporate update transcript",
                source_family=SourceFamily.PERMITTED_SECONDARY,
                published_at=datetime(2026, 2, 25, 17, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 2, 25, 17, 10, tzinfo=timezone.utc),
                event_date=date(2026, 2, 25),
                is_structured_source=False,
                raw_metadata={"demo_variant": "secondary_same_day_loser"},
            ),
            make_mock_raw_candidate(
                raw_candidate_id="investor_000_older",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=request.company_name,
                source_name="Mock Issuer Events Page",
                source_identifier="investor-event-000",
                source_url="https://example.com/mock/investor/older",
                title=f"{request.company_name or request.ticker} fireside chat transcript",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=datetime(2026, 1, 30, 16, 0, tzinfo=timezone.utc),
                updated_at=datetime(2026, 1, 30, 16, 5, tzinfo=timezone.utc),
                event_date=date(2026, 1, 30),
                is_structured_source=False,
                raw_metadata={"demo_variant": "older_valid"},
            ),
        ]

    def build_demo_document_store(self, request: RetrievalRequest) -> dict[str, FetchedDocumentContent]:
        return {
            "investor_002_audio_only": make_mock_fetched_document(
                raw_candidate_id="investor_002_audio_only",
                document_text=None,
                raw_payload={"demo_variant": "audio_only_should_fail"},
            ),
            "investor_001_deck_transcript": make_mock_fetched_document(
                raw_candidate_id="investor_001_deck_transcript",
                document_text=(
                    f"Transcript excerpt for {request.company_name or request.ticker} covered development priorities, financing runway commentary, and near-term milestones."
                ),
            ),
            "investor_001_secondary": make_mock_fetched_document(
                raw_candidate_id="investor_001_secondary",
                document_text=(
                    f"Secondary transcript host mirrored the same investor event for {request.company_name or request.ticker}."
                ),
            ),
            "investor_000_older": make_mock_fetched_document(
                raw_candidate_id="investor_000_older",
                document_text=(
                    f"Older investor transcript for {request.company_name or request.ticker} before the latest corporate update."
                ),
            ),
        }


RETRIEVAL_ADAPTER_REGISTRY: dict[DocumentType, Type[BaseRetrievalAdapter]] = {
    DocumentType.MATERIAL_EVENT: MockMaterialEventRetrievalAdapter,
    DocumentType.CLINICAL_TRIAL_UPDATE: MockClinicalTrialUpdateRetrievalAdapter,
    DocumentType.FDA_REVIEW: MockFDAReviewRetrievalAdapter,
    DocumentType.FINANCING_DILUTION: MockFinancingDilutionRetrievalAdapter,
    DocumentType.INVESTOR_COMMUNICATION: MockInvestorCommunicationRetrievalAdapter,
}


In [ ]:
# === SEC EDGAR Shared Infrastructure ===

class EdgarClient:
    """Shared HTTP client for SEC EDGAR EFTS search and filing retrieval."""

    EFTS_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
    FILING_ARCHIVE_BASE = "https://www.sec.gov/Archives/edgar/data"
    RATE_LIMIT_DELAY = 0.12  # Stay under 10 req/s

    def __init__(self, user_agent: str | None = None):
        self.user_agent = user_agent or SEC_EDGAR_USER_AGENT
        if not self.user_agent:
            raise ValueError("SEC_EDGAR_USER_AGENT must be set for live EDGAR access.")
        self._session = requests.Session()
        self._session.headers.update({
            "User-Agent": self.user_agent,
            "Accept": "application/json",
        })
        self._last_request_time: float = 0.0

    def _rate_limit(self):
        elapsed = time.time() - self._last_request_time
        if elapsed < self.RATE_LIMIT_DELAY:
            time.sleep(self.RATE_LIMIT_DELAY - elapsed)
        self._last_request_time = time.time()

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=30),
        retry=retry_if_exception_type((requests.exceptions.ConnectionError, requests.exceptions.Timeout)),
    )
    def search_efts(
        self,
        *,
        ticker: str | None = None,
        company_name: str | None = None,
        form_types: list[str],
        start_date: date | None = None,
        end_date: date | None = None,
        query_text: str | None = None,
        max_results: int = 10,
    ) -> list[dict[str, Any]]:
        """Search EDGAR EFTS for filings matching the given criteria."""
        self._rate_limit()
        params: dict[str, Any] = {
            "forms": ",".join(form_types),
            "from": 0,
            "size": max_results,
        }
        # Build query parts
        query_parts: list[str] = []
        if query_text:
            query_parts.append(query_text)

        if query_parts:
            params["q"] = " ".join(query_parts)

        if ticker:
            params["tickers"] = ticker.upper()

        if start_date:
            params["startdt"] = start_date.isoformat()
        if end_date:
            params["enddt"] = end_date.isoformat()

        resp = self._session.get(self.EFTS_SEARCH_URL, params=params, timeout=30)
        if resp.status_code == 429:
            logger.warning("EDGAR rate limited (429). Backing off...")
            time.sleep(5)
            resp = self._session.get(self.EFTS_SEARCH_URL, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        return data.get("hits", {}).get("hits", [])

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=30),
        retry=retry_if_exception_type((requests.exceptions.ConnectionError, requests.exceptions.Timeout)),
    )
    def search_submissions(
        self,
        cik: str,
        form_types: list[str],
        *,
        start_date: date | None = None,
        end_date: date | None = None,
        query_text: str | None = None,
        max_results: int = 10,
    ) -> list[dict[str, Any]]:
        """Search filings by CIK via the data.sec.gov submissions API.

        Returns hits in a format compatible with search_efts, so existing
        fetch_filing_index_and_primary_doc works unchanged.
        """
        self._rate_limit()
        cik_padded = cik.lstrip("0").zfill(10)
        url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
        resp = self._session.get(url, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        company_name = data.get("name", "")
        recent = data.get("filings", {}).get("recent", {})
        forms = recent.get("form", [])
        dates = recent.get("filingDate", [])
        accessions = recent.get("accessionNumber", [])
        primary_docs = recent.get("primaryDocument", [])
        form_types_set = set(ft.upper() for ft in form_types)
        query_lower = query_text.lower() if query_text else None
        hits = []
        for i, (form, dt, acc, pdoc) in enumerate(zip(forms, dates, accessions, primary_docs)):
            if form.upper() not in form_types_set:
                continue
            if start_date and dt < start_date.isoformat():
                continue
            if end_date and dt > end_date.isoformat():
                continue
            # Build EFTS-compatible hit dict
            hit = {
                "_id": acc,
                "_source": {
                    "accession_no": acc,
                    "file_date": dt,
                    "form_type": form,
                    "display_names": [company_name],
                    "entity_id": cik.lstrip("0"),
                    "ciks": [cik_padded],
                    "primary_document": pdoc,
                },
            }
            hits.append(hit)
            if len(hits) >= max_results:
                break
        return hits

    def fetch_filing_text(self, filing_url: str, max_chars: int = 80_000) -> str:
        """Fetch filing HTML and return stripped plain text."""
        self._rate_limit()
        resp = self._session.get(filing_url, timeout=30)
        if resp.status_code == 429:
            time.sleep(5)
            resp = self._session.get(filing_url, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "lxml")
        # Remove script and style elements
        for tag in soup(["script", "style", "meta", "link"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        return text[:max_chars]

    def get_filing_primary_doc_url(self, filing_entry: dict[str, Any]) -> str | None:
        """Extract the primary document URL from an EFTS search hit."""
        source = filing_entry.get("_source", {})
        file_num = source.get("file_num", "")
        accession_raw = source.get("accession_no", "") or filing_entry.get("_id", "")
        primary_doc = source.get("file_description", "")

        # Build URL from accession number
        accession_clean = accession_raw.replace("-", "")
        if not accession_clean:
            return None

        # Try to find the primary document from the filing index
        entity_id = source.get("entity_id", "")
        if entity_id and accession_raw:
            return f"https://www.sec.gov/Archives/edgar/data/{entity_id}/{accession_clean}/{accession_raw}.txt"
        return None

    def get_filing_index_url(self, filing_entry: dict[str, Any]) -> str | None:
        """Build the filing index page URL for an EFTS hit."""
        source = filing_entry.get("_source", {})
        accession_raw = source.get("accession_no", "") or filing_entry.get("_id", "")
        entity_id = source.get("entity_id", "")
        accession_clean = accession_raw.replace("-", "")
        if entity_id and accession_clean:
            return f"https://www.sec.gov/Archives/edgar/data/{entity_id}/{accession_clean}/"
        return None

    def fetch_filing_index_and_primary_doc(self, filing_entry: dict[str, Any], max_chars: int = 80_000) -> tuple[str | None, str]:
        """Fetch filing index page, find primary doc link, fetch and return its text."""
        source = filing_entry.get("_source", {})
        # Fast path: if primary_document is known (submissions API), fetch directly
        primary_doc = source.get("primary_document")
        entity_id = source.get("entity_id", "")
        accession_raw = source.get("accession_no", "") or filing_entry.get("_id", "")
        if primary_doc and entity_id and accession_raw:
            acc_nodash = accession_raw.replace("-", "")
            doc_url = f"https://www.sec.gov/Archives/edgar/data/{entity_id}/{acc_nodash}/{primary_doc}"
            try:
                return doc_url, self.fetch_filing_text(doc_url, max_chars)
            except Exception:
                pass  # Fall through to index-page approach

        index_url = self.get_filing_index_url(filing_entry)
        if not index_url:
            return None, ""

        self._rate_limit()
        try:
            resp = self._session.get(index_url, timeout=30)
            resp.raise_for_status()
        except Exception:
            # Fallback: try direct filing text URL
            doc_url = self.get_filing_primary_doc_url(filing_entry)
            if doc_url:
                return doc_url, self.fetch_filing_text(doc_url, max_chars)
            return None, ""

        soup = BeautifulSoup(resp.text, "lxml")
        # Find primary document link in the filing index table
        primary_url = None
        for row in soup.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 3:
                desc = cells[1].get_text(strip=True).lower() if len(cells) > 1 else ""
                link = cells[2].find("a") if len(cells) > 2 else None
                if link is None:
                    link = cells[0].find("a")
                if link and link.get("href"):
                    href = link["href"]
                    # Prefer the main filing document (htm/html)
                    if href.endswith((".htm", ".html", ".txt")):
                        if not primary_url or "primary" in desc or "complete" in desc:
                            if href.startswith("/"):
                                primary_url = f"https://www.sec.gov{href}"
                            elif href.startswith("http"):
                                primary_url = href
                            else:
                                primary_url = index_url + href

        if not primary_url:
            # Fallback: use the direct filing text
            doc_url = self.get_filing_primary_doc_url(filing_entry)
            if doc_url:
                return doc_url, self.fetch_filing_text(doc_url, max_chars)
            return None, ""

        return primary_url, self.fetch_filing_text(primary_url, max_chars)


def _parse_edgar_date(date_str: str | None) -> datetime | None:
    """Parse an EDGAR date string into a UTC datetime."""
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str[:10], "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except (ValueError, TypeError):
        return None


# === Live Retrieval Adapter 1: SEC Material Event (8-K) ===

class SECMaterialEventRetrievalAdapter(BaseRetrievalAdapter):
    adapter_name = "sec_material_event_adapter"
    document_type = DocumentType.MATERIAL_EVENT

    def __init__(self):
        self._client = EdgarClient()

    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        cik = request.filters.get("cik")
        if cik:
            hits = self._client.search_submissions(
                cik=cik, form_types=["8-K", "8-K/A"],
                start_date=request.start_date, end_date=request.end_date,
                max_results=request.max_candidates,
            )
        else:
            hits = self._client.search_efts(
                ticker=request.ticker, form_types=["8-K", "8-K/A"],
                start_date=request.start_date, end_date=request.end_date,
                max_results=request.max_candidates,
            )
        candidates = []
        for hit in hits:
            source = hit.get("_source", {})
            accession = source.get("accession_no", hit.get("_id", ""))
            filed_date = _parse_edgar_date(source.get("file_date") or source.get("period_of_report"))
            display_names = source.get("display_names", [])
            company_display = display_names[0] if display_names else (request.company_name or "")
            form_type = source.get("form_type", "8-K")
            candidates.append(RawRetrievalCandidateMetadata(
                raw_candidate_id=f"edgar_8k_{accession}",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=company_display,
                source_name="SEC EDGAR",
                source_identifier=accession,
                source_url=f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={request.ticker}&type=8-K&dateb=&owner=include&count=10",
                title=f"{request.ticker} {form_type} filing ({accession})",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=filed_date or now_utc(),
                updated_at=filed_date,
                event_date=filed_date.date() if filed_date else None,
                is_structured_source=True,
                raw_metadata={"form_type": form_type, "accession": accession, "_source": source},
            ))
        return candidates

    def fetch_document(self, raw_candidate: RawRetrievalCandidateMetadata, request: RetrievalRequest) -> FetchedDocumentContent:
        hit = {"_source": raw_candidate.raw_metadata.get("_source", {}), "_id": raw_candidate.source_identifier}
        doc_url, text = self._client.fetch_filing_index_and_primary_doc(hit)
        if not text.strip():
            return FetchedDocumentContent(
                raw_candidate_id=raw_candidate.raw_candidate_id,
                document_text=None,
                fetch_status=ProcessingStatus.EXTRACTION_FAILED,
                fetch_notes=["Could not extract text from EDGAR filing."],
                raw_payload={"doc_url": doc_url},
            )
        return FetchedDocumentContent(
            raw_candidate_id=raw_candidate.raw_candidate_id,
            document_text=text,
            content_type="text/plain",
            fetch_status=ProcessingStatus.SUCCESS,
            fetch_notes=[f"Fetched from {doc_url}"],
            raw_payload={"doc_url": doc_url},
        )


# === Live Retrieval Adapter 2: ClinicalTrials.gov ===

class ClinicalTrialsGovRetrievalAdapter(BaseRetrievalAdapter):
    adapter_name = "clinicaltrials_gov_adapter"
    document_type = DocumentType.CLINICAL_TRIAL_UPDATE

    BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=20),
        retry=retry_if_exception_type((requests.exceptions.ConnectionError, requests.exceptions.Timeout)),
    )
    def _search(self, company_name: str, max_results: int) -> list[dict]:
        params = {
            "query.spons": company_name,
            "sort": "LastUpdatePostDate",
            "pageSize": min(max_results, 20),
            "format": "json",
        }
        resp = requests.get(self.BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        return resp.json().get("studies", [])

    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        company = request.company_name or request.ticker
        studies = self._search(company, request.max_candidates)
        candidates = []
        for study in studies:
            proto = study.get("protocolSection", {})
            ident = proto.get("identificationModule", {})
            status_mod = proto.get("statusModule", {})
            nct_id = ident.get("nctId", "")
            title = ident.get("officialTitle") or ident.get("briefTitle") or ""
            org_name = ident.get("organization", {}).get("fullName", "")
            overall_status = status_mod.get("overallStatus", "")
            last_update = status_mod.get("lastUpdateSubmitDate", "")
            start_date_raw = status_mod.get("startDateStruct", {}).get("date", "")

            published = _parse_ct_date(last_update) or now_utc()

            candidates.append(RawRetrievalCandidateMetadata(
                raw_candidate_id=f"ct_{nct_id}",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=org_name or company,
                source_name="ClinicalTrials.gov",
                source_identifier=nct_id,
                source_url=f"https://clinicaltrials.gov/study/{nct_id}",
                title=title,
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=published,
                updated_at=published,
                event_date=published.date() if published else None,
                is_structured_source=True,
                raw_metadata={
                    "nct_id": nct_id,
                    "overall_status": overall_status,
                    "org_name": org_name,
                    "protocol_section": proto,
                },
            ))
        return candidates

    def fetch_document(self, raw_candidate: RawRetrievalCandidateMetadata, request: RetrievalRequest) -> FetchedDocumentContent:
        proto = raw_candidate.raw_metadata.get("protocol_section", {})
        text_parts = []

        # Identification
        ident = proto.get("identificationModule", {})
        text_parts.append(f"Study Title: {ident.get('officialTitle', ident.get('briefTitle', 'N/A'))}")
        text_parts.append(f"NCT ID: {ident.get('nctId', 'N/A')}")
        text_parts.append(f"Organization: {ident.get('organization', {}).get('fullName', 'N/A')}")

        # Status
        status_mod = proto.get("statusModule", {})
        text_parts.append(f"\nStudy Status: {status_mod.get('overallStatus', 'N/A')}")
        start_date = status_mod.get("startDateStruct", {}).get("date", "N/A")
        completion_date = status_mod.get("completionDateStruct", {}).get("date", "N/A")
        text_parts.append(f"Start Date: {start_date}")
        text_parts.append(f"Estimated Completion: {completion_date}")

        # Description
        desc = proto.get("descriptionModule", {})
        if desc.get("briefSummary"):
            text_parts.append(f"\nBrief Summary:\n{desc['briefSummary']}")
        if desc.get("detailedDescription"):
            text_parts.append(f"\nDetailed Description:\n{desc['detailedDescription']}")

        # Design
        design = proto.get("designModule", {})
        if design:
            text_parts.append(f"\nStudy Type: {design.get('studyType', 'N/A')}")
            phases = design.get("phases", [])
            if phases:
                text_parts.append(f"Phase: {', '.join(phases)}")
            enrollment = design.get("enrollmentInfo", {})
            if enrollment:
                text_parts.append(f"Enrollment: {enrollment.get('count', 'N/A')} ({enrollment.get('type', '')})")

        # Arms & Interventions
        arms_mod = proto.get("armsInterventionsModule", {})
        interventions = arms_mod.get("interventions", [])
        if interventions:
            text_parts.append("\nInterventions:")
            for intv in interventions[:5]:
                text_parts.append(f"  - {intv.get('type', '')}: {intv.get('name', '')} — {intv.get('description', '')}")

        # Outcomes
        outcomes_mod = proto.get("outcomesModule", {})
        primary = outcomes_mod.get("primaryOutcomes", [])
        if primary:
            text_parts.append("\nPrimary Outcomes:")
            for out in primary[:5]:
                text_parts.append(f"  - {out.get('measure', '')} (timeframe: {out.get('timeFrame', 'N/A')})")

        # Eligibility
        elig = proto.get("eligibilityModule", {})
        if elig:
            text_parts.append(f"\nEligibility: Ages {elig.get('minimumAge', 'N/A')} to {elig.get('maximumAge', 'N/A')}, Sex: {elig.get('sex', 'N/A')}")

        document_text = "\n".join(text_parts)
        return FetchedDocumentContent(
            raw_candidate_id=raw_candidate.raw_candidate_id,
            document_text=document_text,
            content_type="text/plain",
            fetch_status=ProcessingStatus.SUCCESS,
            fetch_notes=["Synthesized from ClinicalTrials.gov protocol sections."],
            raw_payload={"nct_id": raw_candidate.raw_metadata.get("nct_id")},
        )


def _parse_ct_date(date_str: str | None) -> datetime | None:
    """Parse ClinicalTrials.gov date formats (YYYY-MM-DD or Month YYYY)."""
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    except ValueError:
        pass
    try:
        return datetime.strptime(date_str, "%B %Y").replace(tzinfo=timezone.utc)
    except ValueError:
        pass
    try:
        return datetime.strptime(date_str, "%B %d, %Y").replace(tzinfo=timezone.utc)
    except ValueError:
        return None


# === Live Retrieval Adapter 3: openFDA Drug Review ===

class OpenFDAReviewRetrievalAdapter(BaseRetrievalAdapter):
    adapter_name = "openfda_review_adapter"
    document_type = DocumentType.FDA_REVIEW

    DRUGSFDA_URL = "https://api.fda.gov/drug/drugsfda.json"
    LABEL_URL = "https://api.fda.gov/drug/label.json"

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=20),
        retry=retry_if_exception_type((requests.exceptions.ConnectionError, requests.exceptions.Timeout)),
    )
    def _query_fda(self, url: str, search: str, limit: int) -> list[dict]:
        params: dict[str, Any] = {"search": search, "limit": limit}
        if OPENFDA_API_KEY:
            params["api_key"] = OPENFDA_API_KEY
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 404:
            return []
        resp.raise_for_status()
        return resp.json().get("results", [])

    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        company = request.company_name or request.ticker
        search_query = f'sponsor_name:"{company}"'
        results = self._query_fda(self.DRUGSFDA_URL, search_query, request.max_candidates)
        candidates = []
        for result in results:
            app_num = result.get("application_number", "")
            sponsor = result.get("sponsor_name", company)
            products = result.get("products", [])
            brand_names = [p.get("brand_name", "") for p in products if p.get("brand_name")]
            submissions = result.get("submissions", [])

            # Use most recent submission date
            latest_date = None
            for sub in submissions:
                sub_date = sub.get("submission_status_date")
                parsed = _parse_fda_date(sub_date)
                if parsed and (latest_date is None or parsed > latest_date):
                    latest_date = parsed

            published = latest_date or now_utc()
            title_brand = f" ({', '.join(brand_names[:3])})" if brand_names else ""
            candidates.append(RawRetrievalCandidateMetadata(
                raw_candidate_id=f"fda_{app_num}",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=sponsor,
                source_name="openFDA",
                source_identifier=app_num,
                source_url=f"https://api.fda.gov/drug/drugsfda.json?search=application_number:{app_num}",
                title=f"FDA Application {app_num}{title_brand}",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=published,
                updated_at=published,
                event_date=published.date() if published else None,
                is_structured_source=True,
                raw_metadata={
                    "application_number": app_num,
                    "sponsor_name": sponsor,
                    "products": products,
                    "submissions": submissions,
                },
            ))
        return candidates

    def fetch_document(self, raw_candidate: RawRetrievalCandidateMetadata, request: RetrievalRequest) -> FetchedDocumentContent:
        meta = raw_candidate.raw_metadata
        text_parts = []

        # Application overview
        text_parts.append(f"FDA Application Number: {meta.get('application_number', 'N/A')}")
        text_parts.append(f"Sponsor: {meta.get('sponsor_name', 'N/A')}")

        # Products
        products = meta.get("products", [])
        if products:
            text_parts.append("\nApproved Products:")
            for product in products:
                brand = product.get("brand_name", "N/A")
                active = product.get("active_ingredients", [])
                ingredients = ", ".join(
                    f"{ai.get('name', '')} ({ai.get('strength', '')})"
                    for ai in active
                ) if active else "N/A"
                dosage = product.get("dosage_form", "N/A")
                route = product.get("route", "N/A")
                text_parts.append(f"  - {brand}: {ingredients}, {dosage}, {route}")

        # Submissions history
        submissions = meta.get("submissions", [])
        if submissions:
            text_parts.append("\nSubmission History:")
            for sub in sorted(submissions, key=lambda s: s.get("submission_status_date", ""), reverse=True)[:10]:
                sub_type = sub.get("submission_type", "N/A")
                sub_num = sub.get("submission_number", "")
                sub_status = sub.get("submission_status", "N/A")
                sub_date = sub.get("submission_status_date", "N/A")
                text_parts.append(f"  - {sub_type} #{sub_num}: {sub_status} ({sub_date})")

        # Try to fetch drug label for additional context
        company = request.company_name or request.ticker
        try:
            label_results = self._query_fda(
                self.LABEL_URL,
                f'openfda.manufacturer_name:"{company}"',
                limit=1,
            )
            if label_results:
                label = label_results[0]
                if label.get("indications_and_usage"):
                    indications = label["indications_and_usage"]
                    if isinstance(indications, list):
                        indications = " ".join(indications)
                    text_parts.append(f"\nIndications and Usage:\n{indications[:3000]}")
                if label.get("warnings"):
                    warnings_text = label["warnings"]
                    if isinstance(warnings_text, list):
                        warnings_text = " ".join(warnings_text)
                    text_parts.append(f"\nWarnings:\n{warnings_text[:2000]}")
        except Exception:
            text_parts.append("\n[Drug label data not available]")

        document_text = "\n".join(text_parts)
        return FetchedDocumentContent(
            raw_candidate_id=raw_candidate.raw_candidate_id,
            document_text=document_text if len(document_text.strip()) > 40 else None,
            content_type="text/plain",
            fetch_status=ProcessingStatus.SUCCESS if len(document_text.strip()) > 40 else ProcessingStatus.EXTRACTION_FAILED,
            fetch_notes=["Assembled from openFDA drugsfda and label endpoints."],
            raw_payload={"application_number": meta.get("application_number")},
        )


def _parse_fda_date(date_str: str | None) -> datetime | None:
    """Parse openFDA date format (YYYYMMDD)."""
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str[:8], "%Y%m%d").replace(tzinfo=timezone.utc)
    except (ValueError, TypeError):
        return None


# === Live Retrieval Adapter 4: SEC Financing/Dilution ===

class SECFinancingDilutionRetrievalAdapter(BaseRetrievalAdapter):
    adapter_name = "sec_financing_dilution_adapter"
    document_type = DocumentType.FINANCING_DILUTION

    FORM_TYPES = ["S-3", "S-3/A", "424B5", "424B2", "S-1", "S-1/A"]

    def __init__(self):
        self._client = EdgarClient()

    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        cik = request.filters.get("cik")
        if cik:
            hits = self._client.search_submissions(
                cik=cik, form_types=self.FORM_TYPES,
                start_date=request.start_date, end_date=request.end_date,
                max_results=request.max_candidates,
            )
        else:
            hits = self._client.search_efts(
                ticker=request.ticker, form_types=self.FORM_TYPES,
                start_date=request.start_date, end_date=request.end_date,
                max_results=request.max_candidates,
            )
        candidates = []
        for hit in hits:
            source = hit.get("_source", {})
            accession = source.get("accession_no", hit.get("_id", ""))
            filed_date = _parse_edgar_date(source.get("file_date") or source.get("period_of_report"))
            display_names = source.get("display_names", [])
            company_display = display_names[0] if display_names else (request.company_name or "")
            form_type = source.get("form_type", "S-3")
            candidates.append(RawRetrievalCandidateMetadata(
                raw_candidate_id=f"edgar_fin_{accession}",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=company_display,
                source_name="SEC EDGAR",
                source_identifier=accession,
                source_url=self._client.get_filing_index_url(hit),
                title=f"{request.ticker} {form_type} filing ({accession})",
                source_family=SourceFamily.OFFICIAL_REGULATORY,
                published_at=filed_date or now_utc(),
                updated_at=filed_date,
                event_date=filed_date.date() if filed_date else None,
                is_structured_source=True,
                raw_metadata={"form_type": form_type, "accession": accession, "_source": source},
            ))
        return candidates

    def fetch_document(self, raw_candidate: RawRetrievalCandidateMetadata, request: RetrievalRequest) -> FetchedDocumentContent:
        hit = {"_source": raw_candidate.raw_metadata.get("_source", {}), "_id": raw_candidate.source_identifier}
        doc_url, text = self._client.fetch_filing_index_and_primary_doc(hit)
        if not text.strip():
            return FetchedDocumentContent(
                raw_candidate_id=raw_candidate.raw_candidate_id,
                document_text=None,
                fetch_status=ProcessingStatus.EXTRACTION_FAILED,
                fetch_notes=["Could not extract text from EDGAR filing."],
                raw_payload={"doc_url": doc_url},
            )
        return FetchedDocumentContent(
            raw_candidate_id=raw_candidate.raw_candidate_id,
            document_text=text,
            content_type="text/plain",
            fetch_status=ProcessingStatus.SUCCESS,
            fetch_notes=[f"Fetched from {doc_url}"],
            raw_payload={"doc_url": doc_url},
        )


# === Live Retrieval Adapter 5: SEC Investor Communication ===

class SECInvestorCommunicationRetrievalAdapter(BaseRetrievalAdapter):
    adapter_name = "sec_investor_communication_adapter"
    document_type = DocumentType.INVESTOR_COMMUNICATION

    KEYWORDS = ["earnings", "investor", "presentation", "quarterly", "results of operations"]

    def __init__(self):
        self._client = EdgarClient()

    def search_candidates(self, request: RetrievalRequest) -> list[RawRetrievalCandidateMetadata]:
        cik = request.filters.get("cik")
        if cik:
            hits = self._client.search_submissions(
                cik=cik, form_types=["8-K"],
                start_date=request.start_date, end_date=request.end_date,
                max_results=request.max_candidates * 2,
            )
        else:
            hits = self._client.search_efts(
                ticker=request.ticker, form_types=["8-K"],
                start_date=request.start_date, end_date=request.end_date,
                query_text=" OR ".join(f'"{kw}"' for kw in self.KEYWORDS),
                max_results=request.max_candidates * 2,
            )
        candidates = []
        for hit in hits:
            source = hit.get("_source", {})
            accession = source.get("accession_no", hit.get("_id", ""))
            filed_date = _parse_edgar_date(source.get("file_date") or source.get("period_of_report"))
            display_names = source.get("display_names", [])
            company_display = display_names[0] if display_names else (request.company_name or "")
            form_type = source.get("form_type", "8-K")
            candidates.append(RawRetrievalCandidateMetadata(
                raw_candidate_id=f"edgar_inv_{accession}",
                adapter_name=self.adapter_name,
                document_type=self.document_type,
                ticker=request.ticker,
                company_name=company_display,
                source_name="SEC EDGAR",
                source_identifier=accession,
                source_url=self._client.get_filing_index_url(hit),
                title=f"{request.ticker} {form_type} investor communication ({accession})",
                source_family=SourceFamily.ISSUER_PUBLISHED,
                published_at=filed_date or now_utc(),
                updated_at=filed_date,
                event_date=filed_date.date() if filed_date else None,
                is_structured_source=False,
                raw_metadata={"form_type": form_type, "accession": accession, "_source": source},
            ))
        return candidates[:request.max_candidates]

    def fetch_document(self, raw_candidate: RawRetrievalCandidateMetadata, request: RetrievalRequest) -> FetchedDocumentContent:
        hit = {"_source": raw_candidate.raw_metadata.get("_source", {}), "_id": raw_candidate.source_identifier}
        doc_url, text = self._client.fetch_filing_index_and_primary_doc(hit)
        if not text.strip():
            return FetchedDocumentContent(
                raw_candidate_id=raw_candidate.raw_candidate_id,
                document_text=None,
                fetch_status=ProcessingStatus.EXTRACTION_FAILED,
                fetch_notes=["Could not extract text from EDGAR filing."],
                raw_payload={"doc_url": doc_url},
            )
        return FetchedDocumentContent(
            raw_candidate_id=raw_candidate.raw_candidate_id,
            document_text=text,
            content_type="text/plain",
            fetch_status=ProcessingStatus.SUCCESS,
            fetch_notes=[f"Fetched from {doc_url}"],
            raw_payload={"doc_url": doc_url},
        )


# === Live Retrieval Adapter Registry ===

if not PIPELINE_CONFIG.enable_mock_retrieval:
    RETRIEVAL_ADAPTER_REGISTRY = {
        DocumentType.MATERIAL_EVENT: SECMaterialEventRetrievalAdapter,
        DocumentType.CLINICAL_TRIAL_UPDATE: ClinicalTrialsGovRetrievalAdapter,
        DocumentType.FDA_REVIEW: OpenFDAReviewRetrievalAdapter,
        DocumentType.FINANCING_DILUTION: SECFinancingDilutionRetrievalAdapter,
        DocumentType.INVESTOR_COMMUNICATION: SECInvestorCommunicationRetrievalAdapter,
    }
    logger.info("Live retrieval adapters registered (EDGAR, ClinicalTrials.gov, openFDA).")
else:
    logger.info("Mock retrieval adapters active (enable_mock_retrieval=True).")

## 16. Pipeline State Model

In [ ]:

class EvidenceSnippet(ContractModel):
    """Structured evidence reference that downstream nodes can cite."""

    evidence_id: str | None = None
    document_id: str
    source_url: str | None = None
    source_chunk_id: str | None = None
    source_section_id: str | None = None
    section_title: str | None = None
    evidence_type: "EvidenceType | None" = None
    supported_dimensions: list["AnalysisDimension"] = Field(default_factory=list)
    interpretation: "EvidenceInterpretation | None" = None
    snippet_text: str | None = None
    rationale: str
    start_char: int | None = None
    end_char: int | None = None
    metadata: dict[str, Any] = Field(default_factory=dict)


class WorkerInput(ContractModel):
    """Input contract for a disclosure-specific worker node."""

    run_id: str
    ticker: str
    document_type: DocumentType
    retrieval_result: RetrievalResult
    config: PipelineConfig
    graph_context: dict[str, Any] = Field(default_factory=dict)


class WorkerOutput(ContractModel):
    """Output contract returned by each disclosure-specific worker."""

    worker_name: str
    document_type: DocumentType
    status: ProcessingStatus = ProcessingStatus.PENDING
    summary: str | None = None
    sentiment: SentimentAssessment | None = None
    tone: ToneAssessment | None = None
    key_points: list[str] = Field(default_factory=list)
    evidence: list[EvidenceSnippet] = Field(default_factory=list)
    caveats: list[str] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    confidence: float | None = None

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence")


class ArbiterInput(ContractModel):
    """Input contract for an arbiter node that reviews worker outputs."""

    run_id: str
    ticker: str
    worker_outputs: list[WorkerOutput]
    retrieval_results: list[RetrievalResult] = Field(default_factory=list)
    config: PipelineConfig


class ArbiterOutput(ContractModel):
    """Cross-document synthesis output from a summary or sentiment arbiter."""

    arbiter_name: str
    arbiter_kind: ArbiterKind
    status: ProcessingStatus = ProcessingStatus.PENDING
    summary: str | None = None
    sentiment: SentimentAssessment | None = None
    tone: ToneAssessment | None = None
    consensus_points: list[str] = Field(default_factory=list)
    conflicts: list[str] = Field(default_factory=list)
    evidence: list[EvidenceSnippet] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    confidence: float | None = None

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence")


class MasterInput(ContractModel):
    """Input contract for the single master node."""

    run_id: str
    ticker: str
    worker_outputs: list[WorkerOutput]
    arbiter_outputs: list[ArbiterOutput]
    retrieval_results: list[RetrievalResult]
    config: PipelineConfig


class MasterOutput(ContractModel):
    """Structured master-node output before UI flattening."""

    ticker: str
    status: ProcessingStatus = ProcessingStatus.PENDING
    master_summary: str | None = None
    master_sentiment: SentimentAssessment | None = None
    master_tone: ToneAssessment | None = None
    worker_outputs: list[WorkerOutput] = Field(default_factory=list)
    arbiter_outputs: list[ArbiterOutput] = Field(default_factory=list)
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    ready_for_ui: bool = False
    generated_at: datetime = Field(default_factory=now_utc)


class PipelineState(ContractModel):
    """Shared state container for future orchestration."""

    run_id: str
    ticker: str
    config: PipelineConfig
    retrieval_results: dict[DocumentType, RetrievalResult] = Field(default_factory=dict)
    worker_outputs: dict[DocumentType, WorkerOutput] = Field(default_factory=dict)
    arbiter_outputs: list[ArbiterOutput] = Field(default_factory=list)
    master_output: MasterOutput | None = None
    graph_context: dict[str, Any] = Field(default_factory=dict)
    shared_cache_refs: dict[str, str] = Field(default_factory=dict)
    errors: list[PipelineError] = Field(default_factory=list)
    created_at: datetime = Field(default_factory=now_utc)
    updated_at: datetime = Field(default_factory=now_utc)


## 17. Worker Interface Definitions

In [ ]:
class BaseWorker(ABC):
    """Base interface for one disclosure-specific worker node."""

    worker_name: ClassVar[str] = "base_worker"
    document_type: ClassVar[DocumentType]
    input_model: ClassVar[type[WorkerInput]] = WorkerInput
    output_model: ClassVar[type[WorkerOutput]] = WorkerOutput

    def analyze(self, worker_input: WorkerInput) -> WorkerOutput:
        """Analyze a single disclosure artifact and return a typed worker output."""
        raise NotImplementedError("Worker analysis is intentionally stubbed in this notebook step.")


class MaterialEventWorker(BaseWorker):
    worker_name = "material_event_worker"
    document_type = DocumentType.MATERIAL_EVENT


class ClinicalTrialUpdateWorker(BaseWorker):
    worker_name = "clinical_trial_update_worker"
    document_type = DocumentType.CLINICAL_TRIAL_UPDATE


class FDAReviewWorker(BaseWorker):
    worker_name = "fda_review_worker"
    document_type = DocumentType.FDA_REVIEW


class FinancingDilutionWorker(BaseWorker):
    worker_name = "financing_dilution_worker"
    document_type = DocumentType.FINANCING_DILUTION


class InvestorCommunicationWorker(BaseWorker):
    worker_name = "investor_communication_worker"
    document_type = DocumentType.INVESTOR_COMMUNICATION


WORKER_REGISTRY: dict[DocumentType, Type[BaseWorker]] = {
    DocumentType.MATERIAL_EVENT: MaterialEventWorker,
    DocumentType.CLINICAL_TRIAL_UPDATE: ClinicalTrialUpdateWorker,
    DocumentType.FDA_REVIEW: FDAReviewWorker,
    DocumentType.FINANCING_DILUTION: FinancingDilutionWorker,
    DocumentType.INVESTOR_COMMUNICATION: InvestorCommunicationWorker,
}


## 18. Arbiter Interface Definitions

In [ ]:
class BaseArbiter(ABC):
    """Base interface for arbiters that reconcile worker outputs."""

    arbiter_name: ClassVar[str] = "base_arbiter"
    arbiter_kind: ClassVar[ArbiterKind]
    input_model: ClassVar[type[ArbiterInput]] = ArbiterInput
    output_model: ClassVar[type[ArbiterOutput]] = ArbiterOutput

    def arbitrate(self, arbiter_input: ArbiterInput) -> ArbiterOutput:
        """Review worker outputs and return one typed arbiter result."""
        raise NotImplementedError("Arbiter logic is intentionally stubbed in this notebook step.")


class SummaryArbiter(BaseArbiter):
    arbiter_name = "summary_arbiter"
    arbiter_kind = ArbiterKind.SUMMARY


class SentimentArbiter(BaseArbiter):
    arbiter_name = "sentiment_arbiter"
    arbiter_kind = ArbiterKind.SENTIMENT


ARBITER_REGISTRY: dict[ArbiterKind, Type[BaseArbiter]] = {
    ArbiterKind.SUMMARY: SummaryArbiter,
    ArbiterKind.SENTIMENT: SentimentArbiter,
}


## 19. Master Node Interface Definition

In [ ]:
class MasterNode(ABC):
    """Single master node that packages arbiter outputs into the UI-facing payload.

    Responsibilities in later steps:
    - validate disclosure coverage across retrieval results
    - merge arbiter outputs into a final cross-document view
    - preserve caveats and provenance links
    - emit the only FinalUIPayload returned to downstream consumers
    """

    input_model: ClassVar[type[MasterInput]] = MasterInput
    output_model: ClassVar[type[FinalUIPayload]] = FinalUIPayload

    def build_payload(self, master_input: MasterInput) -> FinalUIPayload:
        """Assemble the final UI payload from worker and arbiter contracts."""
        raise NotImplementedError("Master node packaging is intentionally stubbed in this notebook step.")


## 20. Stub Utility Functions

In [ ]:
def build_document_chunks(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    """Split normalized text into chunks for future embedding and retrieval steps."""
    # TODO: Implement configurable chunking with overlap and traceable offsets.
    raise NotImplementedError("Chunk building is intentionally stubbed in this notebook step.")


def score_sentiment(text: str) -> SentimentAssessment:
    """Score sentiment from document text using a future model-backed analyzer."""
    # TODO: Replace with model-based sentiment scoring that returns evidence-backed output.
    raise NotImplementedError("Sentiment scoring is intentionally stubbed in this notebook step.")


def score_fogging(text: str) -> float | None:
    """Estimate fogging or obfuscation tone using a future model-backed analyzer."""
    # TODO: Replace with model-based fogging analysis and connect it to full tone scoring.
    raise NotImplementedError("Fogging scoring is intentionally stubbed in this notebook step.")


def merge_worker_outputs(worker_outputs: Sequence[WorkerOutput]) -> dict[DocumentType, WorkerOutput]:
    """Build a keyed worker-output mapping without performing any arbitration."""
    # TODO: Add duplicate detection and conflict handling when orchestration is introduced.
    return {output.document_type: output for output in worker_outputs}


def build_ui_payload(master_output: MasterOutput) -> FinalUIPayload:
    """Convert a master output object into the final UI payload shape."""
    # TODO: Flatten master-node results into disclosure cards and overall UI fields.
    raise NotImplementedError("UI payload construction is intentionally stubbed in this notebook step.")


## 21. Shared Processing Design Notes

This notebook stage begins after retrieval has already selected one disclosure candidate for a given disclosure type.

Processing assumptions for this step:

- Text cleanup is deterministic and intentionally conservative.
- Structural cues such as headers, lists, transcript speakers, and table-like lines are preserved where possible.
- Chunking is document-aware and prefers section boundaries before falling back to simple fixed windows.
- Embeddings are optional and safe to stub; the notebook remains runnable even if Ollama is unavailable.
- Graph-aware retrieval remains lightweight and local to a single processed document.
- No worker-specific prompting or analysis is introduced in this step.


In [ ]:
import hashlib
import math
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

GLOBAL_CONFIG.setdefault(
    "processing_defaults",
    {
        "max_header_chars": 120,
        "max_header_words": 14,
        "strip_page_markers": True,
        "normalize_tables": True,
        "preserve_headers": True,
        "collapse_blank_lines": True,
    },
)
GLOBAL_CONFIG.setdefault(
    "chunking_defaults",
    {
        "target_chunk_chars": 700,
        "max_chunk_chars": 900,
        "overlap_chars": 120,
        "min_chunk_chars": 160,
        "context_window_chars": 90,
        "include_section_titles": True,
        "fallback_chunk_chars": 650,
    },
)
GLOBAL_CONFIG.setdefault(
    "embedding_defaults",
    {
        "provider": "voyage",
        "enabled": True,
        "prefer_live_embeddings": True,
        "use_mock_fallback": False,
        "model_name": "voyage-3-large",
        "base_url": "https://api.voyageai.com",
        "api_path": "/v1/embeddings",
        "batch_size": 8,
        "request_timeout_seconds": 15.0,
        "mock_vector_dimensions": 1024,
        "normalize_vectors": True,
    },
)
GLOBAL_CONFIG.setdefault(
    "graph_retrieval_defaults",
    {
        "top_k": 3,
        "candidate_pool_size": 6,
        "neighbor_hops": 1,
        "context_expansion_hops": 1,
        "min_similarity_threshold": 0.0,
        "max_graph_bonus": 0.18,
        "section_title_weight": 0.08,
        "neighbor_similarity_weight": 0.06,
        "cross_reference_bonus": 0.03,
    },
)

## 22. Document Processing Models

In [ ]:
class ProcessingNoteSeverity(str, Enum):
    """Severity levels for deterministic processing warnings and notes."""

    INFO = "info"
    WARNING = "warning"
    ERROR = "error"


class SectionKind(str, Enum):
    """High-level structural section types detected from cleaned text."""

    NARRATIVE = "narrative"
    LIST = "list"
    TABLE = "table"
    TRANSCRIPT = "transcript"
    MIXED = "mixed"
    UNKNOWN = "unknown"


class EmbeddingProvider(str, Enum):
    """Supported embedding providers for the notebook prototype."""

    OLLAMA = "ollama"
    VOYAGE = "voyage"
    MOCK = "mock"


class EmbeddingStatus(str, Enum):
    """Embedding generation status values."""

    SUCCESS = "success"
    MOCK = "mock"
    SKIPPED = "skipped"
    FAILED = "failed"


class GraphNodeType(str, Enum):
    """Node types used by the lightweight single-document graph."""

    DOCUMENT = "document"
    SECTION = "section"
    CHUNK = "chunk"
    ANCHOR = "anchor"


class GraphEdgeType(str, Enum):
    """Edge types used by the lightweight single-document graph."""

    CONTAINS = "contains"
    ADJACENT = "adjacent"
    SAME_SECTION = "same_section"
    SECTION_SEQUENCE = "section_sequence"
    CROSS_REFERENCE = "cross_reference"


class ProcessingConfig(ContractModel):
    """Configuration for deterministic text cleanup and section parsing."""

    max_header_chars: int = 120
    max_header_words: int = 14
    strip_page_markers: bool = True
    normalize_tables: bool = True
    preserve_headers: bool = True
    collapse_blank_lines: bool = True

## 23. Text Normalization Helpers

In [ ]:
HEADER_PREFIX_PATTERN = re.compile(r"^(item|section|part|article)\s+[A-Za-z0-9IVXivx.\-]+", re.IGNORECASE)
NUMBERED_HEADER_PATTERN = re.compile(r"^(?:\d+(?:\.\d+){0,3}|[A-Z]\.|[IVXLC]+\.)\s+\S")
SPEAKER_LABEL_PATTERN = re.compile(r"^[A-Z][A-Za-z .&/\-]{1,40}:\s")
BULLET_START_PATTERN = re.compile(r"^\s*(?:[-*]|\d+[.)]|[•◦▪●])\s+")
TABLE_SPLIT_PATTERN = re.compile(r"(?:\t+|\s{2,})")
PAGE_MARKER_PATTERN = re.compile(r"^(page|slide)\s+\d+(?:\s+of\s+\d+)?$", re.IGNORECASE)
SEPARATOR_LINE_PATTERN = re.compile(r"^[\-=*_]{3,}$")


def approximate_word_count(text: str) -> int:
    """Approximate word count using a simple alphanumeric token regex."""
    return len(re.findall(r"\b\w+\b", text))


def normalize_line_breaks(text: str) -> str:
    """Standardize line breaks without removing paragraph boundaries."""
    return text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")


def normalize_whitespace(text: str) -> str:
    """Trim trailing spaces and collapse excessive blank lines conservatively."""
    cleaned_lines: list[str] = []
    for line in text.split("\n"):
        stripped = line.rstrip()
        if " | " in stripped:
            cleaned_lines.append(stripped)
        else:
            cleaned_lines.append(re.sub(r" {2,}", " ", stripped))
    cleaned_text = "\n".join(cleaned_lines)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)
    return cleaned_text.strip()


def normalize_bullets_and_lists(text: str) -> str:
    """Standardize common bullet characters while preserving list intent."""
    normalized_lines: list[str] = []
    for line in text.split("\n"):
        updated = re.sub(r"^\s*[•◦▪●]\s*", "- ", line)
        updated = re.sub(r"^\s*(\d+)\)\s+", r"\1. ", updated)
        normalized_lines.append(updated)
    return "\n".join(normalized_lines)


def normalize_table_like_text(text: str) -> str:
    """Preserve simple table-like rows by converting wide spacing into pipe separators."""
    normalized_lines: list[str] = []
    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            normalized_lines.append("")
            continue
        if " | " in stripped or BULLET_START_PATTERN.match(stripped):
            normalized_lines.append(stripped)
            continue
        candidate_parts = [part.strip() for part in TABLE_SPLIT_PATTERN.split(stripped) if part.strip()]
        if len(candidate_parts) >= 3 and len(stripped) <= 180 and not stripped.endswith("."):
            normalized_lines.append(" | ".join(candidate_parts))
        else:
            normalized_lines.append(stripped)
    return "\n".join(normalized_lines)


def strip_noise_blocks(text: str, document_id: str | None = None) -> tuple[str, list[ProcessingNote]]:
    """Remove generic layout noise such as page markers and repeated separators."""
    kept_lines: list[str] = []
    notes: list[ProcessingNote] = []
    removed_count = 0
    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            kept_lines.append("")
            continue
        if PAGE_MARKER_PATTERN.match(stripped) or SEPARATOR_LINE_PATTERN.match(stripped):
            removed_count += 1
            continue
        if stripped.isdigit() and len(stripped) <= 3:
            removed_count += 1
            continue
        kept_lines.append(line)
    if removed_count:
        notes.append(
            make_processing_note(
                "noise_lines_removed",
                f"Removed {removed_count} generic layout/noise line(s).",
                severity=ProcessingNoteSeverity.INFO,
                document_id=document_id,
                metadata={"removed_count": removed_count},
            )
        )
    return "\n".join(kept_lines), notes


def is_probable_header_line(line: str, *, previous_blank: bool = True, next_blank: bool = True) -> bool:
    """Detect likely section headers using simple, explicit heuristics."""
    stripped = line.strip()
    if not stripped:
        return False
    if len(stripped) > PROCESSING_CONFIG.max_header_chars:
        return False
    word_count = approximate_word_count(stripped)
    if word_count == 0 or word_count > PROCESSING_CONFIG.max_header_words:
        return False
    if HEADER_PREFIX_PATTERN.match(stripped) or NUMBERED_HEADER_PATTERN.match(stripped):
        return True
    if stripped.endswith(":") and word_count <= PROCESSING_CONFIG.max_header_words:
        return True
    uppercase_ratio = sum(1 for char in stripped if char.isupper()) / max(1, sum(1 for char in stripped if char.isalpha()))
    if uppercase_ratio >= 0.65 and previous_blank:
        return True
    if previous_blank and next_blank and word_count <= 8 and not stripped.endswith("."):
        return True
    return False


def preserve_meaningful_headers(text: str) -> str:
    """Ensure probable headers stay visually separated from surrounding paragraphs."""
    lines = text.split("\n")
    output_lines: list[str] = []
    for index, line in enumerate(lines):
        previous_blank = index == 0 or not lines[index - 1].strip()
        next_blank = index == len(lines) - 1 or not lines[index + 1].strip()
        if is_probable_header_line(line, previous_blank=previous_blank, next_blank=next_blank):
            if output_lines and output_lines[-1].strip():
                output_lines.append("")
            output_lines.append(line.strip())
            if not next_blank:
                output_lines.append("")
        else:
            output_lines.append(line)
    return "\n".join(output_lines)


def safe_text_cleanup(text: str, document_id: str | None = None) -> tuple[str, list[ProcessingNote]]:
    """Run conservative, inspectable text cleanup without destroying structure."""
    notes: list[ProcessingNote] = []
    if not text or not text.strip():
        return "", [
            make_processing_note(
                "empty_text",
                "Document text is empty after initial inspection.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=document_id,
            )
        ]

    working_text = normalize_line_breaks(text)
    working_text = normalize_bullets_and_lists(working_text)
    working_text, stripped_notes = strip_noise_blocks(working_text, document_id=document_id)
    notes.extend(stripped_notes)
    if PROCESSING_CONFIG.normalize_tables:
        working_text = normalize_table_like_text(working_text)
    if PROCESSING_CONFIG.preserve_headers:
        working_text = preserve_meaningful_headers(working_text)
    working_text = normalize_whitespace(working_text)

    notes.append(
        make_processing_note(
            "text_cleanup_complete",
            "Applied deterministic text cleanup and structural preservation steps.",
            severity=ProcessingNoteSeverity.INFO,
            document_id=document_id,
            metadata={
                "raw_char_count": len(text),
                "cleaned_char_count": len(working_text),
            },
        )
    )
    return working_text, notes


def normalize_document_text(raw_text: str | None) -> str | None:
    """Notebook-wide text normalization wrapper used by retrieval and processing."""
    if raw_text is None:
        return None
    cleaned_text, _ = safe_text_cleanup(raw_text)
    return cleaned_text or None


## 24. Section Detection and Structural Parsing

In [ ]:
def infer_section_level(title: str) -> int:
    """Infer a lightweight header depth from numbering patterns when present."""
    stripped = title.strip()
    numbered_match = re.match(r"^(\d+(?:\.\d+)*)", stripped)
    if numbered_match:
        return min(4, numbered_match.group(1).count(".") + 1)
    if HEADER_PREFIX_PATTERN.match(stripped):
        return 1
    return 1


def extract_reference_label(title: str) -> str | None:
    """Extract a stable reference label from a section title when possible."""
    stripped = title.strip()
    for pattern in [r"^(Item\s+[A-Za-z0-9.]+)", r"^(Section\s+[A-Za-z0-9.]+)", r"^(\d+(?:\.\d+)*)"]:
        match = re.match(pattern, stripped, flags=re.IGNORECASE)
        if match:
            return match.group(1)
    return None


def classify_section_kind(section_text: str) -> SectionKind:
    """Classify a section using explicit, shallow structural heuristics."""
    nonblank_lines = [line.strip() for line in section_text.splitlines() if line.strip()]
    if not nonblank_lines:
        return SectionKind.UNKNOWN

    bullet_lines = sum(1 for line in nonblank_lines if BULLET_START_PATTERN.match(line))
    table_lines = sum(1 for line in nonblank_lines if " | " in line)
    speaker_lines = sum(1 for line in nonblank_lines if SPEAKER_LABEL_PATTERN.match(line))
    line_count = len(nonblank_lines)

    ratios = {
        SectionKind.LIST: bullet_lines / line_count,
        SectionKind.TABLE: table_lines / line_count,
        SectionKind.TRANSCRIPT: speaker_lines / line_count,
    }
    dominant_kind = max(ratios, key=ratios.get)
    dominant_ratio = ratios[dominant_kind]

    if dominant_ratio >= 0.45:
        return dominant_kind
    if sum(1 for ratio in ratios.values() if ratio >= 0.20) >= 2:
        return SectionKind.MIXED
    return SectionKind.NARRATIVE


def detect_document_sections(
    selected_document: SelectedDocument,
    cleaned_text: str,
) -> tuple[list[DocumentSection], list[ProcessingNote]]:
    """Detect lightweight sections from cleaned text using explicit header heuristics."""
    notes: list[ProcessingNote] = []
    if not cleaned_text.strip():
        return [], [
            make_processing_note(
                "no_clean_text",
                "No cleaned text was available for section detection.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=selected_document.document_id,
            )
        ]

    lines = cleaned_text.splitlines()
    header_indices: list[int] = []
    for index, line in enumerate(lines):
        previous_blank = index == 0 or not lines[index - 1].strip()
        next_blank = index == len(lines) - 1 or not lines[index + 1].strip()
        if is_probable_header_line(line, previous_blank=previous_blank, next_blank=next_blank):
            header_indices.append(index)

    sections: list[DocumentSection] = []
    if not header_indices:
        notes.append(
            make_processing_note(
                "weak_section_structure",
                "No reliable section headers were detected; a fallback untitled section was created.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=selected_document.document_id,
            )
        )
        fallback_text = cleaned_text.strip()
        sections.append(
            DocumentSection(
                section_id=f"{selected_document.document_id}::section_001",
                document_id=selected_document.document_id,
                section_index=0,
                title=selected_document.title or "Untitled Document Body",
                section_kind=classify_section_kind(fallback_text),
                level=1,
                reference_label=None,
                line_start=0,
                line_end=max(0, len(lines) - 1),
                raw_text=fallback_text,
                cleaned_text=fallback_text,
                char_count=len(fallback_text),
                word_count=approximate_word_count(fallback_text),
                header_detected=False,
                parent_section_id=None,
            )
        )
        return sections, notes

    section_boundaries = header_indices + [len(lines)]
    section_stack: list[tuple[int, str]] = []
    for section_index, start_index in enumerate(header_indices):
        end_index = section_boundaries[section_index + 1]
        title = lines[start_index].strip()
        body_lines = lines[start_index + 1:end_index]
        section_text = "\n".join([title] + body_lines).strip()
        level = infer_section_level(title)
        while section_stack and section_stack[-1][0] >= level:
            section_stack.pop()
        parent_section_id = section_stack[-1][1] if section_stack else None
        section_id = f"{selected_document.document_id}::section_{section_index + 1:03d}"
        section = DocumentSection(
            section_id=section_id,
            document_id=selected_document.document_id,
            section_index=section_index,
            title=title,
            section_kind=classify_section_kind(section_text),
            level=level,
            reference_label=extract_reference_label(title),
            line_start=start_index,
            line_end=max(start_index, end_index - 1),
            raw_text=section_text,
            cleaned_text=section_text,
            char_count=len(section_text),
            word_count=approximate_word_count(section_text),
            header_detected=True,
            parent_section_id=parent_section_id,
        )
        sections.append(section)
        section_stack.append((level, section_id))

    notes.append(
        make_processing_note(
            "section_detection_complete",
            f"Detected {len(sections)} section(s) in the cleaned document.",
            severity=ProcessingNoteSeverity.INFO,
            document_id=selected_document.document_id,
            metadata={"section_count": len(sections)},
        )
    )
    return sections, notes


## 25. Chunking Strategy

In [ ]:
def split_section_into_blocks(section: DocumentSection) -> list[str]:
    """Split a section into paragraph-like blocks while preserving list/table segments."""
    blocks = [block.strip() for block in re.split(r"\n\s*\n", section.cleaned_text) if block.strip()]
    if blocks:
        return blocks
    return [line.strip() for line in section.cleaned_text.splitlines() if line.strip()]


def slice_text_with_overlap(text: str, max_chars: int, overlap_chars: int) -> list[str]:
    """Fallback slicer used when a single block exceeds the preferred chunk size."""
    if len(text) <= max_chars:
        return [text]
    slices: list[str] = []
    start_index = 0
    step_size = max(1, max_chars - overlap_chars)
    while start_index < len(text):
        slice_text = text[start_index : start_index + max_chars].strip()
        if slice_text:
            slices.append(slice_text)
        if start_index + max_chars >= len(text):
            break
        start_index += step_size
    return slices


def summarize_chunk_context(
    chunk_text: str,
    *,
    section_title: str | None = None,
    previous_text: str | None = None,
    next_text: str | None = None,
) -> str:
    """Build a deterministic local-context summary without any model calls."""
    excerpt = chunk_text.replace("\n", " ").strip()[:140]
    parts: list[str] = []
    if section_title:
        parts.append(f"section={section_title}")
    if previous_text:
        parts.append(f"prev={previous_text.replace(chr(10), ' ').strip()[:50]}")
    parts.append(f"excerpt={excerpt}")
    if next_text:
        parts.append(f"next={next_text.replace(chr(10), ' ').strip()[:50]}")
    return " | ".join(parts)


## 26. Chunk Models and Chunk Builders

In [ ]:
def assign_chunk_ids(document_id: str, chunk_count: int) -> list[str]:
    """Assign stable chunk ids for one document."""
    return [f"{document_id}::chunk_{index + 1:03d}" for index in range(chunk_count)]


def build_chunks_from_sections(
    selected_document: SelectedDocument,
    sections: Sequence[DocumentSection],
    chunking_config: ChunkingConfig = CHUNKING_CONFIG,
) -> tuple[list[DocumentChunk], list[ProcessingNote]]:
    """Build stable chunks while respecting section boundaries where possible."""
    chunks: list[DocumentChunk] = []
    notes: list[ProcessingNote] = []

    for section in sections:
        blocks = split_section_into_blocks(section)
        if not blocks:
            continue

        working_blocks = blocks.copy()
        if chunking_config.include_section_titles and section.title and not working_blocks[0].startswith(section.title):
            working_blocks.insert(0, section.title)

        section_chunk_texts: list[str] = []
        current_text = ""
        for block in working_blocks:
            candidate_text = f"{current_text}\n\n{block}".strip() if current_text else block
            if len(candidate_text) <= chunking_config.max_chunk_chars or len(current_text) < chunking_config.min_chunk_chars:
                current_text = candidate_text
                continue
            if current_text:
                section_chunk_texts.append(current_text)
            overlap_prefix = current_text[-chunking_config.overlap_chars :].strip() if current_text else ""
            current_text = f"{overlap_prefix}\n\n{block}".strip() if overlap_prefix else block
            if len(current_text) > chunking_config.max_chunk_chars:
                section_chunk_texts.extend(
                    slice_text_with_overlap(current_text, chunking_config.max_chunk_chars, chunking_config.overlap_chars)
                )
                current_text = ""
        if current_text:
            section_chunk_texts.append(current_text)

        for order_in_section, chunk_text in enumerate(section_chunk_texts):
            chunks.append(
                DocumentChunk(
                    chunk_id="pending",
                    document_id=selected_document.document_id,
                    document_type=selected_document.document_type,
                    chunk_index=len(chunks),
                    order_in_section=order_in_section,
                    parent_section_id=section.section_id,
                    parent_section_title=section.title,
                    section_kind=section.section_kind,
                    text=chunk_text,
                    char_count=len(chunk_text),
                    word_count=approximate_word_count(chunk_text),
                )
            )

    chunk_ids = assign_chunk_ids(selected_document.document_id, len(chunks))
    for index, chunk in enumerate(chunks):
        chunk.chunk_id = chunk_ids[index]
        chunk.chunk_index = index

    for index, chunk in enumerate(chunks):
        previous_chunk = chunks[index - 1] if index > 0 else None
        next_chunk = chunks[index + 1] if index < len(chunks) - 1 else None
        chunk.previous_chunk_id = previous_chunk.chunk_id if previous_chunk else None
        chunk.next_chunk_id = next_chunk.chunk_id if next_chunk else None
        chunk.context_before = previous_chunk.text[-chunking_config.context_window_chars :] if previous_chunk else None
        chunk.context_after = next_chunk.text[: chunking_config.context_window_chars] if next_chunk else None
        chunk.local_context_summary = summarize_chunk_context(
            chunk.text,
            section_title=chunk.parent_section_title,
            previous_text=chunk.context_before,
            next_text=chunk.context_after,
        )

    if chunks:
        notes.append(
            make_processing_note(
                "chunk_build_complete",
                f"Built {len(chunks)} chunk(s) from detected sections.",
                severity=ProcessingNoteSeverity.INFO,
                document_id=selected_document.document_id,
                metadata={"chunk_count": len(chunks)},
            )
        )
    return chunks, notes


def build_fallback_chunks(
    selected_document: SelectedDocument,
    cleaned_text: str,
    chunking_config: ChunkingConfig = CHUNKING_CONFIG,
) -> tuple[list[DocumentChunk], list[ProcessingNote]]:
    """Fallback chunking used when reliable sections are unavailable."""
    notes = [
        make_processing_note(
            "fallback_chunking_used",
            "Fallback chunking was used because the document lacked strong section structure.",
            severity=ProcessingNoteSeverity.WARNING,
            document_id=selected_document.document_id,
        )
    ]
    slices = slice_text_with_overlap(
        cleaned_text,
        chunking_config.fallback_chunk_chars,
        chunking_config.overlap_chars,
    )
    synthetic_section_id = f"{selected_document.document_id}::section_fallback"
    chunks: list[DocumentChunk] = []
    for index, chunk_text in enumerate(slices):
        chunks.append(
            DocumentChunk(
                chunk_id=f"{selected_document.document_id}::chunk_{index + 1:03d}",
                document_id=selected_document.document_id,
                document_type=selected_document.document_type,
                chunk_index=index,
                order_in_section=index,
                parent_section_id=synthetic_section_id,
                parent_section_title=selected_document.title or "Fallback Section",
                section_kind=SectionKind.NARRATIVE,
                text=chunk_text,
                char_count=len(chunk_text),
                word_count=approximate_word_count(chunk_text),
            )
        )
    for index, chunk in enumerate(chunks):
        previous_chunk = chunks[index - 1] if index > 0 else None
        next_chunk = chunks[index + 1] if index < len(chunks) - 1 else None
        chunk.previous_chunk_id = previous_chunk.chunk_id if previous_chunk else None
        chunk.next_chunk_id = next_chunk.chunk_id if next_chunk else None
        chunk.context_before = previous_chunk.text[-chunking_config.context_window_chars :] if previous_chunk else None
        chunk.context_after = next_chunk.text[: chunking_config.context_window_chars] if next_chunk else None
        chunk.local_context_summary = summarize_chunk_context(
            chunk.text,
            section_title=chunk.parent_section_title,
            previous_text=chunk.context_before,
            next_text=chunk.context_after,
        )
    return chunks, notes


def build_document_chunks(
    selected_document: SelectedDocument,
    cleaned_text: str,
    sections: Sequence[DocumentSection],
    chunking_config: ChunkingConfig = CHUNKING_CONFIG,
) -> tuple[list[DocumentChunk], list[ProcessingNote], bool]:
    """Notebook-wide chunk builder that prefers section-aware chunking before fallback windows."""
    if sections:
        chunks, notes = build_chunks_from_sections(selected_document, sections, chunking_config)
        if chunks:
            return chunks, notes, False
    fallback_chunks, fallback_notes = build_fallback_chunks(selected_document, cleaned_text, chunking_config)
    return fallback_chunks, fallback_notes, True


def selected_document_from_candidate(candidate: RetrievalCandidate) -> SelectedDocument:
    """Convert a selected retrieval candidate into the shared SelectedDocument schema."""
    metadata = build_document_metadata_from_candidate(candidate)
    return SelectedDocument(
        document_id=candidate.candidate_id,
        document_type=candidate.document_type,
        ticker=candidate.ticker or metadata.ticker,
        title=candidate.title,
        raw_text=candidate.document_text or "",
        source_name=candidate.source_name,
        source_url=candidate.source_url,
        source_identifier=candidate.source_identifier,
        metadata=metadata,
        provenance=candidate.provenance,
        is_mock_data=candidate.is_mock_data,
    )


def selected_document_from_retrieval_result(result: RetrievalResult) -> SelectedDocument | None:
    """Convert a retrieval result into a SelectedDocument when a winner exists."""
    if result.selected_candidate is None:
        return None
    return selected_document_from_candidate(result.selected_candidate)


def process_selected_document(
    selected_document: SelectedDocument,
    processing_config: ProcessingConfig = PROCESSING_CONFIG,
    chunking_config: ChunkingConfig = CHUNKING_CONFIG,
) -> ProcessedDocument:
    """Run deterministic cleanup, section parsing, and chunk building for one selected document."""
    cleaned_text, cleanup_notes = safe_text_cleanup(selected_document.raw_text, document_id=selected_document.document_id)
    sections, section_notes = detect_document_sections(selected_document, cleaned_text)
    chunks, chunk_notes, used_fallback_chunking = build_document_chunks(
        selected_document,
        cleaned_text,
        sections,
        chunking_config,
    )
    processing_notes = cleanup_notes + section_notes + chunk_notes
    if not cleaned_text:
        processing_notes.append(
            make_processing_note(
                "no_clean_text_after_processing",
                "The document remained empty after processing and cannot be chunked reliably.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=selected_document.document_id,
            )
        )
    return ProcessedDocument(
        document=selected_document,
        cleaned_text=cleaned_text,
        sections=list(sections),
        chunks=chunks,
        processing_notes=processing_notes,
        raw_char_count=len(selected_document.raw_text),
        cleaned_char_count=len(cleaned_text),
        cleaned_word_count=approximate_word_count(cleaned_text),
        section_count=len(sections),
        chunk_count=len(chunks),
        used_fallback_chunking=used_fallback_chunking,
    )


## 27. Embedding Configuration and Ollama Embedding Hooks

In [ ]:
def tokenize_for_embedding(text: str) -> list[str]:
    """Tokenize text for deterministic mock embeddings and lexical overlap checks."""
    return re.findall(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?", text.lower())


def l2_normalize(vector: Sequence[float]) -> list[float]:
    """Return an L2-normalized copy of a vector."""
    norm = math.sqrt(sum(value * value for value in vector))
    if norm == 0.0:
        return [0.0 for _ in vector]
    return [value / norm for value in vector]


class BaseEmbeddingClient(ABC):
    """Base embedding client interface for local or mock-safe embeddings."""

    provider_name: ClassVar[str] = "base"

    @abstractmethod
    def embed_texts(self, texts: Sequence[str], config: EmbeddingConfig) -> EmbeddingBatchResult:
        """Embed a batch of texts using the configured provider."""


class MockHashEmbeddingClient(BaseEmbeddingClient):
    """Deterministic hash-based embedding client used for safe notebook demos."""

    provider_name = EmbeddingProvider.MOCK.value

    def embed_texts(self, texts: Sequence[str], config: EmbeddingConfig) -> EmbeddingBatchResult:
        vectors = [self._mock_embed(text, config.mock_vector_dimensions, config.normalize_vectors) for text in texts]
        notes = [
            make_processing_note(
                "mock_embeddings_used",
                "Generated deterministic mock embeddings because live embeddings were disabled or unavailable.",
                severity=ProcessingNoteSeverity.INFO,
                metadata={"vector_dimension": config.mock_vector_dimensions},
            )
        ]
        return EmbeddingBatchResult(
            provider_name=self.provider_name,
            model_name="mock_hash_embedding",
            status=EmbeddingStatus.MOCK,
            vectors=vectors,
            is_mock_embedding=True,
            notes=notes,
        )

    @staticmethod
    def _mock_embed(text: str, dimensions: int, normalize_vectors: bool) -> list[float]:
        vector = [0.0] * dimensions
        tokens = tokenize_for_embedding(text)
        if not tokens:
            return vector
        for token in tokens:
            digest = hashlib.sha256(token.encode("utf-8")).digest()
            index = int.from_bytes(digest[:4], "big") % dimensions
            sign = 1.0 if digest[4] % 2 == 0 else -1.0
            vector[index] += sign
        return l2_normalize(vector) if normalize_vectors else vector


class OllamaEmbeddingClient(BaseEmbeddingClient):
    """Best-effort Ollama embedding hook with safe fallback behavior."""

    provider_name = EmbeddingProvider.OLLAMA.value

    def embed_texts(self, texts: Sequence[str], config: EmbeddingConfig) -> EmbeddingBatchResult:
        if not texts:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.SKIPPED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "no_texts_to_embed",
                        "No texts were provided to the embedding client.",
                        severity=ProcessingNoteSeverity.INFO,
                    )
                ],
            )

        if not config.enabled:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.SKIPPED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "embeddings_disabled",
                        "Embedding generation is disabled in the current configuration.",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        if not config.prefer_live_embeddings:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.SKIPPED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "live_embeddings_not_requested",
                        "Live Ollama embeddings are not enabled; a mock fallback may be used.",
                        severity=ProcessingNoteSeverity.INFO,
                    )
                ],
            )

        if not config.model_name or config.model_name == "configurable_placeholder":
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.FAILED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "ollama_model_not_configured",
                        "Ollama embedding model name is still a configurable placeholder.",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        payload = json.dumps({"model": config.model_name, "input": list(texts)}).encode("utf-8")
        request = Request(
            config.base_url.rstrip("/") + config.api_path,
            data=payload,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        try:
            with urlopen(request, timeout=config.request_timeout_seconds) as response:
                response_payload = json.loads(response.read().decode("utf-8"))
        except (HTTPError, URLError, TimeoutError, OSError, ValueError) as exc:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.FAILED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "ollama_request_failed",
                        f"Ollama embedding request failed: {exc}",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        embeddings = response_payload.get("embeddings")
        if not isinstance(embeddings, list) or len(embeddings) != len(texts):
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.FAILED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "ollama_response_invalid",
                        "Ollama response did not contain an embeddings array matching the request batch size.",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        normalized_vectors = [l2_normalize(vector) if config.normalize_vectors else vector for vector in embeddings]
        return EmbeddingBatchResult(
            provider_name=self.provider_name,
            model_name=config.model_name,
            status=EmbeddingStatus.SUCCESS,
            vectors=normalized_vectors,
            is_mock_embedding=False,
            notes=[
                make_processing_note(
                    "ollama_embeddings_complete",
                    f"Generated {len(normalized_vectors)} embedding vector(s) via Ollama.",
                    severity=ProcessingNoteSeverity.INFO,
                )
            ],
        )


class VoyageAIEmbeddingClient(BaseEmbeddingClient):
    """Voyage AI embedding client using the REST API."""

    provider_name = EmbeddingProvider.VOYAGE.value

    def embed_texts(self, texts: Sequence[str], config: EmbeddingConfig) -> EmbeddingBatchResult:
        if not texts:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.SKIPPED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "no_texts_to_embed",
                        "No texts were provided to the embedding client.",
                        severity=ProcessingNoteSeverity.INFO,
                    )
                ],
            )

        if not config.enabled:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.SKIPPED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "embeddings_disabled",
                        "Embedding generation is disabled in the current configuration.",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        api_key = os.environ.get("VOYAGE_AI_API_KEY", "")
        if not api_key:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.FAILED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "voyage_api_key_missing",
                        "VOYAGE_AI_API_KEY environment variable is not set.",
                        severity=ProcessingNoteSeverity.ERROR,
                    )
                ],
            )

        url = config.base_url.rstrip("/") + config.api_path
        payload = json.dumps({
            "input": list(texts),
            "model": config.model_name,
            "input_type": "document",
        }).encode("utf-8")
        request = Request(
            url,
            data=payload,
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {api_key}",
            },
            method="POST",
        )
        try:
            with urlopen(request, timeout=config.request_timeout_seconds) as response:
                response_payload = json.loads(response.read().decode("utf-8"))
        except (HTTPError, URLError, TimeoutError, OSError, ValueError) as exc:
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.FAILED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "voyage_request_failed",
                        f"Voyage AI embedding request failed: {exc}",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        data = response_payload.get("data")
        if not isinstance(data, list) or len(data) != len(texts):
            return EmbeddingBatchResult(
                provider_name=self.provider_name,
                model_name=config.model_name,
                status=EmbeddingStatus.FAILED,
                vectors=[],
                notes=[
                    make_processing_note(
                        "voyage_response_invalid",
                        "Voyage AI response did not contain a data array matching the request batch size.",
                        severity=ProcessingNoteSeverity.WARNING,
                    )
                ],
            )

        raw_vectors = [item["embedding"] for item in data]
        normalized_vectors = [l2_normalize(v) if config.normalize_vectors else v for v in raw_vectors]
        return EmbeddingBatchResult(
            provider_name=self.provider_name,
            model_name=config.model_name,
            status=EmbeddingStatus.SUCCESS,
            vectors=normalized_vectors,
            is_mock_embedding=False,
            notes=[
                make_processing_note(
                    "voyage_embeddings_complete",
                    f"Generated {len(normalized_vectors)} embedding vector(s) via Voyage AI.",
                    severity=ProcessingNoteSeverity.INFO,
                )
            ],
        )


def build_embedding_client(config: EmbeddingConfig = EMBEDDING_CONFIG) -> BaseEmbeddingClient:
    """Build the preferred embedding client from configuration."""
    if config.provider == EmbeddingProvider.MOCK:
        return MockHashEmbeddingClient()
    elif config.provider == EmbeddingProvider.VOYAGE:
        return VoyageAIEmbeddingClient()
    return OllamaEmbeddingClient()


def batch_embed_texts(
    texts: Sequence[str],
    config: EmbeddingConfig = EMBEDDING_CONFIG,
    preferred_client: BaseEmbeddingClient | None = None,
) -> EmbeddingBatchResult:
    """Embed text batches with a safe mock fallback when live embeddings fail or are disabled."""
    client = preferred_client or build_embedding_client(config)
    batch_result = client.embed_texts(texts, config)
    if batch_result.status in {EmbeddingStatus.SUCCESS, EmbeddingStatus.MOCK}:
        return batch_result
    if config.use_mock_fallback:
        mock_result = MockHashEmbeddingClient().embed_texts(texts, config)
        mock_result.notes = batch_result.notes + [
            make_processing_note(
                "embedding_fallback_used",
                "Fell back to deterministic mock embeddings after live embedding skip/failure.",
                severity=ProcessingNoteSeverity.WARNING,
            )
        ] + mock_result.notes
        return mock_result
    return batch_result


def build_chunk_embedding_records(
    processed_document: ProcessedDocument,
    config: EmbeddingConfig = EMBEDDING_CONFIG,
    preferred_client: BaseEmbeddingClient | None = None,
) -> tuple[list[ChunkEmbeddingRecord], list[ProcessingNote]]:
    """Build chunk embedding records for a processed document."""
    if not processed_document.chunks:
        notes = [
            make_processing_note(
                "no_chunks_to_embed",
                "Processed document contains no chunks to embed.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=processed_document.document.document_id,
            )
        ]
        return [], notes

    batch_result = batch_embed_texts(
        [chunk.text for chunk in processed_document.chunks],
        config=config,
        preferred_client=preferred_client,
    )
    records: list[ChunkEmbeddingRecord] = []
    for chunk, vector in zip(processed_document.chunks, batch_result.vectors):
        records.append(
            ChunkEmbeddingRecord(
                chunk_id=chunk.chunk_id,
                document_id=chunk.document_id,
                provider_name=batch_result.provider_name,
                model_name=batch_result.model_name,
                status=batch_result.status,
                embedding=vector,
                vector_dimension=len(vector),
                is_mock_embedding=batch_result.is_mock_embedding,
                notes=batch_result.notes,
            )
        )
    return records, batch_result.notes

## 28. Lightweight Graph-Aware Retrieval Models

In [ ]:
def build_document_graph(processed_document: ProcessedDocument) -> GraphDocumentIndex:
    """Build a lightweight graph index from one processed document."""
    nodes: list[GraphNode] = []
    edges: list[GraphEdge] = []
    notes: list[ProcessingNote] = []
    section_to_chunk_ids: dict[str, list[str]] = {}
    chunk_neighbors: dict[str, list[str]] = {chunk.chunk_id: [] for chunk in processed_document.chunks}
    chunk_to_section_id: dict[str, str] = {}
    cross_reference_targets: dict[str, list[str]] = {}

    document_node_id = f"document::{processed_document.document.document_id}"
    nodes.append(
        GraphNode(
            node_id=document_node_id,
            document_id=processed_document.document.document_id,
            node_type=GraphNodeType.DOCUMENT,
            label=processed_document.document.title or processed_document.document.document_id,
            ref_id=processed_document.document.document_id,
        )
    )

    section_lookup: dict[str, DocumentSection] = {section.section_id: section for section in processed_document.sections}
    section_reference_lookup: dict[str, str] = {}

    for section in processed_document.sections:
        section_node_id = f"section::{section.section_id}"
        nodes.append(
            GraphNode(
                node_id=section_node_id,
                document_id=processed_document.document.document_id,
                node_type=GraphNodeType.SECTION,
                label=section.title,
                ref_id=section.section_id,
                metadata={"section_kind": section.section_kind.value, "level": section.level},
            )
        )
        edges.append(
            GraphEdge(
                edge_id=f"edge::{document_node_id}::{section_node_id}",
                document_id=processed_document.document.document_id,
                source_node_id=document_node_id,
                target_node_id=section_node_id,
                edge_type=GraphEdgeType.CONTAINS,
            )
        )
        if section.reference_label:
            section_reference_lookup[normalize_token(section.reference_label)] = section.section_id

    for previous_section, next_section in zip(processed_document.sections, processed_document.sections[1:]):
        edges.append(
            GraphEdge(
                edge_id=f"edge::section_sequence::{previous_section.section_id}::{next_section.section_id}",
                document_id=processed_document.document.document_id,
                source_node_id=f"section::{previous_section.section_id}",
                target_node_id=f"section::{next_section.section_id}",
                edge_type=GraphEdgeType.SECTION_SEQUENCE,
                weight=0.5,
            )
        )

    for chunk in processed_document.chunks:
        chunk_node_id = f"chunk::{chunk.chunk_id}"
        nodes.append(
            GraphNode(
                node_id=chunk_node_id,
                document_id=processed_document.document.document_id,
                node_type=GraphNodeType.CHUNK,
                label=chunk.parent_section_title or chunk.chunk_id,
                ref_id=chunk.chunk_id,
                metadata={"chunk_index": chunk.chunk_index, "section_kind": chunk.section_kind.value},
            )
        )
        if chunk.parent_section_id:
            section_to_chunk_ids.setdefault(chunk.parent_section_id, []).append(chunk.chunk_id)
            chunk_to_section_id[chunk.chunk_id] = chunk.parent_section_id
            edges.append(
                GraphEdge(
                    edge_id=f"edge::contains::{chunk.parent_section_id}::{chunk.chunk_id}",
                    document_id=processed_document.document.document_id,
                    source_node_id=f"section::{chunk.parent_section_id}",
                    target_node_id=chunk_node_id,
                    edge_type=GraphEdgeType.CONTAINS,
                )
            )

        if chunk.previous_chunk_id:
            chunk_neighbors[chunk.chunk_id].append(chunk.previous_chunk_id)
            edges.append(
                GraphEdge(
                    edge_id=f"edge::adjacent::{chunk.previous_chunk_id}::{chunk.chunk_id}",
                    document_id=processed_document.document.document_id,
                    source_node_id=f"chunk::{chunk.previous_chunk_id}",
                    target_node_id=chunk_node_id,
                    edge_type=GraphEdgeType.ADJACENT,
                )
            )
        if chunk.next_chunk_id:
            chunk_neighbors[chunk.chunk_id].append(chunk.next_chunk_id)

    for section_id, chunk_ids in section_to_chunk_ids.items():
        for first_chunk_id, second_chunk_id in zip(chunk_ids, chunk_ids[1:]):
            chunk_neighbors.setdefault(first_chunk_id, []).append(second_chunk_id)
            chunk_neighbors.setdefault(second_chunk_id, []).append(first_chunk_id)
            edges.append(
                GraphEdge(
                    edge_id=f"edge::same_section::{first_chunk_id}::{second_chunk_id}",
                    document_id=processed_document.document.document_id,
                    source_node_id=f"chunk::{first_chunk_id}",
                    target_node_id=f"chunk::{second_chunk_id}",
                    edge_type=GraphEdgeType.SAME_SECTION,
                    weight=0.75,
                )
            )

    for chunk in processed_document.chunks:
        normalized_chunk_text = normalize_token(chunk.text)
        for reference_label, target_section_id in section_reference_lookup.items():
            if reference_label and target_section_id != chunk.parent_section_id and f"section {reference_label}" in normalized_chunk_text:
                cross_reference_targets.setdefault(chunk.chunk_id, []).append(target_section_id)
                edges.append(
                    GraphEdge(
                        edge_id=f"edge::cross_reference::{chunk.chunk_id}::{target_section_id}",
                        document_id=processed_document.document.document_id,
                        source_node_id=f"chunk::{chunk.chunk_id}",
                        target_node_id=f"section::{target_section_id}",
                        edge_type=GraphEdgeType.CROSS_REFERENCE,
                        weight=0.5,
                    )
                )

    if len(processed_document.sections) <= 1:
        notes.append(
            make_processing_note(
                "graph_structure_minimal",
                "Graph structure is minimal because the document has one or fewer sections.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=processed_document.document.document_id,
            )
        )
    else:
        notes.append(
            make_processing_note(
                "graph_index_complete",
                f"Built graph index with {len(nodes)} nodes and {len(edges)} edges.",
                severity=ProcessingNoteSeverity.INFO,
                document_id=processed_document.document.document_id,
            )
        )

    for chunk_id, neighbors in chunk_neighbors.items():
        chunk_neighbors[chunk_id] = sorted(set(neighbors))

    return GraphDocumentIndex(
        document_id=processed_document.document.document_id,
        nodes=nodes,
        edges=edges,
        section_to_chunk_ids=section_to_chunk_ids,
        chunk_neighbors=chunk_neighbors,
        chunk_to_section_id=chunk_to_section_id,
        cross_reference_targets=cross_reference_targets,
        notes=notes,
    )


def get_neighbor_chunks(
    graph_index: GraphDocumentIndex,
    chunk_id: str,
    *,
    max_hops: int = 1,
) -> list[str]:
    """Return neighboring chunk ids using adjacency and same-section links."""
    visited = {chunk_id}
    frontier = {chunk_id}
    neighbors: list[str] = []
    for _ in range(max_hops):
        next_frontier: set[str] = set()
        for current_chunk_id in frontier:
            for neighbor_id in graph_index.chunk_neighbors.get(current_chunk_id, []):
                if neighbor_id not in visited:
                    visited.add(neighbor_id)
                    next_frontier.add(neighbor_id)
                    neighbors.append(neighbor_id)
        frontier = next_frontier
        if not frontier:
            break
    return neighbors


def expand_chunk_context(
    processed_document: ProcessedDocument,
    graph_index: GraphDocumentIndex,
    chunk_id: str,
    *,
    max_hops: int = 1,
) -> list[DocumentChunk]:
    """Expand one chunk into a small neighborhood for richer retrieval context."""
    chunk_lookup = {chunk.chunk_id: chunk for chunk in processed_document.chunks}
    related_ids = [chunk_id] + get_neighbor_chunks(graph_index, chunk_id, max_hops=max_hops)
    related_chunks = [chunk_lookup[candidate_id] for candidate_id in related_ids if candidate_id in chunk_lookup]
    return sorted(related_chunks, key=lambda chunk: chunk.chunk_index)


def score_chunk_with_graph_context(
    *,
    query_text: str,
    chunk: DocumentChunk,
    processed_document: ProcessedDocument,
    graph_index: GraphDocumentIndex,
    base_similarity: float,
    similarity_lookup: dict[str, float],
    retrieval_config: GraphRetrievalConfig = GRAPH_RETRIEVAL_CONFIG,
) -> tuple[float, float, list[ProcessingNote]]:
    """Apply a lightweight graph bonus using section titles and neighbor support."""
    notes: list[ProcessingNote] = []
    query_tokens = set(tokenize_for_embedding(query_text))
    section_title_tokens = set(tokenize_for_embedding(chunk.parent_section_title or ""))
    section_overlap_ratio = 0.0
    if query_tokens and section_title_tokens:
        section_overlap_ratio = len(query_tokens & section_title_tokens) / len(query_tokens | section_title_tokens)

    neighbor_ids = get_neighbor_chunks(graph_index, chunk.chunk_id, max_hops=retrieval_config.neighbor_hops)
    neighbor_similarity = max((similarity_lookup.get(neighbor_id, 0.0) for neighbor_id in neighbor_ids), default=0.0)
    cross_reference_bonus = (
        retrieval_config.cross_reference_bonus if graph_index.cross_reference_targets.get(chunk.chunk_id) else 0.0
    )

    graph_bonus = min(
        retrieval_config.max_graph_bonus,
        (section_overlap_ratio * retrieval_config.section_title_weight)
        + (neighbor_similarity * retrieval_config.neighbor_similarity_weight)
        + cross_reference_bonus,
    )
    adjusted_score = base_similarity + graph_bonus

    if graph_bonus:
        notes.append(
            make_processing_note(
                "graph_bonus_applied",
                "Applied graph-aware retrieval bonus using section title overlap and neighbor support.",
                severity=ProcessingNoteSeverity.INFO,
                document_id=chunk.document_id,
                chunk_id=chunk.chunk_id,
                metadata={
                    "section_overlap_ratio": round(section_overlap_ratio, 4),
                    "neighbor_similarity": round(neighbor_similarity, 4),
                    "cross_reference_bonus": round(cross_reference_bonus, 4),
                },
            )
        )
    return adjusted_score, graph_bonus, notes


## 29. In-Document Semantic Retrieval Functions

In [ ]:
def cosine_similarity(vector_a: Sequence[float], vector_b: Sequence[float]) -> float:
    """Compute cosine similarity for two vectors of equal length."""
    if not vector_a or not vector_b or len(vector_a) != len(vector_b):
        return 0.0
    numerator = sum(left * right for left, right in zip(vector_a, vector_b))
    left_norm = math.sqrt(sum(value * value for value in vector_a))
    right_norm = math.sqrt(sum(value * value for value in vector_b))
    if left_norm == 0.0 or right_norm == 0.0:
        return 0.0
    return numerator / (left_norm * right_norm)


def build_chunk_embedding_lookup(records: Sequence[ChunkEmbeddingRecord]) -> dict[str, list[float]]:
    """Build a chunk-id to embedding vector mapping."""
    return {record.chunk_id: record.embedding for record in records if record.embedding is not None}


def make_text_excerpt(text: str, limit: int = 180) -> str:
    """Build a short, single-line excerpt for notebook display."""
    single_line = " ".join(text.split())
    if len(single_line) <= limit:
        return single_line
    return single_line[: limit - 3] + "..."


def retrieve_relevant_chunks(
    query_text: str,
    processed_document: ProcessedDocument,
    chunk_embeddings: Sequence[ChunkEmbeddingRecord],
    graph_index: GraphDocumentIndex,
    embedding_config: EmbeddingConfig = EMBEDDING_CONFIG,
    retrieval_config: GraphRetrievalConfig = GRAPH_RETRIEVAL_CONFIG,
    preferred_client: BaseEmbeddingClient | None = None,
) -> ChunkRetrievalResult:
    """Retrieve relevant chunks from one processed document using embeddings plus graph context."""
    notes: list[ProcessingNote] = []
    if not processed_document.chunks:
        notes.append(
            make_processing_note(
                "no_chunks_available",
                "No chunks are available for retrieval from this processed document.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=processed_document.document.document_id,
            )
        )
        return ChunkRetrievalResult(
            document_id=processed_document.document.document_id,
            query_text=query_text,
            embedding_provider=embedding_config.provider.value,
            embedding_model_name=embedding_config.model_name,
            query_embedding_status=EmbeddingStatus.SKIPPED,
            hits=[],
            notes=notes,
        )

    query_embedding_batch = batch_embed_texts([query_text], config=embedding_config, preferred_client=preferred_client)
    notes.extend(query_embedding_batch.notes)
    if not query_embedding_batch.vectors:
        notes.append(
            make_processing_note(
                "query_embedding_unavailable",
                "Query embedding could not be generated, so no chunk retrieval was performed.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=processed_document.document.document_id,
            )
        )
        return ChunkRetrievalResult(
            document_id=processed_document.document.document_id,
            query_text=query_text,
            embedding_provider=query_embedding_batch.provider_name,
            embedding_model_name=query_embedding_batch.model_name,
            query_embedding_status=query_embedding_batch.status,
            hits=[],
            notes=notes,
        )

    query_vector = query_embedding_batch.vectors[0]
    chunk_lookup = {chunk.chunk_id: chunk for chunk in processed_document.chunks}
    embedding_lookup = build_chunk_embedding_lookup(chunk_embeddings)
    similarity_lookup: dict[str, float] = {}
    for chunk in processed_document.chunks:
        vector = embedding_lookup.get(chunk.chunk_id)
        if vector is None:
            continue
        similarity_lookup[chunk.chunk_id] = cosine_similarity(query_vector, vector)

    candidate_ids = [
        chunk_id
        for chunk_id, similarity in sorted(similarity_lookup.items(), key=lambda item: (-item[1], item[0]))
        if similarity >= retrieval_config.min_similarity_threshold
    ]

    if not candidate_ids:
        notes.append(
            make_processing_note(
                "no_chunk_matches",
                "Chunk retrieval returned no candidates above the similarity threshold.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=processed_document.document.document_id,
            )
        )
        return ChunkRetrievalResult(
            document_id=processed_document.document.document_id,
            query_text=query_text,
            embedding_provider=query_embedding_batch.provider_name,
            embedding_model_name=query_embedding_batch.model_name,
            query_embedding_status=query_embedding_batch.status,
            hits=[],
            notes=notes,
        )

    candidate_pool_size = max(retrieval_config.top_k, retrieval_config.candidate_pool_size)
    candidate_pool_ids = candidate_ids[:candidate_pool_size]
    hits: list[ChunkRetrievalHit] = []
    for chunk_id in candidate_pool_ids:
        chunk = chunk_lookup[chunk_id]
        base_similarity = similarity_lookup.get(chunk_id, 0.0)
        adjusted_score, graph_bonus, graph_notes = score_chunk_with_graph_context(
            query_text=query_text,
            chunk=chunk,
            processed_document=processed_document,
            graph_index=graph_index,
            base_similarity=base_similarity,
            similarity_lookup=similarity_lookup,
            retrieval_config=retrieval_config,
        )
        expanded_chunks = expand_chunk_context(
            processed_document,
            graph_index,
            chunk_id,
            max_hops=retrieval_config.context_expansion_hops,
        )
        hits.append(
            ChunkRetrievalHit(
                rank=0,
                chunk_id=chunk.chunk_id,
                document_id=chunk.document_id,
                section_id=chunk.parent_section_id,
                section_title=chunk.parent_section_title,
                similarity_score=base_similarity,
                graph_bonus=graph_bonus,
                adjusted_score=adjusted_score,
                text_excerpt=make_text_excerpt(chunk.text),
                expanded_context_chunk_ids=[expanded_chunk.chunk_id for expanded_chunk in expanded_chunks],
                expanded_context_preview=make_text_excerpt("\n\n".join(expanded_chunk.text for expanded_chunk in expanded_chunks), 220),
                notes=graph_notes,
            )
        )

    ranked_hits = sorted(hits, key=lambda hit: (-hit.adjusted_score, hit.chunk_id))[: retrieval_config.top_k]
    for rank, hit in enumerate(ranked_hits, start=1):
        hit.rank = rank

    if len(ranked_hits) < retrieval_config.top_k:
        notes.append(
            make_processing_note(
                "too_few_retrieval_hits",
                f"Only {len(ranked_hits)} chunk hit(s) were available for top_k={retrieval_config.top_k}.",
                severity=ProcessingNoteSeverity.WARNING,
                document_id=processed_document.document.document_id,
            )
        )

    return ChunkRetrievalResult(
        document_id=processed_document.document.document_id,
        query_text=query_text,
        embedding_provider=query_embedding_batch.provider_name,
        embedding_model_name=query_embedding_batch.model_name,
        query_embedding_status=query_embedding_batch.status,
        used_graph_context=True,
        hits=ranked_hits,
        notes=notes,
    )


## 30. Demo Documents and Processing Walkthrough

In [ ]:
def make_demo_selected_document(
    *,
    document_id: str,
    document_type: DocumentType,
    title: str,
    ticker: str,
    raw_text: str,
    source_name: str,
    source_url: str,
) -> SelectedDocument:
    """Build a fake selected document for notebook processing demos."""
    metadata = DocumentMetadata(
        document_id=document_id,
        ticker=ticker,
        document_type=document_type,
        title=title,
        source_name=source_name,
        source_url=source_url,
        source_family=SourceFamily.ISSUER_PUBLISHED,
        retrieved_at=now_utc(),
        is_structured_source=False,
        is_mock_data=True,
        notes=["Synthetic demo selected document."],
    )
    return SelectedDocument(
        document_id=document_id,
        document_type=document_type,
        ticker=ticker,
        title=title,
        raw_text=raw_text,
        source_name=source_name,
        source_url=source_url,
        metadata=metadata,
        is_mock_data=True,
    )


DEMO_SELECTED_DOCUMENTS = {
    "filing_material_event": make_demo_selected_document(
        document_id="demo_selected_001",
        document_type=DocumentType.MATERIAL_EVENT,
        title="FakeBio Therapeutics Files Material Agreement Update",
        ticker="FAKE",
        source_name="Mock Filing Archive",
        source_url="https://example.com/demo/filing-material-event",
        raw_text="""
FORM 8-K
Page 1 of 3

Item 1.01 Entry into a Material Definitive Agreement

On March 1, 2026, FakeBio Therapeutics, Inc. entered into a collaboration agreement with Northwind Biologics.
The agreement covers manufacturing transfer services, a joint steering committee, and staged closing conditions.

Key Terms
• Upfront payment of $25 million at closing
• Joint steering committee oversight for development activities
• Closing remains subject to antitrust clearance and manufacturing transfer milestones

Milestone Schedule      Amount      Timing
Closing Payment      $25 million      At closing
Commercial Milestone      $60 million      Upon first major-market launch

Item 2.03 Creation of a Direct Financial Obligation

The company may owe additional milestone consideration if the licensed program reaches regulatory and sales triggers.
See Section 1.01 for the agreement summary.

Forward-Looking Statements

Management expects the closing process to continue during the second quarter of 2026.
Page 2 of 3
""".strip(),
    ),
    "registry_update": make_demo_selected_document(
        document_id="demo_selected_002",
        document_type=DocumentType.CLINICAL_TRIAL_UPDATE,
        title="FakeBio Therapeutics Phase 2 Registry Update",
        ticker="FAKE",
        source_name="Mock Trial Registry",
        source_url="https://example.com/demo/registry-update",
        raw_text="""
Study Overview

Study Title
Randomized Phase 2 Study of FB-201 in Recurrent Disease

Recruitment Status
Recruiting

Outcome Measures
Primary Outcome Measure
Change from baseline in biomarker score at Week 24
Secondary Outcome Measure
Objective response rate through Month 6

Enrollment Summary
Current Enrollment      84
Planned Enrollment      120
Estimated Primary Completion Date      September 2026

Eligibility Criteria
- Adults aged 18 years and older
- Histologically confirmed recurrent disease
- Prior standard therapy required

Contacts and Locations
Lead site activated in Chicago, Illinois
Additional enrollment expected in Boston and Houston
""".strip(),
    ),
    "investor_transcript": make_demo_selected_document(
        document_id="demo_selected_003",
        document_type=DocumentType.INVESTOR_COMMUNICATION,
        title="FakeBio Therapeutics Corporate Update Transcript",
        ticker="FAKE",
        source_name="Mock Investor Events Page",
        source_url="https://example.com/demo/investor-transcript",
        raw_text="""
FakeBio Therapeutics Corporate Update

Operator:
Good morning, and welcome to the FakeBio Therapeutics corporate update.

Program Priorities
- Enrollment cadence across newly activated sites
- Manufacturing readiness for registration supply
- Regulatory meeting preparation before year-end

Chief Executive Officer:
We remain focused on enrollment execution, manufacturing readiness, and a disciplined data-readout plan.
The operating team expects site activation to support a steadier cadence over the next quarter.

Chief Financial Officer:
Based on current operating assumptions, cash runway extends into the first half of 2027.
We continue to prioritize manufacturing investments and trial execution spending.

Question and Answer
Analyst:
How should investors think about near-term milestones?

Chief Executive Officer:
The next milestones are site activation, enrollment cadence, chemistry manufacturing validation, and trial execution quality.
""".strip(),
    ),
}

DEMO_PROCESSING_QUERIES = {
    "filing_material_event": "closing conditions milestone payment agreement",
    "registry_update": "enrollment primary completion outcome measure",
    "investor_transcript": "cash runway near-term milestones",
}

DEMO_PROCESSED_DOCUMENTS: dict[str, ProcessedDocument] = {}
DEMO_GRAPH_INDICES: dict[str, GraphDocumentIndex] = {}
DEMO_CHUNK_EMBEDDINGS: dict[str, list[ChunkEmbeddingRecord]] = {}
DEMO_CHUNK_RETRIEVAL_RESULTS: dict[str, ChunkRetrievalResult] = {}

for demo_key, selected_document in DEMO_SELECTED_DOCUMENTS.items():
    processed_document = process_selected_document(selected_document)
    graph_index = build_document_graph(processed_document)
    embedding_records, embedding_notes = build_chunk_embedding_records(processed_document, EMBEDDING_CONFIG)
    processed_document.processing_notes.extend(embedding_notes)
    retrieval_result = retrieve_relevant_chunks(
        DEMO_PROCESSING_QUERIES[demo_key],
        processed_document,
        embedding_records,
        graph_index,
        EMBEDDING_CONFIG,
        GRAPH_RETRIEVAL_CONFIG,
    )
    DEMO_PROCESSED_DOCUMENTS[demo_key] = processed_document
    DEMO_GRAPH_INDICES[demo_key] = graph_index
    DEMO_CHUNK_EMBEDDINGS[demo_key] = embedding_records
    DEMO_CHUNK_RETRIEVAL_RESULTS[demo_key] = retrieval_result

PROCESSING_DEMO_SUMMARY = {
    demo_key: {
        "document_type": processed_document.document.document_type.value,
        "section_count": processed_document.section_count,
        "chunk_count": processed_document.chunk_count,
        "used_fallback_chunking": processed_document.used_fallback_chunking,
        "processing_warnings": [
            note.message for note in processed_document.processing_notes if note.severity != ProcessingNoteSeverity.INFO
        ],
        "graph_node_count": len(DEMO_GRAPH_INDICES[demo_key].nodes),
        "graph_edge_count": len(DEMO_GRAPH_INDICES[demo_key].edges),
        "embedding_statuses": sorted({record.status.value for record in DEMO_CHUNK_EMBEDDINGS[demo_key]}),
        "top_hits": [
            {
                "rank": hit.rank,
                "chunk_id": hit.chunk_id,
                "section_title": hit.section_title,
                "similarity_score": round(hit.similarity_score, 4),
                "graph_bonus": round(hit.graph_bonus, 4),
                "adjusted_score": round(hit.adjusted_score, 4),
                "text_excerpt": hit.text_excerpt,
                "expanded_context_chunk_ids": hit.expanded_context_chunk_ids,
            }
            for hit in DEMO_CHUNK_RETRIEVAL_RESULTS[demo_key].hits
        ],
    }
    for demo_key, processed_document in DEMO_PROCESSED_DOCUMENTS.items()
}

print(json.dumps(PROCESSING_DEMO_SUMMARY, indent=2))


## 31. Retrieval Pipeline Demo

In [ ]:
if PIPELINE_CONFIG.enable_mock_retrieval:
    # Mock mode: use FAKE ticker with mock adapters
    demo_requests = [
        RetrievalRequest(
            request_id=f"demo_{document_type.value}",
            ticker="FAKE",
            company_name="FakeBio Therapeutics",
            company_aliases=["FakeBio", "FakeBio Therapeutics"],
            document_type=document_type,
            max_candidates=PIPELINE_CONFIG.default_max_candidates,
            minimum_text_chars=PIPELINE_CONFIG.min_usable_text_chars,
            minimum_relevance_score=PIPELINE_CONFIG.minimum_relevance_score,
            freshness_bonus_max=PIPELINE_CONFIG.freshness_bonus_max,
            freshness_rank_decay=PIPELINE_CONFIG.freshness_rank_decay,
            source_preferences=SourcePreferencePolicy(
                preferred_source_families=PIPELINE_CONFIG.default_source_preferences,
                prefer_structured_source=True,
                allow_secondary_sources=True,
                allow_unknown_sources=True,
                allow_mock_data=True,
            ),
            retrieval_notes=["Notebook demo request using placeholder retrieval data."],
            is_mock_request=True,
        )
        for document_type in DocumentType
    ]
else:
    # Live mode: use real ticker with live adapters
    demo_requests = [
        RetrievalRequest(
            request_id=f"demo_{document_type.value}",
            ticker="GILD",
            company_name="Gilead Sciences, Inc.",
            company_aliases=["Gilead", "Gilead Sciences"],
            document_type=document_type,
            max_candidates=PIPELINE_CONFIG.default_max_candidates,
            minimum_text_chars=PIPELINE_CONFIG.min_usable_text_chars,
            minimum_relevance_score=PIPELINE_CONFIG.minimum_relevance_score,
            freshness_bonus_max=PIPELINE_CONFIG.freshness_bonus_max,
            freshness_rank_decay=PIPELINE_CONFIG.freshness_rank_decay,
            source_preferences=SourcePreferencePolicy(
                preferred_source_families=PIPELINE_CONFIG.default_source_preferences,
                prefer_structured_source=True,
                allow_secondary_sources=True,
                allow_unknown_sources=True,
                allow_mock_data=False,
            ),
            retrieval_notes=["Live demo request using real data sources."],
            is_mock_request=False,
        )
        for document_type in DocumentType
    ]

DEMO_RETRIEVAL_RESULTS: dict[DocumentType, RetrievalResult] = {}
for request in demo_requests:
    adapter_class = RETRIEVAL_ADAPTER_REGISTRY[request.document_type]
    adapter = adapter_class()
    DEMO_RETRIEVAL_RESULTS[request.document_type] = adapter.retrieve(request)

DEMO_RETRIEVAL_OUTPUT = {
    document_type.value: {
        "status": result.status.value,
        "selected_candidate_id": result.selected_candidate.candidate_id if result.selected_candidate else None,
        "selected_title": result.selected_candidate.title if result.selected_candidate else None,
        "selected_source_name": result.selected_candidate.source_name if result.selected_candidate else None,
        "selected_source_url": result.selected_candidate.source_url if result.selected_candidate else None,
        "issue_codes": [str(issue.error_code) for issue in result.issues],
        "ranking_notes": result.selection_decision.ranking_notes,
    }
    for document_type, result in DEMO_RETRIEVAL_RESULTS.items()
}

print(json.dumps(DEMO_RETRIEVAL_OUTPUT, indent=2))



## 32. Worker Specialization Design Notes

This stage begins after one disclosure has already been retrieved, validated, normalized, chunked, and made retrievable at the chunk level.

Prompt 6 design constraints:

- Only the disclosure-specific worker layer is being added here; the shared schemas, shared scoring rubric, and generic parsing logic remain intact.
- Each disclosure type gets one small worker subclass and one document-native master prompt block.
- Prompt assembly is modular so later live model clients can inspect or swap prompt sections without rewriting the worker pipeline.
- Worker judgments must stay evidence-backed and document-native; no brittle phrase dictionaries or undocumented source fields are introduced.
- Retrieval, preprocessing, and chunk evidence selection continue to come from the earlier notebook steps.
- Arbiter logic, master-node aggregation, UI payload assembly, and tests remain deferred.


In [ ]:
from typing import Callable, Mapping


class AnalysisDimension(str, Enum):
    """Document-agnostic dimensions used by the shared worker-analysis layer."""

    SENTIMENT = "sentiment"
    UNCERTAINTY = "uncertainty"
    FOGGING = "fogging"
    HEDGING = "hedging"
    PROMOTIONAL_TONE = "promotional_tone"
    CLARITY = "clarity"
    MATERIALITY = "materiality"
    COMPLETENESS = "completeness"


class EvidenceType(str, Enum):
    """Generic evidence categories supported across disclosure types."""

    DIRECT_QUOTE = "direct_quote"
    NUMERIC_DETAIL = "numeric_detail"
    STATUS_UPDATE = "status_update"
    RISK_DISCLOSURE = "risk_disclosure"
    FORWARD_LOOKING = "forward_looking"
    CONTEXTUAL_SUMMARY = "contextual_summary"


class EvidenceInterpretation(str, Enum):
    """Standardized interpretations attached to one piece of evidence."""

    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"
    UNCERTAINTY = "uncertainty"
    FOGGING = "fogging"
    HEDGING = "hedging"
    PROMOTIONAL = "promotional"
    CLARIFYING = "clarifying"
    MATERIAL = "material"


class AnalysisIssueSeverity(str, Enum):
    """Severity levels for shared analysis warnings and issues."""

    INFO = "info"
    WARNING = "warning"
    ERROR = "error"


ANALYSIS_DIMENSION_SCORE_RANGES: dict[AnalysisDimension, tuple[float, float]] = {
    AnalysisDimension.SENTIMENT: (SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX),
    AnalysisDimension.UNCERTAINTY: (0.0, 1.0),
    AnalysisDimension.FOGGING: (0.0, 1.0),
    AnalysisDimension.HEDGING: (0.0, 1.0),
    AnalysisDimension.PROMOTIONAL_TONE: (0.0, 1.0),
    AnalysisDimension.CLARITY: (0.0, 1.0),
    AnalysisDimension.MATERIALITY: (0.0, 1.0),
    AnalysisDimension.COMPLETENESS: (0.0, 1.0),
}


def get_analysis_dimension_score_range(dimension: AnalysisDimension) -> tuple[float, float]:
    """Return the valid score range for one analysis dimension."""
    return ANALYSIS_DIMENSION_SCORE_RANGES[dimension]


class AnalysisIssue(ContractModel):
    """Structured analysis-layer issue that does not assume a downstream provider."""

    issue_code: str
    message: str
    severity: AnalysisIssueSeverity = AnalysisIssueSeverity.WARNING
    dimension: AnalysisDimension | None = None
    source_chunk_id: str | None = None
    field_name: str | None = None
    recoverable: bool = True
    metadata: dict[str, Any] = Field(default_factory=dict)


class AnalysisWarning(AnalysisIssue):
    """Warning subtype used for recoverable analysis concerns."""

    severity: AnalysisIssueSeverity = AnalysisIssueSeverity.WARNING


class AnalysisFinding(ContractModel):
    """One normalized worker finding backed by cited evidence."""

    finding_id: str
    dimension: AnalysisDimension
    summary: str
    interpretation: EvidenceInterpretation | None = None
    evidence_ids: list[str] = Field(default_factory=list)
    source_chunk_ids: list[str] = Field(default_factory=list)
    score: float | None = None
    rationale: str | None = None
    is_material: bool = False

    @model_validator(mode="after")
    def validate_score_range(self) -> "AnalysisFinding":
        if self.score is None:
            return self
        minimum, maximum = get_analysis_dimension_score_range(self.dimension)
        if not minimum <= self.score <= maximum:
            raise ValueError(
                f"finding score for {self.dimension.value} must be between {minimum} and {maximum}."
            )
        return self


class AnalysisScore(ContractModel):
    """Normalized score object used across all workers before arbitration."""

    dimension: AnalysisDimension
    score: float | None = None
    label: str | None = None
    confidence_score: float | None = None
    rationale: str | None = None
    evidence_ids: list[str] = Field(default_factory=list)
    rubric_notes: list[str] = Field(default_factory=list)

    @model_validator(mode="after")
    def validate_score_range(self) -> "AnalysisScore":
        if self.score is None:
            return self
        minimum, maximum = get_analysis_dimension_score_range(self.dimension)
        if not minimum <= self.score <= maximum:
            raise ValueError(f"score for {self.dimension.value} must be between {minimum} and {maximum}.")
        return self

    @field_validator("confidence_score")
    @classmethod
    def validate_confidence_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence_score")


In [ ]:
class RubricAnchor(ContractModel):
    """Human-readable anchor for a score within one dimension."""

    score: float
    description: str


class RubricDimensionDefinition(ContractModel):
    """Explicit scoring definition for one analysis dimension."""

    dimension: AnalysisDimension
    minimum_score: float
    maximum_score: float
    objective: str
    evidence_expectation: str
    scoring_guidance: list[str] = Field(default_factory=list)
    anchors: list[RubricAnchor] = Field(default_factory=list)


class AnalysisRubric(ContractModel):
    """Inspectable shared rubric applied before worker specialization."""

    rubric_id: str
    version: str
    summary: str
    core_principles: list[str] = Field(default_factory=list)
    sentiment_labels: list[SentimentLabel] = Field(default_factory=list)
    dimension_definitions: list[RubricDimensionDefinition] = Field(default_factory=list)


DEFAULT_ANALYSIS_RUBRIC = AnalysisRubric(
    rubric_id="shared_worker_analysis_rubric",
    version="0.1.0",
    summary="Document-agnostic rubric for evidence-based biotech disclosure worker analysis.",
    core_principles=[
        "Anchor every score to cited evidence rather than unsupported impression.",
        "Separate factual directionality from rhetorical tone and presentation style.",
        "Treat uncertainty as unresolved dependency or missing specificity, not automatic negativity.",
        "Reward balance, specificity, and completeness while penalizing opacity and unsupported emphasis.",
        "Mark missing evidence as a warning instead of fabricating precision.",
    ],
    sentiment_labels=[
        SentimentLabel.POSITIVE,
        SentimentLabel.NEGATIVE,
        SentimentLabel.NEUTRAL,
        SentimentLabel.MIXED,
        SentimentLabel.INSUFFICIENT_EVIDENCE,
    ],
    dimension_definitions=[
        RubricDimensionDefinition(
            dimension=AnalysisDimension.SENTIMENT,
            minimum_score=-1.0,
            maximum_score=1.0,
            objective="Measure the directional implication of disclosed facts, not the emotional tone of wording alone.",
            evidence_expectation="Use outcome-bearing evidence and note opposing facts before assigning a label.",
            scoring_guidance=[
                "Positive sentiment requires support from favorable disclosed facts, not mere optimism.",
                "Negative sentiment should reflect disclosed setbacks, burdens, or downside facts rather than harsh wording alone.",
                "Mixed sentiment is appropriate when material positive and negative evidence coexist.",
                "Neutral sentiment is appropriate when evidence is descriptive without clear directional implication.",
            ],
            anchors=[
                RubricAnchor(score=-1.0, description="Strongly negative disclosed implications with clear evidence."),
                RubricAnchor(score=0.0, description="Directionally neutral or insufficiently directional evidence."),
                RubricAnchor(score=1.0, description="Strongly positive disclosed implications with clear evidence."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.UNCERTAINTY,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure how much the disclosure leaves key outcomes dependent on unresolved events, timing, or missing specifics.",
            evidence_expectation="Reference unresolved milestones, contingencies, missing dates, or acknowledged unknowns.",
            scoring_guidance=[
                "High uncertainty reflects unresolved dependencies, incomplete timing, or significant unknowns.",
                "Moderate uncertainty reflects partial specificity with meaningful open questions remaining.",
                "Low uncertainty reflects concrete timing, scope, and next steps.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low uncertainty; facts and next steps are specific."),
                RubricAnchor(score=0.5, description="Moderate uncertainty; some specifics are present but gaps remain."),
                RubricAnchor(score=1.0, description="High uncertainty; key outcomes remain unresolved or poorly specified."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.FOGGING,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure opacity, evasiveness, or lack of operational specificity in the disclosure.",
            evidence_expectation="Cite vague summaries, missing detail, or unsupported abstraction; do not treat concise clarity as fogging.",
            scoring_guidance=[
                "High fogging requires evidence that material points are obscured or left vague.",
                "Low fogging reflects concrete details, inspectable numbers, or precise procedural descriptions.",
                "Do not treat uncertainty disclosures by themselves as fogging if they are explicit and specific.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low fogging; the disclosure is explicit and inspectable."),
                RubricAnchor(score=0.5, description="Moderate fogging; some material points are underspecified."),
                RubricAnchor(score=1.0, description="High fogging; material detail is obscured or evasive."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.HEDGING,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure how strongly claims are qualified, conditional, or deferred.",
            evidence_expectation="Tie the score to conditionality, caveats, or claim-softening language patterns without assuming deception.",
            scoring_guidance=[
                "High hedging reflects many qualified claims or repeated conditional framing.",
                "Moderate hedging reflects normal caution around forward-looking statements.",
                "Low hedging reflects direct factual reporting with limited qualification.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low hedging; mostly direct factual reporting."),
                RubricAnchor(score=0.5, description="Moderate hedging; caution and conditional framing are noticeable."),
                RubricAnchor(score=1.0, description="High hedging; claims are heavily qualified or deferred."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.PROMOTIONAL_TONE,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure imbalance between claims and support, not mere polish or competent presentation.",
            evidence_expectation="Cite disproportionate emphasis, unsupported upside framing, or selective omission of balancing context.",
            scoring_guidance=[
                "High promotional tone requires evidence of overreach beyond the disclosed support.",
                "Low promotional tone reflects balanced framing and alignment between claims and cited facts.",
                "Polished language alone is not promotional if the document stays balanced and specific.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low promotional tone; framing stays balanced and supported."),
                RubricAnchor(score=0.5, description="Moderate promotional tone; emphasis is noticeable but partly supported."),
                RubricAnchor(score=1.0, description="High promotional tone; claims materially outpace disclosed support."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.CLARITY,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure how clearly the disclosure explains what happened, why it matters, and what comes next.",
            evidence_expectation="Reference organization, specificity, and ease of tracing material points.",
            scoring_guidance=[
                "High clarity reflects coherent structure and concrete explanations.",
                "Low clarity reflects fragmented or difficult-to-follow material points.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low clarity; material meaning is difficult to trace."),
                RubricAnchor(score=0.5, description="Moderate clarity; key facts are present but somewhat diffuse."),
                RubricAnchor(score=1.0, description="High clarity; material facts and next steps are explicit."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.MATERIALITY,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure how consequential the cited facts appear within the document context.",
            evidence_expectation="Use disclosed financial, regulatory, clinical, or operational consequences to support the score.",
            scoring_guidance=[
                "High materiality reflects consequences that could meaningfully affect the issuer, program, or investors.",
                "Low materiality reflects routine or limited-scope information.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low materiality; limited apparent consequence."),
                RubricAnchor(score=0.5, description="Moderate materiality; meaningful but not central consequences."),
                RubricAnchor(score=1.0, description="High materiality; clearly consequential disclosed facts."),
            ],
        ),
        RubricDimensionDefinition(
            dimension=AnalysisDimension.COMPLETENESS,
            minimum_score=0.0,
            maximum_score=1.0,
            objective="Measure whether the disclosure provides enough context, detail, and balance to evaluate the main point.",
            evidence_expectation="Reference presence or absence of timing, scope, tradeoffs, and balancing detail.",
            scoring_guidance=[
                "High completeness reflects balanced context, scope, and next steps.",
                "Low completeness reflects meaningful omissions that limit interpretation.",
            ],
            anchors=[
                RubricAnchor(score=0.0, description="Low completeness; material context is missing."),
                RubricAnchor(score=0.5, description="Moderate completeness; useful context exists but gaps remain."),
                RubricAnchor(score=1.0, description="High completeness; the document supplies enough context to evaluate the point."),
            ],
        ),
    ],
)


ANALYSIS_RUBRIC_BY_DIMENSION = {
    definition.dimension: definition for definition in DEFAULT_ANALYSIS_RUBRIC.dimension_definitions
}


def coerce_float(value: Any) -> float | None:
    """Safely coerce a raw value to float without raising."""
    if value is None or value == "":
        return None
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        normalized = value.strip()
        if not normalized:
            return None
        try:
            return float(normalized)
        except ValueError:
            return None
    return None


def clamp_score_value(
    value: float | None,
    *,
    minimum: float,
    maximum: float,
    warning_code: str,
    dimension: AnalysisDimension,
) -> tuple[float | None, AnalysisWarning | None]:
    """Clamp a raw score into range and emit a structured warning when adjustment is needed."""
    if value is None:
        return None, None
    if minimum <= value <= maximum:
        return value, None
    clamped = max(minimum, min(maximum, value))
    warning = AnalysisWarning(
        issue_code=warning_code,
        message=f"{dimension.value} score {value} was outside [{minimum}, {maximum}] and was clamped.",
        dimension=dimension,
        field_name="score",
        metadata={"raw_score": value, "clamped_score": clamped},
    )
    return clamped, warning


def normalize_sentiment_label(value: Any, score: float | None = None) -> tuple[SentimentLabel, AnalysisWarning | None]:
    """Normalize raw sentiment labels and optionally infer a label from a valid score."""
    if isinstance(value, SentimentLabel):
        return value, None
    if isinstance(value, str):
        normalized = value.strip().lower()
        mapping = {
            "positive": SentimentLabel.POSITIVE,
            "negative": SentimentLabel.NEGATIVE,
            "neutral": SentimentLabel.NEUTRAL,
            "mixed": SentimentLabel.MIXED,
            "insufficient_evidence": SentimentLabel.INSUFFICIENT_EVIDENCE,
            "insufficient evidence": SentimentLabel.INSUFFICIENT_EVIDENCE,
        }
        if normalized in mapping:
            return mapping[normalized], None
    if score is None:
        warning = AnalysisWarning(
            issue_code="sentiment_label_missing",
            message="Sentiment label was missing or invalid and no score was available to infer one.",
            dimension=AnalysisDimension.SENTIMENT,
            field_name="label",
        )
        return SentimentLabel.INSUFFICIENT_EVIDENCE, warning

    if score >= 0.25:
        inferred = SentimentLabel.POSITIVE
    elif score <= -0.25:
        inferred = SentimentLabel.NEGATIVE
    elif abs(score) <= 0.1:
        inferred = SentimentLabel.NEUTRAL
    else:
        inferred = SentimentLabel.MIXED

    warning = AnalysisWarning(
        issue_code="sentiment_label_inferred",
        message="Sentiment label was inferred from the normalized sentiment score using explicit thresholds.",
        dimension=AnalysisDimension.SENTIMENT,
        field_name="label",
        metadata={"inferred_label": inferred.value, "score": score},
    )
    return inferred, warning


def normalize_dimension_label(dimension: AnalysisDimension, score: float | None) -> str | None:
    """Map a normalized score to a small explicit label set."""
    if score is None:
        return None
    if dimension == AnalysisDimension.SENTIMENT:
        sentiment_label, _ = normalize_sentiment_label(None, score)
        return sentiment_label.value
    if score < 0.34:
        return "low"
    if score < 0.67:
        return "moderate"
    return "high"


def build_standardized_analysis_score(
    *,
    dimension: AnalysisDimension,
    raw_score: Any,
    raw_label: Any,
    raw_confidence: Any,
    rationale: Any = None,
    evidence_ids: Sequence[str] | None = None,
    rubric_notes: Sequence[str] | None = None,
) -> tuple[AnalysisScore, list[AnalysisWarning]]:
    """Map raw analysis content into one validated score object plus warnings."""
    warnings: list[AnalysisWarning] = []
    minimum, maximum = get_analysis_dimension_score_range(dimension)

    score_value = coerce_float(raw_score)
    clamped_score, clamp_warning = clamp_score_value(
        score_value,
        minimum=minimum,
        maximum=maximum,
        warning_code="score_out_of_range",
        dimension=dimension,
    )
    if clamp_warning is not None:
        warnings.append(clamp_warning)

    confidence_value = coerce_float(raw_confidence)
    clamped_confidence, confidence_warning = clamp_score_value(
        confidence_value,
        minimum=0.0,
        maximum=1.0,
        warning_code="confidence_out_of_range",
        dimension=dimension,
    )
    if confidence_warning is not None:
        warnings.append(confidence_warning)

    if dimension == AnalysisDimension.SENTIMENT:
        sentiment_label, label_warning = normalize_sentiment_label(raw_label, clamped_score)
        if label_warning is not None:
            warnings.append(label_warning)
        normalized_label = sentiment_label.value
    else:
        normalized_label = normalize_dimension_label(dimension, clamped_score)

    score = AnalysisScore(
        dimension=dimension,
        score=clamped_score,
        label=normalized_label,
        confidence_score=clamped_confidence,
        rationale=str(rationale).strip() if isinstance(rationale, str) and rationale.strip() else None,
        evidence_ids=list(evidence_ids or []),
        rubric_notes=list(rubric_notes or ANALYSIS_RUBRIC_BY_DIMENSION[dimension].scoring_guidance),
    )
    return score, warnings


In [ ]:
class SectionContextSummary(ContractModel):
    """Compact section metadata supplied to the generic worker-analysis input."""

    section_id: str
    title: str
    level: int
    section_kind: SectionKind
    char_count: int
    word_count: int


class ChunkEvidenceBundle(ContractModel):
    """Bundle of chunk text plus local context used as candidate evidence for analysis."""

    bundle_id: str
    document_id: str
    chunk_id: str
    section_id: str | None = None
    section_title: str | None = None
    retrieval_rank: int | None = None
    adjusted_score: float | None = None
    similarity_score: float | None = None
    graph_bonus: float | None = None
    evidence_type: EvidenceType = EvidenceType.CONTEXTUAL_SUMMARY
    primary_text: str
    expanded_context_text: str | None = None
    local_context_summary: str | None = None
    notes: list[str] = Field(default_factory=list)


EvidenceSnippet.model_rebuild()


def infer_evidence_type(chunk: DocumentChunk) -> EvidenceType:
    """Infer a generic evidence type using simple structural cues only."""
    if chunk.section_kind == SectionKind.TABLE or any(char.isdigit() for char in chunk.text):
        return EvidenceType.NUMERIC_DETAIL
    if chunk.section_kind == SectionKind.TRANSCRIPT:
        return EvidenceType.DIRECT_QUOTE
    if chunk.section_kind == SectionKind.LIST:
        return EvidenceType.STATUS_UPDATE
    return EvidenceType.CONTEXTUAL_SUMMARY


def build_section_context_summaries(
    processed_document: ProcessedDocument,
    *,
    max_sections: int = 8,
) -> list[SectionContextSummary]:
    """Build compact section summaries for the shared analysis packet."""
    return [
        SectionContextSummary(
            section_id=section.section_id,
            title=section.title,
            level=section.level,
            section_kind=section.section_kind,
            char_count=section.char_count,
            word_count=section.word_count,
        )
        for section in processed_document.sections[:max_sections]
    ]


def build_chunk_evidence_bundles(
    processed_document: ProcessedDocument,
    chunk_retrieval_result: ChunkRetrievalResult | None = None,
    *,
    max_bundles: int = 4,
) -> tuple[list[ChunkEvidenceBundle], list[AnalysisWarning]]:
    """Convert retrieved chunks into reusable evidence bundles for worker analysis."""
    warnings: list[AnalysisWarning] = []
    chunk_lookup = {chunk.chunk_id: chunk for chunk in processed_document.chunks}
    hit_lookup = {hit.chunk_id: hit for hit in (chunk_retrieval_result.hits if chunk_retrieval_result else [])}

    selected_chunk_ids = [hit.chunk_id for hit in (chunk_retrieval_result.hits if chunk_retrieval_result else [])]
    if not selected_chunk_ids:
        selected_chunk_ids = [chunk.chunk_id for chunk in processed_document.chunks[:max_bundles]]
        warnings.append(
            AnalysisWarning(
                issue_code="insufficient_chunk_evidence",
                message="No ranked chunk hits were available, so the analysis packet used the first processed chunks as fallback evidence.",
                metadata={"fallback_chunk_count": len(selected_chunk_ids)},
            )
        )

    bundles: list[ChunkEvidenceBundle] = []
    for rank, chunk_id in enumerate(selected_chunk_ids[:max_bundles], start=1):
        chunk = chunk_lookup.get(chunk_id)
        if chunk is None:
            warnings.append(
                AnalysisWarning(
                    issue_code="missing_chunk_reference",
                    message=f"Chunk id {chunk_id} was referenced for analysis but was not found in the processed document.",
                    source_chunk_id=chunk_id,
                )
            )
            continue

        hit = hit_lookup.get(chunk_id)
        expanded_context_text = hit.expanded_context_preview if hit is not None else None
        bundles.append(
            ChunkEvidenceBundle(
                bundle_id=f"{processed_document.document.document_id}::bundle_{rank:02d}",
                document_id=processed_document.document.document_id,
                chunk_id=chunk.chunk_id,
                section_id=chunk.parent_section_id,
                section_title=chunk.parent_section_title,
                retrieval_rank=hit.rank if hit is not None else rank,
                adjusted_score=hit.adjusted_score if hit is not None else None,
                similarity_score=hit.similarity_score if hit is not None else None,
                graph_bonus=hit.graph_bonus if hit is not None else None,
                evidence_type=infer_evidence_type(chunk),
                primary_text=chunk.text,
                expanded_context_text=expanded_context_text,
                local_context_summary=chunk.local_context_summary,
                notes=[note.message for note in chunk.notes],
            )
        )

    if len(bundles) < 2:
        warnings.append(
            AnalysisWarning(
                issue_code="limited_evidence_bundle_count",
                message="Fewer than two evidence bundles were available for shared worker analysis.",
                metadata={"bundle_count": len(bundles)},
            )
        )

    return bundles, warnings


def build_evidence_snippet_from_bundle(
    bundle: ChunkEvidenceBundle,
    *,
    source_url: str | None = None,
    rationale: str,
    supported_dimensions: Sequence[AnalysisDimension] | None = None,
    interpretation: EvidenceInterpretation | None = None,
    snippet_text: str | None = None,
) -> EvidenceSnippet:
    """Build a normalized evidence snippet from one chunk evidence bundle."""
    text = snippet_text or make_text_excerpt(bundle.primary_text, 240)
    return EvidenceSnippet(
        evidence_id=f"{bundle.bundle_id}::snippet",
        document_id=bundle.document_id,
        source_url=source_url,
        source_chunk_id=bundle.chunk_id,
        source_section_id=bundle.section_id,
        section_title=bundle.section_title,
        evidence_type=bundle.evidence_type,
        supported_dimensions=list(supported_dimensions or []),
        interpretation=interpretation,
        snippet_text=text,
        rationale=rationale,
        metadata={
            "bundle_id": bundle.bundle_id,
            "retrieval_rank": bundle.retrieval_rank,
            "adjusted_score": bundle.adjusted_score,
            "similarity_score": bundle.similarity_score,
            "graph_bonus": bundle.graph_bonus,
        },
    )


In [ ]:
class WorkerAnalysisInput(ContractModel):
    """Shared analysis packet assembled before any disclosure-specific prompting exists."""

    run_id: str
    ticker: str
    worker_name: str
    document_type: DocumentType
    document_metadata: DocumentMetadata
    document_title: str | None = None
    document_text: str
    document_text_excerpt: str
    section_context: list[SectionContextSummary] = Field(default_factory=list)
    top_chunk_bundles: list[ChunkEvidenceBundle] = Field(default_factory=list)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)
    processing_notes: list[ProcessingNote] = Field(default_factory=list)
    analysis_instructions: str
    expected_output_schema: dict[str, Any] = Field(default_factory=dict)
    retrieval_query: str | None = None
    rubric_id: str
    rubric_version: str
    is_mock_data: bool = False
    metadata: dict[str, Any] = Field(default_factory=dict)


DEFAULT_GENERIC_ANALYSIS_INSTRUCTIONS = """
You are performing shared worker analysis for a biotech disclosure.
Use only the supplied document text, chunk evidence bundles, and provenance.
Produce evidence-backed conclusions.
Keep sentiment separate from uncertainty, fogging, hedging, and promotional tone.
Distinguish specific disclosed facts from rhetorical framing.
If evidence is incomplete, return warnings instead of guessing.
"""


def describe_worker_analysis_output_schema(rubric: AnalysisRubric = DEFAULT_ANALYSIS_RUBRIC) -> dict[str, Any]:
    """Describe the expected normalized worker-analysis output shape for later model clients."""
    return {
        "summary": "Short evidence-backed narrative summary.",
        "confidence_score": "Float in [0.0, 1.0].",
        "scores": [
            {
                "dimension": definition.dimension.value,
                "score_range": [definition.minimum_score, definition.maximum_score],
                "label": "Required for sentiment; optional normalized label for other dimensions.",
                "confidence_score": "Optional float in [0.0, 1.0].",
                "rationale": definition.objective,
            }
            for definition in rubric.dimension_definitions
        ],
        "findings": [
            {
                "dimension": "analysis dimension",
                "summary": "Finding summary tied to evidence.",
                "interpretation": "positive | negative | neutral | uncertainty | fogging | hedging | promotional | clarifying | material",
                "evidence_refs": ["bundle_id or chunk_id references"],
            }
        ],
        "evidence_refs": [
            {
                "chunk_id": "source chunk id",
                "bundle_id": "optional bundle id",
                "dimension": "supported dimension",
                "interpretation": "how the evidence should be read",
                "why_it_matters": "why the citation matters",
            }
        ],
        "reasoning_notes": ["Optional compact notes describing how the evidence was interpreted."],
    }


def build_worker_analysis_input(
    worker_input: WorkerInput,
    processed_document: ProcessedDocument,
    chunk_retrieval_result: ChunkRetrievalResult | None = None,
    *,
    worker_name: str,
    analysis_instructions: str | None = None,
    rubric: AnalysisRubric = DEFAULT_ANALYSIS_RUBRIC,
    max_document_chars: int = 3500,
    max_evidence_bundles: int = 4,
) -> tuple[WorkerAnalysisInput, list[AnalysisWarning]]:
    """Assemble the reusable shared analysis packet for one worker invocation."""
    metadata = processed_document.document.metadata or DocumentMetadata(
        document_id=processed_document.document.document_id,
        ticker=processed_document.document.ticker,
        document_type=processed_document.document.document_type,
        title=processed_document.document.title,
        source_name=processed_document.document.source_name,
        source_url=processed_document.document.source_url,
        is_mock_data=processed_document.document.is_mock_data,
    )
    document_text = processed_document.cleaned_text[:max_document_chars].strip()
    if not document_text:
        document_text = processed_document.document.raw_text[:max_document_chars].strip()

    evidence_bundles, bundle_warnings = build_chunk_evidence_bundles(
        processed_document,
        chunk_retrieval_result,
        max_bundles=max_evidence_bundles,
    )
    if len(processed_document.cleaned_text) > max_document_chars:
        bundle_warnings.append(
            AnalysisWarning(
                issue_code="document_text_truncated",
                message="The document text in the shared analysis packet was truncated for notebook-safe prompt size.",
                metadata={"max_document_chars": max_document_chars, "original_chars": len(processed_document.cleaned_text)},
            )
        )

    analysis_input = WorkerAnalysisInput(
        run_id=worker_input.run_id,
        ticker=worker_input.ticker,
        worker_name=worker_name,
        document_type=worker_input.document_type,
        document_metadata=metadata,
        document_title=processed_document.document.title,
        document_text=document_text,
        document_text_excerpt=make_text_excerpt(document_text, 320),
        section_context=build_section_context_summaries(processed_document),
        top_chunk_bundles=evidence_bundles,
        provenance=list(processed_document.document.provenance),
        processing_notes=list(processed_document.processing_notes),
        analysis_instructions=(analysis_instructions or DEFAULT_GENERIC_ANALYSIS_INSTRUCTIONS).strip(),
        expected_output_schema=describe_worker_analysis_output_schema(rubric),
        retrieval_query=chunk_retrieval_result.query_text if chunk_retrieval_result is not None else None,
        rubric_id=rubric.rubric_id,
        rubric_version=rubric.version,
        is_mock_data=processed_document.document.is_mock_data,
        metadata={
            "chunk_count": processed_document.chunk_count,
            "section_count": processed_document.section_count,
            "used_fallback_chunking": processed_document.used_fallback_chunking,
        },
    )
    return analysis_input, bundle_warnings


In [ ]:
class WorkerReasoningTrace(ContractModel):
    """Trace of the shared worker-analysis path before future specialization."""

    analysis_client_name: str
    model_name: str
    used_mock_client: bool = False
    selected_chunk_ids: list[str] = Field(default_factory=list)
    selected_section_ids: list[str] = Field(default_factory=list)
    reasoning_notes: list[str] = Field(default_factory=list)
    unresolved_items: list[str] = Field(default_factory=list)
    prompt_context_preview: str | None = None
    raw_response_present: bool = False


class WorkerAnalysisOutput(ContractModel):
    """Internal shared analysis result returned before mapping to WorkerOutput."""

    run_id: str
    ticker: str
    worker_name: str
    document_type: DocumentType
    status: ProcessingStatus = ProcessingStatus.PENDING
    summary: str | None = None
    confidence_score: float | None = None
    scores: list[AnalysisScore] = Field(default_factory=list)
    findings: list[AnalysisFinding] = Field(default_factory=list)
    evidence: list[EvidenceSnippet] = Field(default_factory=list)
    warnings: list[AnalysisWarning] = Field(default_factory=list)
    issues: list[AnalysisIssue] = Field(default_factory=list)
    reasoning_trace: WorkerReasoningTrace | None = None
    raw_output: dict[str, Any] = Field(default_factory=dict)
    is_mock_analysis: bool = False
    generated_at: datetime = Field(default_factory=now_utc)

    @field_validator("confidence_score")
    @classmethod
    def validate_confidence_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence_score")


def make_analysis_warning(
    issue_code: str,
    message: str,
    *,
    dimension: AnalysisDimension | None = None,
    source_chunk_id: str | None = None,
    field_name: str | None = None,
    metadata: dict[str, Any] | None = None,
) -> AnalysisWarning:
    """Build a normalized analysis warning."""
    return AnalysisWarning(
        issue_code=issue_code,
        message=message,
        dimension=dimension,
        source_chunk_id=source_chunk_id,
        field_name=field_name,
        metadata=metadata or {},
    )


def make_analysis_issue(
    issue_code: str,
    message: str,
    *,
    severity: AnalysisIssueSeverity = AnalysisIssueSeverity.ERROR,
    dimension: AnalysisDimension | None = None,
    source_chunk_id: str | None = None,
    field_name: str | None = None,
    recoverable: bool = True,
    metadata: dict[str, Any] | None = None,
) -> AnalysisIssue:
    """Build a normalized analysis issue."""
    return AnalysisIssue(
        issue_code=issue_code,
        message=message,
        severity=severity,
        dimension=dimension,
        source_chunk_id=source_chunk_id,
        field_name=field_name,
        recoverable=recoverable,
        metadata=metadata or {},
    )


def coerce_analysis_payload(raw_output: Any) -> tuple[dict[str, Any], list[AnalysisWarning]]:
    """Coerce raw client output into a dictionary when possible."""
    warnings: list[AnalysisWarning] = []
    if raw_output is None:
        warnings.append(make_analysis_warning("empty_analysis_output", "Analysis client returned no output."))
        return {}, warnings
    if isinstance(raw_output, dict):
        return raw_output, warnings
    if isinstance(raw_output, str):
        stripped = raw_output.strip()
        if not stripped:
            warnings.append(make_analysis_warning("empty_analysis_output", "Analysis client returned empty text."))
            return {}, warnings
        try:
            payload = json.loads(stripped)
        except json.JSONDecodeError:
            warnings.append(
                make_analysis_warning(
                    "analysis_output_not_json",
                    "Analysis output was text instead of JSON; the parser will continue with an empty payload.",
                )
            )
            return {}, warnings
        if isinstance(payload, dict):
            return payload, warnings
        warnings.append(
            make_analysis_warning(
                "analysis_output_wrong_shape",
                "Analysis JSON payload was not a dictionary.",
            )
        )
        return {}, warnings
    warnings.append(
        make_analysis_warning(
            "analysis_output_unsupported_type",
            f"Analysis output type {type(raw_output).__name__} is unsupported by the shared parser.",
        )
    )
    return {}, warnings


def parse_worker_analysis_output(
    raw_output: Any,
    analysis_input: WorkerAnalysisInput,
    *,
    client_name: str,
    model_name: str,
    is_mock: bool = False,
    client_warnings: Sequence[AnalysisWarning] | None = None,
    prompt_context: dict[str, Any] | None = None,
) -> WorkerAnalysisOutput:
    """Defensively parse raw worker analysis output into a normalized internal result."""
    payload, parse_warnings = coerce_analysis_payload(raw_output)
    warnings: list[AnalysisWarning] = list(client_warnings or []) + parse_warnings
    issues: list[AnalysisIssue] = []

    confidence_score = coerce_float(payload.get("confidence_score")) if payload else None
    if confidence_score is not None:
        confidence_score, confidence_warning = clamp_score_value(
            confidence_score,
            minimum=0.0,
            maximum=1.0,
            warning_code="confidence_out_of_range",
            dimension=AnalysisDimension.CLARITY,
        )
        if confidence_warning is not None:
            warnings.append(confidence_warning)

    normalized_scores, score_warnings = normalize_analysis_scores(payload, DEFAULT_ANALYSIS_RUBRIC)
    warnings.extend(score_warnings)

    evidence, evidence_warnings = map_raw_evidence_references(payload.get("evidence_refs"), analysis_input)
    warnings.extend(evidence_warnings)

    findings, finding_warnings = extract_structured_findings(
        payload.get("findings"),
        analysis_input=analysis_input,
        normalized_scores=normalized_scores,
        evidence=evidence,
    )
    warnings.extend(finding_warnings)

    summary = payload.get("summary")
    if isinstance(summary, str):
        summary = summary.strip() or None
    else:
        summary = None
    if summary is None and findings:
        summary = findings[0].summary
        warnings.append(
            make_analysis_warning(
                "summary_missing",
                "Analysis summary was missing, so the first finding summary was used as a fallback.",
                field_name="summary",
            )
        )
    elif summary is None:
        warnings.append(make_analysis_warning("summary_missing", "Analysis summary was missing.", field_name="summary"))

    reasoning_notes = payload.get("reasoning_notes")
    if not isinstance(reasoning_notes, list):
        reasoning_notes = []
    reasoning_notes = [str(note).strip() for note in reasoning_notes if str(note).strip()]

    unresolved_items = payload.get("unresolved_items")
    if not isinstance(unresolved_items, list):
        unresolved_items = []
    unresolved_items = [str(item).strip() for item in unresolved_items if str(item).strip()]

    if not reasoning_notes:
        warnings.append(make_analysis_warning("empty_reasoning_notes", "Analysis output did not include reasoning notes."))

    if not evidence:
        warnings.append(
            make_analysis_warning(
                "missing_evidence_references",
                "Analysis output did not contain valid evidence references.",
            )
        )

    if payload and not normalized_scores:
        issues.append(
            make_analysis_issue(
                "score_normalization_failed",
                "No normalized scores could be produced from the raw analysis payload.",
            )
        )

    prompt_context_preview = None
    if prompt_context is not None:
        prompt_context_preview = make_text_excerpt(json.dumps(prompt_context, default=str), 320)

    reasoning_trace = WorkerReasoningTrace(
        analysis_client_name=client_name,
        model_name=model_name,
        used_mock_client=is_mock,
        selected_chunk_ids=[bundle.chunk_id for bundle in analysis_input.top_chunk_bundles],
        selected_section_ids=[bundle.section_id for bundle in analysis_input.top_chunk_bundles if bundle.section_id],
        reasoning_notes=reasoning_notes,
        unresolved_items=unresolved_items,
        prompt_context_preview=prompt_context_preview,
        raw_response_present=bool(payload),
    )

    status = ProcessingStatus.SUCCESS
    if issues:
        status = ProcessingStatus.ANALYSIS_FAILED
    elif warnings:
        status = ProcessingStatus.PARTIAL
    if not payload:
        status = ProcessingStatus.ANALYSIS_FAILED

    return WorkerAnalysisOutput(
        run_id=analysis_input.run_id,
        ticker=analysis_input.ticker,
        worker_name=analysis_input.worker_name,
        document_type=analysis_input.document_type,
        status=status,
        summary=summary,
        confidence_score=confidence_score,
        scores=normalized_scores,
        findings=findings,
        evidence=evidence,
        warnings=warnings,
        issues=issues,
        reasoning_trace=reasoning_trace,
        raw_output=payload,
        is_mock_analysis=is_mock,
    )


In [ ]:
class AnalysisClientRequest(ContractModel):
    """Normalized request sent to a shared analysis client."""

    worker_name: str
    document_type: DocumentType
    analysis_input: WorkerAnalysisInput
    prompt_context: dict[str, Any] = Field(default_factory=dict)
    prompt_text: str | None = None
    rubric: AnalysisRubric
    metadata: dict[str, Any] = Field(default_factory=dict)


class AnalysisClientResponse(ContractModel):
    """Provider-agnostic response returned by a shared analysis client."""

    client_name: str
    model_name: str
    status: ProcessingStatus
    raw_output: dict[str, Any] | None = None
    raw_text: str | None = None
    warnings: list[AnalysisWarning] = Field(default_factory=list)
    is_mock: bool = False
    generated_at: datetime = Field(default_factory=now_utc)


class BaseAnalysisClient(ABC):
    """Base interface for local, mock, or wrapped analysis clients."""

    client_name: ClassVar[str] = "base_analysis_client"
    model_name: ClassVar[str] = "configurable_placeholder"

    @abstractmethod
    def run_analysis(self, request: AnalysisClientRequest) -> AnalysisClientResponse:
        """Run one analysis request and return a normalized response object."""


class MockAnalysisClient(BaseAnalysisClient):
    """Deterministic placeholder client that keeps the notebook runnable."""

    client_name = "mock_analysis_client"

    def __init__(self, model_name: str = "mock_structured_worker_v1") -> None:
        self.model_name = model_name

    def run_analysis(self, request: AnalysisClientRequest) -> AnalysisClientResponse:
        bundles = request.analysis_input.top_chunk_bundles[:3]
        evidence_refs: list[dict[str, Any]] = []
        findings: list[dict[str, Any]] = []
        for index, bundle in enumerate(bundles, start=1):
            if index == 1:
                dimension = AnalysisDimension.MATERIALITY
                interpretation = EvidenceInterpretation.MATERIAL
            elif index == 2:
                dimension = AnalysisDimension.UNCERTAINTY
                interpretation = EvidenceInterpretation.UNCERTAINTY
            else:
                dimension = AnalysisDimension.CLARITY
                interpretation = EvidenceInterpretation.CLARIFYING

            evidence_refs.append(
                {
                    "chunk_id": bundle.chunk_id,
                    "bundle_id": bundle.bundle_id,
                    "dimension": dimension.value,
                    "interpretation": interpretation.value,
                    "why_it_matters": f"Mock citation built from ranked evidence bundle {index}.",
                }
            )
            findings.append(
                {
                    "dimension": dimension.value,
                    "summary": f"Mock finding {index}: {make_text_excerpt(bundle.primary_text, 120)}",
                    "interpretation": interpretation.value,
                    "evidence_refs": [bundle.bundle_id],
                    "rationale": "Placeholder finding derived from the top-ranked evidence bundle.",
                    "is_material": index == 1,
                }
            )

        raw_output = {
            "summary": f"Mock shared analysis scaffold for {request.analysis_input.document_title or request.document_type.value}.",
            "confidence_score": 0.35,
            "scores": [
                {
                    "dimension": AnalysisDimension.SENTIMENT.value,
                    "score": 0.0,
                    "label": SentimentLabel.NEUTRAL.value,
                    "confidence_score": 0.35,
                    "rationale": "Mock client does not infer directional sentiment beyond supplied evidence bundles.",
                },
                {
                    "dimension": AnalysisDimension.UNCERTAINTY.value,
                    "score": min(0.75, 0.25 + (0.1 * len(bundles))),
                    "confidence_score": 0.35,
                    "rationale": "Placeholder uncertainty score based on evidence-bundle coverage rather than substantive interpretation.",
                },
                {
                    "dimension": AnalysisDimension.FOGGING.value,
                    "score": 0.25,
                    "confidence_score": 0.3,
                    "rationale": "Mock placeholder score; no real fogging judgment was performed.",
                },
                {
                    "dimension": AnalysisDimension.HEDGING.value,
                    "score": 0.3,
                    "confidence_score": 0.3,
                    "rationale": "Mock placeholder score; no real hedging judgment was performed.",
                },
                {
                    "dimension": AnalysisDimension.PROMOTIONAL_TONE.value,
                    "score": 0.2,
                    "confidence_score": 0.3,
                    "rationale": "Mock placeholder score; polished wording is not evaluated here.",
                },
                {
                    "dimension": AnalysisDimension.CLARITY.value,
                    "score": 0.6 if bundles else 0.4,
                    "confidence_score": 0.35,
                    "rationale": "Mock score based on whether chunks and local summaries were available.",
                },
                {
                    "dimension": AnalysisDimension.MATERIALITY.value,
                    "score": 0.65 if bundles else 0.3,
                    "confidence_score": 0.35,
                    "rationale": "Mock score based on the presence of retrieved evidence bundles only.",
                },
                {
                    "dimension": AnalysisDimension.COMPLETENESS.value,
                    "score": 0.5 if len(bundles) >= 2 else 0.35,
                    "confidence_score": 0.35,
                    "rationale": "Mock completeness score reflects evidence bundle count, not substantive adequacy.",
                },
            ],
            "findings": findings,
            "evidence_refs": evidence_refs,
            "reasoning_notes": [
                "Mock analysis client returned a deterministic placeholder payload.",
                "Scores and findings should not be treated as reliable disclosure judgments.",
            ],
            "unresolved_items": [
                "Worker-specific prompts and provider-backed reasoning are not implemented in this step.",
            ],
        }
        warnings = [
            make_analysis_warning(
                "mock_analysis_used",
                "Mock analysis client was used because no live worker-analysis model is configured in this notebook step.",
            )
        ]
        return AnalysisClientResponse(
            client_name=self.client_name,
            model_name=self.model_name,
            status=ProcessingStatus.SUCCESS,
            raw_output=raw_output,
            warnings=warnings,
            is_mock=True,
        )


class CallableAnalysisClient(BaseAnalysisClient):
    """Wrap a callable or `.invoke(...)` object without assuming any provider-specific payload shape."""

    client_name = "callable_analysis_client"

    def __init__(
        self,
        invoke_target: Callable[[dict[str, Any]], Any] | Any,
        *,
        client_name: str = "callable_analysis_client",
        model_name: str = "callable_placeholder",
    ) -> None:
        self.invoke_target = invoke_target
        self.client_name = client_name
        self.model_name = model_name

    def run_analysis(self, request: AnalysisClientRequest) -> AnalysisClientResponse:
        request_payload = request.model_dump(mode="json")
        try:
            if hasattr(self.invoke_target, "invoke"):
                raw_result = self.invoke_target.invoke(request_payload)
            elif callable(self.invoke_target):
                raw_result = self.invoke_target(request_payload)
            else:
                raise TypeError("invoke_target must be callable or expose an invoke method.")
        except Exception as exc:
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.ANALYSIS_FAILED,
                raw_text=str(exc),
                warnings=[
                    make_analysis_warning(
                        "analysis_client_invocation_failed",
                        f"Callable analysis client failed: {exc}",
                    )
                ],
                is_mock=False,
            )

        if isinstance(raw_result, AnalysisClientResponse):
            return raw_result
        if isinstance(raw_result, dict):
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.SUCCESS,
                raw_output=raw_result,
            )
        return AnalysisClientResponse(
            client_name=self.client_name,
            model_name=self.model_name,
            status=ProcessingStatus.SUCCESS,
            raw_text=str(raw_result),
        )


def build_analysis_client(
    config: PipelineConfig,
    preferred_client: BaseAnalysisClient | None = None,
) -> BaseAnalysisClient:
    """Build the default shared analysis client for the current notebook configuration."""
    if preferred_client is not None:
        return preferred_client
    mode = getattr(config, "analysis_client_mode", "mock")
    if mode == "mock" or not config.enable_analysis or config.enable_mock_analysis:
        return MockAnalysisClient(model_name=config.worker_model_name or "mock_structured_worker_v1")
    if mode == "moonshot":
        return MoonshotAnalysisClient(model_name=config.worker_model_name or "moonshot-v1-128k")
    if mode == "cli":
        return ClaudeCliAnalysisClient(model_name=config.worker_model_name)
    if mode == "api":
        return AnthropicAnalysisClient(model_name=config.worker_model_name)
    return MockAnalysisClient(model_name=config.worker_model_name or "mock_structured_worker_v1")

In [ ]:
# === Live Anthropic Analysis Client ===

class AnthropicAnalysisClient(BaseAnalysisClient):
    """Live analysis client using the Anthropic Claude API."""

    client_name = "anthropic_analysis_client"

    def __init__(
        self,
        *,
        api_key: str | None = None,
        model_name: str = "claude-opus-4-20250514",
        max_tokens: int = 4096,
        temperature: float = 0.2,
    ):
        self._api_key = api_key or ANTHROPIC_API_KEY or os.environ.get("ANTHROPIC_API_KEY", "")
        if not self._api_key:
            raise ValueError("ANTHROPIC_API_KEY must be set for live analysis.")
        self._client = anthropic.Anthropic(api_key=self._api_key)
        self.model_name = model_name
        self.max_tokens = max_tokens
        self.temperature = temperature

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=2, min=4, max=60),
        retry=retry_if_exception_type(anthropic.RateLimitError),
    )
    def _call_api(self, system_prompt: str, user_message: str) -> str:
        """Call the Anthropic messages API with retry on rate limits."""
        response = self._client.messages.create(
            model=self.model_name,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": user_message}],
        )
        return response.content[0].text

    def _extract_json(self, raw_text: str) -> dict[str, Any] | None:
        """Parse JSON from LLM response, handling both raw JSON and ```json``` blocks."""
        text = raw_text.strip()
        # Try direct JSON parse
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
        # Try extracting from ```json ... ``` block
        import re
        match = re.search(r"```(?:json)?\s*\n?(.*?)\n?\s*```", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                pass
        # Try finding the first { ... } block
        brace_start = text.find("{")
        brace_end = text.rfind("}")
        if brace_start != -1 and brace_end > brace_start:
            try:
                return json.loads(text[brace_start:brace_end + 1])
            except json.JSONDecodeError:
                pass
        return None

    def run_analysis(self, request: AnalysisClientRequest) -> AnalysisClientResponse:
        """Run one analysis request against Claude and parse the structured response."""
        # Extract system prompt from the analysis instructions
        system_prompt = request.prompt_context.get("analysis_instructions", "")
        if not system_prompt:
            system_prompt = "You are a biotech disclosure analysis worker. Analyze the provided document and return structured JSON output."

        user_message = request.prompt_text or json.dumps(
            request.prompt_context, default=str, indent=2
        )

        try:
            raw_text = self._call_api(system_prompt, user_message)
        except anthropic.APIError as exc:
            logger.error("Anthropic API error: %s", exc)
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.ANALYSIS_FAILED,
                raw_text=str(exc),
                warnings=[
                    make_analysis_warning(
                        "anthropic_api_error",
                        f"Anthropic API call failed: {exc}",
                    )
                ],
                is_mock=False,
            )

        parsed = self._extract_json(raw_text)
        if parsed is None:
            logger.warning("Could not parse JSON from Claude response. Returning raw text.")
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.PARTIAL,
                raw_output=None,
                raw_text=raw_text,
                warnings=[
                    make_analysis_warning(
                        "json_parse_failed",
                        "LLM response could not be parsed as JSON. Raw text preserved.",
                    )
                ],
                is_mock=False,
            )

        return AnalysisClientResponse(
            client_name=self.client_name,
            model_name=self.model_name,
            status=ProcessingStatus.SUCCESS,
            raw_output=parsed,
            raw_text=raw_text,
            is_mock=False,
        )


logger.info("AnthropicAnalysisClient defined.")

# === Moonshot AI (Kimi) Analysis Client ===
import httpx  # required for MoonshotAnalysisClient HTTP calls


class _MoonshotRateLimitError(Exception):
    """Raised on 429 to trigger tenacity retry; not raised for other HTTP errors."""


class MoonshotAnalysisClient(BaseAnalysisClient):
    """Live analysis client using the Moonshot AI (Kimi) OpenAI-compatible API."""

    client_name = "moonshot_analysis_client"

    def __init__(
        self,
        *,
        api_key: str | None = None,
        model_name: str = "moonshot-v1-128k",
        max_tokens: int = 4096,
        temperature: float = 0.2,
        base_url: str = "https://api.moonshot.ai/v1",
    ):
        self._api_key = api_key or MOONSHOT_API_KEY or os.environ.get("MOONSHOT_API_KEY", "")
        if not self._api_key:
            raise ValueError("MOONSHOT_API_KEY must be set for Moonshot analysis.")
        self.model_name = model_name
        self.max_tokens = max_tokens
        self.temperature = temperature
        self._base_url = base_url
        self._http = httpx.Client(timeout=180.0)

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=2, min=4, max=60),
        retry=retry_if_exception_type(_MoonshotRateLimitError),
    )
    def _call_api(self, system_prompt: str, user_message: str) -> str:
        """Call Moonshot chat completions API with retry only on 429 rate limits."""
        response = self._http.post(
            f"{self._base_url}/chat/completions",
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self._api_key}",
            },
            json={
                "model": self.model_name,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message},
                ],
                "max_tokens": self.max_tokens,
                "temperature": self.temperature,
            },
        )
        if response.status_code == 429:
            logger.warning("Moonshot rate limit hit (429), will retry...")
            raise _MoonshotRateLimitError(f"Rate limited (429): {response.text[:200]}")
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]

    def _extract_json(self, raw_text: str) -> dict[str, Any] | None:
        """Parse JSON from LLM response — same 3-strategy parser."""
        import re
        text = raw_text.strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
        match = re.search(r"```(?:json)?\s*\n?(.*?)\n?\s*```", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                pass
        brace_start = text.find("{")
        brace_end = text.rfind("}")
        if brace_start != -1 and brace_end > brace_start:
            try:
                return json.loads(text[brace_start:brace_end + 1])
            except json.JSONDecodeError:
                pass
        return None

    def run_analysis(self, request: AnalysisClientRequest) -> AnalysisClientResponse:
        """Run one analysis request against Moonshot AI and parse the structured response."""
        system_prompt = request.prompt_context.get("analysis_instructions", "")
        if not system_prompt:
            system_prompt = "You are a biotech disclosure analysis worker. Analyze the provided document and return structured JSON output."

        user_message = request.prompt_text or json.dumps(
            request.prompt_context, default=str, indent=2
        )

        try:
            raw_text = self._call_api(system_prompt, user_message)
        except (httpx.HTTPStatusError, _MoonshotRateLimitError, Exception) as exc:
            logger.error("Moonshot API error: %s", exc)
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.ANALYSIS_FAILED,
                raw_text=str(exc),
                warnings=[
                    make_analysis_warning(
                        "moonshot_api_error",
                        f"Moonshot API call failed: {exc}",
                    )
                ],
                is_mock=False,
            )

        parsed = self._extract_json(raw_text)
        if parsed is None:
            logger.warning("Could not parse JSON from Moonshot response. Returning raw text.")
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.PARTIAL,
                raw_output=None,
                raw_text=raw_text,
                warnings=[
                    make_analysis_warning(
                        "json_parse_failed",
                        "LLM response could not be parsed as JSON. Raw text preserved.",
                    )
                ],
                is_mock=False,
            )

        return AnalysisClientResponse(
            client_name=self.client_name,
            model_name=self.model_name,
            status=ProcessingStatus.SUCCESS,
            raw_output=parsed,
            raw_text=raw_text,
            is_mock=False,
        )


logger.info("MoonshotAnalysisClient defined.")

# === Claude CLI Analysis Client ===

class ClaudeCliAnalysisClient(BaseAnalysisClient):
    """Analysis client that invokes the Claude CLI (claude -p) via subprocess.
    
    Uses the user's Claude subscription instead of per-token API fees.
    """

    client_name = "claude_cli_analysis_client"

    def __init__(
        self,
        *,
        cli_path: str = CLAUDE_CLI_PATH,
        model_name: str = "claude-opus-4-20250514",
        timeout_seconds: int = 300,
        max_retries: int = 3,
    ):
        self._cli_path = cli_path
        if not Path(cli_path).is_file():
            raise FileNotFoundError(f"Claude CLI not found at {cli_path}")
        self.model_name = model_name
        self._timeout = timeout_seconds
        self._max_retries = max_retries

    def _build_clean_env(self) -> dict[str, str]:
        """Build a subprocess environment with all Claude Code nesting guards removed."""
        env = os.environ.copy()
        for key in list(env):
            if key.startswith("CLAUDE"):
                env.pop(key, None)
        return env

    def _call_cli(self, system_prompt: str, user_message: str) -> str:
        """Invoke claude -p and return raw stdout."""
        cmd = [
            self._cli_path, "-p",
            "--output-format", "json",
            "--no-session-persistence",
            "--model", self.model_name,
            "--max-turns", "1",
            "--system-prompt", system_prompt,
        ]
        result = subprocess.run(
            cmd,
            input=user_message,
            capture_output=True,
            text=True,
            timeout=self._timeout,
            env=self._build_clean_env(),
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"Claude CLI exited with code {result.returncode}: {result.stderr[:500]}"
            )
        if not result.stdout.strip():
            raise RuntimeError("Claude CLI returned empty output")
        return result.stdout

    def _parse_cli_json_output(self, raw_stdout: str) -> str:
        """Extract the LLM text from the CLI JSON envelope."""
        try:
            envelope = json.loads(raw_stdout)
            if isinstance(envelope, dict) and "result" in envelope:
                return envelope["result"]
            if isinstance(envelope, dict) and "content" in envelope:
                return envelope["content"]
            if isinstance(envelope, list):
                # CLI may return list of content blocks
                texts = [b.get("text", "") for b in envelope if isinstance(b, dict) and b.get("type") == "text"]
                if texts:
                    return "\n".join(texts)
        except (json.JSONDecodeError, TypeError, KeyError):
            pass
        # Fallback: treat raw stdout as LLM text
        return raw_stdout.strip()

    def _extract_json(self, raw_text: str) -> dict[str, Any] | None:
        """Parse JSON from LLM response — same 3-strategy parser as AnthropicAnalysisClient."""
        import re
        text = raw_text.strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass
        match = re.search(r"```(?:json)?\s*\n?(.*?)\n?\s*```", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1).strip())
            except json.JSONDecodeError:
                pass
        brace_start = text.find("{")
        brace_end = text.rfind("}")
        if brace_start != -1 and brace_end > brace_start:
            try:
                return json.loads(text[brace_start:brace_end + 1])
            except json.JSONDecodeError:
                pass
        return None

    def run_analysis(self, request: AnalysisClientRequest) -> AnalysisClientResponse:
        """Run one analysis request via Claude CLI with retry."""
        system_prompt = request.prompt_context.get("analysis_instructions", "")
        if not system_prompt:
            system_prompt = "You are a biotech disclosure analysis worker. Analyze the provided document and return structured JSON output."

        user_message = request.prompt_text or json.dumps(
            request.prompt_context, default=str, indent=2
        )

        raw_stdout = None
        last_error = None
        for attempt in range(1, self._max_retries + 1):
            try:
                raw_stdout = self._call_cli(system_prompt, user_message)
                break
            except (subprocess.TimeoutExpired, RuntimeError) as exc:
                last_error = exc
                wait_time = 2 ** attempt
                logger.warning(
                    "CLI attempt %d/%d failed: %s — retrying in %ds",
                    attempt, self._max_retries, exc, wait_time,
                )
                time.sleep(wait_time)

        if raw_stdout is None:
            logger.error("All %d CLI attempts failed: %s", self._max_retries, last_error)
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.ANALYSIS_FAILED,
                raw_text=str(last_error),
                warnings=[
                    make_analysis_warning(
                        "cli_all_retries_failed",
                        f"Claude CLI failed after {self._max_retries} attempts: {last_error}",
                    )
                ],
                is_mock=False,
            )

        llm_text = self._parse_cli_json_output(raw_stdout)
        parsed = self._extract_json(llm_text)

        if parsed is None:
            logger.warning("Could not parse JSON from CLI response. Returning raw text.")
            return AnalysisClientResponse(
                client_name=self.client_name,
                model_name=self.model_name,
                status=ProcessingStatus.PARTIAL,
                raw_output=None,
                raw_text=llm_text,
                warnings=[
                    make_analysis_warning(
                        "json_parse_failed",
                        "CLI response could not be parsed as JSON. Raw text preserved.",
                    )
                ],
                is_mock=False,
            )

        return AnalysisClientResponse(
            client_name=self.client_name,
            model_name=self.model_name,
            status=ProcessingStatus.SUCCESS,
            raw_output=parsed,
            raw_text=llm_text,
            is_mock=False,
        )


logger.info("ClaudeCliAnalysisClient defined.")

In [ ]:
def build_generic_analysis_query(processed_document: ProcessedDocument) -> str:
    """Build a document-agnostic retrieval query for shared worker analysis."""
    title = processed_document.document.title or processed_document.document.document_type.value.replace("_", " ")
    dimensions = ", ".join(
        [
            AnalysisDimension.SENTIMENT.value,
            AnalysisDimension.UNCERTAINTY.value,
            AnalysisDimension.FOGGING.value,
            AnalysisDimension.HEDGING.value,
            AnalysisDimension.PROMOTIONAL_TONE.value,
            AnalysisDimension.CLARITY.value,
            AnalysisDimension.MATERIALITY.value,
            AnalysisDimension.COMPLETENESS.value,
        ]
    )
    return f"{title} evidence for {dimensions}"


def retrieve_analysis_context(
    processed_document: ProcessedDocument,
    *,
    graph_index: GraphDocumentIndex | None = None,
    chunk_embeddings: Sequence[ChunkEmbeddingRecord] | None = None,
    chunk_retrieval_result: ChunkRetrievalResult | None = None,
    query_text: str | None = None,
    embedding_config: EmbeddingConfig = EMBEDDING_CONFIG,
    retrieval_config: GraphRetrievalConfig = GRAPH_RETRIEVAL_CONFIG,
    preferred_embedding_client: BaseEmbeddingClient | None = None,
) -> tuple[ChunkRetrievalResult, GraphDocumentIndex, list[ChunkEmbeddingRecord], list[AnalysisWarning]]:
    """Resolve the chunk-level context that shared worker analysis will inspect."""
    warnings: list[AnalysisWarning] = []
    resolved_graph_index = graph_index or build_document_graph(processed_document)
    resolved_chunk_embeddings = list(chunk_embeddings or [])

    if chunk_retrieval_result is not None:
        return chunk_retrieval_result, resolved_graph_index, resolved_chunk_embeddings, warnings

    if not resolved_chunk_embeddings:
        resolved_chunk_embeddings, embedding_notes = build_chunk_embedding_records(
            processed_document,
            embedding_config,
            preferred_client=preferred_embedding_client,
        )
        processed_document.processing_notes.extend(embedding_notes)
        if not resolved_chunk_embeddings:
            warnings.append(
                make_analysis_warning(
                    "missing_chunk_embeddings",
                    "Shared analysis context had to proceed without precomputed chunk embeddings.",
                )
            )

    effective_query = query_text or build_generic_analysis_query(processed_document)
    retrieval_result = retrieve_relevant_chunks(
        effective_query,
        processed_document,
        resolved_chunk_embeddings,
        resolved_graph_index,
        embedding_config,
        retrieval_config,
        preferred_client=preferred_embedding_client,
    )
    if not retrieval_result.hits:
        warnings.append(
            make_analysis_warning(
                "no_ranked_chunk_hits",
                "Chunk retrieval did not return ranked hits for shared worker analysis.",
            )
        )
    return retrieval_result, resolved_graph_index, resolved_chunk_embeddings, warnings


def build_analysis_prompt_context(
    analysis_input: WorkerAnalysisInput,
    rubric: AnalysisRubric = DEFAULT_ANALYSIS_RUBRIC,
) -> dict[str, Any]:
    """Build a compact provider-agnostic prompt context object."""
    return {
        "worker_name": analysis_input.worker_name,
        "document_type": analysis_input.document_type.value,
        "ticker": analysis_input.ticker,
        "document_title": analysis_input.document_title,
        "document_excerpt": analysis_input.document_text_excerpt,
        "analysis_instructions": analysis_input.analysis_instructions,
        "rubric": {
            "rubric_id": rubric.rubric_id,
            "version": rubric.version,
            "summary": rubric.summary,
            "core_principles": rubric.core_principles,
            "dimensions": [
                {
                    "dimension": definition.dimension.value,
                    "objective": definition.objective,
                    "evidence_expectation": definition.evidence_expectation,
                }
                for definition in rubric.dimension_definitions
            ],
        },
        "sections": [section.model_dump(mode="json") for section in analysis_input.section_context],
        "evidence_bundles": [
            {
                "bundle_id": bundle.bundle_id,
                "chunk_id": bundle.chunk_id,
                "section_title": bundle.section_title,
                "retrieval_rank": bundle.retrieval_rank,
                "evidence_type": bundle.evidence_type.value,
                "primary_text": bundle.primary_text,
                "expanded_context_text": bundle.expanded_context_text,
                "local_context_summary": bundle.local_context_summary,
            }
            for bundle in analysis_input.top_chunk_bundles
        ],
        "expected_output_schema": analysis_input.expected_output_schema,
    }


def normalize_analysis_scores(
    raw_payload: Mapping[str, Any] | dict[str, Any],
    rubric: AnalysisRubric = DEFAULT_ANALYSIS_RUBRIC,
) -> tuple[list[AnalysisScore], list[AnalysisWarning]]:
    """Normalize raw score content from a worker-analysis payload."""
    warnings: list[AnalysisWarning] = []
    normalized_scores: list[AnalysisScore] = []

    raw_scores = raw_payload.get("scores") if raw_payload else None
    score_lookup: dict[str, Any] = {}
    if isinstance(raw_scores, Mapping):
        score_lookup = {str(key): value for key, value in raw_scores.items()}
    elif isinstance(raw_scores, list):
        for item in raw_scores:
            if isinstance(item, Mapping) and item.get("dimension"):
                score_lookup[str(item["dimension"])] = item
    elif raw_scores is not None:
        warnings.append(
            make_analysis_warning(
                "scores_wrong_shape",
                "Raw score payload was neither a dictionary nor a list of dimension entries.",
                field_name="scores",
            )
        )

    for definition in rubric.dimension_definitions:
        raw_entry = score_lookup.get(definition.dimension.value)
        raw_score = None
        raw_label = None
        raw_confidence = raw_payload.get("confidence_score") if raw_payload else None
        rationale = None
        evidence_ids: list[str] = []

        if isinstance(raw_entry, Mapping):
            raw_score = raw_entry.get("score")
            raw_label = raw_entry.get("label")
            raw_confidence = raw_entry.get("confidence_score", raw_confidence)
            rationale = raw_entry.get("rationale")
            if isinstance(raw_entry.get("evidence_ids"), list):
                evidence_ids = [str(item) for item in raw_entry["evidence_ids"] if str(item).strip()]
        else:
            if definition.dimension == AnalysisDimension.SENTIMENT:
                raw_score = raw_payload.get("sentiment_score")
                raw_label = raw_payload.get("sentiment_label")
            else:
                raw_score = raw_payload.get(f"{definition.dimension.value}_score")

        score, score_warnings = build_standardized_analysis_score(
            dimension=definition.dimension,
            raw_score=raw_score,
            raw_label=raw_label,
            raw_confidence=raw_confidence,
            rationale=rationale,
            evidence_ids=evidence_ids,
            rubric_notes=definition.scoring_guidance,
        )
        if score.score is None:
            warnings.append(
                make_analysis_warning(
                    "missing_dimension_score",
                    f"No explicit score was provided for {definition.dimension.value}.",
                    dimension=definition.dimension,
                    field_name="scores",
                )
            )
        warnings.extend(score_warnings)
        normalized_scores.append(score)

    return normalized_scores, warnings


def map_raw_evidence_references(
    raw_references: Any,
    analysis_input: WorkerAnalysisInput,
) -> tuple[list[EvidenceSnippet], list[AnalysisWarning]]:
    """Map raw evidence references back to structured evidence snippets."""
    warnings: list[AnalysisWarning] = []
    evidence: list[EvidenceSnippet] = []

    bundle_lookup = {bundle.bundle_id: bundle for bundle in analysis_input.top_chunk_bundles}
    chunk_lookup = {bundle.chunk_id: bundle for bundle in analysis_input.top_chunk_bundles}

    if not isinstance(raw_references, list):
        return evidence, warnings

    for index, reference in enumerate(raw_references, start=1):
        if not isinstance(reference, Mapping):
            warnings.append(
                make_analysis_warning(
                    "invalid_evidence_reference",
                    "One evidence reference was not a dictionary and was skipped.",
                    field_name="evidence_refs",
                    metadata={"index": index},
                )
            )
            continue

        bundle = None
        bundle_id = reference.get("bundle_id")
        chunk_id = reference.get("chunk_id")
        if isinstance(bundle_id, str):
            bundle = bundle_lookup.get(bundle_id)
        if bundle is None and isinstance(chunk_id, str):
            bundle = chunk_lookup.get(chunk_id)
        if bundle is None:
            warnings.append(
                make_analysis_warning(
                    "missing_evidence_reference",
                    "Evidence reference did not match any prepared evidence bundle.",
                    field_name="evidence_refs",
                    source_chunk_id=str(chunk_id) if chunk_id is not None else None,
                )
            )
            continue

        dimension_raw = reference.get("dimension")
        interpretation_raw = reference.get("interpretation")
        supported_dimensions: list[AnalysisDimension] = []
        interpretation = None
        if isinstance(dimension_raw, str):
            try:
                supported_dimensions = [AnalysisDimension(dimension_raw)]
            except ValueError:
                warnings.append(
                    make_analysis_warning(
                        "unknown_evidence_dimension",
                        f"Evidence dimension {dimension_raw!r} is not recognized.",
                        field_name="dimension",
                        source_chunk_id=bundle.chunk_id,
                    )
                )
        if isinstance(interpretation_raw, str):
            try:
                interpretation = EvidenceInterpretation(interpretation_raw)
            except ValueError:
                warnings.append(
                    make_analysis_warning(
                        "unknown_evidence_interpretation",
                        f"Evidence interpretation {interpretation_raw!r} is not recognized.",
                        field_name="interpretation",
                        source_chunk_id=bundle.chunk_id,
                    )
                )

        why_it_matters = reference.get("why_it_matters")
        if not isinstance(why_it_matters, str) or not why_it_matters.strip():
            why_it_matters = "Evidence reference returned by the shared worker-analysis client."

        evidence.append(
            build_evidence_snippet_from_bundle(
                bundle,
                source_url=analysis_input.document_metadata.source_url,
                rationale=why_it_matters.strip(),
                supported_dimensions=supported_dimensions,
                interpretation=interpretation,
                snippet_text=str(reference.get("evidence_text")).strip() if reference.get("evidence_text") else None,
            )
        )

    return evidence, warnings


def extract_structured_findings(
    raw_findings: Any,
    *,
    analysis_input: WorkerAnalysisInput,
    normalized_scores: Sequence[AnalysisScore],
    evidence: Sequence[EvidenceSnippet],
) -> tuple[list[AnalysisFinding], list[AnalysisWarning]]:
    """Normalize raw findings or build fallback findings from available evidence."""
    warnings: list[AnalysisWarning] = []
    findings: list[AnalysisFinding] = []
    evidence_lookup = {item.evidence_id: item for item in evidence if item.evidence_id}
    score_lookup = {score.dimension: score for score in normalized_scores}

    if isinstance(raw_findings, list):
        for index, raw_finding in enumerate(raw_findings, start=1):
            if not isinstance(raw_finding, Mapping):
                warnings.append(
                    make_analysis_warning(
                        "invalid_finding_entry",
                        "One finding entry was not a dictionary and was skipped.",
                        field_name="findings",
                        metadata={"index": index},
                    )
                )
                continue

            dimension_raw = raw_finding.get("dimension")
            if not isinstance(dimension_raw, str):
                warnings.append(
                    make_analysis_warning(
                        "finding_dimension_missing",
                        "A finding was missing its analysis dimension and was skipped.",
                        field_name="findings",
                        metadata={"index": index},
                    )
                )
                continue
            try:
                dimension = AnalysisDimension(dimension_raw)
            except ValueError:
                warnings.append(
                    make_analysis_warning(
                        "finding_dimension_unknown",
                        f"Finding dimension {dimension_raw!r} is not recognized.",
                        field_name="findings",
                        metadata={"index": index},
                    )
                )
                continue

            interpretation = None
            interpretation_raw = raw_finding.get("interpretation")
            if isinstance(interpretation_raw, str):
                try:
                    interpretation = EvidenceInterpretation(interpretation_raw)
                except ValueError:
                    warnings.append(
                        make_analysis_warning(
                            "finding_interpretation_unknown",
                            f"Finding interpretation {interpretation_raw!r} is not recognized.",
                            field_name="findings",
                            metadata={"index": index},
                        )
                    )

            evidence_refs = raw_finding.get("evidence_refs")
            if not isinstance(evidence_refs, list):
                evidence_refs = []
            evidence_ids = [str(item) for item in evidence_refs if str(item).strip() and str(item) in evidence_lookup]
            source_chunk_ids = [
                evidence_lookup[evidence_id].source_chunk_id
                for evidence_id in evidence_ids
                if evidence_lookup[evidence_id].source_chunk_id is not None
            ]
            summary = str(raw_finding.get("summary") or "").strip()
            if not summary:
                warnings.append(
                    make_analysis_warning(
                        "finding_summary_missing",
                        "A finding was missing summary text and was skipped.",
                        field_name="findings",
                        metadata={"index": index},
                    )
                )
                continue

            findings.append(
                AnalysisFinding(
                    finding_id=f"{analysis_input.run_id}::{analysis_input.worker_name}::finding_{len(findings) + 1:02d}",
                    dimension=dimension,
                    summary=summary,
                    interpretation=interpretation,
                    evidence_ids=evidence_ids,
                    source_chunk_ids=[chunk_id for chunk_id in source_chunk_ids if chunk_id],
                    score=score_lookup.get(dimension).score if dimension in score_lookup else None,
                    rationale=str(raw_finding.get("rationale")).strip() if raw_finding.get("rationale") else None,
                    is_material=bool(raw_finding.get("is_material", False)),
                )
            )

    if findings:
        return findings, warnings

    fallback_evidence = list(evidence)[:3]
    if not fallback_evidence:
        warnings.append(
            make_analysis_warning(
                "no_findings_available",
                "No normalized findings could be built because both findings and evidence were empty.",
            )
        )
        return findings, warnings

    for index, evidence_item in enumerate(fallback_evidence, start=1):
        dimension = (
            evidence_item.supported_dimensions[0]
            if evidence_item.supported_dimensions
            else AnalysisDimension.MATERIALITY
        )
        findings.append(
            AnalysisFinding(
                finding_id=f"{analysis_input.run_id}::{analysis_input.worker_name}::fallback_finding_{index:02d}",
                dimension=dimension,
                summary=evidence_item.rationale,
                interpretation=evidence_item.interpretation,
                evidence_ids=[evidence_item.evidence_id] if evidence_item.evidence_id else [],
                source_chunk_ids=[evidence_item.source_chunk_id] if evidence_item.source_chunk_id else [],
                score=score_lookup.get(dimension).score if dimension in score_lookup else None,
                rationale="Fallback finding built from structured evidence because raw findings were missing.",
                is_material=dimension == AnalysisDimension.MATERIALITY,
            )
        )
    warnings.append(
        make_analysis_warning(
            "findings_built_from_evidence",
            "Raw findings were missing, so fallback findings were built from the normalized evidence snippets.",
        )
    )
    return findings, warnings


def deduplicate_analysis_items[T](items: Sequence[T], key_builder: Callable[[T], tuple[Any, ...]]) -> list[T]:
    """Deduplicate warnings or issues while preserving order."""
    seen: set[tuple[Any, ...]] = set()
    deduplicated: list[T] = []
    for item in items:
        key = key_builder(item)
        if key in seen:
            continue
        seen.add(key)
        deduplicated.append(item)
    return deduplicated


def collect_analysis_warnings(analysis_output: WorkerAnalysisOutput) -> list[AnalysisWarning]:
    """Add secondary warnings after parsing and score normalization."""
    warnings = list(analysis_output.warnings)
    if analysis_output.confidence_score is not None and analysis_output.confidence_score < 0.4:
        warnings.append(
            make_analysis_warning(
                "low_confidence_output",
                "Shared worker analysis returned low confidence and should be treated as provisional.",
            )
        )

    sentiment_score = next(
        (score for score in analysis_output.scores if score.dimension == AnalysisDimension.SENTIMENT),
        None,
    )
    if sentiment_score is not None and sentiment_score.label == SentimentLabel.MIXED.value and sentiment_score.score is not None:
        if abs(sentiment_score.score) < 0.2:
            warnings.append(
                make_analysis_warning(
                    "ambiguous_sentiment_label",
                    "Sentiment was labeled mixed while the normalized sentiment score remained close to zero.",
                    dimension=AnalysisDimension.SENTIMENT,
                )
            )

    if analysis_output.reasoning_trace is None or not analysis_output.reasoning_trace.reasoning_notes:
        warnings.append(
            make_analysis_warning(
                "empty_reasoning_notes",
                "No reasoning notes were captured in the shared worker-analysis trace.",
            )
        )

    if len(analysis_output.evidence) < 2:
        warnings.append(
            make_analysis_warning(
                "insufficient_chunk_evidence",
                "Shared worker analysis finished with fewer than two structured evidence snippets.",
            )
        )

    return deduplicate_analysis_items(
        warnings,
        lambda item: (item.issue_code, item.message, item.dimension, item.source_chunk_id, item.field_name),
    )


def analysis_scores_to_lookup(scores: Sequence[AnalysisScore]) -> dict[AnalysisDimension, AnalysisScore]:
    """Build a dimension lookup for normalized scores."""
    return {score.dimension: score for score in scores}


def build_sentiment_assessment_from_scores(scores: Sequence[AnalysisScore]) -> SentimentAssessment | None:
    """Convert normalized analysis scores into the legacy sentiment assessment shape."""
    sentiment_score = analysis_scores_to_lookup(scores).get(AnalysisDimension.SENTIMENT)
    if sentiment_score is None:
        return None
    sentiment_label, _ = normalize_sentiment_label(sentiment_score.label, sentiment_score.score)
    return SentimentAssessment(
        label=sentiment_label,
        score=sentiment_score.score,
        confidence=sentiment_score.confidence_score,
        rationale=sentiment_score.rationale,
    )


def build_tone_assessment_from_scores(scores: Sequence[AnalysisScore]) -> ToneAssessment | None:
    """Convert normalized analysis scores into the legacy tone assessment shape."""
    lookup = analysis_scores_to_lookup(scores)
    if not any(dimension in lookup for dimension in (AnalysisDimension.FOGGING, AnalysisDimension.HEDGING, AnalysisDimension.PROMOTIONAL_TONE)):
        return None
    confidence_candidates = [
        lookup[dimension].confidence_score
        for dimension in (AnalysisDimension.FOGGING, AnalysisDimension.HEDGING, AnalysisDimension.PROMOTIONAL_TONE)
        if dimension in lookup and lookup[dimension].confidence_score is not None
    ]
    return ToneAssessment(
        fogging_score=lookup.get(AnalysisDimension.FOGGING).score if AnalysisDimension.FOGGING in lookup else None,
        hedging_score=lookup.get(AnalysisDimension.HEDGING).score if AnalysisDimension.HEDGING in lookup else None,
        promotional_score=(
            lookup.get(AnalysisDimension.PROMOTIONAL_TONE).score
            if AnalysisDimension.PROMOTIONAL_TONE in lookup
            else None
        ),
        confidence=(sum(confidence_candidates) / len(confidence_candidates)) if confidence_candidates else None,
        rationale="Shared worker-analysis tone scores normalized from the common rubric.",
    )


def analysis_issue_to_pipeline_error(issue: AnalysisIssue, document_type: DocumentType) -> PipelineError:
    """Map an analysis-layer issue into the pipeline-wide error shape."""
    return PipelineError(
        error_code=issue.issue_code,
        message=issue.message,
        stage="worker_analysis",
        document_type=document_type,
        recoverable=issue.recoverable,
        details={
            "severity": issue.severity.value,
            "dimension": issue.dimension.value if issue.dimension else None,
            "source_chunk_id": issue.source_chunk_id,
            "field_name": issue.field_name,
            **issue.metadata,
        },
    )


def assemble_worker_output(analysis_output: WorkerAnalysisOutput) -> WorkerOutput:
    """Map the shared internal analysis result into the existing worker output contract."""
    key_points = [finding.summary for finding in analysis_output.findings[:5]]
    caveats = [warning.message for warning in analysis_output.warnings]
    issues = [analysis_issue_to_pipeline_error(issue, analysis_output.document_type) for issue in analysis_output.issues]
    return WorkerOutput(
        worker_name=analysis_output.worker_name,
        document_type=analysis_output.document_type,
        status=analysis_output.status,
        summary=analysis_output.summary,
        sentiment=build_sentiment_assessment_from_scores(analysis_output.scores),
        tone=build_tone_assessment_from_scores(analysis_output.scores),
        key_points=key_points,
        evidence=list(analysis_output.evidence),
        caveats=caveats,
        issues=issues,
        confidence=analysis_output.confidence_score,
    )


def run_generic_analysis(
    analysis_input: WorkerAnalysisInput,
    *,
    analysis_client: BaseAnalysisClient,
    rubric: AnalysisRubric = DEFAULT_ANALYSIS_RUBRIC,
    prompt_context: dict[str, Any] | None = None,
    prompt_text: str | None = None,
) -> WorkerAnalysisOutput:
    """Run the shared worker-analysis path with the configured analysis client."""
    resolved_prompt_context = prompt_context or build_analysis_prompt_context(analysis_input, rubric)
    resolved_prompt_text = prompt_text or json.dumps(resolved_prompt_context, default=str, indent=2)
    request = AnalysisClientRequest(
        worker_name=analysis_input.worker_name,
        document_type=analysis_input.document_type,
        analysis_input=analysis_input,
        prompt_context=resolved_prompt_context,
        prompt_text=resolved_prompt_text,
        rubric=rubric,
    )
    response = analysis_client.run_analysis(request)
    raw_output = response.raw_output if response.raw_output is not None else response.raw_text
    analysis_output = parse_worker_analysis_output(
        raw_output,
        analysis_input,
        client_name=response.client_name,
        model_name=response.model_name,
        is_mock=response.is_mock,
        client_warnings=response.warnings,
        prompt_context=resolved_prompt_context,
    )
    analysis_output.warnings = collect_analysis_warnings(analysis_output)
    if response.status == ProcessingStatus.ANALYSIS_FAILED and analysis_output.status != ProcessingStatus.ANALYSIS_FAILED:
        analysis_output.status = ProcessingStatus.PARTIAL
    return analysis_output



## 33. Shared Prompt Assembly Helpers

These helpers keep prompt construction explicit and inspectable. Each specialized worker reuses the same shared base instruction, prompt assembly flow, and warning model while supplying only its document-native master prompt block and analysis query.


In [ ]:
from textwrap import dedent


WORKER_PROMPT_REQUIRED_SECTIONS = (
    "## Shared Worker Base Instruction",
    "## Worker-Specific Analysis Instruction",
    "## Input Metadata",
    "## Evidence and Context",
    "## Output Schema",
)


def render_prompt_bullet_list(items: Sequence[str]) -> str:
    """Render one bullet list for prompt text blocks."""
    cleaned_items = [str(item).strip() for item in items if str(item).strip()]
    return "\n".join(f"- {item}" for item in cleaned_items) if cleaned_items else "- none"


def format_optional_datetime(value: datetime | None) -> str:
    """Format optional datetimes consistently for prompt metadata blocks."""
    return value.isoformat() if isinstance(value, datetime) else "n/a"


def format_optional_date(value: date | None) -> str:
    """Format optional dates consistently for prompt metadata blocks."""
    return value.isoformat() if isinstance(value, date) else "n/a"


def build_shared_worker_instruction(rubric: AnalysisRubric = DEFAULT_ANALYSIS_RUBRIC) -> str:
    """Build the common instruction block that every disclosure worker shares."""
    dimension_lines = [
        f"{definition.dimension.value}: {definition.objective}"
        for definition in rubric.dimension_definitions
    ]
    return dedent(
        f"""
        ## Shared Worker Base Instruction

        You are one disclosure-specific worker inside a biotech disclosure analysis system.
        Analyze exactly one document of the assigned type.
        Use only the supplied document text, section context, chunk evidence bundles, and provenance.
        Keep every conclusion proportional to the cited evidence.
        Separate disclosed facts from management framing and separate directional implication from tone.
        If the evidence does not support a firm inference, state the uncertainty directly instead of guessing.
        Do not rely on canned phrase matching, undocumented metadata, or outside assumptions.
        Return structured output only in the requested schema.

        ### Shared Rubric Priorities
        {render_prompt_bullet_list(rubric.core_principles)}

        ### Shared Dimensions
        {render_prompt_bullet_list(dimension_lines)}
        """
    ).strip()


def build_worker_specific_instruction(
    *,
    worker_label: str,
    worker_job: str,
    document_native_focus: str,
    what_to_look_for: Sequence[str],
    fogging_guidance: Sequence[str],
    signal_guidance: Sequence[str],
    output_requirements: Sequence[str] | None = None,
) -> str:
    """Build the worker-specific instruction block for one disclosure type."""
    resolved_output_requirements = list(
        output_requirements
        or [
            "Tie every material conclusion to cited evidence bundle ids or chunk ids.",
            "Call out missing expected detail explicitly rather than filling gaps with outside knowledge.",
            "Keep positive, negative, mixed, or neutral signal grounded in disclosed facts, not wording polish alone.",
            "If the evidence is thin or ambiguous, reflect that explicitly in uncertainty, caveats, and confidence.",
        ]
    )
    return dedent(
        f"""
        ## Worker-Specific Analysis Instruction

        ### Worker
        {worker_label}

        ### Your Job
        {worker_job}

        ### Document-Native Focus
        {document_native_focus}

        ### What To Look For
        {render_prompt_bullet_list(what_to_look_for)}

        ### Fogging and Evasiveness
        {render_prompt_bullet_list(fogging_guidance)}

        ### Positive vs Negative Signal
        {render_prompt_bullet_list(signal_guidance)}

        ### Output Discipline
        {render_prompt_bullet_list(resolved_output_requirements)}
        """
    ).strip()


def render_input_metadata_for_prompt(analysis_input: WorkerAnalysisInput) -> str:
    """Render one compact metadata block for a disclosure-specific worker prompt."""
    metadata = analysis_input.document_metadata
    metadata_lines = [
        f"- run_id: {analysis_input.run_id}",
        f"- worker_name: {analysis_input.worker_name}",
        f"- document_type: {analysis_input.document_type.value}",
        f"- ticker: {analysis_input.ticker}",
        f"- company_name: {metadata.company_name or 'n/a'}",
        f"- document_id: {metadata.document_id}",
        f"- title: {analysis_input.document_title or metadata.title or 'n/a'}",
        f"- source_name: {metadata.source_name or 'n/a'}",
        f"- source_family: {metadata.source_family.value}",
        f"- source_url: {metadata.source_url or 'n/a'}",
        f"- published_at: {format_optional_datetime(metadata.published_at)}",
        f"- updated_at: {format_optional_datetime(metadata.updated_at)}",
        f"- event_date: {format_optional_date(metadata.event_date)}",
        f"- version_label: {metadata.version_label or 'n/a'}",
        f"- section_count: {analysis_input.metadata.get('section_count', 'n/a')}",
        f"- chunk_count: {analysis_input.metadata.get('chunk_count', 'n/a')}",
        f"- retrieval_query: {analysis_input.retrieval_query or 'n/a'}",
        f"- mock_data: {analysis_input.is_mock_data}",
    ]
    processing_lines = [
        f"- {note.code}: {note.message}"
        for note in analysis_input.processing_notes[:5]
    ]
    if not processing_lines:
        processing_lines = ["- none"]
    return "\n".join(metadata_lines + ["", "### Processing Notes", *processing_lines])


def render_analysis_context_for_prompt(analysis_input: WorkerAnalysisInput) -> str:
    """Render the evidence and context block used by every specialized worker prompt."""
    section_lines = [
        (
            f"- {section.section_id} | title={section.title!r} | kind={section.section_kind.value} "
            f"| level={section.level} | words={section.word_count}"
        )
        for section in analysis_input.section_context
    ]
    if not section_lines:
        section_lines = ["- none"]

    evidence_blocks: list[str] = []
    for bundle in analysis_input.top_chunk_bundles:
        bundle_lines = [
            f"- bundle_id: {bundle.bundle_id}",
            f"  chunk_id: {bundle.chunk_id}",
            f"  section_title: {bundle.section_title or 'n/a'}",
            f"  retrieval_rank: {bundle.retrieval_rank if bundle.retrieval_rank is not None else 'n/a'}",
            f"  evidence_type: {bundle.evidence_type.value}",
            f"  adjusted_score: {bundle.adjusted_score if bundle.adjusted_score is not None else 'n/a'}",
            f"  primary_text: {make_text_excerpt(bundle.primary_text, 280)}",
        ]
        if bundle.expanded_context_text:
            bundle_lines.append(f"  expanded_context: {make_text_excerpt(bundle.expanded_context_text, 220)}")
        if bundle.local_context_summary:
            bundle_lines.append(f"  local_context_summary: {bundle.local_context_summary}")
        if bundle.notes:
            bundle_lines.append(f"  notes: {' | '.join(bundle.notes[:3])}")
        evidence_blocks.append("\n".join(bundle_lines))
    if not evidence_blocks:
        evidence_blocks = ["- none"]

    provenance_lines = [
        (
            f"- stage={record.stage} | adapter={record.adapter_name} | "
            f"source={record.source_name or 'n/a'} | note={record.note or 'n/a'}"
        )
        for record in analysis_input.provenance[:5]
    ]
    if not provenance_lines:
        provenance_lines = ["- none"]

    return "\n".join(
        [
            "### Document Text",
            analysis_input.document_text,
            "",
            "### Section Context",
            *section_lines,
            "",
            "### Evidence Bundles",
            *evidence_blocks,
            "",
            "### Provenance",
            *provenance_lines,
        ]
    )


def render_output_schema_for_prompt(expected_output_schema: dict[str, Any]) -> str:
    """Render the structured output contract for prompt assembly."""
    return "\n".join(
        [
            "Return one JSON object only that matches this schema exactly.",
            "```json",
            json.dumps(expected_output_schema, indent=2, default=str),
            "```",
        ]
    )


def build_worker_prompt(
    *,
    shared_instruction: str,
    worker_specific_instruction: str,
    analysis_input: WorkerAnalysisInput,
) -> str:
    """Assemble the full prompt text passed to one specialized worker."""
    return "\n\n".join(
        [
            shared_instruction.strip(),
            worker_specific_instruction.strip(),
            "## Input Metadata\n\n" + render_input_metadata_for_prompt(analysis_input),
            "## Evidence and Context\n\n" + render_analysis_context_for_prompt(analysis_input),
            "## Output Schema\n\n" + render_output_schema_for_prompt(analysis_input.expected_output_schema),
        ]
    ).strip()


def validate_worker_prompt_sections(prompt_text: str) -> list[AnalysisWarning]:
    """Warn when an assembled worker prompt is missing required structural blocks."""
    missing_sections = [section for section in WORKER_PROMPT_REQUIRED_SECTIONS if section not in prompt_text]
    if not missing_sections:
        return []
    return [
        make_analysis_warning(
            "worker_prompt_missing_required_sections",
            "Assembled worker prompt is missing one or more required sections.",
            field_name="prompt_text",
            metadata={"missing_sections": missing_sections},
        )
    ]


def validate_prompt_analysis_context(analysis_input: WorkerAnalysisInput) -> list[AnalysisWarning]:
    """Warn when the assembled prompt context is too thin for confident analysis."""
    warnings: list[AnalysisWarning] = []
    if len(analysis_input.top_chunk_bundles) < 2:
        warnings.append(
            make_analysis_warning(
                "worker_evidence_bundle_too_thin",
                "Disclosure-specific analysis is proceeding with fewer than two evidence bundles, so confidence should remain conservative.",
                field_name="top_chunk_bundles",
                metadata={"bundle_count": len(analysis_input.top_chunk_bundles)},
            )
        )
    if len(analysis_input.document_text.strip()) < 250 or not analysis_input.section_context:
        warnings.append(
            make_analysis_warning(
                "prompt_context_too_sparse",
                "Prompt context is sparse because the document excerpt or section context is limited.",
                field_name="document_text",
                metadata={
                    "document_char_count": len(analysis_input.document_text.strip()),
                    "section_count": len(analysis_input.section_context),
                },
            )
        )
    if not analysis_input.expected_output_schema:
        warnings.append(
            make_analysis_warning(
                "expected_output_schema_missing",
                "Prompt assembly did not receive the expected structured output schema.",
                field_name="expected_output_schema",
            )
        )
    return warnings


class BaseAnalysisWorker(BaseWorker):
    """Reusable shared worker implementation with modular prompt assembly."""

    rubric: ClassVar[AnalysisRubric] = DEFAULT_ANALYSIS_RUBRIC
    worker_label: ClassVar[str] = "Base Analysis Worker"
    worker_specific_instruction: ClassVar[str] = ""
    analysis_focus_query: ClassVar[str | None] = None

    def __init__(
        self,
        *,
        analysis_client: BaseAnalysisClient | None = None,
        rubric: AnalysisRubric | None = None,
        analysis_instructions: str | None = None,
    ) -> None:
        self.analysis_client = analysis_client
        self.runtime_rubric = rubric or self.rubric
        self.analysis_instructions = analysis_instructions

    def get_analysis_client(self, worker_input: WorkerInput) -> BaseAnalysisClient:
        """Resolve the effective analysis client for this worker invocation."""
        return build_analysis_client(worker_input.config, preferred_client=self.analysis_client)

    def build_analysis_query(self, processed_document: ProcessedDocument) -> str:
        """Build the retrieval query used to select chunk evidence for this worker."""
        return self.analysis_focus_query or build_generic_analysis_query(processed_document)

    def get_worker_specific_instruction(self, worker_input: WorkerInput | None = None) -> str:
        """Return the disclosure-type-specific master prompt block."""
        return self.worker_specific_instruction.strip()

    def get_analysis_instructions(self, worker_input: WorkerInput) -> str:
        """Return the shared plus worker-specific instruction text for the analysis packet."""
        if self.analysis_instructions:
            return self.analysis_instructions.strip()
        blocks = [
            build_shared_worker_instruction(self.runtime_rubric),
            self.get_worker_specific_instruction(worker_input),
        ]
        return "\n\n".join(block for block in blocks if block.strip()).strip()

    def validate_worker_input(
        self,
        worker_input: WorkerInput,
        selected_document: SelectedDocument,
    ) -> list[AnalysisWarning]:
        """Warn when worker input or selected-document types do not match the specialized worker."""
        warnings: list[AnalysisWarning] = []
        if worker_input.document_type != self.document_type:
            warnings.append(
                make_analysis_warning(
                    "worker_document_type_mismatch",
                    f"{self.worker_name} expected {self.document_type.value} but received worker input for {worker_input.document_type.value}.",
                    field_name="document_type",
                    metadata={
                        "expected_document_type": self.document_type.value,
                        "received_document_type": worker_input.document_type.value,
                    },
                )
            )
        if selected_document.document_type != self.document_type:
            warnings.append(
                make_analysis_warning(
                    "selected_document_type_mismatch",
                    f"Selected document type {selected_document.document_type.value} does not match {self.worker_name} specialization {self.document_type.value}.",
                    field_name="selected_document.document_type",
                    metadata={
                        "expected_document_type": self.document_type.value,
                        "selected_document_type": selected_document.document_type.value,
                    },
                )
            )
        return warnings

    def build_prompt_text(self, analysis_input: WorkerAnalysisInput) -> str:
        """Build the final prompt text for one worker invocation."""
        return build_worker_prompt(
            shared_instruction=build_shared_worker_instruction(self.runtime_rubric),
            worker_specific_instruction=self.get_worker_specific_instruction(),
            analysis_input=analysis_input,
        )

    def build_prompt_context(
        self,
        analysis_input: WorkerAnalysisInput,
        prompt_text: str,
    ) -> dict[str, Any]:
        """Build a compact structured context object alongside the final prompt text."""
        prompt_context = build_analysis_prompt_context(analysis_input, self.runtime_rubric)
        prompt_context.update(
            {
                "worker_label": self.worker_label,
                "analysis_focus_query": analysis_input.retrieval_query,
                "prompt_required_sections": list(WORKER_PROMPT_REQUIRED_SECTIONS),
                "worker_specific_instruction_excerpt": make_text_excerpt(self.get_worker_specific_instruction(), 360),
                "assembled_prompt_excerpt": make_text_excerpt(prompt_text, 900),
            }
        )
        return prompt_context

    def build_prompt_package(
        self,
        analysis_input: WorkerAnalysisInput,
    ) -> tuple[str, dict[str, Any], list[AnalysisWarning]]:
        """Assemble prompt text, structured prompt context, and prompt-level warnings."""
        prompt_text = self.build_prompt_text(analysis_input)
        prompt_context = self.build_prompt_context(analysis_input, prompt_text)
        prompt_warnings = deduplicate_analysis_items(
            validate_worker_prompt_sections(prompt_text) + validate_prompt_analysis_context(analysis_input),
            lambda item: (item.issue_code, item.message, item.dimension, item.source_chunk_id, item.field_name),
        )
        return prompt_text, prompt_context, prompt_warnings

    def prepare_analysis_input(
        self,
        worker_input: WorkerInput,
    ) -> tuple[WorkerAnalysisInput | None, list[AnalysisWarning], list[AnalysisIssue]]:
        """Prepare the shared analysis packet from retrieval output plus processed context."""
        selected_document = selected_document_from_retrieval_result(worker_input.retrieval_result)
        if selected_document is None:
            return None, [], [
                make_analysis_issue(
                    "no_selected_document",
                    "Worker analysis could not start because retrieval did not produce a selected document.",
                    recoverable=True,
                )
            ]

        input_warnings = self.validate_worker_input(worker_input, selected_document)
        graph_context = worker_input.graph_context
        processed_document = graph_context.get("processed_document")
        if not isinstance(processed_document, ProcessedDocument):
            processed_document = process_selected_document(selected_document)

        graph_index = graph_context.get("graph_index")
        if not isinstance(graph_index, GraphDocumentIndex):
            graph_index = build_document_graph(processed_document)

        chunk_embeddings = graph_context.get("chunk_embeddings")
        chunk_retrieval_result = graph_context.get("chunk_retrieval_result")
        explicit_query = graph_context.get("analysis_query")
        analysis_query = explicit_query if isinstance(explicit_query, str) else self.build_analysis_query(processed_document)
        analysis_context, _, _, context_warnings = retrieve_analysis_context(
            processed_document,
            graph_index=graph_index,
            chunk_embeddings=chunk_embeddings if isinstance(chunk_embeddings, list) else None,
            chunk_retrieval_result=(
                chunk_retrieval_result if isinstance(chunk_retrieval_result, ChunkRetrievalResult) else None
            ),
            query_text=analysis_query,
        )

        analysis_input, builder_warnings = build_worker_analysis_input(
            worker_input,
            processed_document,
            analysis_context,
            worker_name=self.worker_name,
            analysis_instructions=self.get_analysis_instructions(worker_input),
            rubric=self.runtime_rubric,
        )
        return analysis_input, input_warnings + context_warnings + builder_warnings, []

    def analyze_to_internal_output(self, worker_input: WorkerInput) -> WorkerAnalysisOutput:
        """Run the shared worker-analysis path and keep the richer internal output."""
        analysis_input, preparation_warnings, preparation_issues = self.prepare_analysis_input(worker_input)
        if analysis_input is None:
            issue_messages = [issue.message for issue in preparation_issues]
            return WorkerAnalysisOutput(
                run_id=worker_input.run_id,
                ticker=worker_input.ticker,
                worker_name=self.worker_name,
                document_type=worker_input.document_type,
                status=ProcessingStatus.NO_DOCUMENT,
                summary=None,
                confidence_score=0.0,
                warnings=[],
                issues=preparation_issues,
                reasoning_trace=WorkerReasoningTrace(
                    analysis_client_name="none",
                    model_name="none",
                    used_mock_client=False,
                    reasoning_notes=[],
                    unresolved_items=issue_messages,
                    raw_response_present=False,
                ),
                raw_output={},
                is_mock_analysis=False,
            )

        prompt_text, prompt_context, prompt_warnings = self.build_prompt_package(analysis_input)
        analysis_output = run_generic_analysis(
            analysis_input,
            analysis_client=self.get_analysis_client(worker_input),
            rubric=self.runtime_rubric,
            prompt_context=prompt_context,
            prompt_text=prompt_text,
        )
        analysis_output.warnings = deduplicate_analysis_items(
            list(preparation_warnings) + list(prompt_warnings) + list(analysis_output.warnings),
            lambda item: (item.issue_code, item.message, item.dimension, item.source_chunk_id, item.field_name),
        )
        return analysis_output

    def build_worker_output(self, analysis_output: WorkerAnalysisOutput) -> WorkerOutput:
        """Map the internal analysis result into the shared worker output contract."""
        return assemble_worker_output(analysis_output)

    def analyze(self, worker_input: WorkerInput) -> WorkerOutput:
        """Run disclosure-specific shared analysis end to end for one selected disclosure."""
        return self.build_worker_output(self.analyze_to_internal_output(worker_input))



## 34. Worker 1: Material Event 8-K / Reg FD Press Release

This worker treats the disclosure as a material-event explanation: what happened, what changed, why it matters, what depends on it, and what expected detail is still missing.


In [ ]:

MATERIAL_EVENT_MASTER_PROMPT = build_worker_specific_instruction(
    worker_label="Material Event 8-K / Reg FD Press Release",
    worker_job=(
        "Evaluate one material-event disclosure as an analyst would: isolate the triggering event, "
        "the announced change, the operational and financial consequences, the dependency structure, "
        "and the detail the disclosure does or does not provide."
    ),
    document_native_focus=(
        "Treat the document as a formal disclosure of a material event, amendment, termination, obligation, "
        "or Reg FD update that should make a concrete development inspectable."
    ),
    what_to_look_for=[
        "Identify the triggering event or decision and the specific part of the business, program, agreement, obligation, or governance process it affects.",
        "Explain what actually changed relative to the prior state described in the document, not what the tone implies.",
        "Evaluate whether the disclosure makes the economic, operational, legal, regulatory, or strategic consequences inspectable.",
        "Note next steps, closing conditions, contingencies, milestones, or third-party dependencies that drive what happens next.",
        "Flag expected details that are missing even though the event is presented as material.",
    ],
    fogging_guidance=[
        "Treat the disclosure as evasive when it announces a material development but leaves scope, economics, counterparties, timing, or consequences too abstract to inspect.",
        "Do not call it fogging merely because some facts remain confidential if the document clearly explains what is known, what is withheld, and why.",
        "Separate defensive legal framing from true operational opacity; a cautious tone alone is not enough."
    ],
    signal_guidance=[
        "Positive signal comes from favorable disclosed changes that are concrete, causal, and supported by inspectable consequences or next steps.",
        "Negative signal comes from disclosed burdens, delays, defaults, lost rights, operational disruption, downside economics, or newly introduced contingencies.",
        "Mixed signal is appropriate when beneficial and adverse consequences coexist. If the facts are incomplete, mark uncertainty explicitly instead of forcing directionality.",
    ],
)


class MaterialEventWorker(BaseAnalysisWorker):
    worker_name = "material_event_worker"
    worker_label = "Material Event 8-K / Reg FD Press Release"
    document_type = DocumentType.MATERIAL_EVENT
    analysis_focus_query = (
        "Triggering event, disclosed change, consequences, dependencies, next steps, framing quality, and missing material detail."
    )
    worker_specific_instruction = MATERIAL_EVENT_MASTER_PROMPT



## 35. Worker 2: Clinical Trial Registry Update / ClinicalTrials.gov

This worker treats the disclosure as a structured registry record whose value comes from coherent metadata maintenance, clear study-state changes, and preserved interpretability.


In [ ]:

CLINICAL_TRIAL_UPDATE_MASTER_PROMPT = build_worker_specific_instruction(
    worker_label="Clinical Trial Registry Update / ClinicalTrials.gov",
    worker_job=(
        "Evaluate one structured registry update by focusing on study status, timing, design, enrollment, endpoints, results status, "
        "record coherence, and whether the update improves or reduces analyst clarity."
    ),
    document_native_focus=(
        "Treat the document as a structured registry record rather than a narrative press release. "
        "Prioritize operational state, metadata quality, and interpretability over rhetorical tone."
    ),
    what_to_look_for=[
        "Identify what appears changed or newly specified in status, timing, enrollment, eligibility, design, endpoints, locations, or results fields.",
        "Assess whether the update increases clarity, reduces clarity, or leaves key comparability questions unresolved.",
        "Evaluate whether execution appears stable, drifting, paused, or only partially explained based on the disclosed metadata.",
        "Check whether the registry record is internally coherent across dates, statuses, enrollment figures, endpoint wording, and site information.",
        "Note where partial reporting, metadata evolution, or missing historical context limits interpretability."
    ],
    fogging_guidance=[
        "Treat the record as evasive when study-state implications are material but the metadata shifts are not explained enough to assess what changed or why.",
        "Do not confuse ordinary structured brevity with fogging if the fields are precise, internally consistent, and sufficient to interpret the study state.",
        "If prior-state comparison is uncertain because the earlier registry version is absent, say so explicitly instead of inventing a delta."
    ],
    signal_guidance=[
        "Positive signal comes from cleaner metadata, more specific timing or endpoint definitions, operational stabilization, or improved comparability and transparency.",
        "Negative signal comes from drift, delayed timelines, enrollment slippage, endpoint ambiguity, incoherent field updates, or partial reporting that reduces interpretability.",
        "A registry update can be neutral in business direction but still positive or negative for clarity; keep those ideas distinct.",
    ],
)


class ClinicalTrialUpdateWorker(BaseAnalysisWorker):
    worker_name = "clinical_trial_update_worker"
    worker_label = "Clinical Trial Registry Update / ClinicalTrials.gov"
    document_type = DocumentType.CLINICAL_TRIAL_UPDATE
    analysis_focus_query = (
        "Study status, timing, design, enrollment, endpoints, results status, metadata coherence, clarity change, and interpretability."
    )
    worker_specific_instruction = CLINICAL_TRIAL_UPDATE_MASTER_PROMPT



## 36. Worker 3: FDA Review Materials / Regulatory Review Documents

This worker reads the document as disciplined regulatory evidence: posture, benefit-risk framing, unresolved review issues, and implications for label scope, manufacturing readiness, and timing.


In [ ]:

FDA_REVIEW_MASTER_PROMPT = build_worker_specific_instruction(
    worker_label="FDA Review Materials / Regulatory Review Documents",
    worker_job=(
        "Evaluate one FDA review or regulatory review document with disciplined regulatory interpretation. "
        "Focus on posture, evidence strength, unresolved issues, safety, CMC readiness, label scope, and timing implications."
    ),
    document_native_focus=(
        "Treat the document as a regulatory review artifact, not an emotional tone document. "
        "The goal is to interpret how the review record frames benefit, risk, remaining deficiencies, and likely pathway implications."
    ),
    what_to_look_for=[
        "Assess overall regulatory posture from the evidence presented, including where the review appears supportive, conditional, unresolved, or constrained.",
        "Evaluate the strength and limitations of the efficacy and safety evidence as described in the review materials.",
        "Identify unresolved review issues, especially safety, tolerability, CMC/manufacturing, process validation, inspection, or post-marketing obligation implications.",
        "Note label or use constraints, population narrowing, monitoring requirements, or pathway implications that affect practical approval value.",
        "Distinguish factual review conclusions from sponsor framing if both appear in the same package."
    ],
    fogging_guidance=[
        "Treat the document as evasive only when material review implications are abstracted away, contradictory, or not linked clearly to the cited evidence.",
        "Do not translate normal regulatory caution into negativity by default; regulatory specificity can be cautious and still clear.",
        "If timing implications depend on unresolved items, state the dependency explicitly rather than pretending the timetable is known."
    ],
    signal_guidance=[
        "Positive signal comes from supportive benefit-risk framing, strong evidence, manageable safety findings, resolved CMC questions, or clear approval pathway support.",
        "Negative signal comes from unresolved deficiencies, meaningful safety liabilities, weak evidence, manufacturing readiness gaps, restrictive labeling, or timing risks tied to open review issues.",
        "Mixed signal is appropriate when the review supports some elements but limits scope, timing, or commercial usefulness."
    ],
)


class FDAReviewWorker(BaseAnalysisWorker):
    worker_name = "fda_review_worker"
    worker_label = "FDA Review Materials / Regulatory Review Documents"
    document_type = DocumentType.FDA_REVIEW
    analysis_focus_query = (
        "Regulatory posture, benefit-risk framing, evidence strength, unresolved issues, safety, manufacturing readiness, label scope, and timing implications."
    )
    worker_specific_instruction = FDA_REVIEW_MASTER_PROMPT



## 37. Worker 4: Financing / Dilution Disclosure

This worker analyzes financing structure and disclosure quality: what capital was raised, on what terms, under what constraints, and what the deal implies about dilution, runway, and negotiating position.


In [ ]:

FINANCING_DILUTION_MASTER_PROMPT = build_worker_specific_instruction(
    worker_label="Financing / Dilution Disclosure",
    worker_job=(
        "Evaluate one financing or dilution-bearing disclosure by analyzing structure, economic implications, use-of-proceeds clarity, restrictions, "
        "runway implications, and what the deal suggests about urgency versus opportunism."
    ),
    document_native_focus=(
        "Treat the document as a capital-structure disclosure rather than a generic good-news or bad-news announcement. "
        "Focus on terms, contingencies, counterparties, dilution mechanics, and downstream constraints."
    ),
    what_to_look_for=[
        "Identify the financing structure, security mix, pricing mechanics, warrants, prefunded instruments, tranches, closing conditions, and counterparties when disclosed.",
        "Assess dilution, overhang, anti-dilution protections, variable-rate exposure, or structural complexity that changes the economic meaning of the financing.",
        "Evaluate use-of-proceeds specificity, liquidity implications, runway claims, and whether management makes the capital need inspectable.",
        "Consider what the financing suggests about bargaining position, urgency, optionality, and whether the company appears constrained or opportunistic.",
        "Note disclosure gaps around fees, restrictions, covenants, registration obligations, lockups, or contingency structure if those omissions matter."
    ],
    fogging_guidance=[
        "Treat the disclosure as evasive when economic meaning is hard to inspect because pricing, dilution mechanics, contingencies, counterparties, or restrictions remain too vague.",
        "Do not label a financing negative solely because it is dilutive; judge the structure, necessity, and clarity of what was disclosed.",
        "A concise term summary can still be clear if it makes proceeds, securities, and constraints inspectable."
    ],
    signal_guidance=[
        "Positive signal comes from clearly structured capital with understandable dilution, credible runway support, and limited hidden constraints.",
        "Negative signal comes from distressed structure, heavy overhang, opaque economics, restrictive terms, or financing that implies a weak bargaining position.",
        "Mixed signal is appropriate when the company secures needed capital but at meaningful cost, complexity, or strategic constraint."
    ],
)


class FinancingDilutionWorker(BaseAnalysisWorker):
    worker_name = "financing_dilution_worker"
    worker_label = "Financing / Dilution Disclosure"
    document_type = DocumentType.FINANCING_DILUTION
    analysis_focus_query = (
        "Financing structure, dilution, overhang, use of proceeds, urgency versus opportunism, restrictions, runway, and bargaining position."
    )
    worker_specific_instruction = FINANCING_DILUTION_MASTER_PROMPT



## 38. Worker 5: Investor Presentation / Corporate Update / Transcript

This worker evaluates management narrative quality: the story being told, what is emphasized or omitted, how specific the update is, and whether the communication improves analyst clarity or mostly steers perception.


In [ ]:

INVESTOR_COMMUNICATION_MASTER_PROMPT = build_worker_specific_instruction(
    worker_label="Investor Presentation / Corporate Update / Transcript",
    worker_job=(
        "Evaluate one investor-facing communication by separating rhetoric from substance, identifying the story management is trying to tell, "
        "and judging whether the communication actually improves analyst clarity."
    ),
    document_native_focus=(
        "Treat the document as a narrative communication that may mix useful operational updates, milestone framing, risk discussion, and presentational polish. "
        "The task is to assess substance, internal consistency, omissions, and framing choices."
    ),
    what_to_look_for=[
        "Identify the central story management is advancing and the specific facts used to support it.",
        "Note what is newly emphasized, de-emphasized, or omitted relative to the rest of the document's disclosed priorities and risk discussion.",
        "Evaluate specificity versus polish: milestones, timelines, operational facts, and quantified statements should carry more weight than presentation quality.",
        "Assess risk treatment, internal consistency across speakers or slides, and whether answers actually resolve analyst-relevant questions.",
        "Distinguish genuine clarity gains from perception steering that leaves the underlying operating picture largely unchanged."
    ],
    fogging_guidance=[
        "Treat the communication as evasive when it repeats high-level themes but avoids operational specifics, tradeoffs, or risk-bearing detail that analysts would reasonably expect.",
        "Do not penalize ordinary presentation structure; polished delivery is only a concern when it substitutes for inspectable substance.",
        "If the communication is balanced and specific about risks, that should lower fogging even when management remains optimistic."
    ],
    signal_guidance=[
        "Positive signal comes from concrete, balanced, internally consistent communication that makes milestones, risks, and priorities more inspectable.",
        "Negative signal comes from selective framing, omitted counterpoints, unsupported confidence, or narrative drift that weakens analyst usefulness.",
        "Mixed signal is appropriate when management provides some clarity while still steering attention away from unresolved operational or financial risks."
    ],
)


class InvestorCommunicationWorker(BaseAnalysisWorker):
    worker_name = "investor_communication_worker"
    worker_label = "Investor Presentation / Corporate Update / Transcript"
    document_type = DocumentType.INVESTOR_COMMUNICATION
    analysis_focus_query = (
        "Management narrative, emphasis versus omission, specificity, milestone framing, risk treatment, consistency, and analyst-useful clarity."
    )
    worker_specific_instruction = INVESTOR_COMMUNICATION_MASTER_PROMPT



## 39. Worker Registry Update

The registry is re-bound here to the specialized Prompt 6 worker implementations so later orchestration can inspect one explicit disclosure-type-to-worker mapping.


In [ ]:

WORKER_REGISTRY: dict[DocumentType, Type[BaseWorker]] = {
    DocumentType.MATERIAL_EVENT: MaterialEventWorker,
    DocumentType.CLINICAL_TRIAL_UPDATE: ClinicalTrialUpdateWorker,
    DocumentType.FDA_REVIEW: FDAReviewWorker,
    DocumentType.FINANCING_DILUTION: FinancingDilutionWorker,
    DocumentType.INVESTOR_COMMUNICATION: InvestorCommunicationWorker,
}

WORKER_REGISTRY_SUMMARY = {
    document_type.value: {
        "worker_name": worker_class.worker_name,
        "class_name": worker_class.__name__,
        "worker_label": getattr(worker_class, "worker_label", worker_class.worker_name),
        "analysis_focus_query": getattr(worker_class, "analysis_focus_query", None),
    }
    for document_type, worker_class in WORKER_REGISTRY.items()
}

print(json.dumps(WORKER_REGISTRY_SUMMARY, indent=2))



## 40. Demo Worker Runs

This demo creates one mock selected document for each disclosure type, prepares chunk evidence through the existing processing stack, runs each specialized worker with the mock analysis client, and prints the structured `WorkerOutput` plus prompt excerpts.


In [ ]:

def make_worker_demo_selected_document(
    *,
    document_id: str,
    document_type: DocumentType,
    title: str,
    ticker: str,
    raw_text: str,
    source_name: str,
    source_url: str,
    source_family: SourceFamily,
    is_structured_source: bool,
    company_name: str = "FakeBio Therapeutics",
    source_identifier: str | None = None,
    published_at: datetime | None = None,
    updated_at: datetime | None = None,
    event_date: date | None = None,
    raw_metadata: dict[str, Any] | None = None,
) -> SelectedDocument:
    """Create one richer mock selected document specifically for Prompt 6 worker demos."""
    selected_document = make_demo_selected_document(
        document_id=document_id,
        document_type=document_type,
        title=title,
        ticker=ticker,
        raw_text=raw_text,
        source_name=source_name,
        source_url=source_url,
    )
    if selected_document.metadata is None:
        raise ValueError("Prompt 6 worker demos require metadata on the mock selected document.")
    selected_document.source_identifier = source_identifier
    selected_document.metadata.company_name = company_name
    selected_document.metadata.source_identifier = source_identifier
    selected_document.metadata.source_family = source_family
    selected_document.metadata.is_structured_source = is_structured_source
    selected_document.metadata.published_at = published_at
    selected_document.metadata.updated_at = updated_at
    selected_document.metadata.event_date = event_date
    selected_document.metadata.raw_metadata = {"prompt6_demo": True, **(raw_metadata or {})}
    selected_document.metadata.notes.append("Prompt 6 worker demo document.")
    selected_document.provenance.append(
        build_provenance_record(
            stage="demo_worker_document",
            adapter_name="prompt6_demo_source",
            candidate_id=document_id,
            document_type=document_type,
            source_name=source_name,
            source_identifier=source_identifier,
            source_url=source_url,
            note="Synthetic Prompt 6 worker demo selected document.",
            is_mock_data=True,
            metadata={"prompt6_demo": True},
        )
    )
    return selected_document


def build_mock_retrieval_result_from_selected_document(
    selected_document: SelectedDocument,
    *,
    request_id: str,
    company_name: str | None = None,
) -> RetrievalResult:
    """Wrap one mock selected document in the existing retrieval-result contract."""
    metadata = selected_document.metadata or DocumentMetadata(
        document_id=selected_document.document_id,
        ticker=selected_document.ticker,
        document_type=selected_document.document_type,
        title=selected_document.title,
        source_name=selected_document.source_name,
        source_identifier=selected_document.source_identifier,
        source_url=selected_document.source_url,
        is_mock_data=True,
    )
    effective_timestamp = metadata.updated_at or metadata.published_at
    if effective_timestamp is None and metadata.event_date is not None:
        effective_timestamp = datetime.combine(metadata.event_date, time.min, tzinfo=timezone.utc)
    request = RetrievalRequest(
        request_id=request_id,
        ticker=selected_document.ticker,
        company_name=company_name or metadata.company_name,
        company_aliases=[selected_document.ticker, company_name or metadata.company_name or selected_document.ticker],
        document_type=selected_document.document_type,
        max_candidates=1,
        minimum_text_chars=PIPELINE_CONFIG.min_usable_text_chars,
        minimum_relevance_score=PIPELINE_CONFIG.minimum_relevance_score,
        freshness_bonus_max=PIPELINE_CONFIG.freshness_bonus_max,
        freshness_rank_decay=PIPELINE_CONFIG.freshness_rank_decay,
        source_preferences=SourcePreferencePolicy(allow_mock_data=True),
        retrieval_notes=["Synthetic Prompt 6 worker demo retrieval request."],
        is_mock_request=True,
    )
    candidate = RetrievalCandidate(
        candidate_id=selected_document.document_id,
        adapter_name="prompt6_demo_adapter",
        document_type=selected_document.document_type,
        ticker=selected_document.ticker,
        company_name=company_name or metadata.company_name,
        source_name=selected_document.source_name,
        source_identifier=selected_document.source_identifier,
        source_url=selected_document.source_url,
        source_family=metadata.source_family,
        title=selected_document.title,
        published_at=metadata.published_at,
        updated_at=metadata.updated_at,
        event_date=metadata.event_date,
        retrieved_at=now_utc(),
        document_text=selected_document.raw_text,
        is_structured_source=metadata.is_structured_source,
        is_mock_data=True,
        relevance_notes=["Synthetic Prompt 6 demo candidate was created directly from the selected document."],
        freshness_notes=["Single demo candidate; freshness comparison is intentionally skipped."],
        validation_notes=["Prompt 6 demo candidate includes usable text and matching metadata."],
        selection_notes=["Selected directly for specialized worker demo."],
        raw_metadata=dict(metadata.raw_metadata),
        provenance=list(selected_document.provenance),
        validation=DocumentValidationResult(
            candidate_id=selected_document.document_id,
            status=ProcessingStatus.SUCCESS,
            is_valid=True,
            is_partial=False,
            has_usable_text=True,
            has_minimum_metadata=True,
            type_matches_request=True,
            identity_matches_request=True,
            validation_notes=["Synthetic Prompt 6 demo candidate validated successfully."],
        ),
        evaluation=CandidateEvaluation(
            candidate_id=selected_document.document_id,
            is_relevant=True,
            relevance_score=100.0,
            freshness_score=100.0,
            total_score=100.0,
            type_compatibility_score=1.0,
            identity_alignment_score=1.0,
            title_alignment_score=1.0,
            structured_source_bonus=0.05 if metadata.is_structured_source else 0.0,
            source_preference_bonus=0.05,
            effective_timestamp=effective_timestamp,
            evaluation_notes=["Synthetic Prompt 6 demo candidate marked fully relevant for worker testing."],
        ),
    )
    return RetrievalResult(
        request=request,
        adapter_name="prompt6_demo_adapter",
        status=ProcessingStatus.SUCCESS,
        selected_candidate=candidate,
        baseline_candidate=candidate,
        candidates=[candidate],
        selection_decision=RetrievalSelectionDecision(
            selected_candidate_id=candidate.candidate_id,
            selected_reason="Synthetic Prompt 6 demo selected document.",
            ranking_notes=["Only one mock candidate was supplied for this specialized worker demo."],
        ),
        provenance=list(selected_document.provenance),
        is_mock_result=True,
    )


DEMO_WORKER_SELECTED_DOCUMENTS: dict[DocumentType, SelectedDocument] = {
    DocumentType.MATERIAL_EVENT: make_worker_demo_selected_document(
        document_id="prompt6_demo_material_event",
        document_type=DocumentType.MATERIAL_EVENT,
        title="FakeBio Announces Manufacturing Transfer and Collaboration Reset",
        ticker="FAKE",
        source_name="Mock SEC Filing Feed",
        source_url="https://example.com/prompt6/material-event",
        source_identifier="prompt6-material-event-001",
        source_family=SourceFamily.OFFICIAL_REGULATORY,
        is_structured_source=True,
        published_at=datetime(2026, 3, 3, 16, 0, tzinfo=timezone.utc),
        updated_at=datetime(2026, 3, 3, 18, 0, tzinfo=timezone.utc),
        event_date=date(2026, 3, 3),
        raw_text="""
FORM 8-K

Item 1.01 Entry into a Material Definitive Agreement
On March 3, 2026, FakeBio Therapeutics terminated its prior ex-U.S. commercialization option with Alder Peak and entered a replacement co-development and supply agreement with Northwind Biologics for FB-401.
Northwind will pay $18 million upfront at closing and reimburse transition costs up to $6 million.
Closing is subject to transfer of manufacturing records, antitrust clearance, and assignment of certain third-party licenses.
If closing is delayed beyond June 30, 2026, either party may terminate.

Operational Effects
FakeBio will move drug-substance manufacturing from its internal pilot plant to Northwind's commercial network and expects one quarter of transition work.
The company paused enrollment expansion in Cohort B until comparability data are reviewed.

Next Steps
Management expects to file a Form 8-K amendment with schedules after confidentiality review.
Forward-looking statements apply.
""".strip(),
    ),
    DocumentType.CLINICAL_TRIAL_UPDATE: make_worker_demo_selected_document(
        document_id="prompt6_demo_clinical_trial",
        document_type=DocumentType.CLINICAL_TRIAL_UPDATE,
        title="FakeBio FB-201 Phase 2 Registry Update",
        ticker="FAKE",
        source_name="Mock Clinical Trial Registry",
        source_url="https://example.com/prompt6/clinical-trial-update",
        source_identifier="prompt6-trial-update-001",
        source_family=SourceFamily.OFFICIAL_REGULATORY,
        is_structured_source=True,
        published_at=datetime(2026, 3, 2, 9, 0, tzinfo=timezone.utc),
        updated_at=datetime(2026, 3, 2, 14, 0, tzinfo=timezone.utc),
        event_date=date(2026, 3, 2),
        raw_text="""
Study Record
Study Title: Randomized Phase 2 Study of FB-201 in Recurrent Disease
Overall Recruitment Status: Active, not recruiting
Last Update Posted: March 2, 2026

Enrollment
Actual Enrollment: 84
Planned Enrollment: 120
Primary Completion Date: December 2026

Study Design
Randomized, open-label, dose-optimization expansion added in Cohort 3.
Steroid washout requirement clarified to 14 days.

Outcome Measures
Primary Outcome: Change from baseline in biomarker score at Week 24
Secondary Outcomes: Confirmed objective response rate through Month 9; durability follow-up through Month 12
No results have been posted.

Contacts and Locations
Boston site withdrawn.
Houston and Denver sites listed as recruiting pending IRB activation.
Imaging schedule updated in the protocol record.
""".strip(),
    ),
    DocumentType.FDA_REVIEW: make_worker_demo_selected_document(
        document_id="prompt6_demo_fda_review",
        document_type=DocumentType.FDA_REVIEW,
        title="FDA Multidisciplinary Review Excerpt for FB-401",
        ticker="FAKE",
        source_name="Mock FDA Review Archive",
        source_url="https://example.com/prompt6/fda-review",
        source_identifier="prompt6-fda-review-001",
        source_family=SourceFamily.OFFICIAL_REGULATORY,
        is_structured_source=True,
        published_at=datetime(2026, 2, 28, 8, 0, tzinfo=timezone.utc),
        updated_at=datetime(2026, 2, 28, 8, 30, tzinfo=timezone.utc),
        event_date=date(2026, 2, 28),
        raw_text="""
FDA Multidisciplinary Review

Clinical Review
Study 301 met the primary endpoint, but confirmatory support from subgroup analyses remains limited.
The review team noted a clinically meaningful response pattern in biomarker-positive patients.

Safety Review
Grade 3 hepatic laboratory abnormalities occurred in 8% of treated patients versus 2% in control.
The review recommends dose interruption and monitoring language in labeling.

CMC Review
Commercial process validation lots were representative, but one comparability package for a post-approval manufacturing site remains under review.

Labeling Considerations
The proposed indication should be limited to biomarker-positive adult patients after prior therapy.
First-line use is not supported by the submitted data.

Action Timing
No late-cycle meeting issue was identified, but approval timing depends on closure of the manufacturing comparability item.
""".strip(),
    ),
    DocumentType.FINANCING_DILUTION: make_worker_demo_selected_document(
        document_id="prompt6_demo_financing",
        document_type=DocumentType.FINANCING_DILUTION,
        title="FakeBio Registered Direct Offering and Warrant Financing",
        ticker="FAKE",
        source_name="Mock SEC Filing Feed",
        source_url="https://example.com/prompt6/financing",
        source_identifier="prompt6-financing-001",
        source_family=SourceFamily.OFFICIAL_REGULATORY,
        is_structured_source=True,
        published_at=datetime(2026, 3, 6, 16, 0, tzinfo=timezone.utc),
        updated_at=datetime(2026, 3, 6, 16, 25, tzinfo=timezone.utc),
        event_date=date(2026, 3, 6),
        raw_text="""
Item 1.01 Securities Purchase Agreement
FakeBio Therapeutics entered into a registered direct offering with two healthcare-focused investors.
The company agreed to sell 12.5 million common shares and prefunded warrants for 5.0 million shares at a combined purchase price of $3.20.
Investors also received five-year warrants for 50% of the purchased share count with an exercise price of $4.10.
Gross proceeds are expected to be approximately $56 million before placement agent fees and expenses.

Use of Proceeds
Net proceeds are expected to support Phase 2 execution, process validation work, and general corporate purposes.

Closing Conditions and Restrictions
Closing is expected on March 9, 2026, subject to customary closing conditions.
The company agreed not to issue variable-rate securities for 45 days after closing.
Management stated that the financing extends runway into the third quarter of 2027 under the current operating plan.
""".strip(),
    ),
    DocumentType.INVESTOR_COMMUNICATION: make_worker_demo_selected_document(
        document_id="prompt6_demo_investor_update",
        document_type=DocumentType.INVESTOR_COMMUNICATION,
        title="FakeBio Corporate Update Transcript",
        ticker="FAKE",
        source_name="Mock Investor Events Page",
        source_url="https://example.com/prompt6/investor-update",
        source_identifier="prompt6-investor-update-001",
        source_family=SourceFamily.ISSUER_PUBLISHED,
        is_structured_source=False,
        published_at=datetime(2026, 3, 5, 13, 0, tzinfo=timezone.utc),
        updated_at=datetime(2026, 3, 5, 13, 10, tzinfo=timezone.utc),
        event_date=date(2026, 3, 5),
        raw_text="""
FakeBio Therapeutics Corporate Update Transcript

Chief Executive Officer:
We are entering 2026 with three priorities: finish site activation, complete process validation, and prepare an end-of-phase 2 package.
The main change since last quarter is slower-than-planned activation at two European sites, partly offset by stronger U.S. screening.

Slide 7 Milestones
Top-line data in the fourth quarter of 2026.
Regulatory meeting by year-end.
Manufacturing readiness work continues in parallel.

Chief Financial Officer:
Cash runway extends into the third quarter of 2027 after the March financing.
We are sequencing spend to support trial execution and CMC readiness.

Analyst Question:
What are the biggest remaining risks?

Chief Executive Officer:
Enrollment consistency, assay turnaround time, and manufacturing comparability remain the main execution risks.
We are not assuming partnership revenue in the current operating plan.
""".strip(),
    ),
}

DEMO_WORKER_ANALYSIS_INPUTS: dict[DocumentType, WorkerAnalysisInput] = {}
DEMO_WORKER_PROMPTS: dict[DocumentType, str] = {}
DEMO_WORKER_PROMPT_CONTEXTS: dict[DocumentType, dict[str, Any]] = {}
DEMO_WORKER_INTERNAL_OUTPUTS: dict[DocumentType, WorkerAnalysisOutput] = {}
DEMO_WORKER_OUTPUTS: dict[DocumentType, WorkerOutput] = {}
DEMO_WORKER_RUN_SUMMARY: dict[str, Any] = {}

for document_type, worker_class in WORKER_REGISTRY.items():
    selected_document = DEMO_WORKER_SELECTED_DOCUMENTS[document_type]
    worker = worker_class(analysis_client=MockAnalysisClient())
    processed_document = process_selected_document(selected_document)
    graph_index = build_document_graph(processed_document)
    chunk_embeddings, embedding_notes = build_chunk_embedding_records(processed_document, EMBEDDING_CONFIG)
    processed_document.processing_notes.extend(embedding_notes)
    analysis_query = worker.build_analysis_query(processed_document)
    chunk_retrieval_result = retrieve_relevant_chunks(
        analysis_query,
        processed_document,
        chunk_embeddings,
        graph_index,
        EMBEDDING_CONFIG,
        GRAPH_RETRIEVAL_CONFIG,
    )
    retrieval_result = build_mock_retrieval_result_from_selected_document(
        selected_document,
        request_id=f"prompt6_demo_{document_type.value}",
        company_name=selected_document.metadata.company_name if selected_document.metadata else None,
    )
    worker_input = WorkerInput(
        run_id=f"prompt6_demo_analysis_{document_type.value}",
        ticker=selected_document.ticker,
        document_type=document_type,
        retrieval_result=retrieval_result,
        config=PIPELINE_CONFIG,
        graph_context={
            "processed_document": processed_document,
            "graph_index": graph_index,
            "chunk_embeddings": chunk_embeddings,
            "chunk_retrieval_result": chunk_retrieval_result,
            "analysis_query": analysis_query,
        },
    )
    analysis_input, preparation_warnings, preparation_issues = worker.prepare_analysis_input(worker_input)
    if analysis_input is None:
        raise RuntimeError(
            f"Expected Prompt 6 analysis input for {document_type.value}. Issues: {preparation_issues}"
        )

    prompt_text, prompt_context, prompt_warnings = worker.build_prompt_package(analysis_input)
    analysis_output = worker.analyze_to_internal_output(worker_input)
    worker_output = worker.build_worker_output(analysis_output)

    DEMO_WORKER_ANALYSIS_INPUTS[document_type] = analysis_input
    DEMO_WORKER_PROMPTS[document_type] = prompt_text
    DEMO_WORKER_PROMPT_CONTEXTS[document_type] = prompt_context
    DEMO_WORKER_INTERNAL_OUTPUTS[document_type] = analysis_output
    DEMO_WORKER_OUTPUTS[document_type] = worker_output
    DEMO_WORKER_RUN_SUMMARY[document_type.value] = {
        "worker_name": worker.worker_name,
        "worker_label": worker.worker_label,
        "analysis_query": analysis_query,
        "prompt_sections_present": {
            section: section in prompt_text for section in WORKER_PROMPT_REQUIRED_SECTIONS
        },
        "prompt_excerpt": make_text_excerpt(prompt_text, 720),
        "worker_specific_instruction_excerpt": make_text_excerpt(worker.get_worker_specific_instruction(), 260),
        "evidence_bundle_count": len(analysis_input.top_chunk_bundles),
        "selected_chunk_ids": [bundle.chunk_id for bundle in analysis_input.top_chunk_bundles],
        "preparation_warning_codes": [warning.issue_code for warning in preparation_warnings],
        "prompt_warning_codes": [warning.issue_code for warning in prompt_warnings],
        "analysis_warning_codes": [warning.issue_code for warning in analysis_output.warnings],
        "worker_output": worker_output.model_dump(mode="json"),
    }

print(json.dumps(DEMO_WORKER_RUN_SUMMARY, indent=2))


## 41. Prompt 6 Step Completion Notes

Prompt 6 completed the worker-specialization layer:
- modular prompt assembly helpers that build shared instructions, worker-specific master prompts, metadata/context blocks, and output-schema blocks
- a prompt-aware `BaseAnalysisWorker` path that assembles specialized prompt text/context without changing the shared `WorkerOutput` contract
- five disclosure-specific worker subclasses for material events, clinical trial registry updates, FDA review materials, financing disclosures, and investor communications
- explicit worker registry updates plus runnable mock demos for all five disclosure types
- structured warnings for prompt gaps, thin evidence, sparse prompt context, worker-type mismatches, and missing output fields

Prompt 7 begins below and adds the cross-document arbiter layer.


## 42. Arbiter Design Notes

Prompt 7 adds the arbiter layer and keeps it heuristic-based, explicit, and inspectable.

Design choices in this step:
- The arbiter consumes only structured `WorkerOutput` objects.
- Cross-document comparison stays rule-based so agreement, divergence, and contradiction remain auditable.
- Confidence reconciliation is additive and documented rather than opaque.
- Provenance is preserved back to worker name, document type, and evidence references.
- The arbiter emits an intermediate `ArbiterOutput` for downstream master-node use only.

Still deferred:
- final master-node aggregation
- final UI payload assembly
- tests, which should continue to live in separate notebooks


## 43. Arbiter Models and Enums


In [ ]:
from collections import defaultdict
from itertools import combinations
from typing import Hashable

EXPECTED_DOCUMENT_TYPES = list(DocumentType)
USABLE_WORKER_STATUSES = {ProcessingStatus.SUCCESS, ProcessingStatus.PARTIAL}
STRUCTURED_DOCUMENT_TYPES = {
    DocumentType.MATERIAL_EVENT,
    DocumentType.CLINICAL_TRIAL_UPDATE,
    DocumentType.FDA_REVIEW,
    DocumentType.FINANCING_DILUTION,
}
SOFT_DOCUMENT_TYPES = {DocumentType.INVESTOR_COMMUNICATION}


class ArbiterKind(str, Enum):
    SUMMARY = "summary"
    SENTIMENT = "sentiment"
    CROSS_DOCUMENT = "cross_document"


class NormalizedSignalDirection(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    MIXED = "mixed"
    NEUTRAL = "neutral"
    UNCERTAIN = "uncertain"


class ArbiterSignalCategory(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    UNCERTAINTY = "uncertainty"
    FOGGING = "fogging"


class ArbiterDecisionType(str, Enum):
    ALIGNED_SIGNAL = "aligned_signal"
    CONTRADICTORY_SIGNAL = "contradictory_signal"
    UNRESOLVED_AMBIGUITY = "unresolved_ambiguity"
    MATERIAL_CONCERN = "material_concern"
    MATERIAL_POSITIVE = "material_positive"
    CROSS_DOCUMENT_UNCERTAINTY = "cross_document_uncertainty"
    STORY_SUBSTANCE_MISMATCH = "story_vs_substance_mismatch"


class CrossDocumentTheme(str, Enum):
    CLINICAL_EXECUTION = "clinical_execution"
    REGULATORY_POSTURE = "regulatory_posture"
    FINANCING_AND_RUNWAY = "financing_and_runway"
    OPERATIONAL_EXECUTION = "operational_execution"
    NARRATIVE_CREDIBILITY = "narrative_credibility"


THEME_LABELS: dict[CrossDocumentTheme, str] = {
    CrossDocumentTheme.CLINICAL_EXECUTION: "Clinical execution",
    CrossDocumentTheme.REGULATORY_POSTURE: "Regulatory posture",
    CrossDocumentTheme.FINANCING_AND_RUNWAY: "Financing and runway",
    CrossDocumentTheme.OPERATIONAL_EXECUTION: "Operational execution",
    CrossDocumentTheme.NARRATIVE_CREDIBILITY: "Narrative credibility",
}

THEME_DOCUMENT_RELEVANCE: dict[CrossDocumentTheme, dict[DocumentType, float]] = {
    CrossDocumentTheme.CLINICAL_EXECUTION: {
        DocumentType.CLINICAL_TRIAL_UPDATE: 1.00,
        DocumentType.MATERIAL_EVENT: 0.75,
        DocumentType.FDA_REVIEW: 0.65,
        DocumentType.INVESTOR_COMMUNICATION: 0.45,
    },
    CrossDocumentTheme.REGULATORY_POSTURE: {
        DocumentType.FDA_REVIEW: 1.00,
        DocumentType.MATERIAL_EVENT: 0.75,
        DocumentType.CLINICAL_TRIAL_UPDATE: 0.55,
        DocumentType.INVESTOR_COMMUNICATION: 0.40,
    },
    CrossDocumentTheme.FINANCING_AND_RUNWAY: {
        DocumentType.FINANCING_DILUTION: 1.00,
        DocumentType.MATERIAL_EVENT: 0.70,
        DocumentType.INVESTOR_COMMUNICATION: 0.50,
    },
    CrossDocumentTheme.OPERATIONAL_EXECUTION: {
        DocumentType.MATERIAL_EVENT: 0.85,
        DocumentType.CLINICAL_TRIAL_UPDATE: 0.90,
        DocumentType.FDA_REVIEW: 0.70,
        DocumentType.FINANCING_DILUTION: 0.55,
        DocumentType.INVESTOR_COMMUNICATION: 0.45,
    },
    CrossDocumentTheme.NARRATIVE_CREDIBILITY: {
        DocumentType.INVESTOR_COMMUNICATION: 1.00,
        DocumentType.MATERIAL_EVENT: 0.80,
        DocumentType.CLINICAL_TRIAL_UPDATE: 0.80,
        DocumentType.FDA_REVIEW: 0.85,
        DocumentType.FINANCING_DILUTION: 0.80,
    },
}


class ArbiterIssue(ContractModel):
    issue_code: str
    message: str
    severity: AnalysisIssueSeverity = AnalysisIssueSeverity.WARNING
    document_types: list[DocumentType] = Field(default_factory=list)
    worker_names: list[str] = Field(default_factory=list)
    recoverable: bool = True
    metadata: dict[str, Any] = Field(default_factory=dict)


class ArbiterWarning(ArbiterIssue):
    severity: AnalysisIssueSeverity = AnalysisIssueSeverity.WARNING


class ArbiterEvidenceReference(ContractModel):
    worker_name: str
    document_type: DocumentType
    evidence_id: str | None = None
    document_id: str
    source_url: str | None = None
    source_chunk_id: str | None = None
    source_section_id: str | None = None
    section_title: str | None = None
    interpretation: EvidenceInterpretation | None = None
    snippet_text: str | None = None
    rationale: str | None = None


class NormalizedWorkerOutput(ContractModel):
    worker_name: str
    document_type: DocumentType
    status: ProcessingStatus
    summary: str | None = None
    sentiment_label: SentimentLabel = SentimentLabel.INSUFFICIENT_EVIDENCE
    sentiment_score: float | None = None
    worker_confidence_raw: float | None = None
    normalized_confidence: float = 0.0
    evidence_density: float = 0.0
    key_points: list[str] = Field(default_factory=list)
    caveats: list[str] = Field(default_factory=list)
    evidence_count: int = 0
    key_point_count: int = 0
    caveat_count: int = 0
    issue_count: int = 0
    direction: NormalizedSignalDirection = NormalizedSignalDirection.UNCERTAIN
    fogging_score: float | None = None
    hedging_score: float | None = None
    promotional_score: float | None = None
    is_structured_document: bool = False
    is_soft_document: bool = False
    warnings: list[ArbiterWarning] = Field(default_factory=list)
    issues: list[ArbiterIssue] = Field(default_factory=list)
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)
    normalization_notes: list[str] = Field(default_factory=list)

    @field_validator("normalized_confidence", "evidence_density")
    @classmethod
    def validate_unit_interval(cls, value: float) -> float:
        return _validate_optional_range(value, 0.0, 1.0, "arbiter normalized value")


class ThematicWorkerSignal(ContractModel):
    theme: CrossDocumentTheme
    worker_name: str
    document_type: DocumentType
    direction: NormalizedSignalDirection
    relevance_weight: float
    adjusted_confidence: float
    evidence_density: float
    sentiment_score: float | None = None
    fogging_score: float | None = None
    hedging_score: float | None = None
    promotional_score: float | None = None
    is_structured_document: bool = False
    is_soft_document: bool = False
    summary: str | None = None
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)

    @field_validator("relevance_weight", "adjusted_confidence", "evidence_density")
    @classmethod
    def validate_unit_interval(cls, value: float) -> float:
        return _validate_optional_range(value, 0.0, 1.0, "arbiter thematic value")


class ArbiterFinding(ContractModel):
    finding_id: str
    category: ArbiterSignalCategory
    decision_type: ArbiterDecisionType
    theme: CrossDocumentTheme | None = None
    title: str
    summary: str
    supporting_document_types: list[DocumentType] = Field(default_factory=list)
    contradicting_document_types: list[DocumentType] = Field(default_factory=list)
    worker_names: list[str] = Field(default_factory=list)
    confidence: float | None = None
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)
    reasoning_notes: list[str] = Field(default_factory=list)

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "finding confidence")


class ArbiterSignalGroup(ContractModel):
    group_id: str
    category: ArbiterSignalCategory
    theme: CrossDocumentTheme | None = None
    title: str
    description: str
    document_types: list[DocumentType] = Field(default_factory=list)
    worker_names: list[str] = Field(default_factory=list)
    findings: list[ArbiterFinding] = Field(default_factory=list)
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)
    confidence: float | None = None
    reasoning_notes: list[str] = Field(default_factory=list)

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "signal group confidence")


class ArbiterConflict(ContractModel):
    conflict_id: str
    theme: CrossDocumentTheme
    title: str
    description: str
    positive_document_types: list[DocumentType] = Field(default_factory=list)
    negative_document_types: list[DocumentType] = Field(default_factory=list)
    worker_names: list[str] = Field(default_factory=list)
    high_confidence_conflict: bool = False
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)
    reasoning_notes: list[str] = Field(default_factory=list)


class CrossDocumentJudgment(ContractModel):
    judgment_id: str
    theme: CrossDocumentTheme
    decision_type: ArbiterDecisionType
    direction: NormalizedSignalDirection
    summary: str
    supporting_document_types: list[DocumentType] = Field(default_factory=list)
    opposing_document_types: list[DocumentType] = Field(default_factory=list)
    worker_names: list[str] = Field(default_factory=list)
    confidence: float | None = None
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)
    reasoning_notes: list[str] = Field(default_factory=list)

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "judgment confidence")


class ArbiterConfidenceAdjustment(ContractModel):
    factor_name: str
    adjustment: float
    rationale: str


class ArbiterConfidenceAssessment(ContractModel):
    base_confidence: float
    final_confidence: float
    worker_confidence_average: float | None = None
    evidence_density_average: float | None = None
    structured_support_ratio: float | None = None
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    high_confidence_conflict_count: int = 0
    adjustments: list[ArbiterConfidenceAdjustment] = Field(default_factory=list)
    reasoning_notes: list[str] = Field(default_factory=list)

    @field_validator(
        "base_confidence",
        "final_confidence",
        "worker_confidence_average",
        "evidence_density_average",
        "structured_support_ratio",
    )
    @classmethod
    def validate_unit_interval(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "arbiter confidence value")


class ArbiterOutput(ContractModel):
    arbiter_id: str
    arbiter_name: str
    arbiter_kind: ArbiterKind = ArbiterKind.CROSS_DOCUMENT
    status: ProcessingStatus = ProcessingStatus.PENDING
    summary: str | None = None
    sentiment: SentimentAssessment | None = None
    tone: ToneAssessment | None = None
    covered_document_types: list[DocumentType] = Field(default_factory=list)
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    cross_document_judgments: list[CrossDocumentJudgment] = Field(default_factory=list)
    positive_signal_groups: list[ArbiterSignalGroup] = Field(default_factory=list)
    negative_signal_groups: list[ArbiterSignalGroup] = Field(default_factory=list)
    aligned_signals: list[ArbiterSignalGroup] = Field(default_factory=list)
    conflicting_signals: list[ArbiterConflict] = Field(default_factory=list)
    unresolved_uncertainties: list[ArbiterFinding] = Field(default_factory=list)
    fogging_or_story_substance_flags: list[ArbiterFinding] = Field(default_factory=list)
    confidence_assessment: ArbiterConfidenceAssessment | None = None
    missing_coverage_notes: list[str] = Field(default_factory=list)
    evidence_references: list[ArbiterEvidenceReference] = Field(default_factory=list)
    warnings: list[ArbiterWarning] = Field(default_factory=list)
    issues: list[ArbiterIssue] = Field(default_factory=list)
    reasoning_notes: list[str] = Field(default_factory=list)
    consensus_points: list[str] = Field(default_factory=list)
    conflicts: list[str] = Field(default_factory=list)
    evidence: list[EvidenceSnippet] = Field(default_factory=list)
    confidence: float | None = None

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "arbiter output confidence")


class MasterInput(ContractModel):
    run_id: str
    ticker: str
    worker_outputs: list[WorkerOutput]
    arbiter_outputs: list[ArbiterOutput]
    retrieval_results: list[RetrievalResult]
    config: PipelineConfig


class MasterOutput(ContractModel):
    ticker: str
    status: ProcessingStatus = ProcessingStatus.PENDING
    master_summary: str | None = None
    master_sentiment: SentimentAssessment | None = None
    master_tone: ToneAssessment | None = None
    worker_outputs: list[WorkerOutput] = Field(default_factory=list)
    arbiter_outputs: list[ArbiterOutput] = Field(default_factory=list)
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    ready_for_ui: bool = False
    generated_at: datetime = Field(default_factory=now_utc)


class PipelineState(ContractModel):
    run_id: str
    ticker: str
    config: PipelineConfig
    retrieval_results: dict[DocumentType, RetrievalResult] = Field(default_factory=dict)
    worker_outputs: dict[DocumentType, WorkerOutput] = Field(default_factory=dict)
    arbiter_outputs: list[ArbiterOutput] = Field(default_factory=list)
    master_output: MasterOutput | None = None
    graph_context: dict[str, Any] = Field(default_factory=dict)
    shared_cache_refs: dict[str, str] = Field(default_factory=dict)
    errors: list[PipelineError] = Field(default_factory=list)
    created_at: datetime = Field(default_factory=now_utc)
    updated_at: datetime = Field(default_factory=now_utc)


## 44. Worker Output Normalization Helpers


In [ ]:
def clamp_value(value: float, minimum: float, maximum: float) -> float:
    return max(minimum, min(maximum, value))


def safe_mean(values: Sequence[float | None]) -> float | None:
    present_values = [value for value in values if value is not None]
    if not present_values:
        return None
    return sum(present_values) / len(present_values)


def stable_unique(items: Sequence[Hashable]) -> list[Any]:
    seen: set[Hashable] = set()
    ordered: list[Any] = []
    for item in items:
        if item in seen:
            continue
        seen.add(item)
        ordered.append(item)
    return ordered


def document_type_label(document_type: DocumentType) -> str:
    return DISCLOSURE_TYPE_LABELS.get(document_type.value, document_type.value.replace("_", " "))


def make_arbiter_issue(
    issue_code: str,
    message: str,
    *,
    severity: AnalysisIssueSeverity = AnalysisIssueSeverity.WARNING,
    document_types: Sequence[DocumentType] | None = None,
    worker_names: Sequence[str] | None = None,
    recoverable: bool = True,
    metadata: dict[str, Any] | None = None,
) -> ArbiterIssue:
    return ArbiterIssue(
        issue_code=issue_code,
        message=message,
        severity=severity,
        document_types=list(document_types or []),
        worker_names=list(worker_names or []),
        recoverable=recoverable,
        metadata=metadata or {},
    )


def make_arbiter_warning(
    issue_code: str,
    message: str,
    *,
    document_types: Sequence[DocumentType] | None = None,
    worker_names: Sequence[str] | None = None,
    metadata: dict[str, Any] | None = None,
) -> ArbiterWarning:
    return ArbiterWarning(
        issue_code=issue_code,
        message=message,
        document_types=list(document_types or []),
        worker_names=list(worker_names or []),
        metadata=metadata or {},
    )


def worker_status_rank(status: ProcessingStatus) -> int:
    ranking = {
        ProcessingStatus.SUCCESS: 5,
        ProcessingStatus.PARTIAL: 4,
        ProcessingStatus.PENDING: 3,
        ProcessingStatus.NO_DOCUMENT: 2,
        ProcessingStatus.RETRIEVAL_FAILED: 1,
        ProcessingStatus.SELECTION_FAILED: 1,
        ProcessingStatus.EXTRACTION_FAILED: 1,
        ProcessingStatus.ANALYSIS_FAILED: 1,
    }
    return ranking.get(status, 0)


def worker_output_quality_key(worker_output: WorkerOutput) -> tuple[float, float, int, int]:
    sentiment_confidence = worker_output.sentiment.confidence if worker_output.sentiment else None
    confidence = worker_output.confidence if worker_output.confidence is not None else sentiment_confidence or 0.0
    return (float(worker_status_rank(worker_output.status)), float(confidence), len(worker_output.evidence), len(worker_output.key_points))


def select_best_worker_output_per_type(
    worker_outputs: Sequence[WorkerOutput],
) -> tuple[list[WorkerOutput], list[ArbiterWarning]]:
    grouped: dict[DocumentType, list[WorkerOutput]] = defaultdict(list)
    for worker_output in worker_outputs:
        grouped[worker_output.document_type].append(worker_output)

    selected_outputs: list[WorkerOutput] = []
    warnings: list[ArbiterWarning] = []
    for document_type in EXPECTED_DOCUMENT_TYPES:
        group = grouped.get(document_type, [])
        if not group:
            continue
        if len(group) > 1:
            ordered_group = sorted(group, key=worker_output_quality_key, reverse=True)
            selected_outputs.append(ordered_group[0])
            warnings.append(
                make_arbiter_warning(
                    "duplicate_worker_outputs",
                    f"Multiple worker outputs were provided for {document_type_label(document_type)}; the strongest one was kept.",
                    document_types=[document_type],
                    worker_names=[output.worker_name for output in group],
                    metadata={"kept_worker": ordered_group[0].worker_name},
                )
            )
            continue
        selected_outputs.append(group[0])

    return selected_outputs, warnings


def flatten_worker_evidence_references(worker_output: WorkerOutput) -> list[ArbiterEvidenceReference]:
    return [
        ArbiterEvidenceReference(
            worker_name=worker_output.worker_name,
            document_type=worker_output.document_type,
            evidence_id=evidence.evidence_id,
            document_id=evidence.document_id,
            source_url=evidence.source_url,
            source_chunk_id=evidence.source_chunk_id,
            source_section_id=evidence.source_section_id,
            section_title=evidence.section_title,
            interpretation=evidence.interpretation,
            snippet_text=evidence.snippet_text,
            rationale=evidence.rationale,
        )
        for evidence in worker_output.evidence
    ]


def deduplicate_arbiter_evidence(
    evidence_references: Sequence[ArbiterEvidenceReference],
) -> list[ArbiterEvidenceReference]:
    deduplicated: dict[tuple[Any, ...], ArbiterEvidenceReference] = {}
    for reference in evidence_references:
        reference_key = (
            reference.worker_name,
            reference.document_type.value,
            reference.document_id,
            reference.evidence_id,
            reference.source_chunk_id,
            reference.snippet_text,
        )
        if reference_key not in deduplicated:
            deduplicated[reference_key] = reference
    return list(deduplicated.values())


def deduplicate_evidence_snippets(evidence_snippets: Sequence[EvidenceSnippet]) -> list[EvidenceSnippet]:
    deduplicated: dict[tuple[Any, ...], EvidenceSnippet] = {}
    for snippet in evidence_snippets:
        snippet_key = (
            snippet.document_id,
            snippet.evidence_id,
            snippet.source_chunk_id,
            snippet.snippet_text,
            snippet.rationale,
        )
        if snippet_key not in deduplicated:
            deduplicated[snippet_key] = snippet
    return list(deduplicated.values())


def convert_worker_pipeline_issue_to_arbiter_issue(
    worker_output: WorkerOutput,
    issue: PipelineError,
) -> ArbiterIssue:
    severity_value = str(issue.details.get("severity", AnalysisIssueSeverity.WARNING.value))
    severity = AnalysisIssueSeverity(severity_value) if severity_value in AnalysisIssueSeverity._value2member_map_ else AnalysisIssueSeverity.WARNING
    return make_arbiter_issue(
        issue.error_code,
        issue.message,
        severity=severity,
        document_types=[worker_output.document_type] if worker_output.document_type else [],
        worker_names=[worker_output.worker_name],
        recoverable=issue.recoverable,
        metadata={"stage": issue.stage, **issue.details},
    )


def normalize_worker_sentiment_label(worker_output: WorkerOutput) -> SentimentLabel:
    if worker_output.sentiment is None:
        return SentimentLabel.INSUFFICIENT_EVIDENCE
    return worker_output.sentiment.label


def estimate_worker_evidence_density(worker_output: WorkerOutput) -> float:
    if worker_output.status == ProcessingStatus.NO_DOCUMENT:
        return 0.0

    density = 0.0
    density += min(0.70, len(worker_output.evidence) * 0.20)
    density += min(0.20, len(worker_output.key_points) * 0.04)
    density += 0.10 if worker_output.summary else 0.0
    if worker_output.status == ProcessingStatus.PARTIAL:
        density -= 0.10
    elif worker_output.status not in USABLE_WORKER_STATUSES:
        density -= 0.20
    density -= min(0.10, len(worker_output.issues) * 0.02)
    return clamp_value(density, 0.0, 1.0)


def validate_worker_output_for_arbiter(
    worker_output: WorkerOutput,
) -> tuple[list[ArbiterWarning], list[ArbiterIssue]]:
    warnings: list[ArbiterWarning] = []
    issues: list[ArbiterIssue] = [
        convert_worker_pipeline_issue_to_arbiter_issue(worker_output, issue) for issue in worker_output.issues
    ]

    if worker_output.status not in USABLE_WORKER_STATUSES:
        warnings.append(
            make_arbiter_warning(
                "worker_output_not_usable",
                f"{worker_output.worker_name} returned status {worker_output.status.value}; its signal will be treated cautiously.",
                document_types=[worker_output.document_type],
                worker_names=[worker_output.worker_name],
            )
        )
    if not worker_output.summary:
        warnings.append(make_arbiter_warning("worker_output_missing_summary", "Worker output is missing a summary field.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name]))
    if worker_output.sentiment is None:
        warnings.append(make_arbiter_warning("worker_output_missing_sentiment", "Worker output is missing sentiment data.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name]))
    if not worker_output.evidence:
        warnings.append(make_arbiter_warning("worker_output_missing_evidence", "Worker output has no cited evidence.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name]))
    if not worker_output.key_points:
        warnings.append(make_arbiter_warning("worker_output_missing_key_points", "Worker output has no key points for comparison.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name]))

    confidence_candidates = [worker_output.confidence, worker_output.sentiment.confidence if worker_output.sentiment else None, worker_output.tone.confidence if worker_output.tone else None]
    if safe_mean(confidence_candidates) is None:
        warnings.append(make_arbiter_warning("worker_output_missing_confidence", "Worker output is missing both direct and nested confidence fields.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name]))

    if worker_output.sentiment is not None and worker_output.sentiment.score is not None:
        score = worker_output.sentiment.score
        label = worker_output.sentiment.label
        incompatible_label = (
            (label == SentimentLabel.POSITIVE and score < -0.10)
            or (label == SentimentLabel.NEGATIVE and score > 0.10)
            or (label == SentimentLabel.NEUTRAL and abs(score) > 0.35)
        )
        if incompatible_label:
            warnings.append(make_arbiter_warning("sentiment_label_score_mismatch", "Worker sentiment label and score point in materially different directions.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name], metadata={"sentiment_label": label.value, "sentiment_score": score}))

    evidence_density = estimate_worker_evidence_density(worker_output)
    if evidence_density < 0.30 and worker_output.status in USABLE_WORKER_STATUSES:
        warnings.append(make_arbiter_warning("worker_output_sparse_signal", "Worker output has sparse evidence density for cross-document comparison.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name], metadata={"evidence_density": round(evidence_density, 3)}))

    return warnings, issues


def normalize_signal_direction(worker_output: WorkerOutput, *, normalized_confidence: float) -> NormalizedSignalDirection:
    if worker_output.status not in USABLE_WORKER_STATUSES:
        return NormalizedSignalDirection.UNCERTAIN
    sentiment = worker_output.sentiment
    if sentiment is None:
        return NormalizedSignalDirection.UNCERTAIN
    if sentiment.label == SentimentLabel.POSITIVE:
        return NormalizedSignalDirection.POSITIVE
    if sentiment.label == SentimentLabel.NEGATIVE:
        return NormalizedSignalDirection.NEGATIVE
    if sentiment.label == SentimentLabel.MIXED:
        return NormalizedSignalDirection.MIXED
    if sentiment.label == SentimentLabel.NEUTRAL:
        return NormalizedSignalDirection.NEUTRAL if normalized_confidence >= 0.55 else NormalizedSignalDirection.UNCERTAIN
    return NormalizedSignalDirection.UNCERTAIN


def normalize_worker_output_for_arbitration(worker_output: WorkerOutput) -> NormalizedWorkerOutput:
    warnings, issues = validate_worker_output_for_arbiter(worker_output)
    sentiment_label = normalize_worker_sentiment_label(worker_output)
    sentiment_score = worker_output.sentiment.score if worker_output.sentiment else None
    evidence_density = estimate_worker_evidence_density(worker_output)
    evidence_references = flatten_worker_evidence_references(worker_output)
    confidence_candidates = [worker_output.confidence, worker_output.sentiment.confidence if worker_output.sentiment else None, worker_output.tone.confidence if worker_output.tone else None]
    worker_confidence_raw = safe_mean(confidence_candidates)
    normalized_confidence = worker_confidence_raw if worker_confidence_raw is not None else 0.35
    normalization_notes: list[str] = []

    if worker_output.status == ProcessingStatus.PARTIAL:
        normalized_confidence = clamp_value(normalized_confidence - 0.08, 0.0, 1.0)
        normalization_notes.append("Reduced confidence slightly because the worker status is partial.")
    elif worker_output.status not in USABLE_WORKER_STATUSES:
        normalized_confidence = min(normalized_confidence, 0.25)
        normalization_notes.append("Capped confidence because the worker did not return a usable status.")
    if evidence_density < 0.30:
        normalized_confidence = clamp_value(normalized_confidence - 0.07, 0.0, 1.0)
        normalization_notes.append("Reduced confidence because evidence density is sparse.")

    direction = normalize_signal_direction(worker_output, normalized_confidence=normalized_confidence)
    if normalized_confidence < 0.45:
        warnings.append(make_arbiter_warning("worker_output_low_confidence", "Worker output carries low confidence after arbiter normalization.", document_types=[worker_output.document_type], worker_names=[worker_output.worker_name], metadata={"normalized_confidence": round(normalized_confidence, 3)}))

    return NormalizedWorkerOutput(
        worker_name=worker_output.worker_name,
        document_type=worker_output.document_type,
        status=worker_output.status,
        summary=worker_output.summary,
        sentiment_label=sentiment_label,
        sentiment_score=sentiment_score,
        worker_confidence_raw=worker_confidence_raw,
        normalized_confidence=normalized_confidence,
        evidence_density=evidence_density,
        key_points=list(worker_output.key_points),
        caveats=list(worker_output.caveats),
        evidence_count=len(worker_output.evidence),
        key_point_count=len(worker_output.key_points),
        caveat_count=len(worker_output.caveats),
        issue_count=len(worker_output.issues),
        direction=direction,
        fogging_score=worker_output.tone.fogging_score if worker_output.tone else None,
        hedging_score=worker_output.tone.hedging_score if worker_output.tone else None,
        promotional_score=worker_output.tone.promotional_score if worker_output.tone else None,
        is_structured_document=worker_output.document_type in STRUCTURED_DOCUMENT_TYPES,
        is_soft_document=worker_output.document_type in SOFT_DOCUMENT_TYPES,
        warnings=warnings,
        issues=issues,
        evidence_references=evidence_references,
        normalization_notes=normalization_notes,
    )


def extract_worker_signal_profile(normalized_output: NormalizedWorkerOutput) -> dict[str, Any]:
    return {
        "worker_name": normalized_output.worker_name,
        "document_type": normalized_output.document_type.value,
        "status": normalized_output.status.value,
        "direction": normalized_output.direction.value,
        "sentiment_label": normalized_output.sentiment_label.value,
        "sentiment_score": normalized_output.sentiment_score,
        "normalized_confidence": round(normalized_output.normalized_confidence, 3),
        "evidence_density": round(normalized_output.evidence_density, 3),
        "evidence_count": normalized_output.evidence_count,
        "key_point_count": normalized_output.key_point_count,
        "fogging_score": normalized_output.fogging_score,
        "hedging_score": normalized_output.hedging_score,
        "promotional_score": normalized_output.promotional_score,
    }


## 45. Cross-Document Comparison Logic


In [ ]:
def build_cross_document_themes(
    normalized_outputs: Sequence[NormalizedWorkerOutput],
) -> dict[CrossDocumentTheme, list[ThematicWorkerSignal]]:
    theme_map: dict[CrossDocumentTheme, list[ThematicWorkerSignal]] = defaultdict(list)
    for normalized_output in normalized_outputs:
        for theme, relevance_map in THEME_DOCUMENT_RELEVANCE.items():
            relevance_weight = relevance_map.get(normalized_output.document_type, 0.0)
            if relevance_weight <= 0.0:
                continue
            adjusted_confidence = clamp_value(
                normalized_output.normalized_confidence * relevance_weight * (0.50 + (0.50 * normalized_output.evidence_density)),
                0.0,
                1.0,
            )
            theme_map[theme].append(
                ThematicWorkerSignal(
                    theme=theme,
                    worker_name=normalized_output.worker_name,
                    document_type=normalized_output.document_type,
                    direction=normalized_output.direction,
                    relevance_weight=relevance_weight,
                    adjusted_confidence=adjusted_confidence,
                    evidence_density=normalized_output.evidence_density,
                    sentiment_score=normalized_output.sentiment_score,
                    fogging_score=normalized_output.fogging_score,
                    hedging_score=normalized_output.hedging_score,
                    promotional_score=normalized_output.promotional_score,
                    is_structured_document=normalized_output.is_structured_document,
                    is_soft_document=normalized_output.is_soft_document,
                    summary=normalized_output.summary,
                    evidence_references=list(normalized_output.evidence_references),
                )
            )
    return dict(theme_map)


def summarize_theme_support(theme_signals: Sequence[ThematicWorkerSignal]) -> dict[str, Any]:
    support: dict[NormalizedSignalDirection, float] = defaultdict(float)
    supporting_docs: dict[NormalizedSignalDirection, list[DocumentType]] = defaultdict(list)
    supporting_workers: dict[NormalizedSignalDirection, list[str]] = defaultdict(list)
    supporting_evidence: dict[NormalizedSignalDirection, list[ArbiterEvidenceReference]] = defaultdict(list)
    for signal in theme_signals:
        support[signal.direction] += signal.adjusted_confidence
        supporting_docs[signal.direction].append(signal.document_type)
        supporting_workers[signal.direction].append(signal.worker_name)
        supporting_evidence[signal.direction].extend(signal.evidence_references[:2])
    return {
        "support": support,
        "supporting_docs": {direction: stable_unique(values) for direction, values in supporting_docs.items()},
        "supporting_workers": {direction: stable_unique(values) for direction, values in supporting_workers.items()},
        "supporting_evidence": {direction: deduplicate_arbiter_evidence(values) for direction, values in supporting_evidence.items()},
    }


def build_theme_judgment(theme: CrossDocumentTheme, theme_signals: Sequence[ThematicWorkerSignal]) -> CrossDocumentJudgment:
    theme_support = summarize_theme_support(theme_signals)
    support = theme_support["support"]
    positive_support = float(support.get(NormalizedSignalDirection.POSITIVE, 0.0))
    negative_support = float(support.get(NormalizedSignalDirection.NEGATIVE, 0.0))
    mixed_support = float(support.get(NormalizedSignalDirection.MIXED, 0.0))
    uncertain_support = float(support.get(NormalizedSignalDirection.UNCERTAIN, 0.0))
    neutral_support = float(support.get(NormalizedSignalDirection.NEUTRAL, 0.0))
    participating_document_types = stable_unique([signal.document_type for signal in theme_signals])
    participating_worker_names = stable_unique([signal.worker_name for signal in theme_signals])
    strongest_support = max(positive_support, negative_support, mixed_support, uncertain_support, neutral_support)
    confidence = clamp_value(strongest_support / max(1.0, len(participating_document_types)), 0.0, 1.0)
    positive_docs = theme_support["supporting_docs"].get(NormalizedSignalDirection.POSITIVE, [])
    negative_docs = theme_support["supporting_docs"].get(NormalizedSignalDirection.NEGATIVE, [])
    uncertain_docs = theme_support["supporting_docs"].get(NormalizedSignalDirection.UNCERTAIN, [])

    if positive_support >= 0.45 and negative_support >= 0.45:
        decision_type = ArbiterDecisionType.CONTRADICTORY_SIGNAL
        direction = NormalizedSignalDirection.MIXED
        summary = f"{THEME_LABELS[theme]} is contradictory across the current disclosures rather than converging on one direction."
        supporting_document_types = positive_docs
        opposing_document_types = negative_docs
        evidence_references = deduplicate_arbiter_evidence(theme_support["supporting_evidence"].get(NormalizedSignalDirection.POSITIVE, []) + theme_support["supporting_evidence"].get(NormalizedSignalDirection.NEGATIVE, []))
    elif negative_support >= 0.60 and negative_support > positive_support + 0.15:
        has_structured_negative_support = any(signal.is_structured_document and signal.direction == NormalizedSignalDirection.NEGATIVE and signal.adjusted_confidence >= 0.30 for signal in theme_signals)
        decision_type = ArbiterDecisionType.MATERIAL_CONCERN if has_structured_negative_support else ArbiterDecisionType.ALIGNED_SIGNAL
        direction = NormalizedSignalDirection.NEGATIVE
        summary = f"{THEME_LABELS[theme]} skews negative across the currently available disclosures."
        supporting_document_types = negative_docs
        opposing_document_types = positive_docs
        evidence_references = theme_support["supporting_evidence"].get(NormalizedSignalDirection.NEGATIVE, [])
    elif positive_support >= 0.60 and positive_support > negative_support + 0.15:
        decision_type = ArbiterDecisionType.MATERIAL_POSITIVE if len(positive_docs) >= 2 else ArbiterDecisionType.ALIGNED_SIGNAL
        direction = NormalizedSignalDirection.POSITIVE
        summary = f"{THEME_LABELS[theme]} is supported positively across the currently available disclosures."
        supporting_document_types = positive_docs
        opposing_document_types = negative_docs
        evidence_references = theme_support["supporting_evidence"].get(NormalizedSignalDirection.POSITIVE, [])
    elif uncertain_support + mixed_support >= max(positive_support, negative_support):
        decision_type = ArbiterDecisionType.CROSS_DOCUMENT_UNCERTAINTY if len(participating_document_types) >= 2 else ArbiterDecisionType.UNRESOLVED_AMBIGUITY
        direction = NormalizedSignalDirection.UNCERTAIN
        summary = f"{THEME_LABELS[theme]} remains uncertain because current disclosures do not resolve the key open questions."
        supporting_document_types = uncertain_docs
        opposing_document_types = positive_docs + negative_docs
        evidence_references = deduplicate_arbiter_evidence(theme_support["supporting_evidence"].get(NormalizedSignalDirection.UNCERTAIN, []) + theme_support["supporting_evidence"].get(NormalizedSignalDirection.MIXED, []))
    elif len(participating_document_types) < 2:
        decision_type = ArbiterDecisionType.UNRESOLVED_AMBIGUITY
        direction = NormalizedSignalDirection.UNCERTAIN
        summary = f"{THEME_LABELS[theme]} has only thin cross-document coverage so far."
        supporting_document_types = participating_document_types
        opposing_document_types = []
        evidence_references = deduplicate_arbiter_evidence([reference for signal in theme_signals for reference in signal.evidence_references[:2]])
    else:
        decision_type = ArbiterDecisionType.UNRESOLVED_AMBIGUITY
        direction = NormalizedSignalDirection.MIXED
        summary = f"{THEME_LABELS[theme]} is directionally mixed and does not yet support a clean synthesis."
        supporting_document_types = positive_docs + negative_docs
        opposing_document_types = uncertain_docs
        evidence_references = deduplicate_arbiter_evidence([reference for signal in theme_signals for reference in signal.evidence_references[:2]])

    return CrossDocumentJudgment(
        judgment_id=f"judgment_{theme.value}",
        theme=theme,
        decision_type=decision_type,
        direction=direction,
        summary=summary,
        supporting_document_types=stable_unique(supporting_document_types),
        opposing_document_types=stable_unique(opposing_document_types),
        worker_names=participating_worker_names,
        confidence=confidence,
        evidence_references=deduplicate_arbiter_evidence(evidence_references),
        reasoning_notes=[
            f"positive_support={positive_support:.2f}",
            f"negative_support={negative_support:.2f}",
            f"mixed_support={mixed_support:.2f}",
            f"uncertain_support={uncertain_support:.2f}",
        ],
    )


def detect_cross_document_alignment(judgments: Sequence[CrossDocumentJudgment]) -> list[CrossDocumentJudgment]:
    aligned_types = {ArbiterDecisionType.ALIGNED_SIGNAL, ArbiterDecisionType.MATERIAL_CONCERN, ArbiterDecisionType.MATERIAL_POSITIVE}
    return [judgment for judgment in judgments if judgment.decision_type in aligned_types]


def signals_form_conflict(left_signal: ThematicWorkerSignal, right_signal: ThematicWorkerSignal) -> bool:
    direction_pair = {left_signal.direction, right_signal.direction}
    if direction_pair == {NormalizedSignalDirection.POSITIVE, NormalizedSignalDirection.NEGATIVE}:
        return True
    if direction_pair == {NormalizedSignalDirection.POSITIVE, NormalizedSignalDirection.UNCERTAIN}:
        return left_signal.is_soft_document != right_signal.is_soft_document
    return False


def detect_cross_document_conflicts(
    theme_map: dict[CrossDocumentTheme, list[ThematicWorkerSignal]],
) -> list[ArbiterConflict]:
    conflicts: list[ArbiterConflict] = []
    for theme, theme_signals in theme_map.items():
        for left_signal, right_signal in combinations(theme_signals, 2):
            if not signals_form_conflict(left_signal, right_signal):
                continue
            if min(left_signal.adjusted_confidence, right_signal.adjusted_confidence) < 0.20:
                continue
            positive_documents = []
            negative_documents = []
            if left_signal.direction == NormalizedSignalDirection.POSITIVE:
                positive_documents.append(left_signal.document_type)
            elif left_signal.direction in {NormalizedSignalDirection.NEGATIVE, NormalizedSignalDirection.UNCERTAIN}:
                negative_documents.append(left_signal.document_type)
            if right_signal.direction == NormalizedSignalDirection.POSITIVE:
                positive_documents.append(right_signal.document_type)
            elif right_signal.direction in {NormalizedSignalDirection.NEGATIVE, NormalizedSignalDirection.UNCERTAIN}:
                negative_documents.append(right_signal.document_type)
            conflicts.append(
                ArbiterConflict(
                    conflict_id=f"conflict_{theme.value}_{left_signal.document_type.value}_{right_signal.document_type.value}",
                    theme=theme,
                    title=f"{THEME_LABELS[theme]} conflict",
                    description=f"{document_type_label(left_signal.document_type)} and {document_type_label(right_signal.document_type)} do not support the same story for {THEME_LABELS[theme].lower()}.",
                    positive_document_types=stable_unique(positive_documents),
                    negative_document_types=stable_unique(negative_documents),
                    worker_names=stable_unique([left_signal.worker_name, right_signal.worker_name]),
                    high_confidence_conflict=left_signal.adjusted_confidence >= 0.45 and right_signal.adjusted_confidence >= 0.45,
                    evidence_references=deduplicate_arbiter_evidence(left_signal.evidence_references[:2] + right_signal.evidence_references[:2]),
                    reasoning_notes=[
                        f"left_direction={left_signal.direction.value}",
                        f"right_direction={right_signal.direction.value}",
                        f"left_adjusted_confidence={left_signal.adjusted_confidence:.2f}",
                        f"right_adjusted_confidence={right_signal.adjusted_confidence:.2f}",
                    ],
                )
            )
    deduplicated_conflicts: dict[tuple[Any, ...], ArbiterConflict] = {}
    for conflict in conflicts:
        conflict_key = (conflict.theme.value, tuple(sorted(document_type.value for document_type in conflict.positive_document_types)), tuple(sorted(document_type.value for document_type in conflict.negative_document_types)))
        if conflict_key not in deduplicated_conflicts:
            deduplicated_conflicts[conflict_key] = conflict
    return list(deduplicated_conflicts.values())


def compare_worker_outputs(normalized_outputs: Sequence[NormalizedWorkerOutput]) -> dict[str, Any]:
    theme_map = build_cross_document_themes(normalized_outputs)
    judgments = [build_theme_judgment(theme, signals) for theme, signals in theme_map.items()]
    return {
        "themes": theme_map,
        "judgments": judgments,
        "aligned_judgments": detect_cross_document_alignment(judgments),
        "conflicts": detect_cross_document_conflicts(theme_map),
    }


## 46. Signal Grouping and Conflict Detection


In [ ]:
def make_finding_from_judgment(judgment: CrossDocumentJudgment, *, category: ArbiterSignalCategory) -> ArbiterFinding:
    title_prefix = {
        ArbiterSignalCategory.POSITIVE: "Positive signal",
        ArbiterSignalCategory.NEGATIVE: "Negative signal",
        ArbiterSignalCategory.UNCERTAINTY: "Uncertainty",
        ArbiterSignalCategory.FOGGING: "Narrative concern",
    }[category]
    return ArbiterFinding(
        finding_id=f"{category.value}_{judgment.theme.value}",
        category=category,
        decision_type=judgment.decision_type,
        theme=judgment.theme,
        title=f"{title_prefix}: {THEME_LABELS[judgment.theme]}",
        summary=judgment.summary,
        supporting_document_types=list(judgment.supporting_document_types),
        contradicting_document_types=list(judgment.opposing_document_types),
        worker_names=list(judgment.worker_names),
        confidence=judgment.confidence,
        evidence_references=list(judgment.evidence_references),
        reasoning_notes=list(judgment.reasoning_notes),
    )


def make_signal_group_from_judgment(judgment: CrossDocumentJudgment, *, category: ArbiterSignalCategory) -> ArbiterSignalGroup:
    finding = make_finding_from_judgment(judgment, category=category)
    return ArbiterSignalGroup(
        group_id=f"group_{category.value}_{judgment.theme.value}",
        category=category,
        theme=judgment.theme,
        title=finding.title,
        description=finding.summary,
        document_types=stable_unique(list(judgment.supporting_document_types) + list(judgment.opposing_document_types)),
        worker_names=list(judgment.worker_names),
        findings=[finding],
        evidence_references=list(judgment.evidence_references),
        confidence=judgment.confidence,
        reasoning_notes=list(judgment.reasoning_notes),
    )


def group_positive_signals(judgments: Sequence[CrossDocumentJudgment]) -> list[ArbiterSignalGroup]:
    positive_decision_types = {ArbiterDecisionType.ALIGNED_SIGNAL, ArbiterDecisionType.MATERIAL_POSITIVE}
    return [make_signal_group_from_judgment(judgment, category=ArbiterSignalCategory.POSITIVE) for judgment in judgments if judgment.direction == NormalizedSignalDirection.POSITIVE and judgment.decision_type in positive_decision_types]


def group_negative_signals(judgments: Sequence[CrossDocumentJudgment]) -> list[ArbiterSignalGroup]:
    negative_decision_types = {ArbiterDecisionType.ALIGNED_SIGNAL, ArbiterDecisionType.MATERIAL_CONCERN}
    return [make_signal_group_from_judgment(judgment, category=ArbiterSignalCategory.NEGATIVE) for judgment in judgments if judgment.direction == NormalizedSignalDirection.NEGATIVE and judgment.decision_type in negative_decision_types]


def build_sparse_worker_uncertainty_finding(normalized_output: NormalizedWorkerOutput) -> ArbiterFinding:
    return ArbiterFinding(
        finding_id=f"worker_uncertainty_{normalized_output.document_type.value}",
        category=ArbiterSignalCategory.UNCERTAINTY,
        decision_type=ArbiterDecisionType.CROSS_DOCUMENT_UNCERTAINTY,
        theme=None,
        title=f"Sparse signal: {document_type_label(normalized_output.document_type)}",
        summary=f"{document_type_label(normalized_output.document_type)} contributes only thin support because confidence or evidence density is limited.",
        supporting_document_types=[normalized_output.document_type],
        contradicting_document_types=[],
        worker_names=[normalized_output.worker_name],
        confidence=min(normalized_output.normalized_confidence, normalized_output.evidence_density),
        evidence_references=list(normalized_output.evidence_references[:2]),
        reasoning_notes=list(normalized_output.normalization_notes),
    )


def group_uncertainties(
    judgments: Sequence[CrossDocumentJudgment],
    normalized_outputs: Sequence[NormalizedWorkerOutput],
    missing_document_types: Sequence[DocumentType],
) -> list[ArbiterFinding]:
    uncertainty_findings = [make_finding_from_judgment(judgment, category=ArbiterSignalCategory.UNCERTAINTY) for judgment in judgments if judgment.decision_type in {ArbiterDecisionType.UNRESOLVED_AMBIGUITY, ArbiterDecisionType.CROSS_DOCUMENT_UNCERTAINTY}]
    for normalized_output in normalized_outputs:
        if normalized_output.status not in USABLE_WORKER_STATUSES:
            continue
        if normalized_output.normalized_confidence < 0.45 or normalized_output.evidence_density < 0.30:
            uncertainty_findings.append(build_sparse_worker_uncertainty_finding(normalized_output))
    for document_type in missing_document_types:
        uncertainty_findings.append(
            ArbiterFinding(
                finding_id=f"missing_coverage_{document_type.value}",
                category=ArbiterSignalCategory.UNCERTAINTY,
                decision_type=ArbiterDecisionType.CROSS_DOCUMENT_UNCERTAINTY,
                theme=None,
                title=f"Missing coverage: {document_type_label(document_type)}",
                summary=f"No usable worker output is available for {document_type_label(document_type)}.",
                supporting_document_types=[document_type],
                contradicting_document_types=[],
                worker_names=[],
                confidence=0.0,
                evidence_references=[],
                reasoning_notes=["Missing coverage lowers cross-document certainty."],
            )
        )
    deduplicated_findings: dict[str, ArbiterFinding] = {}
    for finding in uncertainty_findings:
        if finding.finding_id not in deduplicated_findings:
            deduplicated_findings[finding.finding_id] = finding
    return list(deduplicated_findings.values())


def detect_story_vs_substance_mismatch(normalized_outputs: Sequence[NormalizedWorkerOutput]) -> list[ArbiterFinding]:
    investor_outputs = [output for output in normalized_outputs if output.document_type == DocumentType.INVESTOR_COMMUNICATION and output.status in USABLE_WORKER_STATUSES]
    structured_outputs = [output for output in normalized_outputs if output.document_type in STRUCTURED_DOCUMENT_TYPES and output.status in USABLE_WORKER_STATUSES]
    if not investor_outputs or not structured_outputs:
        return []
    hard_negative_or_uncertain = [output for output in structured_outputs if output.direction in {NormalizedSignalDirection.NEGATIVE, NormalizedSignalDirection.UNCERTAIN} and output.normalized_confidence >= 0.45]
    findings: list[ArbiterFinding] = []
    for investor_output in investor_outputs:
        positive_narrative = investor_output.direction == NormalizedSignalDirection.POSITIVE
        elevated_promotion = (investor_output.promotional_score or 0.0) >= 0.65
        thin_support = investor_output.evidence_density < 0.35
        if not hard_negative_or_uncertain:
            continue
        if not (positive_narrative or elevated_promotion or thin_support):
            continue
        findings.append(
            ArbiterFinding(
                finding_id=f"story_substance_{investor_output.document_type.value}",
                category=ArbiterSignalCategory.FOGGING,
                decision_type=ArbiterDecisionType.STORY_SUBSTANCE_MISMATCH,
                theme=CrossDocumentTheme.NARRATIVE_CREDIBILITY,
                title="Story-vs-substance mismatch",
                summary="Investor-facing narrative is more favorable or more polished than the harder disclosures support.",
                supporting_document_types=[investor_output.document_type],
                contradicting_document_types=stable_unique([output.document_type for output in hard_negative_or_uncertain]),
                worker_names=stable_unique([investor_output.worker_name] + [output.worker_name for output in hard_negative_or_uncertain]),
                confidence=clamp_value(safe_mean([investor_output.normalized_confidence] + [output.normalized_confidence for output in hard_negative_or_uncertain]) or 0.0, 0.0, 1.0),
                evidence_references=deduplicate_arbiter_evidence(investor_output.evidence_references[:2] + [reference for output in hard_negative_or_uncertain for reference in output.evidence_references[:1]]),
                reasoning_notes=[
                    f"investor_direction={investor_output.direction.value}",
                    f"investor_promotional_score={investor_output.promotional_score}",
                    f"investor_evidence_density={investor_output.evidence_density:.2f}",
                ],
            )
        )
    return findings


def group_fogging_concerns(
    normalized_outputs: Sequence[NormalizedWorkerOutput],
    story_mismatch_findings: Sequence[ArbiterFinding],
) -> list[ArbiterFinding]:
    fogging_findings = list(story_mismatch_findings)
    for normalized_output in normalized_outputs:
        highest_tone_score = max(normalized_output.fogging_score or 0.0, normalized_output.hedging_score or 0.0, normalized_output.promotional_score or 0.0)
        if highest_tone_score < 0.65 or normalized_output.status not in USABLE_WORKER_STATUSES:
            continue
        fogging_findings.append(
            ArbiterFinding(
                finding_id=f"tone_flag_{normalized_output.document_type.value}",
                category=ArbiterSignalCategory.FOGGING,
                decision_type=ArbiterDecisionType.STORY_SUBSTANCE_MISMATCH,
                theme=CrossDocumentTheme.NARRATIVE_CREDIBILITY,
                title=f"Elevated tone concern: {document_type_label(normalized_output.document_type)}",
                summary=f"{document_type_label(normalized_output.document_type)} carries elevated fogging, hedging, or promotional tone relative to its arbiter-ready evidence.",
                supporting_document_types=[normalized_output.document_type],
                contradicting_document_types=[],
                worker_names=[normalized_output.worker_name],
                confidence=clamp_value(highest_tone_score, 0.0, 1.0),
                evidence_references=list(normalized_output.evidence_references[:2]),
                reasoning_notes=[
                    f"fogging_score={normalized_output.fogging_score}",
                    f"hedging_score={normalized_output.hedging_score}",
                    f"promotional_score={normalized_output.promotional_score}",
                ],
            )
        )
    deduplicated_findings: dict[str, ArbiterFinding] = {}
    for finding in fogging_findings:
        if finding.finding_id not in deduplicated_findings:
            deduplicated_findings[finding.finding_id] = finding
    return list(deduplicated_findings.values())


## 47. Confidence Reconciliation Logic


In [ ]:
def make_confidence_adjustment(factor_name: str, adjustment: float, rationale: str) -> ArbiterConfidenceAdjustment:
    return ArbiterConfidenceAdjustment(factor_name=factor_name, adjustment=adjustment, rationale=rationale)


def adjust_confidence_for_evidence_density(normalized_outputs: Sequence[NormalizedWorkerOutput]) -> ArbiterConfidenceAdjustment:
    evidence_density_average = safe_mean([output.evidence_density for output in normalized_outputs]) or 0.0
    adjustment = clamp_value((evidence_density_average - 0.50) * 0.30, -0.15, 0.15)
    return make_confidence_adjustment("evidence_density", adjustment, f"Average evidence density was {evidence_density_average:.2f}.")


def adjust_confidence_for_conflict(conflicts: Sequence[ArbiterConflict]) -> ArbiterConfidenceAdjustment:
    high_confidence_conflict_count = sum(1 for conflict in conflicts if conflict.high_confidence_conflict)
    adjustment = -min(0.25, (len(conflicts) * 0.06) + (high_confidence_conflict_count * 0.05))
    return make_confidence_adjustment("explicit_conflicts", adjustment, f"Detected {len(conflicts)} explicit conflicts, including {high_confidence_conflict_count} high-confidence conflicts.")


def adjust_confidence_for_missing_coverage(missing_document_types: Sequence[DocumentType]) -> ArbiterConfidenceAdjustment:
    adjustment = -min(0.20, len(missing_document_types) * 0.04)
    return make_confidence_adjustment("missing_coverage", adjustment, f"Missing usable coverage for {len(missing_document_types)} expected document types.")


def adjust_confidence_for_structured_support(normalized_outputs: Sequence[NormalizedWorkerOutput]) -> tuple[ArbiterConfidenceAdjustment, float]:
    usable_outputs = [output for output in normalized_outputs if output.status in USABLE_WORKER_STATUSES]
    if not usable_outputs:
        return make_confidence_adjustment("structured_support", -0.10, "No usable worker outputs were available."), 0.0
    weighted_supports = [output.normalized_confidence * (0.50 + (0.50 * output.evidence_density)) for output in usable_outputs]
    total_support = sum(weighted_supports)
    structured_support = sum(output.normalized_confidence * (0.50 + (0.50 * output.evidence_density)) for output in usable_outputs if output.is_structured_document)
    structured_support_ratio = structured_support / total_support if total_support > 0.0 else 0.0
    adjustment = clamp_value((structured_support_ratio - 0.50) * 0.20, -0.10, 0.10)
    return make_confidence_adjustment("structured_support", adjustment, f"Structured-document support ratio was {structured_support_ratio:.2f}."), structured_support_ratio


def assess_cross_document_confidence(
    normalized_outputs: Sequence[NormalizedWorkerOutput],
    conflicts: Sequence[ArbiterConflict],
    missing_document_types: Sequence[DocumentType],
) -> ArbiterConfidenceAssessment:
    usable_outputs = [output for output in normalized_outputs if output.status in USABLE_WORKER_STATUSES]
    worker_confidence_average = safe_mean([output.normalized_confidence for output in usable_outputs]) or 0.25
    evidence_density_average = safe_mean([output.evidence_density for output in usable_outputs]) or 0.0
    base_confidence = clamp_value(worker_confidence_average, 0.0, 1.0)
    adjustments = [
        adjust_confidence_for_evidence_density(usable_outputs),
        adjust_confidence_for_conflict(conflicts),
        adjust_confidence_for_missing_coverage(missing_document_types),
    ]
    structured_support_adjustment, structured_support_ratio = adjust_confidence_for_structured_support(usable_outputs)
    adjustments.append(structured_support_adjustment)
    final_confidence = clamp_value(base_confidence + sum(adjustment.adjustment for adjustment in adjustments), 0.0, 1.0)
    return ArbiterConfidenceAssessment(
        base_confidence=base_confidence,
        final_confidence=final_confidence,
        worker_confidence_average=worker_confidence_average,
        evidence_density_average=evidence_density_average,
        structured_support_ratio=structured_support_ratio,
        missing_document_types=list(missing_document_types),
        high_confidence_conflict_count=sum(1 for conflict in conflicts if conflict.high_confidence_conflict),
        adjustments=adjustments,
        reasoning_notes=[
            f"base_confidence={base_confidence:.2f}",
            f"worker_confidence_average={worker_confidence_average:.2f}",
            f"evidence_density_average={evidence_density_average:.2f}",
            f"structured_support_ratio={structured_support_ratio:.2f}",
        ],
    )


def build_missing_coverage_notes(
    normalized_outputs: Sequence[NormalizedWorkerOutput],
    missing_document_types: Sequence[DocumentType],
) -> list[str]:
    notes: list[str] = []
    if missing_document_types:
        notes.append("Missing or unusable worker coverage for: " + ", ".join(document_type_label(document_type) for document_type in missing_document_types) + ".")
    structured_coverage_present = any(output.is_structured_document and output.status in USABLE_WORKER_STATUSES for output in normalized_outputs)
    if not structured_coverage_present:
        notes.append("No successful structured-document worker output is available; current arbitration is dominated by softer narrative evidence.")
    usable_outputs = [output for output in normalized_outputs if output.status in USABLE_WORKER_STATUSES]
    if usable_outputs and all(output.is_soft_document for output in usable_outputs):
        notes.append("All usable support currently comes from softer narrative documents rather than harder disclosures.")
    return notes


def aggregate_arbiter_sentiment(
    normalized_outputs: Sequence[NormalizedWorkerOutput],
    conflicts: Sequence[ArbiterConflict],
    confidence_assessment: ArbiterConfidenceAssessment,
) -> SentimentAssessment | None:
    weighted_scores: list[float] = []
    weights: list[float] = []
    for normalized_output in normalized_outputs:
        if normalized_output.status not in USABLE_WORKER_STATUSES or normalized_output.sentiment_score is None:
            continue
        weight = normalized_output.normalized_confidence * (0.50 + (0.50 * normalized_output.evidence_density))
        if normalized_output.is_soft_document:
            weight *= 0.85
        weighted_scores.append(normalized_output.sentiment_score * weight)
        weights.append(weight)
    if not weights or sum(weights) == 0.0:
        return SentimentAssessment(label=SentimentLabel.INSUFFICIENT_EVIDENCE, score=None, confidence=confidence_assessment.final_confidence, rationale="Arbiter did not receive enough usable worker sentiment data to infer a directional score.")
    aggregate_score = sum(weighted_scores) / sum(weights)
    positive_present = any(output.direction == NormalizedSignalDirection.POSITIVE for output in normalized_outputs)
    negative_present = any(output.direction == NormalizedSignalDirection.NEGATIVE for output in normalized_outputs)
    if conflicts and positive_present and negative_present:
        label = SentimentLabel.MIXED
    elif aggregate_score >= 0.20:
        label = SentimentLabel.POSITIVE
    elif aggregate_score <= -0.20:
        label = SentimentLabel.NEGATIVE
    elif abs(aggregate_score) <= 0.10:
        label = SentimentLabel.NEUTRAL
    else:
        label = SentimentLabel.MIXED
    return SentimentAssessment(label=label, score=clamp_value(aggregate_score, SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX), confidence=confidence_assessment.final_confidence, rationale="Intermediate arbiter sentiment derived from weighted worker sentiment; the master node will decide final UI-facing sentiment later.")


def aggregate_arbiter_tone(normalized_outputs: Sequence[NormalizedWorkerOutput], confidence_assessment: ArbiterConfidenceAssessment) -> ToneAssessment | None:
    usable_outputs = [output for output in normalized_outputs if output.status in USABLE_WORKER_STATUSES]
    if not usable_outputs:
        return None
    def weighted_tone_mean(attribute_name: str) -> float | None:
        weighted_values: list[float] = []
        weights: list[float] = []
        for normalized_output in usable_outputs:
            tone_value = getattr(normalized_output, attribute_name)
            if tone_value is None:
                continue
            weight = normalized_output.normalized_confidence
            weighted_values.append(tone_value * weight)
            weights.append(weight)
        if not weights or sum(weights) == 0.0:
            return None
        return clamp_value(sum(weighted_values) / sum(weights), 0.0, 1.0)
    fogging_score = weighted_tone_mean("fogging_score")
    hedging_score = weighted_tone_mean("hedging_score")
    promotional_score = weighted_tone_mean("promotional_score")
    if fogging_score is None and hedging_score is None and promotional_score is None:
        return None
    return ToneAssessment(fogging_score=fogging_score, hedging_score=hedging_score, promotional_score=promotional_score, confidence=confidence_assessment.final_confidence, rationale="Intermediate arbiter tone derived from weighted worker tone outputs.")


def build_arbiter_warnings(
    normalized_outputs: Sequence[NormalizedWorkerOutput],
    conflicts: Sequence[ArbiterConflict],
    missing_document_types: Sequence[DocumentType],
    confidence_assessment: ArbiterConfidenceAssessment,
) -> list[ArbiterWarning]:
    warnings: list[ArbiterWarning] = [warning for normalized_output in normalized_outputs for warning in normalized_output.warnings]
    usable_outputs = [output for output in normalized_outputs if output.status in USABLE_WORKER_STATUSES]
    if len(usable_outputs) < 2:
        warnings.append(make_arbiter_warning("too_few_worker_outputs", "Too few usable worker outputs are available for a strong cross-document judgment.", document_types=[output.document_type for output in usable_outputs], worker_names=[output.worker_name for output in usable_outputs]))
    if any(conflict.high_confidence_conflict for conflict in conflicts):
        warnings.append(make_arbiter_warning("high_confidence_worker_conflict", "At least two higher-confidence worker outputs materially disagree.", document_types=stable_unique([document_type for conflict in conflicts for document_type in conflict.positive_document_types + conflict.negative_document_types]), worker_names=stable_unique([worker_name for conflict in conflicts for worker_name in conflict.worker_names])))
    if any(document_type in STRUCTURED_DOCUMENT_TYPES for document_type in missing_document_types):
        warnings.append(make_arbiter_warning("missing_hard_document_coverage", "One or more structured disclosure types are missing or unusable for arbitration.", document_types=[document_type for document_type in missing_document_types if document_type in STRUCTURED_DOCUMENT_TYPES]))
    if usable_outputs and all(output.is_soft_document for output in usable_outputs):
        warnings.append(make_arbiter_warning("soft_document_only_support", "All usable evidence currently comes from softer narrative documents.", document_types=[output.document_type for output in usable_outputs], worker_names=[output.worker_name for output in usable_outputs]))
    evidence_density_average = safe_mean([output.evidence_density for output in usable_outputs]) or 0.0
    if usable_outputs and evidence_density_average < 0.30:
        warnings.append(make_arbiter_warning("low_evidence_density", "Average evidence density across available worker outputs is low.", document_types=[output.document_type for output in usable_outputs], worker_names=[output.worker_name for output in usable_outputs], metadata={"evidence_density_average": round(evidence_density_average, 3)}))
    if confidence_assessment.final_confidence < 0.35:
        warnings.append(make_arbiter_warning("low_arbiter_confidence", "Cross-document confidence is low after reconciliation.", metadata={"final_confidence": round(confidence_assessment.final_confidence, 3)}))
    deduplicated_warnings: dict[tuple[Any, ...], ArbiterWarning] = {}
    for warning in warnings:
        warning_key = (warning.issue_code, warning.message, tuple(document_type.value for document_type in warning.document_types), tuple(warning.worker_names))
        if warning_key not in deduplicated_warnings:
            deduplicated_warnings[warning_key] = warning
    return list(deduplicated_warnings.values())


## 48. Arbiter Classes / Interfaces


In [ ]:
class BaseArbiter(ABC):
    arbiter_name: ClassVar[str] = "base_arbiter"
    arbiter_kind: ClassVar[ArbiterKind]
    input_model: ClassVar[type[ArbiterInput]] = ArbiterInput
    output_model: ClassVar[type[ArbiterOutput]] = ArbiterOutput

    @abstractmethod
    def arbitrate(self, arbiter_input: ArbiterInput) -> ArbiterOutput:
        raise NotImplementedError


class MasterNode(ABC):
    input_model: ClassVar[type[MasterInput]] = MasterInput
    output_model: ClassVar[type[FinalUIPayload]] = FinalUIPayload

    def build_payload(self, master_input: MasterInput) -> FinalUIPayload:
        raise NotImplementedError("Master node packaging is intentionally deferred to a later prompt.")


class CrossDocumentArbiter(BaseArbiter):
    arbiter_name = "cross_document_arbiter"
    arbiter_kind = ArbiterKind.CROSS_DOCUMENT

    def arbitrate(self, arbiter_input: ArbiterInput) -> ArbiterOutput:
        selected_worker_outputs, selection_warnings = select_best_worker_output_per_type(arbiter_input.worker_outputs)
        normalized_outputs = [normalize_worker_output_for_arbitration(worker_output) for worker_output in selected_worker_outputs]
        covered_document_types = [output.document_type for output in normalized_outputs if output.status in USABLE_WORKER_STATUSES]
        missing_document_types = [document_type for document_type in EXPECTED_DOCUMENT_TYPES if document_type not in covered_document_types]
        comparison = compare_worker_outputs(normalized_outputs)
        story_mismatch_findings = detect_story_vs_substance_mismatch(normalized_outputs)
        positive_signal_groups = group_positive_signals(comparison["judgments"])
        negative_signal_groups = group_negative_signals(comparison["judgments"])
        unresolved_uncertainties = group_uncertainties(comparison["judgments"], normalized_outputs, missing_document_types)
        fogging_findings = group_fogging_concerns(normalized_outputs, story_mismatch_findings)
        confidence_assessment = assess_cross_document_confidence(normalized_outputs, comparison["conflicts"], missing_document_types)
        warnings = selection_warnings + build_arbiter_warnings(normalized_outputs, comparison["conflicts"], missing_document_types, confidence_assessment)
        issues = [issue for normalized_output in normalized_outputs for issue in normalized_output.issues]
        missing_coverage_notes = build_missing_coverage_notes(normalized_outputs, missing_document_types)
        evidence_references = deduplicate_arbiter_evidence(
            [reference for judgment in comparison["judgments"] for reference in judgment.evidence_references]
            + [reference for group in positive_signal_groups for reference in group.evidence_references]
            + [reference for group in negative_signal_groups for reference in group.evidence_references]
            + [reference for conflict in comparison["conflicts"] for reference in conflict.evidence_references]
            + [reference for finding in unresolved_uncertainties for reference in finding.evidence_references]
            + [reference for finding in fogging_findings for reference in finding.evidence_references]
        )
        legacy_evidence = deduplicate_evidence_snippets([evidence_snippet for worker_output in selected_worker_outputs for evidence_snippet in worker_output.evidence])
        status = ProcessingStatus.SUCCESS
        if missing_document_types or comparison["conflicts"] or unresolved_uncertainties:
            status = ProcessingStatus.PARTIAL
        if not normalized_outputs:
            status = ProcessingStatus.NO_DOCUMENT
        aligned_signals = positive_signal_groups + negative_signal_groups
        return ArbiterOutput(
            arbiter_id=f"{arbiter_input.run_id}_cross_document",
            arbiter_name=self.arbiter_name,
            arbiter_kind=self.arbiter_kind,
            status=status,
            summary=f"Intermediate arbiter output covering {len(covered_document_types)} of {len(EXPECTED_DOCUMENT_TYPES)} expected disclosure types.",
            sentiment=aggregate_arbiter_sentiment(normalized_outputs, comparison["conflicts"], confidence_assessment),
            tone=aggregate_arbiter_tone(normalized_outputs, confidence_assessment),
            covered_document_types=covered_document_types,
            missing_document_types=missing_document_types,
            cross_document_judgments=comparison["judgments"],
            positive_signal_groups=positive_signal_groups,
            negative_signal_groups=negative_signal_groups,
            aligned_signals=aligned_signals,
            conflicting_signals=comparison["conflicts"],
            unresolved_uncertainties=unresolved_uncertainties,
            fogging_or_story_substance_flags=fogging_findings,
            confidence_assessment=confidence_assessment,
            missing_coverage_notes=missing_coverage_notes,
            evidence_references=evidence_references,
            warnings=warnings,
            issues=issues,
            reasoning_notes=[
                f"received_worker_outputs={len(arbiter_input.worker_outputs)}",
                f"normalized_outputs={len(normalized_outputs)}",
                f"judgments={len(comparison['judgments'])}",
                f"conflicts={len(comparison['conflicts'])}",
                f"missing_document_types={len(missing_document_types)}",
            ],
            consensus_points=[group.description for group in aligned_signals[:4]],
            conflicts=[conflict.description for conflict in comparison["conflicts"]],
            evidence=legacy_evidence,
            confidence=confidence_assessment.final_confidence,
        )


ARBITER_REGISTRY: dict[ArbiterKind, Type[BaseArbiter]] = {
    ArbiterKind.CROSS_DOCUMENT: CrossDocumentArbiter,
}

print(json.dumps({arbiter_kind.value: arbiter_class.__name__ for arbiter_kind, arbiter_class in ARBITER_REGISTRY.items()}, indent=2))


## 49. Demo Arbiter Runs


In [ ]:
def make_demo_arbiter_evidence(
    *,
    document_id: str,
    chunk_id: str,
    rationale: str,
    snippet_text: str,
    interpretation: EvidenceInterpretation,
    source_url: str,
    section_title: str = "Demo Section",
) -> EvidenceSnippet:
    return EvidenceSnippet(
        evidence_id=f"{document_id}_{chunk_id}",
        document_id=document_id,
        source_url=source_url,
        source_chunk_id=chunk_id,
        source_section_id=f"{chunk_id}_section",
        section_title=section_title,
        evidence_type=EvidenceType.CONTEXTUAL_SUMMARY,
        supported_dimensions=[AnalysisDimension.SENTIMENT, AnalysisDimension.CLARITY],
        interpretation=interpretation,
        snippet_text=snippet_text,
        rationale=rationale,
        metadata={"prompt7_demo": True},
    )


def make_demo_worker_output(
    *,
    worker_name: str,
    document_type: DocumentType,
    status: ProcessingStatus,
    summary: str | None,
    sentiment_label: SentimentLabel | None = None,
    sentiment_score: float | None = None,
    confidence: float | None = None,
    key_points: Sequence[str] | None = None,
    caveats: Sequence[str] | None = None,
    evidence_specs: Sequence[dict[str, Any]] | None = None,
    fogging_score: float | None = None,
    hedging_score: float | None = None,
    promotional_score: float | None = None,
    tone_confidence: float | None = None,
) -> WorkerOutput:
    evidence = [make_demo_arbiter_evidence(**evidence_spec) for evidence_spec in (evidence_specs or [])]
    sentiment = None
    if sentiment_label is not None:
        sentiment = SentimentAssessment(label=sentiment_label, score=sentiment_score, confidence=confidence, rationale="Synthetic Prompt 7 worker-demo sentiment.")
    tone = None
    if any(value is not None for value in [fogging_score, hedging_score, promotional_score, tone_confidence]):
        tone = ToneAssessment(fogging_score=fogging_score, hedging_score=hedging_score, promotional_score=promotional_score, confidence=tone_confidence, rationale="Synthetic Prompt 7 worker-demo tone.")
    return WorkerOutput(worker_name=worker_name, document_type=document_type, status=status, summary=summary, sentiment=sentiment, tone=tone, key_points=list(key_points or []), evidence=evidence, caveats=list(caveats or []), issues=[], confidence=confidence)


ARBITER_DEMO_CASES: dict[str, list[WorkerOutput]] = {
    "mostly_aligned": [
        make_demo_worker_output(
            worker_name="material_event_worker",
            document_type=DocumentType.MATERIAL_EVENT,
            status=ProcessingStatus.SUCCESS,
            summary="The 8-K frames the catalyst as on track and adds concrete next steps.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.55,
            confidence=0.78,
            key_points=["Company disclosed a partner opt-in milestone.", "Near-term execution steps were made explicit."],
            caveats=["Manufacturing readiness still needs execution follow-through."],
            evidence_specs=[
                {"document_id": "aligned_material_001", "chunk_id": "material_chunk_1", "rationale": "8-K provides concrete milestone detail.", "snippet_text": "Partner opted in after reviewing the updated package.", "interpretation": EvidenceInterpretation.POSITIVE, "source_url": "https://example.com/aligned/material"},
                {"document_id": "aligned_material_001", "chunk_id": "material_chunk_2", "rationale": "Next operational steps are explicitly listed.", "snippet_text": "Site activation and process validation continue in parallel.", "interpretation": EvidenceInterpretation.CLARIFYING, "source_url": "https://example.com/aligned/material"},
            ],
            fogging_score=0.15,
            hedging_score=0.20,
            promotional_score=0.25,
            tone_confidence=0.70,
        ),
        make_demo_worker_output(
            worker_name="clinical_trial_update_worker",
            document_type=DocumentType.CLINICAL_TRIAL_UPDATE,
            status=ProcessingStatus.SUCCESS,
            summary="Registry update clarifies enrollment status without pushing out the primary completion date.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.45,
            confidence=0.82,
            key_points=["Enrollment remains active.", "Primary completion timing was preserved."],
            evidence_specs=[
                {"document_id": "aligned_clinical_001", "chunk_id": "clinical_chunk_1", "rationale": "Registry confirms continued enrollment.", "snippet_text": "Estimated primary completion remains December 2026.", "interpretation": EvidenceInterpretation.POSITIVE, "source_url": "https://example.com/aligned/clinical"},
                {"document_id": "aligned_clinical_001", "chunk_id": "clinical_chunk_2", "rationale": "Endpoint and enrollment fields were clarified.", "snippet_text": "Secondary biomarker analysis plan was updated for clarity.", "interpretation": EvidenceInterpretation.CLARIFYING, "source_url": "https://example.com/aligned/clinical"},
            ],
            fogging_score=0.10,
            hedging_score=0.18,
            promotional_score=0.10,
            tone_confidence=0.75,
        ),
        make_demo_worker_output(
            worker_name="fda_review_worker",
            document_type=DocumentType.FDA_REVIEW,
            status=ProcessingStatus.SUCCESS,
            summary="Regulatory materials are broadly supportive but still preserve label and CMC follow-up items.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.30,
            confidence=0.76,
            key_points=["Benefit-risk framing remains supportive.", "Outstanding items look manageable rather than thesis-breaking."],
            caveats=["Label scope still depends on ongoing discussion."],
            evidence_specs=[
                {"document_id": "aligned_fda_001", "chunk_id": "fda_chunk_1", "rationale": "Review package supports benefit-risk balance.", "snippet_text": "The efficacy package supports a favorable benefit-risk assessment.", "interpretation": EvidenceInterpretation.POSITIVE, "source_url": "https://example.com/aligned/fda"},
                {"document_id": "aligned_fda_001", "chunk_id": "fda_chunk_2", "rationale": "Remaining issues are specific and bounded.", "snippet_text": "CMC validation data will be reviewed in the final labeling cycle.", "interpretation": EvidenceInterpretation.UNCERTAINTY, "source_url": "https://example.com/aligned/fda"},
            ],
            fogging_score=0.12,
            hedging_score=0.28,
            promotional_score=0.08,
            tone_confidence=0.72,
        ),
        make_demo_worker_output(
            worker_name="financing_dilution_worker",
            document_type=DocumentType.FINANCING_DILUTION,
            status=ProcessingStatus.SUCCESS,
            summary="Financing looks opportunistic rather than distressed and extends runway into the data window.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.28,
            confidence=0.74,
            key_points=["Runway extends beyond expected catalyst timing.", "Use of proceeds is tied to trial and CMC execution."],
            evidence_specs=[
                {"document_id": "aligned_financing_001", "chunk_id": "financing_chunk_1", "rationale": "Runway extension reduces near-term financing pressure.", "snippet_text": "Cash runway extends into the third quarter of 2027.", "interpretation": EvidenceInterpretation.POSITIVE, "source_url": "https://example.com/aligned/financing"},
                {"document_id": "aligned_financing_001", "chunk_id": "financing_chunk_2", "rationale": "Use of proceeds is disclosed concretely.", "snippet_text": "Proceeds support enrollment, assay work, and process validation.", "interpretation": EvidenceInterpretation.CLARIFYING, "source_url": "https://example.com/aligned/financing"},
            ],
            fogging_score=0.18,
            hedging_score=0.15,
            promotional_score=0.18,
            tone_confidence=0.68,
        ),
        make_demo_worker_output(
            worker_name="investor_communication_worker",
            document_type=DocumentType.INVESTOR_COMMUNICATION,
            status=ProcessingStatus.SUCCESS,
            summary="Management narrative matches the harder documents reasonably well and keeps remaining risks visible.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.35,
            confidence=0.63,
            key_points=["Management emphasizes execution milestones already disclosed elsewhere.", "Risk section still mentions enrollment and manufacturing dependencies."],
            caveats=["Narrative remains somewhat polished."],
            evidence_specs=[
                {"document_id": "aligned_investor_001", "chunk_id": "investor_chunk_1", "rationale": "Investor communication repeats the same core milestones.", "snippet_text": "Top-line data remains expected in the fourth quarter of 2026.", "interpretation": EvidenceInterpretation.POSITIVE, "source_url": "https://example.com/aligned/investor"},
                {"document_id": "aligned_investor_001", "chunk_id": "investor_chunk_2", "rationale": "Management still discloses key execution risks.", "snippet_text": "Enrollment consistency and manufacturing comparability remain important risks.", "interpretation": EvidenceInterpretation.CLARIFYING, "source_url": "https://example.com/aligned/investor"},
            ],
            fogging_score=0.32,
            hedging_score=0.30,
            promotional_score=0.38,
            tone_confidence=0.60,
        ),
    ],
    "contradictory": [
        make_demo_worker_output(
            worker_name="material_event_worker",
            document_type=DocumentType.MATERIAL_EVENT,
            status=ProcessingStatus.SUCCESS,
            summary="Press release claims the program remains on track and highlights upcoming milestones.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.60,
            confidence=0.72,
            key_points=["Management presents timing as intact."],
            evidence_specs=[{"document_id": "contradict_material_001", "chunk_id": "material_chunk_1", "rationale": "Issuer framing remains favorable.", "snippet_text": "The company remains confident in the regulatory path.", "interpretation": EvidenceInterpretation.POSITIVE, "source_url": "https://example.com/contradict/material"}],
            fogging_score=0.35,
            hedging_score=0.25,
            promotional_score=0.52,
            tone_confidence=0.66,
        ),
        make_demo_worker_output(
            worker_name="clinical_trial_update_worker",
            document_type=DocumentType.CLINICAL_TRIAL_UPDATE,
            status=ProcessingStatus.SUCCESS,
            summary="Registry update shows slower enrollment and pushes out primary completion timing.",
            sentiment_label=SentimentLabel.NEGATIVE,
            sentiment_score=-0.55,
            confidence=0.85,
            key_points=["Enrollment target was reduced.", "Primary completion moved out by two quarters."],
            evidence_specs=[
                {"document_id": "contradict_clinical_001", "chunk_id": "clinical_chunk_1", "rationale": "Registry timing slipped materially.", "snippet_text": "Estimated primary completion moved from Q4 2026 to Q2 2027.", "interpretation": EvidenceInterpretation.NEGATIVE, "source_url": "https://example.com/contradict/clinical"},
                {"document_id": "contradict_clinical_001", "chunk_id": "clinical_chunk_2", "rationale": "Enrollment assumptions weakened.", "snippet_text": "Target enrollment was revised downward after slower site activation.", "interpretation": EvidenceInterpretation.NEGATIVE, "source_url": "https://example.com/contradict/clinical"},
            ],
            fogging_score=0.12,
            hedging_score=0.20,
            promotional_score=0.08,
            tone_confidence=0.80,
        ),
        make_demo_worker_output(
            worker_name="fda_review_worker",
            document_type=DocumentType.FDA_REVIEW,
            status=ProcessingStatus.SUCCESS,
            summary="Regulatory materials identify unresolved safety and CMC issues that could constrain timing and label scope.",
            sentiment_label=SentimentLabel.NEGATIVE,
            sentiment_score=-0.70,
            confidence=0.88,
            key_points=["Safety signal remains under review.", "CMC comparability package is incomplete."],
            caveats=["Benefit-risk is not yet clearly resolved."],
            evidence_specs=[
                {"document_id": "contradict_fda_001", "chunk_id": "fda_chunk_1", "rationale": "Review documents highlight unresolved risk.", "snippet_text": "Outstanding safety concerns require additional analysis before approval.", "interpretation": EvidenceInterpretation.NEGATIVE, "source_url": "https://example.com/contradict/fda"},
                {"document_id": "contradict_fda_001", "chunk_id": "fda_chunk_2", "rationale": "Manufacturing package is not yet complete.", "snippet_text": "CMC comparability remains an approval-critical deficiency.", "interpretation": EvidenceInterpretation.NEGATIVE, "source_url": "https://example.com/contradict/fda"},
            ],
            fogging_score=0.10,
            hedging_score=0.22,
            promotional_score=0.05,
            tone_confidence=0.82,
        ),
        make_demo_worker_output(
            worker_name="financing_dilution_worker",
            document_type=DocumentType.FINANCING_DILUTION,
            status=ProcessingStatus.SUCCESS,
            summary="Financing looks urgent, highly dilutive, and priced from a weak bargaining position.",
            sentiment_label=SentimentLabel.NEGATIVE,
            sentiment_score=-0.60,
            confidence=0.79,
            key_points=["Discount is deep relative to prior trading levels.", "Warrant coverage increases future overhang."],
            evidence_specs=[
                {"document_id": "contradict_financing_001", "chunk_id": "financing_chunk_1", "rationale": "Financing terms are punitive.", "snippet_text": "Offering priced at a 22 percent discount with full warrant coverage.", "interpretation": EvidenceInterpretation.NEGATIVE, "source_url": "https://example.com/contradict/financing"},
                {"document_id": "contradict_financing_001", "chunk_id": "financing_chunk_2", "rationale": "Use of proceeds prioritizes near-term liquidity rather than optionality.", "snippet_text": "Proceeds are needed primarily to maintain ongoing operations.", "interpretation": EvidenceInterpretation.NEGATIVE, "source_url": "https://example.com/contradict/financing"},
            ],
            fogging_score=0.18,
            hedging_score=0.12,
            promotional_score=0.14,
            tone_confidence=0.74,
        ),
        make_demo_worker_output(
            worker_name="investor_communication_worker",
            document_type=DocumentType.INVESTOR_COMMUNICATION,
            status=ProcessingStatus.SUCCESS,
            summary="Management narrative stays strongly upbeat and emphasizes upside despite pressure in harder disclosures.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.70,
            confidence=0.68,
            key_points=["Management emphasizes catalyst potential and de-emphasizes timing risk.", "Presentation highlights broad market opportunity rather than current constraints."],
            evidence_specs=[{"document_id": "contradict_investor_001", "chunk_id": "investor_chunk_1", "rationale": "Investor narrative remains strongly promotional.", "snippet_text": "We remain uniquely positioned for a best-in-class launch trajectory.", "interpretation": EvidenceInterpretation.PROMOTIONAL, "source_url": "https://example.com/contradict/investor"}],
            fogging_score=0.62,
            hedging_score=0.40,
            promotional_score=0.78,
            tone_confidence=0.67,
        ),
    ],
    "sparse_missing": [
        make_demo_worker_output(worker_name="material_event_worker", document_type=DocumentType.MATERIAL_EVENT, status=ProcessingStatus.NO_DOCUMENT, summary=None, sentiment_label=None, confidence=None, key_points=[], caveats=["No current material-event document was retrieved."]),
        make_demo_worker_output(
            worker_name="financing_dilution_worker",
            document_type=DocumentType.FINANCING_DILUTION,
            status=ProcessingStatus.PARTIAL,
            summary="Financing disclosure exists, but terms and use-of-proceeds detail are thin.",
            sentiment_label=SentimentLabel.INSUFFICIENT_EVIDENCE,
            sentiment_score=None,
            confidence=0.38,
            key_points=["Runway benefit cannot be sized confidently from the available text."],
            caveats=["Material financing terms are incomplete in the available excerpt."],
            evidence_specs=[{"document_id": "sparse_financing_001", "chunk_id": "financing_chunk_1", "rationale": "Only limited financing detail is available.", "snippet_text": "The company completed a registered direct offering.", "interpretation": EvidenceInterpretation.UNCERTAINTY, "source_url": "https://example.com/sparse/financing"}],
            fogging_score=0.35,
            hedging_score=0.45,
            promotional_score=0.20,
            tone_confidence=0.40,
        ),
        make_demo_worker_output(
            worker_name="investor_communication_worker",
            document_type=DocumentType.INVESTOR_COMMUNICATION,
            status=ProcessingStatus.SUCCESS,
            summary="Management remains optimistic, but the communication is light on inspectable support.",
            sentiment_label=SentimentLabel.POSITIVE,
            sentiment_score=0.20,
            confidence=0.31,
            key_points=["Narrative reiterates milestones without adding concrete support."],
            evidence_specs=[{"document_id": "sparse_investor_001", "chunk_id": "investor_chunk_1", "rationale": "Narrative remains optimistic with limited detail.", "snippet_text": "We see multiple value-inflection opportunities ahead.", "interpretation": EvidenceInterpretation.PROMOTIONAL, "source_url": "https://example.com/sparse/investor"}],
            fogging_score=0.58,
            hedging_score=0.42,
            promotional_score=0.72,
            tone_confidence=0.36,
        ),
    ],
}

DEMO_ARBITER_OUTPUTS: dict[str, ArbiterOutput] = {}
cross_document_arbiter = CrossDocumentArbiter()

for case_name, worker_outputs in ARBITER_DEMO_CASES.items():
    arbiter_input = ArbiterInput(run_id=f"prompt7_demo_{case_name}", ticker="FAKE", worker_outputs=worker_outputs, retrieval_results=[], config=PIPELINE_CONFIG)
    arbiter_output = cross_document_arbiter.arbitrate(arbiter_input)
    DEMO_ARBITER_OUTPUTS[case_name] = arbiter_output
    print()
    print(f"=== Prompt 7 Arbiter Demo: {case_name} ===")
    print(json.dumps(arbiter_output.model_dump(mode="json"), indent=2))


## 50. Step Completion Notes

Implemented in this step:
- internal arbiter enums and models for normalized worker profiles, judgments, grouped signals, conflicts, confidence assessment, warnings, and structured arbiter output
- worker-output normalization helpers that validate completeness, flatten evidence references, preserve provenance, and attach sparse-signal or low-confidence warnings
- explicit cross-document comparison logic for alignment, divergence, contradiction, and story-vs-substance mismatch detection
- grouped positive signals, negative signals, unresolved uncertainties, fogging or perception-management concerns, and missing-coverage notes
- transparent confidence reconciliation that adjusts for worker confidence, evidence density, structured-vs-soft support, conflicts, and missing coverage
- one practical `CrossDocumentArbiter` plus runnable demo cases for mostly aligned, contradictory, and sparse or missing-document scenarios

Still deferred to later steps:
- final master-node aggregation across arbiter outputs
- final UI payload assembly emitted only by the master node
- tests, which should remain in separate notebooks


## 51. Final Integration Contract Consolidation

This final section consolidates the notebook's latest contracts for `WorkerOutput`, the master-node payload models, and the end-to-end orchestration glue.

Earlier scaffold-only payload stubs remain above as part of the notebook history, but the definitions below are the final contracts to use for integration and testing.

In [ ]:
class WorkerOutput(ContractModel):
    """Final reconciled worker output contract used by arbiter and master stages."""

    worker_name: str
    document_type: DocumentType
    status: ProcessingStatus = ProcessingStatus.PENDING
    summary: str | None = None
    sentiment: SentimentAssessment | None = None
    tone: ToneAssessment | None = None
    key_points: list[str] = Field(default_factory=list)
    evidence: list[EvidenceSnippet] = Field(default_factory=list)
    caveats: list[str] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    confidence: float | None = None
    warnings: list[PipelineError] = Field(default_factory=list)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)
    document_metadata: DocumentMetadata | None = None
    reasoning_notes: list[str] = Field(default_factory=list)

    @field_validator("confidence")
    @classmethod
    def validate_confidence(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "confidence")


class DisclosureCardPayload(ContractModel):
    """Per-disclosure review entry emitted only by the master node."""

    document_type: DocumentType
    worker_name: str
    status: ProcessingStatus
    title: str | None = None
    source_name: str | None = None
    source_url: str | None = None
    summary: str | None = None
    sentiment_label: SentimentLabel | None = None
    sentiment_score: float | None = None
    confidence: float | None = None
    fogging_score: float | None = None
    hedging_score: float | None = None
    promotional_score: float | None = None
    key_points: list[str] = Field(default_factory=list)
    caveats: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    evidence: list[EvidenceSnippet] = Field(default_factory=list)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)

    @field_validator("sentiment_score")
    @classmethod
    def validate_sentiment_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX, "sentiment_score")

    @field_validator("confidence", "fogging_score", "hedging_score", "promotional_score")
    @classmethod
    def validate_optional_unit_interval(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "disclosure payload score")


class FinalUIPayload(ContractModel):
    """Final UI-facing payload emitted exclusively by the master node."""

    ticker: str
    generated_at: datetime = Field(default_factory=now_utc)
    status: ProcessingStatus = ProcessingStatus.PENDING
    overall_summary: str | None = None
    overall_sentiment_label: SentimentLabel | None = None
    overall_sentiment_score: float | None = None
    overall_confidence: float | None = None
    overall_fogging_score: float | None = None
    overall_hedging_score: float | None = None
    overall_promotional_score: float | None = None
    key_positive_signals: list[str] = Field(default_factory=list)
    key_negative_signals: list[str] = Field(default_factory=list)
    key_uncertainties: list[str] = Field(default_factory=list)
    fogging_or_story_substance_flags: list[str] = Field(default_factory=list)
    disclosures: list[DisclosureCardPayload] = Field(default_factory=list)
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    system_warnings: list[str] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)

    @field_validator("overall_sentiment_score")
    @classmethod
    def validate_overall_sentiment_score(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, SENTIMENT_SCORE_MIN, SENTIMENT_SCORE_MAX, "overall_sentiment_score")

    @field_validator("overall_confidence", "overall_fogging_score", "overall_hedging_score", "overall_promotional_score")
    @classmethod
    def validate_optional_unit_interval(cls, value: float | None) -> float | None:
        return _validate_optional_range(value, 0.0, 1.0, "overall payload score")


class MasterInput(ContractModel):
    """Input contract for the single master node."""

    run_id: str
    ticker: str
    worker_outputs: list[WorkerOutput]
    arbiter_outputs: list[ArbiterOutput]
    retrieval_results: list[RetrievalResult]
    config: PipelineConfig


class MasterOutput(ContractModel):
    """Structured master-node output before final UI flattening."""

    ticker: str
    status: ProcessingStatus = ProcessingStatus.PENDING
    master_summary: str | None = None
    master_sentiment: SentimentAssessment | None = None
    master_tone: ToneAssessment | None = None
    disclosures: list[DisclosureCardPayload] = Field(default_factory=list)
    missing_document_types: list[DocumentType] = Field(default_factory=list)
    key_positive_signals: list[str] = Field(default_factory=list)
    key_negative_signals: list[str] = Field(default_factory=list)
    key_uncertainties: list[str] = Field(default_factory=list)
    fogging_or_story_substance_flags: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)
    issues: list[PipelineError] = Field(default_factory=list)
    provenance: list[ProvenanceRecord] = Field(default_factory=list)
    ready_for_ui: bool = False
    generated_at: datetime = Field(default_factory=now_utc)


class MasterNode(ABC):
    """Single master node that emits the only UI-facing payload."""

    output_model: ClassVar[type[FinalUIPayload]] = FinalUIPayload

    @abstractmethod
    def build_payload(self, master_input: MasterInput) -> FinalUIPayload:
        raise NotImplementedError


def analysis_warning_to_pipeline_error(warning: AnalysisWarning, document_type: DocumentType) -> PipelineError:
    """Map an analysis warning into the pipeline-wide warning shape."""
    return PipelineError(
        error_code=warning.issue_code,
        message=warning.message,
        stage="worker_analysis",
        document_type=document_type,
        recoverable=True,
        details={
            "severity": warning.severity.value,
            "dimension": warning.dimension.value if warning.dimension else None,
            "source_chunk_id": warning.source_chunk_id,
            "field_name": warning.field_name,
            **warning.metadata,
        },
    )


def assemble_worker_output(
    analysis_output: WorkerAnalysisOutput,
    *,
    worker_input: WorkerInput | None = None,
) -> WorkerOutput:
    """Final worker-output assembly with warnings, document metadata, and provenance."""
    key_points = [finding.summary for finding in analysis_output.findings[:5]]
    caveats = [warning.message for warning in analysis_output.warnings]
    warnings = [analysis_warning_to_pipeline_error(warning, analysis_output.document_type) for warning in analysis_output.warnings]
    issues = [analysis_issue_to_pipeline_error(issue, analysis_output.document_type) for issue in analysis_output.issues]

    selected_document = None
    retrieval_provenance: list[ProvenanceRecord] = []
    document_metadata = None
    if worker_input is not None:
        retrieval_provenance = list(worker_input.retrieval_result.provenance)
        selected_document = selected_document_from_retrieval_result(worker_input.retrieval_result)
        if selected_document is not None:
            document_metadata = selected_document.metadata
            retrieval_provenance = list(selected_document.provenance) + retrieval_provenance

    reasoning_notes = list(analysis_output.reasoning_trace.reasoning_notes) if analysis_output.reasoning_trace else []
    unresolved_items = list(analysis_output.reasoning_trace.unresolved_items) if analysis_output.reasoning_trace else []
    provenance = retrieval_provenance + [
        build_provenance_record(
            stage="worker_analysis",
            adapter_name=analysis_output.reasoning_trace.analysis_client_name if analysis_output.reasoning_trace else analysis_output.worker_name,
            candidate_id=document_metadata.document_id if document_metadata else None,
            document_type=analysis_output.document_type,
            source_name=document_metadata.source_name if document_metadata else None,
            source_identifier=document_metadata.source_identifier if document_metadata else None,
            source_url=document_metadata.source_url if document_metadata else None,
            note="Worker output assembled from the shared analysis path.",
            metadata={
                "worker_name": analysis_output.worker_name,
                "model_name": analysis_output.reasoning_trace.model_name if analysis_output.reasoning_trace else None,
                "used_mock_client": analysis_output.reasoning_trace.used_mock_client if analysis_output.reasoning_trace else None,
                "unresolved_items": unresolved_items,
            },
        )
    ]

    return WorkerOutput(
        worker_name=analysis_output.worker_name,
        document_type=analysis_output.document_type,
        status=analysis_output.status,
        summary=analysis_output.summary,
        sentiment=build_sentiment_assessment_from_scores(analysis_output.scores),
        tone=build_tone_assessment_from_scores(analysis_output.scores),
        key_points=key_points,
        evidence=list(analysis_output.evidence),
        caveats=caveats,
        issues=issues,
        confidence=analysis_output.confidence_score,
        warnings=warnings,
        provenance=provenance,
        document_metadata=document_metadata,
        reasoning_notes=reasoning_notes + unresolved_items,
    )


def _base_analysis_worker_build_worker_output(self: BaseAnalysisWorker, analysis_output: WorkerAnalysisOutput, worker_input: WorkerInput | None = None) -> WorkerOutput:
    return assemble_worker_output(analysis_output, worker_input=worker_input)


def _base_analysis_worker_analyze(self: BaseAnalysisWorker, worker_input: WorkerInput) -> WorkerOutput:
    analysis_output = self.analyze_to_internal_output(worker_input)
    return self.build_worker_output(analysis_output, worker_input=worker_input)


BaseAnalysisWorker.build_worker_output = _base_analysis_worker_build_worker_output
BaseAnalysisWorker.analyze = _base_analysis_worker_analyze
BaseWorker.output_model = WorkerOutput

In [ ]:
class ArbiterInput(ContractModel):
    """Final reconciled arbiter input using the latest WorkerOutput contract."""

    run_id: str
    ticker: str
    worker_outputs: list[WorkerOutput]
    retrieval_results: list[RetrievalResult] = Field(default_factory=list)
    config: PipelineConfig


class PipelineState(ContractModel):
    """Shared state container aligned to the final worker and master contracts."""

    run_id: str
    ticker: str
    config: PipelineConfig
    retrieval_results: dict[DocumentType, RetrievalResult] = Field(default_factory=dict)
    worker_outputs: dict[DocumentType, WorkerOutput] = Field(default_factory=dict)
    arbiter_outputs: list[ArbiterOutput] = Field(default_factory=list)
    master_output: MasterOutput | None = None
    graph_context: dict[str, Any] = Field(default_factory=dict)
    shared_cache_refs: dict[str, str] = Field(default_factory=dict)
    errors: list[PipelineError] = Field(default_factory=list)
    created_at: datetime = Field(default_factory=now_utc)
    updated_at: datetime = Field(default_factory=now_utc)

## 52. Master Node Helpers and Final Payload Assembly

The master node consumes `ArbiterOutput` plus worker outputs and retrieval metadata, preserves missing coverage and conflict visibility, and emits the only UI-facing payload.

In [ ]:
def arbiter_issue_to_pipeline_error(arbiter_issue: ArbiterIssue) -> PipelineError:
    return PipelineError(
        error_code=arbiter_issue.issue_code,
        message=arbiter_issue.message,
        stage="arbiter",
        document_type=arbiter_issue.document_types[0] if arbiter_issue.document_types else None,
        recoverable=arbiter_issue.recoverable,
        details={
            "severity": arbiter_issue.severity.value,
            "document_types": [document_type.value for document_type in arbiter_issue.document_types],
            "worker_names": list(arbiter_issue.worker_names),
            **arbiter_issue.metadata,
        },
    )


def build_disclosure_card_payload(
    worker_output: WorkerOutput,
    retrieval_result: RetrievalResult | None = None,
) -> DisclosureCardPayload:
    metadata = worker_output.document_metadata
    if metadata is None and retrieval_result is not None and retrieval_result.selected_candidate is not None:
        metadata = build_document_metadata_from_candidate(retrieval_result.selected_candidate)

    return DisclosureCardPayload(
        document_type=worker_output.document_type,
        worker_name=worker_output.worker_name,
        status=worker_output.status,
        title=metadata.title if metadata else None,
        source_name=metadata.source_name if metadata else None,
        source_url=metadata.source_url if metadata else None,
        summary=worker_output.summary,
        sentiment_label=worker_output.sentiment.label if worker_output.sentiment else None,
        sentiment_score=worker_output.sentiment.score if worker_output.sentiment else None,
        confidence=worker_output.confidence,
        fogging_score=worker_output.tone.fogging_score if worker_output.tone else None,
        hedging_score=worker_output.tone.hedging_score if worker_output.tone else None,
        promotional_score=worker_output.tone.promotional_score if worker_output.tone else None,
        key_points=list(worker_output.key_points),
        caveats=list(worker_output.caveats),
        warnings=[warning.message for warning in worker_output.warnings],
        evidence=list(worker_output.evidence),
        provenance=list(worker_output.provenance),
    )


def extract_positive_signals(arbiter_output: ArbiterOutput | None, worker_outputs: Sequence[WorkerOutput]) -> list[str]:
    signals: list[str] = []
    if arbiter_output is not None:
        signals.extend(group.description for group in arbiter_output.positive_signal_groups)
        signals.extend(judgment.summary for judgment in arbiter_output.cross_document_judgments if judgment.direction == NormalizedSignalDirection.POSITIVE)
    if not signals:
        signals.extend(output.key_points[0] for output in worker_outputs if output.key_points and output.sentiment and output.sentiment.label == SentimentLabel.POSITIVE)
    return stable_unique([signal for signal in signals if signal])


def extract_negative_signals(arbiter_output: ArbiterOutput | None, worker_outputs: Sequence[WorkerOutput]) -> list[str]:
    signals: list[str] = []
    if arbiter_output is not None:
        signals.extend(group.description for group in arbiter_output.negative_signal_groups)
        signals.extend(conflict.description for conflict in arbiter_output.conflicting_signals)
        signals.extend(judgment.summary for judgment in arbiter_output.cross_document_judgments if judgment.direction == NormalizedSignalDirection.NEGATIVE)
    if not signals:
        signals.extend(output.key_points[0] for output in worker_outputs if output.key_points and output.sentiment and output.sentiment.label == SentimentLabel.NEGATIVE)
    return stable_unique([signal for signal in signals if signal])


def extract_uncertainties(arbiter_output: ArbiterOutput | None, worker_outputs: Sequence[WorkerOutput]) -> list[str]:
    signals: list[str] = []
    if arbiter_output is not None:
        signals.extend(finding.summary for finding in arbiter_output.unresolved_uncertainties)
        signals.extend(arbiter_output.missing_coverage_notes)
    if not signals:
        signals.extend(output.caveats[0] for output in worker_outputs if output.caveats)
    return stable_unique([signal for signal in signals if signal])


def extract_story_substance_flags(arbiter_output: ArbiterOutput | None, worker_outputs: Sequence[WorkerOutput]) -> list[str]:
    flags: list[str] = []
    if arbiter_output is not None:
        flags.extend(finding.summary for finding in arbiter_output.fogging_or_story_substance_flags)
    if not flags:
        flags.extend(output.caveats[0] for output in worker_outputs if output.caveats)
    return stable_unique([flag for flag in flags if flag])


def build_master_summary(
    arbiter_output: ArbiterOutput | None,
    worker_outputs: Sequence[WorkerOutput],
    *,
    positive_signals: Sequence[str],
    negative_signals: Sequence[str],
    uncertainties: Sequence[str],
) -> str:
    summary_parts: list[str] = []
    if arbiter_output is not None and arbiter_output.summary:
        summary_parts.append(arbiter_output.summary)
    else:
        usable_outputs = [output for output in worker_outputs if output.status in USABLE_WORKER_STATUSES]
        summary_parts.append(f"Integrated review covered {len(usable_outputs)} disclosure(s) with explicit worker, arbiter, and master handoff.")
    if positive_signals:
        summary_parts.append(f"Positive emphasis: {positive_signals[0]}")
    if negative_signals:
        summary_parts.append(f"Negative emphasis: {negative_signals[0]}")
    if uncertainties:
        summary_parts.append(f"Primary uncertainty: {uncertainties[0]}")
    return " ".join(summary_parts).strip()


def aggregate_master_sentiment(
    arbiter_output: ArbiterOutput | None,
    worker_outputs: Sequence[WorkerOutput],
) -> SentimentAssessment | None:
    if arbiter_output is not None and arbiter_output.sentiment is not None:
        return arbiter_output.sentiment
    usable_scores = [output.sentiment.score for output in worker_outputs if output.sentiment and output.sentiment.score is not None]
    if not usable_scores:
        return None
    aggregate_score = safe_mean(usable_scores)
    has_positive = any(output.sentiment and output.sentiment.label == SentimentLabel.POSITIVE for output in worker_outputs)
    has_negative = any(output.sentiment and output.sentiment.label == SentimentLabel.NEGATIVE for output in worker_outputs)
    return SentimentAssessment(
        label=normalize_sentiment_label(None, aggregate_score)[0],
        score=aggregate_score,
        confidence=safe_mean([output.confidence for output in worker_outputs]),
        rationale="Fallback master sentiment aggregated directly from worker outputs when no arbiter sentiment was supplied.",
    )


def aggregate_master_tone(
    arbiter_output: ArbiterOutput | None,
    worker_outputs: Sequence[WorkerOutput],
) -> ToneAssessment | None:
    if arbiter_output is not None and arbiter_output.tone is not None:
        return arbiter_output.tone
    tone_values = [output.tone for output in worker_outputs if output.tone is not None]
    if not tone_values:
        return None
    return ToneAssessment(
        fogging_score=safe_mean([tone.fogging_score for tone in tone_values]),
        hedging_score=safe_mean([tone.hedging_score for tone in tone_values]),
        promotional_score=safe_mean([tone.promotional_score for tone in tone_values]),
        confidence=safe_mean([tone.confidence for tone in tone_values]),
        rationale="Fallback master tone aggregated directly from worker outputs.",
    )


class IntegratedMasterNode(MasterNode):
    """Concrete master node that emits the final UI-facing payload."""

    def build_master_output(self, master_input: MasterInput) -> MasterOutput:
        retrieval_result_map = {result.request.document_type: result for result in master_input.retrieval_results}
        worker_outputs = sorted(master_input.worker_outputs, key=lambda output: EXPECTED_DOCUMENT_TYPES.index(output.document_type) if output.document_type in EXPECTED_DOCUMENT_TYPES else len(EXPECTED_DOCUMENT_TYPES))
        arbiter_output = master_input.arbiter_outputs[0] if master_input.arbiter_outputs else None

        disclosures = [
            build_disclosure_card_payload(worker_output, retrieval_result_map.get(worker_output.document_type))
            for worker_output in worker_outputs
        ]
        positive_signals = extract_positive_signals(arbiter_output, worker_outputs)
        negative_signals = extract_negative_signals(arbiter_output, worker_outputs)
        uncertainties = extract_uncertainties(arbiter_output, worker_outputs)
        story_flags = extract_story_substance_flags(arbiter_output, worker_outputs)
        master_sentiment = aggregate_master_sentiment(arbiter_output, worker_outputs)
        master_tone = aggregate_master_tone(arbiter_output, worker_outputs)
        missing_document_types = list(arbiter_output.missing_document_types) if arbiter_output is not None else [document_type for document_type in EXPECTED_DOCUMENT_TYPES if document_type not in {output.document_type for output in worker_outputs}]

        warnings = stable_unique(
            [warning.message for output in worker_outputs for warning in output.warnings]
            + ([warning.message for warning in arbiter_output.warnings] if arbiter_output is not None else [])
        )
        issues = [issue for output in worker_outputs for issue in output.issues]
        if arbiter_output is not None:
            issues.extend(arbiter_issue_to_pipeline_error(issue) for issue in arbiter_output.issues)

        provenance = [
            build_provenance_record(
                stage="master_payload",
                adapter_name="master_node",
                document_type=None,
                note="Master node assembled the only UI-facing payload.",
                metadata={
                    "worker_output_count": len(worker_outputs),
                    "arbiter_output_count": len(master_input.arbiter_outputs),
                    "missing_document_type_count": len(missing_document_types),
                },
            )
        ]
        for disclosure in disclosures:
            provenance.extend(disclosure.provenance[:1])
        if arbiter_output is not None:
            provenance.append(
                build_provenance_record(
                    stage="master_payload",
                    adapter_name=arbiter_output.arbiter_name,
                    note="Master node consumed ArbiterOutput for final UI assembly.",
                    metadata={"arbiter_id": arbiter_output.arbiter_id},
                )
            )

        status = ProcessingStatus.SUCCESS
        if missing_document_types or negative_signals or uncertainties or story_flags or warnings:
            status = ProcessingStatus.PARTIAL
        if not worker_outputs:
            status = ProcessingStatus.NO_DOCUMENT

        return MasterOutput(
            ticker=master_input.ticker,
            status=status,
            master_summary=build_master_summary(
                arbiter_output,
                worker_outputs,
                positive_signals=positive_signals,
                negative_signals=negative_signals,
                uncertainties=uncertainties,
            ),
            master_sentiment=master_sentiment,
            master_tone=master_tone,
            disclosures=disclosures,
            missing_document_types=missing_document_types,
            key_positive_signals=positive_signals[:6],
            key_negative_signals=negative_signals[:6],
            key_uncertainties=uncertainties[:6],
            fogging_or_story_substance_flags=story_flags[:6],
            warnings=warnings,
            issues=issues,
            provenance=provenance,
            ready_for_ui=True,
        )

    def build_payload(self, master_input: MasterInput) -> FinalUIPayload:
        master_output = self.build_master_output(master_input)
        master_sentiment = master_output.master_sentiment
        master_tone = master_output.master_tone
        return FinalUIPayload(
            ticker=master_output.ticker,
            generated_at=master_output.generated_at,
            status=master_output.status,
            overall_summary=master_output.master_summary,
            overall_sentiment_label=master_sentiment.label if master_sentiment else None,
            overall_sentiment_score=master_sentiment.score if master_sentiment else None,
            overall_confidence=master_sentiment.confidence if master_sentiment else None,
            overall_fogging_score=master_tone.fogging_score if master_tone else None,
            overall_hedging_score=master_tone.hedging_score if master_tone else None,
            overall_promotional_score=master_tone.promotional_score if master_tone else None,
            key_positive_signals=list(master_output.key_positive_signals),
            key_negative_signals=list(master_output.key_negative_signals),
            key_uncertainties=list(master_output.key_uncertainties),
            fogging_or_story_substance_flags=list(master_output.fogging_or_story_substance_flags),
            disclosures=list(master_output.disclosures),
            missing_document_types=list(master_output.missing_document_types),
            system_warnings=list(master_output.warnings),
            issues=list(master_output.issues),
            provenance=list(master_output.provenance),
        )

## 53. Final End-to-End Demo Flow and Notebook Export Namespace

This final demo runs one coherent retrieval -> worker -> arbiter -> master flow using the notebook's mock retrieval results and fallback-safe worker-analysis path.

In [ ]:
# ── helpers: resolve_company_from_ticker, build_retrieval_requests, run_retrieval ──

def resolve_company_from_ticker(ticker: str) -> tuple[str, list[str], str | None]:
    """Resolve ticker → (company_name, aliases, cik) via SEC EDGAR company_tickers.json.

    Uses the authoritative ticker→CIK→name mapping. Returns CIK for downstream
    EDGAR searches. Falls back to (ticker, [ticker], None) on failure.
    """
    ticker_upper = ticker.strip().upper()
    try:
        client = EdgarClient()
        client._rate_limit()
        resp = client._session.get(
            "https://www.sec.gov/files/company_tickers.json", timeout=15
        )
        resp.raise_for_status()
        for entry in resp.json().values():
            if entry.get("ticker") == ticker_upper:
                company_name = entry["title"]
                cik = str(entry["cik_str"])
                aliases = list(dict.fromkeys([company_name, ticker_upper]))
                logger.info(f"Resolved {ticker_upper} → {company_name} (CIK {cik})")
                return company_name, aliases, cik
    except Exception as exc:
        logger.warning(f"EDGAR company_tickers.json lookup failed for {ticker_upper}: {exc}")
    return ticker_upper, [ticker_upper], None


def build_retrieval_requests(
    ticker: str,
    company_name: str,
    company_aliases: list[str] | None = None,
    *,
    cik: str | None = None,
    config: PipelineConfig = PIPELINE_CONFIG,
    run_id: str = "pipeline",
) -> list[RetrievalRequest]:
    """Build one RetrievalRequest per DocumentType, parameterized by ticker/company."""
    aliases = company_aliases or [company_name, ticker]
    is_mock = config.enable_mock_retrieval
    filters: dict[str, Any] = {}
    if cik:
        filters["cik"] = cik
    return [
        RetrievalRequest(
            request_id=f"{run_id}_{document_type.value}",
            ticker=ticker,
            company_name=company_name,
            company_aliases=aliases,
            document_type=document_type,
            max_candidates=config.default_max_candidates,
            minimum_text_chars=config.min_usable_text_chars,
            minimum_relevance_score=config.minimum_relevance_score,
            freshness_bonus_max=config.freshness_bonus_max,
            freshness_rank_decay=config.freshness_rank_decay,
            source_preferences=SourcePreferencePolicy(
                preferred_source_families=config.default_source_preferences,
                prefer_structured_source=True,
                allow_secondary_sources=True,
                allow_unknown_sources=True,
                allow_mock_data=is_mock,
            ),
            retrieval_notes=[f"{'Mock' if is_mock else 'Live'} pipeline request for {ticker}."],
            is_mock_request=is_mock,
            filters=filters,
        )
        for document_type in DocumentType
    ]


def run_retrieval(
    requests: list[RetrievalRequest],
) -> dict[DocumentType, RetrievalResult]:
    """Execute retrieval for all requests with per-adapter error isolation.

    If one adapter crashes, its result gets status=RETRIEVAL_FAILED instead of
    killing the whole pipeline. The arbiter already handles missing_document_types.
    """
    results: dict[DocumentType, RetrievalResult] = {}
    for request in requests:
        try:
            adapter_class = RETRIEVAL_ADAPTER_REGISTRY[request.document_type]
            adapter = adapter_class()
            results[request.document_type] = adapter.retrieve(request)
        except Exception as exc:
            logger.error(f"Retrieval adapter failed for {request.document_type.value}: {exc}")
            results[request.document_type] = RetrievalResult(
                request=request,
                adapter_name=RETRIEVAL_ADAPTER_REGISTRY[request.document_type].__name__,
                status=ProcessingStatus.RETRIEVAL_FAILED,
                issues=[
                    make_retrieval_error(
                        RetrievalErrorCode.ADAPTER_FAILURE,
                        message=f"Adapter crashed: {exc}",
                        stage="retrieval",
                        document_type=request.document_type,
                        adapter_name=RETRIEVAL_ADAPTER_REGISTRY[request.document_type].__name__,
                        recoverable=False,
                    )
                ],
            )
    return results


# ── main entry point ──

def run_full_pipeline(
    ticker: str,
    company_name: str | None = None,
    *,
    run_id: str | None = None,
    config: PipelineConfig = PIPELINE_CONFIG,
    analysis_client: BaseAnalysisClient | None = None,
) -> dict[str, Any]:
    """True end-to-end pipeline: ticker → retrieval → analysis → arbitration → FinalUIPayload.

    Args:
        ticker: Stock ticker symbol (e.g. "GILD", "MRNA").
        company_name: Override company name. Auto-resolved via EDGAR if None.
        run_id: Pipeline run identifier. Auto-generated if None.
        config: Pipeline configuration. Defaults to PIPELINE_CONFIG.
        analysis_client: Override analysis client. Uses build_analysis_client(config) if None.
    """
    effective_run_id = run_id or f"pipeline_{ticker.upper()}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
    print(f"[run_full_pipeline] Starting pipeline for {ticker} (run_id={effective_run_id})")

    # 1. Resolve company name and CIK
    if company_name is None:
        resolved_name, aliases, cik = resolve_company_from_ticker(ticker)
    else:
        resolved_name = company_name
        aliases = [company_name, ticker]
        cik = None
    print(f"[run_full_pipeline] Company: {resolved_name}" + (f" (CIK {cik})" if cik else ""))

    # 2. Build retrieval requests
    requests = build_retrieval_requests(
        ticker=ticker,
        company_name=resolved_name,
        company_aliases=aliases,
        cik=cik,
        config=config,
        run_id=effective_run_id,
    )

    # 3. Run retrieval with per-adapter error isolation
    print(f"[run_full_pipeline] Running retrieval for {len(requests)} document types...")
    retrieval_results = run_retrieval(requests)
    success_count = sum(1 for r in retrieval_results.values() if r.status == ProcessingStatus.SUCCESS)
    print(f"[run_full_pipeline] Retrieval complete: {success_count}/{len(requests)} succeeded")

    # 4. Build analysis client (never silently mock)
    effective_client = analysis_client or build_analysis_client(config)
    print(f"[run_full_pipeline] Analysis client: {effective_client.client_name}")

    # 5. Run 5 workers
    print(f"[run_full_pipeline] Running {len(EXPECTED_DOCUMENT_TYPES)} analysis workers...")
    worker_outputs: dict[DocumentType, WorkerOutput] = {}
    for document_type in EXPECTED_DOCUMENT_TYPES:
        worker_class = WORKER_REGISTRY[document_type]
        worker = worker_class(analysis_client=effective_client)
        retrieval_result = retrieval_results[document_type]
        worker_outputs[document_type] = worker.analyze(
            WorkerInput(
                run_id=effective_run_id,
                ticker=ticker,
                document_type=document_type,
                retrieval_result=retrieval_result,
                config=config,
                graph_context={},
            )
        )
    print(f"[run_full_pipeline] Workers complete")

    # 6. Run arbiter
    print(f"[run_full_pipeline] Running cross-document arbiter...")
    arbiter = CrossDocumentArbiter()
    arbiter_output = arbiter.arbitrate(
        ArbiterInput(
            run_id=effective_run_id,
            ticker=ticker,
            worker_outputs=list(worker_outputs.values()),
            retrieval_results=list(retrieval_results.values()),
            config=config,
        )
    )

    # 7. Run master node
    print(f"[run_full_pipeline] Building final payload...")
    master_node = IntegratedMasterNode()
    final_payload = master_node.build_payload(
        MasterInput(
            run_id=effective_run_id,
            ticker=ticker,
            worker_outputs=list(worker_outputs.values()),
            arbiter_outputs=[arbiter_output],
            retrieval_results=list(retrieval_results.values()),
            config=config,
        )
    )

    print(f"[run_full_pipeline] Done. Final status: {final_payload.status.value}")
    return {
        "ticker": ticker,
        "company_name": resolved_name,
        "run_id": effective_run_id,
        "retrieval_results": retrieval_results,
        "worker_outputs": worker_outputs,
        "arbiter_output": arbiter_output,
        "final_payload": final_payload,
    }


# ── legacy entry point (fixed defaults) ──

def run_integrated_demo_pipeline(
    retrieval_results: Mapping[DocumentType, RetrievalResult] | None = None,
    *,
    run_id: str = "final_integration_demo",
    ticker: str | None = None,
    company_name: str | None = None,
    analysis_client: BaseAnalysisClient | None = None,
) -> dict[str, Any]:
    resolved_retrieval_results = dict(retrieval_results or DEMO_RETRIEVAL_RESULTS)
    if ticker is None:
        sample_result = next(iter(resolved_retrieval_results.values()))
        ticker = sample_result.request.ticker
    if company_name is None:
        company_name = ticker

    effective_analysis_client = analysis_client or build_analysis_client(PIPELINE_CONFIG)
    worker_outputs: dict[DocumentType, WorkerOutput] = {}
    for document_type in EXPECTED_DOCUMENT_TYPES:
        worker_class = WORKER_REGISTRY[document_type]
        worker = worker_class(analysis_client=effective_analysis_client)
        retrieval_result = resolved_retrieval_results[document_type]
        worker_outputs[document_type] = worker.analyze(
            WorkerInput(
                run_id=run_id,
                ticker=ticker,
                document_type=document_type,
                retrieval_result=retrieval_result,
                config=PIPELINE_CONFIG,
                graph_context={},
            )
        )

    arbiter = CrossDocumentArbiter()
    arbiter_output = arbiter.arbitrate(
        ArbiterInput(
            run_id=run_id,
            ticker=ticker,
            worker_outputs=list(worker_outputs.values()),
            retrieval_results=list(resolved_retrieval_results.values()),
            config=PIPELINE_CONFIG,
        )
    )

    master_node = IntegratedMasterNode()
    final_payload = master_node.build_payload(
        MasterInput(
            run_id=run_id,
            ticker=ticker,
            worker_outputs=list(worker_outputs.values()),
            arbiter_outputs=[arbiter_output],
            retrieval_results=list(resolved_retrieval_results.values()),
            config=PIPELINE_CONFIG,
        )
    )

    return {
        "ticker": ticker,
        "company_name": company_name,
        "retrieval_results": resolved_retrieval_results,
        "worker_outputs": worker_outputs,
        "arbiter_output": arbiter_output,
        "final_payload": final_payload,
    }


# ── exports ──

NOTEBOOK_EXPORTS = {
    "PIPELINE_CONFIG": PIPELINE_CONFIG,
    "DocumentType": DocumentType,
    "ProcessingStatus": ProcessingStatus,
    "SentimentLabel": SentimentLabel,
    "SourceFamily": SourceFamily,
    "DocumentMetadata": DocumentMetadata,
    "RetrievalRequest": RetrievalRequest,
    "RetrievedCandidate": RetrievalCandidate if "RetrievalCandidate" in globals() else None,
    "RetrievalCandidate": RetrievalCandidate,
    "RetrievalResult": RetrievalResult,
    "DocumentValidationResult": DocumentValidationResult,
    "ProcessedDocument": ProcessedDocument,
    "DocumentSection": DocumentSection,
    "DocumentChunk": DocumentChunk,
    "WorkerAnalysisInput": WorkerAnalysisInput,
    "WorkerOutput": WorkerOutput,
    "ArbiterOutput": ArbiterOutput,
    "FinalUIPayload": FinalUIPayload,
    "validate_candidate_document": validate_candidate_document,
    "select_most_recent_relevant_candidate": select_most_recent_relevant_candidate,
    "normalize_line_breaks": normalize_line_breaks,
    "normalize_whitespace": normalize_whitespace,
    "normalize_bullets_and_lists": normalize_bullets_and_lists,
    "normalize_table_like_text": normalize_table_like_text,
    "detect_document_sections": detect_document_sections,
    "process_selected_document": process_selected_document,
    "clamp_value": clamp_value,
    "normalize_sentiment_label": normalize_sentiment_label,
    "build_standardized_analysis_score": build_standardized_analysis_score,
    "selected_document_from_retrieval_result": selected_document_from_retrieval_result,
    "build_document_metadata_from_candidate": build_document_metadata_from_candidate,
    "CrossDocumentArbiter": CrossDocumentArbiter,
    "IntegratedMasterNode": IntegratedMasterNode,
    "run_full_pipeline": run_full_pipeline,
    "run_integrated_demo_pipeline": run_integrated_demo_pipeline,
    "resolve_company_from_ticker": resolve_company_from_ticker,
    "build_retrieval_requests": build_retrieval_requests,
    "run_retrieval": run_retrieval,
    "make_demo_worker_output": make_demo_worker_output,
    "ARBITER_DEMO_CASES": ARBITER_DEMO_CASES,
    "DEMO_RETRIEVAL_RESULTS": DEMO_RETRIEVAL_RESULTS,
    "DEMO_WORKER_OUTPUTS": DEMO_WORKER_OUTPUTS,
    "DEMO_ARBITER_OUTPUTS": DEMO_ARBITER_OUTPUTS,
    "EXPECTED_DOCUMENT_TYPES": EXPECTED_DOCUMENT_TYPES,
}


_demo_ticker = "GILD" if not PIPELINE_CONFIG.enable_mock_retrieval else "FAKE"
_demo_company = "Gilead Sciences, Inc." if not PIPELINE_CONFIG.enable_mock_retrieval else "FakeBio Therapeutics"
_demo_client = build_analysis_client(PIPELINE_CONFIG)
FINAL_INTEGRATED_DEMO = run_integrated_demo_pipeline(
    ticker=_demo_ticker,
    company_name=_demo_company,
    analysis_client=_demo_client,
)
FINAL_INTEGRATED_DEMO_SUMMARY = {
    "retrieval": {
        document_type.value: {
            "status": result.status.value,
            "selected_candidate_id": result.selection_decision.selected_candidate_id,
        }
        for document_type, result in FINAL_INTEGRATED_DEMO["retrieval_results"].items()
    },
    "workers": {
        document_type.value: {
            "status": output.status.value,
            "sentiment_label": output.sentiment.label.value if output.sentiment else None,
            "confidence": output.confidence,
            "warning_count": len(output.warnings),
            "issue_count": len(output.issues),
        }
        for document_type, output in FINAL_INTEGRATED_DEMO["worker_outputs"].items()
    },
    "arbiter": {
        "status": FINAL_INTEGRATED_DEMO["arbiter_output"].status.value,
        "sentiment_label": FINAL_INTEGRATED_DEMO["arbiter_output"].sentiment.label.value if FINAL_INTEGRATED_DEMO["arbiter_output"].sentiment else None,
        "conflict_count": len(FINAL_INTEGRATED_DEMO["arbiter_output"].conflicting_signals),
        "uncertainty_count": len(FINAL_INTEGRATED_DEMO["arbiter_output"].unresolved_uncertainties),
    },
    "final_payload": {
        "status": FINAL_INTEGRATED_DEMO["final_payload"].status.value,
        "overall_sentiment_label": FINAL_INTEGRATED_DEMO["final_payload"].overall_sentiment_label.value if FINAL_INTEGRATED_DEMO["final_payload"].overall_sentiment_label else None,
        "disclosure_count": len(FINAL_INTEGRATED_DEMO["final_payload"].disclosures),
        "missing_document_types": [document_type.value for document_type in FINAL_INTEGRATED_DEMO["final_payload"].missing_document_types],
    },
}
print(json.dumps(FINAL_INTEGRATED_DEMO_SUMMARY, indent=2))
print(json.dumps(FINAL_INTEGRATED_DEMO["final_payload"].model_dump(mode="json"), indent=2))


## 54. Final Step Completion Notes

- The notebook now supports end-to-end mock/demo execution across retrieval candidate selection, shared document processing, worker analysis, arbiter reconciliation, and final master-node payload assembly.
- Live retrieval providers, live source-document acquisition, and live model-provider integrations remain future work behind the notebook's existing adapter and client boundaries.
- Separate testing notebooks contain unit validation and pipeline/scenario validation coverage.
- Major assumptions and fallback-safe areas are explicit: mock retrieval fixtures, mock analysis output, and safe embedding fallback paths remain documented rather than hidden.